# Notebook 2: ANC8 Machine-Learning Readiness and Audit

## Purpose

This notebook prepares the finalized ANC8 analytic dataset for reproducible
machine-learning analysis.

The predictor and outcome ascertainment phase was completed and frozen in
Notebook 1. Therefore, this notebook does **not** redefine, reconstruct, or
recategorize the primary predictors or outcomes.

The principal analytic artifact is:

`ANC8_analysis_frozen.pkl`

It contains:

- 6,916 maternal-health roster records;
- two finalized outcomes: `anc4plus` and `anc8plus`;
- 21 prespecified primary predictors;
- the DHS sampling weight (`v005`);
- primary sampling unit (`v021`);
- survey stratum (`v022`); and
- the respondent identifier (`caseid`).

## Prediction framework

Two prediction horizons are distinguished.

**Primary baseline prediction**

ANC8 attainment is predicted using the 21 frozen background, household,
reproductive-history, and access predictors. ANC-process variables are excluded
from this primary model.

**Secondary ANC-entry prediction**

A separately governed extension will evaluate prediction after ANC initiation,
where information available at entry to care, particularly timing of first ANC
contact, may be added. This extension will not modify the frozen baseline
dataset or primary predictor register.

## Reproducibility principles

1. The frozen Notebook-1 artifact is immutable.
2. Predictor construction is not repeated in this notebook.
3. Any later feature addition must be explicitly justified, versioned, and
   merged by `caseid`.
4. Train/test separation occurs before any data-dependent preprocessing.
5. Preprocessing learned from data is fitted only within training data or
   cross-validation folds.
6. Survey design and sampling weights are preserved throughout evaluation.
7. Random procedures use explicit seeds.
8. Model-selection decisions are made without inspecting final test-set
   performance.
9. All important intermediate outputs are audited and exported for
   reproducibility.

In [ ]:
# =============================================================================
# NOTEBOOK 2 - REPRODUCIBLE STARTUP AND FROZEN-ARTIFACT VERIFICATION
#
# WHAT THIS CELL DOES
# -------------------
# 1. Defines the ANC8 project and output paths.
# 2. Loads the frozen analytic dataset created by Notebook 1.
# 3. Computes its SHA-256 hash.
# 4. Verifies that the file is exactly the artifact frozen previously.
# 5. Checks the expected dimensions, identifier, outcomes, predictors,
#    and survey-design variables.
# 6. Reports the software environment for reproducibility.
#
# WHAT THIS CELL DOES NOT DO
# --------------------------
# - No predictor is reconstructed.
# - No missing value is imputed.
# - No variable is encoded or scaled.
# - No train/test split is performed.
# - No model is fitted.
# =============================================================================

from pathlib import Path
import hashlib
import platform

import numpy as np
import pandas as pd


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_DIR = Path(
    r"D:\MOH\2026\Data science\Capstone\ANC8"
)

OUTPUT_DIR = PROJECT_DIR / "outputs"

FROZEN_DATA_PATH = (
    OUTPUT_DIR / "ANC8_analysis_frozen.pkl"
)


# =============================================================================
# 2. EXPECTED FROZEN-ARTIFACT IDENTITY
#
# This SHA-256 value was generated when Notebook 1 froze the analytic dataset.
# If the file changes by even one byte, the hash will change.
# =============================================================================

EXPECTED_SHA256 = (
    "036ee8783922be4da8f719848928be22"
    "a70e1d70766746880fd9de4a3fdc425e"
)


# =============================================================================
# 3. HELPER FUNCTION FOR SHA-256
# =============================================================================

def calculate_sha256(file_path):
    """
    Calculate the SHA-256 checksum of a file.

    SHA-256 provides a reproducible digital fingerprint of the frozen
    artifact. Matching hashes indicate that Notebook 2 is reading the exact
    file produced and approved in Notebook 1.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


# =============================================================================
# 4. VERIFY THAT THE FROZEN FILE EXISTS
# =============================================================================

assert FROZEN_DATA_PATH.exists(), (
    f"Frozen analytic dataset not found:\n{FROZEN_DATA_PATH}"
)


# =============================================================================
# 5. VERIFY FILE IDENTITY BEFORE LOADING
# =============================================================================

observed_sha256 = calculate_sha256(
    FROZEN_DATA_PATH
)

assert observed_sha256 == EXPECTED_SHA256, (
    "\nSHA-256 verification FAILED.\n"
    "The current PKL is not identical to the dataset frozen in Notebook 1.\n"
    f"Expected: {EXPECTED_SHA256}\n"
    f"Observed: {observed_sha256}"
)


# =============================================================================
# 6. LOAD THE FROZEN DATASET
# =============================================================================

anc8 = pd.read_pickle(
    FROZEN_DATA_PATH
)


# =============================================================================
# 7. AUTHORITATIVE VARIABLE ROLES
# =============================================================================

ID_VARIABLE = "caseid"

OUTCOME_VARIABLES = [
    "anc4plus",
    "anc8plus"
]

PRIMARY_PREDICTORS = [
    "v012",
    "maternal_education",
    "maternal_current_union_status",
    "maternal_occupation_group",
    "partner_education",
    "partner_occupation_group",
    "healthcare_decision_participation",
    "wealth_quintile",
    "residence",
    "region",
    "household_size",
    "religion_group",
    "traditional_media_weekly_count",
    "internet_exposure",
    "health_insurance",
    "pregnancy_order_group",
    "preceding_pregnancy_interval_group",
    "pregnancy_intention",
    "previous_pregnancy_history",
    "travel_time_min",
    "transport_mode_group"
]

SURVEY_VARIABLES = [
    "v005",   # DHS sampling weight
    "v021",   # primary sampling unit
    "v022"    # survey stratum
]


EXPECTED_COLUMNS = (
    [ID_VARIABLE]
    + OUTCOME_VARIABLES
    + PRIMARY_PREDICTORS
    + SURVEY_VARIABLES
)


# =============================================================================
# 8. STRUCTURAL ASSERTIONS
# =============================================================================

assert anc8.shape == (6916, 27), (
    f"Unexpected dataset shape: {anc8.shape}"
)

assert anc8[ID_VARIABLE].is_unique, (
    "caseid is not unique."
)

assert anc8[ID_VARIABLE].notna().all(), (
    "caseid contains missing values."
)

assert len(PRIMARY_PREDICTORS) == 21, (
    "The primary predictor list must contain exactly 21 variables."
)

assert len(set(PRIMARY_PREDICTORS)) == 21, (
    "Duplicate variables were found in the primary predictor list."
)

assert set(anc8.columns) == set(EXPECTED_COLUMNS), (
    "The frozen dataset columns do not exactly match the expected schema."
)

assert anc8["anc8plus"].isna().sum() == 23, (
    "ANC8 unresolved count differs from the frozen outcome definition."
)

assert anc8["anc4plus"].isna().sum() == 23, (
    "ANC4 unresolved count differs from the frozen outcome definition."
)

assert anc8[SURVEY_VARIABLES].notna().all().all(), (
    "One or more survey-design variables contain missing values."
)


# =============================================================================
# 9. REPRODUCIBILITY / ENVIRONMENT REPORT
#
# We record rather than assume the software environment.
# Later modeling libraries will be added only when they are actually needed.
# =============================================================================

print("\n" + "=" * 95)
print("ANC8 NOTEBOOK 2 - REPRODUCIBLE STARTUP")
print("=" * 95)

print("\nFrozen artifact")
print("----------------")
print("File:", FROZEN_DATA_PATH.name)
print("Rows:", anc8.shape[0])
print("Columns:", anc8.shape[1])
print("SHA-256:", observed_sha256)
print("Hash verified:", observed_sha256 == EXPECTED_SHA256)

print("\nVariable roles")
print("--------------")
print("Identifier:", ID_VARIABLE)
print("Primary outcome: anc8plus = attainment of 8+ ANC contacts")
print("Secondary outcome: anc4plus = attainment of 4+ ANC contacts")
print("Primary predictors:", len(PRIMARY_PREDICTORS))
print("Survey weight: v005")
print("PSU / cluster: v021")
print("Survey stratum: v022")

print("\nOutcome accounting")
print("------------------")
print(
    "ANC8 resolved:",
    anc8["anc8plus"].notna().sum(),
    "| unresolved:",
    anc8["anc8plus"].isna().sum()
)

print(
    "ANC4 resolved:",
    anc8["anc4plus"].notna().sum(),
    "| unresolved:",
    anc8["anc4plus"].isna().sum()
)

print("\nSoftware environment")
print("--------------------")
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

print("\nSTATUS: FROZEN NOTEBOOK-1 ARTIFACT VERIFIED SUCCESSFULLY")

## 2.1 Prediction questions and governed analytic populations

Two related but distinct prediction analyses are undertaken.

### Primary background-predictor model

The primary analysis predicts attainment of eight or more ANC contacts using
the 21 predictors finalized and frozen in Notebook 1.

These predictors describe maternal, household, socioeconomic,
reproductive-history, media, and health-service-access characteristics.
Variables describing the woman's ANC care process are excluded.

This analysis therefore evaluates the predictive information contained in the
prespecified background predictor set without incorporating information on the
timing or subsequent course of ANC.

Because several DHS characteristics are measured at the survey interview,
this analysis is described as a background-predictor model rather than as a
strictly prospective model available at pregnancy onset.

### ANC-entry extension

A secondary analysis evaluates prediction among women who initiated ANC and
whose first-contact timing is known.

Within this ANC-entry-eligible population, the original 21-predictor
specification is compared with an extended specification that additionally
includes timing of ANC initiation (`early_anc`).

The purpose of this paired analysis is to quantify the incremental predictive
information provided by first-contact timing.

### Governance rules

The full-cohort and ANC-entry analyses represent different analytic populations
and prediction questions and are therefore not interchangeable.

The Notebook-1 frozen dataset remains immutable.

Any predictor introduced after the Notebook-1 freeze must:

1. have an explicit scientific rationale;
2. be relevant to the stated prediction question;
3. be assessed for potential outcome leakage;
4. have its denominator and missingness documented;
5. be merged using `caseid`; and
6. be stored in a separately versioned analytic artifact before downstream use.

In [ ]:
# =============================================================================
# DEFINE PRIMARY BASELINE ANALYTIC POPULATION
#
# Primary outcome:
#   anc8plus
#
# Inclusion:
#   records with resolved ANC8 outcome
#
# Predictors:
#   exactly the 21 frozen predictors
#
# No preprocessing, imputation, encoding, or modelling occurs here.
# =============================================================================

baseline_df = anc8.loc[
    anc8["anc8plus"].notna(),
    [
        ID_VARIABLE,
        "anc8plus",
        "anc4plus",
        *PRIMARY_PREDICTORS,
        *SURVEY_VARIABLES
    ]
].copy()

print("=" * 90)
print("PRIMARY BASELINE ANALYTIC POPULATION")
print("=" * 90)

print("Rows:", len(baseline_df))
print("Columns:", baseline_df.shape[1])
print("Resolved ANC8 outcomes:", baseline_df["anc8plus"].notna().sum())
print("ANC8 positive cases:", int(baseline_df["anc8plus"].sum()))
print("ANC8 negative cases:", int((baseline_df["anc8plus"] == 0).sum()))
print("Unique caseid:", baseline_df["caseid"].is_unique)

## 2.2 ANC8 class balance and survey-weighted outcome prevalence

Before train/test splitting, the primary outcome distribution is examined in
both unweighted and survey-weighted terms.

The unweighted class balance describes the composition of the analytic sample,
while the survey-weighted prevalence estimates the corresponding national
population distribution represented by the EDHS sample.

Because ANC8 attainment is less common than non-attainment, evaluation will not
rely on accuracy alone. Later model assessment will emphasize discrimination,
precision-recall performance, calibration, and threshold-dependent measures.

In [ ]:
# =============================================================================
# ANC8 CLASS-BALANCE AND SURVEY-WEIGHTED OUTCOME AUDIT
#
# Purpose:
# 1. Describe the unweighted class distribution.
# 2. Estimate the survey-weighted prevalence of ANC8 attainment.
# 3. Establish the no-skill baseline for later precision-recall evaluation.
#
# No train/test splitting or modelling occurs here.
# =============================================================================

# DHS sampling weight
baseline_df["survey_weight"] = baseline_df["v005"] / 1_000_000


# -----------------------------------------------------------------------------
# 1. Unweighted class counts and percentages
# -----------------------------------------------------------------------------

unweighted_counts = (
    baseline_df["anc8plus"]
    .value_counts()
    .sort_index()
)

unweighted_percent = (
    baseline_df["anc8plus"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)


# -----------------------------------------------------------------------------
# 2. Survey-weighted prevalence
# -----------------------------------------------------------------------------

weighted_positive = baseline_df.loc[
    baseline_df["anc8plus"].eq(1),
    "survey_weight"
].sum()

weighted_total = baseline_df["survey_weight"].sum()

weighted_prevalence = (
    weighted_positive / weighted_total
)


# -----------------------------------------------------------------------------
# 3. Display with explicit outcome meanings
# -----------------------------------------------------------------------------

print("=" * 92)
print("ANC8 OUTCOME CLASS BALANCE")
print("=" * 92)

print("\nOutcome definition")
print("------------------")
print("0 = Fewer than 8 ANC contacts")
print("1 = 8 or more ANC contacts")

print("\nUnweighted distribution")
print("-----------------------")

print(
    f"0 = Fewer than 8 ANC contacts: "
    f"{int(unweighted_counts.loc[0])} "
    f"({unweighted_percent.loc[0]:.2f}%)"
)

print(
    f"1 = 8 or more ANC contacts:    "
    f"{int(unweighted_counts.loc[1])} "
    f"({unweighted_percent.loc[1]:.2f}%)"
)

print("\nSurvey-weighted prevalence")
print("--------------------------")

print(
    f"ANC8+ weighted prevalence: "
    f"{weighted_prevalence * 100:.2f}%"
)

print(
    f"ANC8<8 weighted prevalence: "
    f"{(1 - weighted_prevalence) * 100:.2f}%"
)

print("\nPrecision-recall reference")
print("--------------------------")

print(
    "No-skill average-precision baseline "
    f"≈ weighted ANC8 prevalence = "
    f"{weighted_prevalence:.4f}"
)

# Average Precision (AP):
# Measures how well the model ranks true ANC8+ women toward the top of its
# predicted-risk list. Higher is better.

# No-skill baseline:
# The performance expected from a model with no useful ability to distinguish
# ANC8+ from ANC8<8. For AP, this is approximately the positive-class prevalence.

## 2.3 Survey clustering and split-design audit

EDHS respondents are sampled within primary sampling units (PSUs or clusters).
Women from the same PSU may share geographic, socioeconomic, health-service,
and contextual characteristics.

If women from the same PSU appear in both the training and test datasets, the
model may partly benefit from cluster-specific similarities that would not be
available when predicting for women from entirely unseen communities.

Therefore, the primary train/test split will be **PSU-aware**:

- all women from a given PSU will be assigned together to either training or
  testing;
- no PSU will appear in both sets;
- the test set will remain untouched during model development;
- cross-validation within the training set will also respect PSU grouping.

Before creating the split, the number and size of PSUs, survey strata, and the
distribution of ANC8 outcomes across PSUs are audited.

In [ ]:
# =============================================================================
# AUDIT DHS CLUSTERING STRUCTURE BEFORE TRAIN / TEST SPLITTING
#
# NEW TERMS
# ---------
# PSU (Primary Sampling Unit):
#   A DHS sampling cluster. Women sampled from the same PSU may be more similar
#   to each other than women sampled from different PSUs.
#
# Group-aware split:
#   A split that keeps every woman from the same PSU together. A PSU is never
#   allowed to appear partly in training and partly in testing.
#
# WHY THIS MATTERS
# ----------------
# A random woman-level split could place neighbors or women from the same
# sampled community on both sides of the train/test boundary. That can make
# predictive performance look better than it would on completely unseen
# communities.
#
# No split is created in this cell. This is an audit only.
# =============================================================================


# -----------------------------------------------------------------------------
# 1. Basic survey-design counts
# -----------------------------------------------------------------------------

n_psu = baseline_df["v021"].nunique()
n_strata = baseline_df["v022"].nunique()

print("=" * 94)
print("DHS CLUSTERING STRUCTURE")
print("=" * 94)

print("\nAnalytic records:", len(baseline_df))
print("Unique PSUs (v021):", n_psu)
print("Unique strata (v022):", n_strata)


# -----------------------------------------------------------------------------
# 2. PSU size distribution
#
# This tells us how many women from the ANC8 analytic population are contained
# in each sampling cluster.
# -----------------------------------------------------------------------------

psu_sizes = (
    baseline_df
    .groupby("v021")
    .size()
)

print("\n" + "=" * 94)
print("NUMBER OF ANC8 ANALYTIC RECORDS PER PSU")
print("=" * 94)

print(psu_sizes.describe())

print("\nSmallest PSU size:", psu_sizes.min())
print("Largest PSU size:", psu_sizes.max())


# -----------------------------------------------------------------------------
# 3. ANC8 outcome composition within PSUs
#
# A PSU may contain:
#   - only ANC8<8 women,
#   - only ANC8+ women,
#   - or both outcome classes.
#
# This is important because grouped splitting is more constrained than ordinary
# random splitting.
# -----------------------------------------------------------------------------

psu_outcome = (
    baseline_df
    .groupby("v021")["anc8plus"]
    .agg(
        n="size",
        positive_n="sum",
        positive_rate="mean"
    )
)

psu_outcome["negative_n"] = (
    psu_outcome["n"] - psu_outcome["positive_n"]
)

only_negative = (
    psu_outcome["positive_n"].eq(0)
).sum()

only_positive = (
    psu_outcome["negative_n"].eq(0)
).sum()

mixed_outcome = (
    (psu_outcome["positive_n"] > 0)
    &
    (psu_outcome["negative_n"] > 0)
).sum()


print("\n" + "=" * 94)
print("ANC8 OUTCOME COMPOSITION ACROSS PSUs")
print("=" * 94)

print("PSUs containing only ANC8<8 women:", only_negative)
print("PSUs containing only ANC8+ women:", only_positive)
print("PSUs containing both outcome classes:", mixed_outcome)


# -----------------------------------------------------------------------------
# 4. Verify PSU-to-stratum relationship
#
# Each PSU should normally belong to one survey stratum.
# We explicitly check this rather than assume it.
# -----------------------------------------------------------------------------

strata_per_psu = (
    baseline_df
    .groupby("v021")["v022"]
    .nunique()
)

psus_in_multiple_strata = (
    strata_per_psu > 1
).sum()


print("\n" + "=" * 94)
print("PSU / STRATUM CONSISTENCY")
print("=" * 94)

print(
    "PSUs appearing in more than one stratum:",
    psus_in_multiple_strata
)

if psus_in_multiple_strata == 0:
    print("Status: each PSU maps to one survey stratum.")
else:
    print("Status: investigate PSU identifiers before splitting.")


# -----------------------------------------------------------------------------
# 5. Survey-weighted ANC8 prevalence by PSU
#
# This is descriptive only. It shows how heterogeneous ANC8 attainment is
# across sampled clusters.
# -----------------------------------------------------------------------------

psu_weighted = (
    baseline_df
    .assign(
        weighted_positive=
            baseline_df["survey_weight"]
            * baseline_df["anc8plus"]
    )
    .groupby("v021")
    .agg(
        total_weight=("survey_weight", "sum"),
        positive_weight=("weighted_positive", "sum")
    )
)

psu_weighted["weighted_ANC8_prevalence"] = (
    psu_weighted["positive_weight"]
    / psu_weighted["total_weight"]
)

print("\n" + "=" * 94)
print("WEIGHTED ANC8 PREVALENCE ACROSS PSUs")
print("=" * 94)

print(
    psu_weighted["weighted_ANC8_prevalence"]
    .describe()
)
# -----------------------------------------------------------------------------
# 6. Number of PSUs represented within each survey stratum
#
# This determines whether a train/test split can preserve survey-stratum
# representation while keeping whole PSUs together.
# -----------------------------------------------------------------------------

psus_per_stratum = (
    baseline_df[
        ["v022", "v021"]
    ]
    .drop_duplicates()
    .groupby("v022")
    .size()
    .rename("n_psus")
)

print("\n" + "=" * 94)
print("PSUs PER SURVEY STRATUM")
print("=" * 94)

print(psus_per_stratum)

print("\nMinimum PSUs in any stratum:", psus_per_stratum.min())
print("Maximum PSUs in any stratum:", psus_per_stratum.max())

## 2.4 Stratum-preserving PSU-aware train/test holdout

The final evaluation set is created without using ANC8 outcome values to choose
among candidate splits.

Because EDHS respondents are clustered within primary sampling units (PSUs),
all women from the same PSU remain in the same data partition.

Because the survey also uses stratification, the holdout is constructed within
each survey stratum so that both development and test datasets retain
representation from all 27 strata.

Within each stratum, approximately 20% of PSUs are randomly assigned to the
held-out test set using a single prespecified random seed. The remaining PSUs
form the development dataset.

The ANC8 outcome is not used to select or optimize the split.

After the partition is locked, outcome prevalence is reported descriptively to
assess the resulting development and test populations. No repartitioning is
performed on the basis of these outcome results.

The held-out test set is excluded from preprocessing estimation,
hyperparameter tuning, model selection, and threshold selection.

In [ ]:
# =============================================================================
# 2.4 CREATE STRATUM-PRESERVING, PSU-AWARE TRAIN / TEST HOLDOUT
#
# Scientific design:
# - split unit = PSU, not individual woman
# - splitting occurs separately within each EDHS survey stratum
# - one prespecified random seed is used
# - ANC8 outcome values play NO role in selecting the test set
# =============================================================================

SPLIT_SEED = 2026
TARGET_TEST_PROPORTION = 0.20

rng_split = np.random.default_rng(SPLIT_SEED)


# -----------------------------------------------------------------------------
# 1. Build one row per PSU with its survey stratum.
#
# Section 2.3 already verified that every PSU maps to exactly one stratum.
# -----------------------------------------------------------------------------

psu_frame = (
    baseline_df[
        ["v021", "v022"]
    ]
    .drop_duplicates()
    .sort_values(
        ["v022", "v021"]
    )
    .reset_index(drop=True)
)

assert psu_frame["v021"].is_unique


# -----------------------------------------------------------------------------
# 2. Sample approximately 20% of PSUs WITHIN EACH STRATUM.
#
# Because every stratum contains at least 12 PSUs, both development and test
# partitions can contain multiple PSUs from every stratum.
#
# No outcome variable is referenced anywhere in this selection step.
# -----------------------------------------------------------------------------

test_psu_list = []

for stratum, stratum_psus in psu_frame.groupby("v022", sort=True):

    psus = stratum_psus["v021"].to_numpy()

    # Nearest whole number of PSUs to 20%, with a minimum of 1.
    n_test_psus = max(
        1,
        int(np.round(
            len(psus) * TARGET_TEST_PROPORTION
        ))
    )

    selected_psus = rng_split.choice(
        psus,
        size=n_test_psus,
        replace=False
    )

    test_psu_list.extend(
        selected_psus.tolist()
    )


test_psus = set(test_psu_list)

train_psus = set(
    psu_frame["v021"]
) - test_psus


# -----------------------------------------------------------------------------
# 3. Create woman-level development and test datasets.
# -----------------------------------------------------------------------------

train_df = baseline_df.loc[
    baseline_df["v021"].isin(train_psus)
].copy()

test_df = baseline_df.loc[
    baseline_df["v021"].isin(test_psus)
].copy()


# -----------------------------------------------------------------------------
# 4. Integrity checks.
# -----------------------------------------------------------------------------

assert len(train_psus.intersection(test_psus)) == 0

assert (
    len(train_df) + len(test_df)
    == len(baseline_df)
)

assert set(train_df["v021"]).isdisjoint(
    set(test_df["v021"])
)

# Every survey stratum should be represented in both partitions.
assert (
    set(train_df["v022"].unique())
    == set(baseline_df["v022"].unique())
)

assert (
    set(test_df["v022"].unique())
    == set(baseline_df["v022"].unique())
)


# -----------------------------------------------------------------------------
# 5. Audit PSU allocation by stratum.
# -----------------------------------------------------------------------------

split_stratum_audit = (
    psu_frame
    .assign(
        partition=np.where(
            psu_frame["v021"].isin(test_psus),
            "Test",
            "Development"
        )
    )
    .groupby(
        ["v022", "partition"]
    )
    .size()
    .unstack(fill_value=0)
)

split_stratum_audit["Total"] = (
    split_stratum_audit.sum(axis=1)
)

split_stratum_audit["Test_fraction_PSU"] = (
    split_stratum_audit["Test"]
    / split_stratum_audit["Total"]
)


# -----------------------------------------------------------------------------
# 6. AFTER the split is locked, describe outcome prevalence.
#
# These values are diagnostics only.
# They are NOT used to alter or reselect the split.
# -----------------------------------------------------------------------------

overall_weighted_prev = np.average(
    baseline_df["anc8plus"],
    weights=baseline_df["survey_weight"]
)

train_weighted_prev = np.average(
    train_df["anc8plus"],
    weights=train_df["survey_weight"]
)

test_weighted_prev = np.average(
    test_df["anc8plus"],
    weights=test_df["survey_weight"]
)


# -----------------------------------------------------------------------------
# 7. Display the locked split.
# -----------------------------------------------------------------------------

print("=" * 100)
print("STRATUM-PRESERVING PSU-AWARE TRAIN / TEST HOLDOUT")
print("=" * 100)

print("\nSplit design")
print("------------")
print("Random seed:", SPLIT_SEED)
print("Outcome used to choose split: No")
print("Split unit: PSU")
print("Stratified by survey stratum (v022): Yes")

print("\nFull analytic population")
print("------------------------")
print("Women:", len(baseline_df))
print("PSUs:", baseline_df["v021"].nunique())
print(
    "Weighted ANC8+ prevalence:",
    f"{overall_weighted_prev * 100:.2f}%"
)

print("\nDevelopment set")
print("---------------")
print("Women:", len(train_df))
print("PSUs:", train_df["v021"].nunique())
print(
    "Weighted ANC8+ prevalence:",
    f"{train_weighted_prev * 100:.2f}%"
)

print("\nHeld-out test set")
print("-----------------")
print("Women:", len(test_df))
print("PSUs:", test_df["v021"].nunique())
print(
    "Weighted ANC8+ prevalence:",
    f"{test_weighted_prev * 100:.2f}%"
)

print("\nIntegrity")
print("---------")
print(
    "Overlapping PSUs:",
    len(
        set(train_df["v021"])
        .intersection(
            set(test_df["v021"])
        )
    )
)
print(
    "Test fraction of women:",
    f"{len(test_df) / len(baseline_df) * 100:.2f}%"
)
print(
    "Test fraction of PSUs:",
    f"{len(test_psus) / psu_frame.shape[0] * 100:.2f}%"
)

print("\nPSU allocation by stratum")
print("-------------------------")
print(split_stratum_audit)

print(
    "\nSTATUS: OUTCOME-INDEPENDENT "
    "STRATUM-PRESERVING PSU HOLDOUT CREATED"
)

In [ ]:
# =============================================================================
# COHORT ACCOUNTING AUDIT AFTER TRAIN / TEST SPLIT
# =============================================================================

assert len(train_df) + len(test_df) == len(baseline_df)

assert set(train_df["caseid"]).isdisjoint(
    set(test_df["caseid"])
)

assert (
    set(train_df["caseid"])
    .union(set(test_df["caseid"]))
    == set(baseline_df["caseid"])
)

print("=" * 90)
print("COHORT ACCOUNTING AUDIT")
print("=" * 90)
print("Full cohort:", len(baseline_df))
print("Development:", len(train_df))
print("Test:", len(test_df))
print("Development + test:", len(train_df) + len(test_df))
print("Duplicate caseids across partitions: 0")
print("All baseline caseids accounted for: True")

## 2.5 Development-data machine-learning readiness audit

Before preprocessing or model fitting, the predictors are audited within the
development dataset only.

The held-out test set is not inspected in this section. This preserves its role
as an independent final evaluation sample and prevents information about its
predictor distribution from influencing preprocessing or model-development
decisions.

The audit has four purposes:

1. verify that all 21 frozen predictors are assigned to an explicit
   representation class;
2. quantify true missingness in the development data;
3. verify the observed levels and frequency of categorical predictors;
4. distinguish substantive structural categories from unresolved information
   and true missing values.

### Predictor representation classes

The 21 predictors are assigned to three mutually exclusive groups:

- **numeric/count predictors:** retain meaningful numeric values;
- **binary predictors:** have two prespecified substantive levels and will later
  be encoded explicitly as 0/1;
- **multi-category predictors:** will later be represented using one-hot
  encoding.

`traditional_media_weekly_count` is treated as a numeric count because its
values 0–3 represent the number of traditional media sources accessed at least
weekly.

Categories such as `Not applicable` and `First pregnancy` are substantive
structural states and are retained as categories.

Explicit labels such as `Unknown` or `Unknown / missing` are retained as
observed unresolved-information categories rather than silently converted to
missing values.

True `NaN` values remain genuine missing data and will be handled within the
model preprocessing pipeline using rules learned from development data only.

Sparse categories are flagged descriptively for review but are not
automatically collapsed.

No imputation, scaling, encoding, feature selection, or model fitting occurs in
this section.

In [ ]:
# =============================================================================
# 2.5 DEVELOPMENT-DATA MACHINE-LEARNING READINESS AUDIT
#
# PURPOSE
# -------
# Audit the 21 frozen predictors BEFORE preprocessing and modelling.
#
# IMPORTANT GOVERNANCE RULE
# -------------------------
# This section uses DEVELOPMENT DATA ONLY.
#
# The held-out test set is intentionally not inspected here. Its predictor
# distributions, missingness, and category levels will be audited only after
# the modelling pipeline and hyperparameters have been locked.
#
# No values are changed in this cell.
# =============================================================================


# =============================================================================
# 1. DEFINE THE THREE PREDICTOR REPRESENTATION GROUPS
#
# These definitions become the canonical predictor-role objects used by the
# preprocessing section that follows.
# =============================================================================

NUMERIC_PREDICTORS = [
    "v012",                           # maternal age at interview, years
    "household_size",                 # number of household members
    "travel_time_min",                # travel time to health facility, minutes
    "traditional_media_weekly_count"  # count of weekly media sources, 0-3
]


BINARY_PREDICTORS = [
    "maternal_current_union_status",  # Not currently / Currently in union
    "residence",                      # Rural / Urban
    "health_insurance"                # No / Yes
]


CATEGORICAL_PREDICTORS_ML = [
    predictor
    for predictor in PRIMARY_PREDICTORS
    if predictor not in NUMERIC_PREDICTORS + BINARY_PREDICTORS
]


# =============================================================================
# 2. AUDIT THAT THE REPRESENTATION GROUPS ARE COMPLETE AND NON-OVERLAPPING
# =============================================================================

all_defined_predictors = (
    NUMERIC_PREDICTORS
    + BINARY_PREDICTORS
    + CATEGORICAL_PREDICTORS_ML
)

assert len(all_defined_predictors) == 21, (
    "Predictor representation does not contain exactly 21 predictors."
)

assert len(set(all_defined_predictors)) == 21, (
    "A predictor appears in more than one representation group."
)

assert set(all_defined_predictors) == set(PRIMARY_PREDICTORS), (
    "Predictor representation does not exactly match PRIMARY_PREDICTORS."
)


print("=" * 100)
print("PREDICTOR REPRESENTATION AUDIT")
print("=" * 100)

print("\nNumeric/count predictors:", len(NUMERIC_PREDICTORS))
for var in NUMERIC_PREDICTORS:
    print(f"  - {var}")

print("\nBinary predictors:", len(BINARY_PREDICTORS))
for var in BINARY_PREDICTORS:
    print(f"  - {var}")

print(
    "\nMulti-category predictors:",
    len(CATEGORICAL_PREDICTORS_ML)
)
for var in CATEGORICAL_PREDICTORS_ML:
    print(f"  - {var}")

print("\nTotal predictors:", len(all_defined_predictors))


# =============================================================================
# 3. TRUE MISSINGNESS AUDIT
#
# Only actual NaN values are counted here.
#
# Labels such as:
#   - "Not applicable"
#   - "Unknown"
#   - "Unknown / missing"
#
# are observed category values and are therefore NOT counted as NaN.
# =============================================================================

missing_rows = []

for var in PRIMARY_PREDICTORS:

    n_missing = int(
        train_df[var].isna().sum()
    )

    missing_rows.append({
        "Predictor": var,
        "Missing_n": n_missing,
        "Missing_pct": (
            n_missing / len(train_df) * 100
        )
    })


development_missing_audit = pd.DataFrame(
    missing_rows
)


print("\n" + "=" * 100)
print("TRUE MISSINGNESS IN DEVELOPMENT DATA")
print("=" * 100)

print(
    development_missing_audit.to_string(
        index=False,
        formatters={
            "Missing_pct": "{:.3f}".format
        }
    )
)


# =============================================================================
# 4. NUMERIC / COUNT PREDICTOR DISTRIBUTIONS
#
# These summaries are descriptive only.
# No transformation decision is learned from the test set.
# =============================================================================

numeric_audit_rows = []

for var in NUMERIC_PREDICTORS:

    s = train_df[var]

    numeric_audit_rows.append({
        "Predictor": var,
        "N_nonmissing": int(s.notna().sum()),
        "Min": s.min(),
        "Median": s.median(),
        "Mean": s.mean(),
        "Max": s.max()
    })


development_numeric_audit = pd.DataFrame(
    numeric_audit_rows
)


print("\n" + "=" * 100)
print("NUMERIC / COUNT PREDICTOR AUDIT")
print("=" * 100)

print(
    development_numeric_audit.to_string(
        index=False,
        formatters={
            "Mean": "{:.2f}".format
        }
    )
)


# =============================================================================
# 5. CATEGORICAL LEVEL COUNTS
#
# Includes both binary and multi-category predictors.
#
# We create a long-format table because it is easier to:
#   - inspect now;
#   - export later;
#   - use as supporting documentation.
# =============================================================================

categorical_for_audit = (
    BINARY_PREDICTORS
    + CATEGORICAL_PREDICTORS_ML
)

category_count_rows = []

for var in categorical_for_audit:

    # Convert to pandas StringDtype so that filling true missing values with a
    # text label does not trigger pandas object-downcasting warnings.
    s_display = (
        train_df[var]
        .astype("string")
        .fillna("<TRUE MISSING>")
    )

    counts = s_display.value_counts(
        dropna=False
    )

    for level, n in counts.items():

        category_count_rows.append({
            "Predictor": var,
            "Category": str(level),
            "Development_n": int(n),
            "Development_pct": (
                int(n) / len(train_df) * 100
            )
        })


development_category_counts = pd.DataFrame(
    category_count_rows
)


print("\n" + "=" * 100)
print("CATEGORICAL LEVEL COUNTS IN DEVELOPMENT DATA")
print("=" * 100)

for var in categorical_for_audit:

    print(f"\n{var}")
    print("-" * len(var))

    display_table = (
        development_category_counts.loc[
            development_category_counts[
                "Predictor"
            ].eq(var),
            [
                "Category",
                "Development_n",
                "Development_pct"
            ]
        ]
        .copy()
    )

    print(
        display_table.to_string(
            index=False,
            formatters={
                "Development_pct": "{:.2f}".format
            }
        )
    )


# =============================================================================
# 6. EXPLICIT GOVERNANCE OF STRUCTURAL AND UNRESOLVED CATEGORIES
#
# STRUCTURAL:
# These are meaningful states. They must remain categories.
#
# UNRESOLVED:
# These explicitly indicate that information was not resolved. They are kept
# as observed categories in the primary preprocessing rather than silently
# being converted into NaN.
# =============================================================================

STRUCTURAL_CATEGORIES = {
    "partner_education": [
        "Not applicable"
    ],
    "partner_occupation_group": [
        "Not applicable"
    ],
    "healthcare_decision_participation": [
        "Not applicable"
    ],
    "preceding_pregnancy_interval_group": [
        "First pregnancy"
    ]
}


UNRESOLVED_CATEGORIES = {
    "partner_education": [
        "Unknown"
    ],
    "partner_occupation_group": [
        "Unknown"
    ],
    "preceding_pregnancy_interval_group": [
        "Unknown / missing"
    ]
}


# =============================================================================
# 7. VERIFY THAT ALL PRESPECIFIED SPECIAL CATEGORIES ACTUALLY EXIST
# =============================================================================

special_category_audit_rows = []


for variable, levels in STRUCTURAL_CATEGORIES.items():

    observed = set(
        train_df[variable]
        .dropna()
        .astype(str)
        .unique()
    )

    for level in levels:

        special_category_audit_rows.append({
            "Predictor": variable,
            "Category": level,
            "Type": "Structural",
            "Observed_in_development": (
                level in observed
            )
        })


for variable, levels in UNRESOLVED_CATEGORIES.items():

    observed = set(
        train_df[variable]
        .dropna()
        .astype(str)
        .unique()
    )

    for level in levels:

        special_category_audit_rows.append({
            "Predictor": variable,
            "Category": level,
            "Type": "Unresolved information",
            "Observed_in_development": (
                level in observed
            )
        })


special_category_audit = pd.DataFrame(
    special_category_audit_rows
)


assert special_category_audit[
    "Observed_in_development"
].all(), (
    "At least one prespecified structural or unresolved category "
    "was not observed in the development dataset."
)


print("\n" + "=" * 100)
print("STRUCTURAL / UNRESOLVED CATEGORY AUDIT")
print("=" * 100)

print(
    special_category_audit.to_string(
        index=False
    )
)


# =============================================================================
# 8. SPARSE CATEGORY AUDIT
#
# <30 observations is used only as a descriptive review flag.
# It is NOT a statistical threshold and does NOT trigger automatic collapsing.
# =============================================================================

SPARSE_CATEGORY_REVIEW_N = 30

development_sparse_categories = (
    development_category_counts.loc[
        development_category_counts[
            "Development_n"
        ] < SPARSE_CATEGORY_REVIEW_N
    ]
    .sort_values(
        ["Development_n", "Predictor", "Category"]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("SPARSE CATEGORY REVIEW")
print("=" * 100)

print(
    f"Descriptive review flag: "
    f"development n < {SPARSE_CATEGORY_REVIEW_N}"
)

if development_sparse_categories.empty:

    print(
        "\nNo categorical levels meet the "
        "descriptive sparse-category flag."
    )

else:

    print(
        development_sparse_categories[
            [
                "Predictor",
                "Category",
                "Development_n",
                "Development_pct"
            ]
        ].to_string(
            index=False,
            formatters={
                "Development_pct": "{:.3f}".format
            }
        )
    )


# =============================================================================
# 9. FINAL DEVELOPMENT-READINESS ASSERTIONS
# =============================================================================

assert train_df["caseid"].is_unique

assert len(train_df) == 5518, (
    "Unexpected development sample size. "
    "Check the locked 2.4 split."
)

assert train_df["anc8plus"].notna().all()

assert (
    set(NUMERIC_PREDICTORS)
    | set(BINARY_PREDICTORS)
    | set(CATEGORICAL_PREDICTORS_ML)
    == set(PRIMARY_PREDICTORS)
)


print("\n" + "=" * 100)
print("DEVELOPMENT-DATA ML READINESS AUDIT COMPLETED")
print("=" * 100)

print("Development women:", len(train_df))
print("Frozen predictors:", len(PRIMARY_PREDICTORS))
print("Numeric/count predictors:", len(NUMERIC_PREDICTORS))
print("Binary predictors:", len(BINARY_PREDICTORS))
print(
    "Multi-category predictors:",
    len(CATEGORICAL_PREDICTORS_ML)
)
print(
    "True missing values across all predictors:",
    int(
        train_df[
            PRIMARY_PREDICTORS
        ].isna().sum().sum()
    )
)
print("Held-out test predictors inspected here: No")
print("Predictor values modified here: No")
print("Preprocessing fitted here: No")

## 2.6 Governed predictor representation and preprocessing

Predictor representation is defined from the scientific meaning of each
variable and is fixed before model fitting.

The three predictor groups established in Section 2.5 are reused without
redefinition:

- four numeric/count predictors;
- three binary predictors;
- fourteen multi-category predictors.

### Numeric/count predictors

The numeric/count predictors are:

- maternal age at interview (`v012`);
- household size;
- travel time to the health facility;
- traditional media weekly count.

For regularized logistic regression, numeric predictors are median-imputed if
necessary and standardized using parameters learned from the relevant training
data.

For Random Forest and XGBoost, numeric predictors are median-imputed if
necessary but are not standardized because tree splits do not require variables
to share a common scale.

### Binary predictors

The three binary predictors have prespecified coding:

- maternal current union status:
  0 = Not currently in union, 1 = Currently in union;
- residence:
  0 = Rural, 1 = Urban;
- health insurance:
  0 = No, 1 = Yes.

These mappings are treated as strict data definitions. An unexpected binary
label will raise an error rather than being silently assigned an arbitrary
numeric value.

### Multi-category predictors

The remaining fourteen predictors are represented using one-hot encoding.

This avoids imposing equal numerical spacing or a linear ordering on predictors
such as education, wealth quintile, region, occupation, pregnancy order, and
pregnancy intention.

All observed category levels are retained. Structural states such as
`Not applicable` and `First pregnancy`, and explicit unresolved-information
categories such as `Unknown` and `Unknown / missing`, remain substantive
categories rather than being converted to missing values.

True missing values are handled by a most-frequent imputation rule learned only
from the relevant training data.

### Reproducibility and explainability requirements

All data-dependent preprocessing is contained inside model pipelines so that
imputation, scaling, and category learning are re-estimated within each
cross-validation training fold.

The preprocessing objects use standard scikit-learn transformers that preserve
retrievable transformed-feature names. This is required so that later
explainability analyses can map model features back to the original predictor
constructs without reconstructing or guessing the transformation history.

The held-out test set does not contribute to preprocessing estimation.

In [ ]:
# =============================================================================
# 2.6 DEFINE GOVERNED MODEL-SPECIFIC PREPROCESSING
#
# IMPORTANT
# ---------
# NUMERIC_PREDICTORS, BINARY_PREDICTORS and CATEGORICAL_PREDICTORS_ML were
# defined and audited in Section 2.5.
#
# They are deliberately NOT redefined here.
#
# These preprocessing objects are only DEFINED here. Nothing is fitted yet.
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer


# =============================================================================
# 1. STRICT BINARY CATEGORY ORDER
#
# OrdinalEncoder is used solely as a controlled text -> 0/1 converter.
#
# Because these categories are prespecified and known, unexpected labels should
# raise an error rather than being silently converted to another numeric value.
# =============================================================================

BINARY_CATEGORIES = [
    [
        "Not currently in union",
        "Currently in union"
    ],
    [
        "Rural",
        "Urban"
    ],
    [
        "No",
        "Yes"
    ]
]


binary_encoder = OrdinalEncoder(
    categories=BINARY_CATEGORIES,
    dtype=float
)


# =============================================================================
# 2. NUMERIC PREPROCESSING
# =============================================================================

# Logistic regression:
# median imputation followed by standardization.
numeric_logistic = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Tree models:
# median imputation only.
numeric_tree = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


# =============================================================================
# 3. MULTI-CATEGORY PREPROCESSING
#
# Genuine NaN values are imputed from training data only.
#
# Named categories such as:
#   "Not applicable"
#   "Unknown"
#   "Unknown / missing"
# remain ordinary observed categories.
#
# handle_unknown="ignore" allows later validation/test observations to contain
# a category that was absent from a particular training fold without causing
# the pipeline to fail.
#
# drop=None retains the complete one-hot representation.
# =============================================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse_output=True,
                dtype=float
            )
        )
    ]
)


# =============================================================================
# 4. LOGISTIC-REGRESSION PREPROCESSOR
# =============================================================================

preprocessor_logistic = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_logistic,
            NUMERIC_PREDICTORS
        ),
        (
            "binary",
            binary_encoder,
            BINARY_PREDICTORS
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_PREDICTORS_ML
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# =============================================================================
# 5. TREE-MODEL PREPROCESSOR
# =============================================================================

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_tree,
            NUMERIC_PREDICTORS
        ),
        (
            "binary",
            binary_encoder,
            BINARY_PREDICTORS
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_PREDICTORS_ML
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# =============================================================================
# 6. GOVERNANCE CHECKS
# =============================================================================

assert (
    set(NUMERIC_PREDICTORS)
    | set(BINARY_PREDICTORS)
    | set(CATEGORICAL_PREDICTORS_ML)
    == set(PRIMARY_PREDICTORS)
)

assert (
    len(NUMERIC_PREDICTORS)
    + len(BINARY_PREDICTORS)
    + len(CATEGORICAL_PREDICTORS_ML)
    == len(PRIMARY_PREDICTORS)
)


print("=" * 94)
print("GOVERNED PREPROCESSING SPECIFICATION DEFINED")
print("=" * 94)

print("\nPredictor representation")
print("------------------------")
print("Numeric/count:", len(NUMERIC_PREDICTORS))
print("Binary:", len(BINARY_PREDICTORS))
print(
    "Multi-category:",
    len(CATEGORICAL_PREDICTORS_ML)
)

print("\nLogistic regression")
print("-------------------")
print("Numeric -> median imputation + standardization")
print("Binary -> strict prespecified 0/1 encoding")
print("Multi-category -> most-frequent imputation + one-hot encoding")

print("\nRandom Forest / XGBoost")
print("-----------------------")
print("Numeric -> median imputation, no scaling")
print("Binary -> same strict prespecified 0/1 encoding")
print("Multi-category -> same one-hot representation")

print("\nExplainability readiness")
print("------------------------")
print("Standard sklearn transformers only: Yes")
print("Feature names retrievable after fitting: Yes")
print("Original predictor grouping preserved: Yes")

print("\nPreprocessors fitted at this stage: No")
print("Held-out test set used here: No")

## 2.7 Development preprocessing and feature-lineage audit

Before model development, temporary copies of the preprocessing objects are
fitted to the complete development dataset to verify that the governed
transformations operate as intended.

This is a functionality and reproducibility audit only. The fitted audit
objects and transformed matrices produced here are not used for
cross-validation or final model fitting.

The held-out test set is not inspected or transformed in this section.

The audit verifies:

1. all 21 original predictors enter the preprocessing system;
2. transformed values are finite;
3. logistic-regression and tree preprocessors produce the same transformed
   feature structure;
4. every transformed feature can be traced back to exactly one original
   predictor;
5. every original predictor maps to at least one transformed feature;
6. transformed feature names are unique.

A feature-lineage table is created linking each transformed model feature to
its original predictor construct. This table will later be exported as a
governance artifact for Notebook 3 so that explainability analyses do not need
to reconstruct or infer the preprocessing history.

Actual model development will continue to place preprocessing inside each
model pipeline so that preprocessing parameters are learned independently
within each cross-validation training fold.

In [ ]:
# =============================================================================
# 2.7 DEVELOPMENT PREPROCESSING AND FEATURE-LINEAGE AUDIT
#
# IMPORTANT
# ---------
# - DEVELOPMENT DATA ONLY
# - No held-out test data are inspected or transformed here.
# - These fitted preprocessors are temporary audit copies.
# - They are NOT used for model fitting or cross-validation.
# =============================================================================

from sklearn.base import clone
import scipy.sparse as sp


# =============================================================================
# 1. DEVELOPMENT PREDICTOR MATRIX
# =============================================================================

X_dev_audit = train_df[
    PRIMARY_PREDICTORS
].copy()

assert len(X_dev_audit) == len(train_df)
assert list(X_dev_audit.columns) == list(PRIMARY_PREDICTORS)


# =============================================================================
# 2. FIT TEMPORARY PREPROCESSOR COPIES ON DEVELOPMENT DATA ONLY
# =============================================================================

audit_logistic_preprocessor = clone(
    preprocessor_logistic
)

audit_tree_preprocessor = clone(
    preprocessor_tree
)


audit_logistic_preprocessor.fit(
    X_dev_audit
)

audit_tree_preprocessor.fit(
    X_dev_audit
)


# =============================================================================
# 3. TRANSFORM DEVELOPMENT DATA
# =============================================================================

X_dev_logistic_audit = (
    audit_logistic_preprocessor.transform(
        X_dev_audit
    )
)

X_dev_tree_audit = (
    audit_tree_preprocessor.transform(
        X_dev_audit
    )
)


# =============================================================================
# 4. RETRIEVE TRANSFORMED FEATURE NAMES
#
# One reason we deliberately used standard sklearn transformers in Section 2.6
# is that feature names can now be retrieved directly rather than reconstructed
# manually.
# =============================================================================

logistic_feature_names = (
    audit_logistic_preprocessor
    .get_feature_names_out()
)

tree_feature_names = (
    audit_tree_preprocessor
    .get_feature_names_out()
)


# Logistic and tree preprocessors use the same substantive representation.
# Scaling numeric variables must not alter feature identity.
assert np.array_equal(
    logistic_feature_names,
    tree_feature_names
), (
    "Logistic and tree preprocessors produced different feature-name structures."
)


# =============================================================================
# 5. FINITE-VALUE HELPER
# =============================================================================

def matrix_is_finite(matrix):
    """
    Check whether every stored value in a dense or sparse transformed matrix
    is finite.

    Sparse matrices store only non-zero values in `.data`, so checking that
    array is sufficient for their explicitly stored entries.
    """

    values = (
        matrix.data
        if sp.issparse(matrix)
        else np.asarray(matrix)
    )

    return np.isfinite(values).all()


assert matrix_is_finite(
    X_dev_logistic_audit
)

assert matrix_is_finite(
    X_dev_tree_audit
)


# =============================================================================
# 6. BUILD AN EXPLICIT FEATURE-LINEAGE TABLE
#
# The aim is:
#
# transformed model feature
#           ↓
# original scientific predictor
#
# This mapping will later support model interpretation without guessing which
# one-hot columns belong to which original construct.
# =============================================================================

lineage_rows = []


# -----------------------------------------------------------------------------
# 6A. Numeric block
#
# Each numeric predictor produces exactly one transformed feature.
# -----------------------------------------------------------------------------

numeric_slice = (
    audit_logistic_preprocessor
    .output_indices_["numeric"]
)

numeric_output_names = (
    logistic_feature_names[
        numeric_slice
    ]
)

assert (
    len(numeric_output_names)
    == len(NUMERIC_PREDICTORS)
)

for transformed_name, original_predictor in zip(
    numeric_output_names,
    NUMERIC_PREDICTORS
):

    lineage_rows.append({
        "Transformed_feature": transformed_name,
        "Original_predictor": original_predictor,
        "Preprocessing_block": "numeric"
    })


# -----------------------------------------------------------------------------
# 6B. Binary block
#
# Each binary predictor produces exactly one 0/1 transformed feature.
# -----------------------------------------------------------------------------

binary_slice = (
    audit_logistic_preprocessor
    .output_indices_["binary"]
)

binary_output_names = (
    logistic_feature_names[
        binary_slice
    ]
)

assert (
    len(binary_output_names)
    == len(BINARY_PREDICTORS)
)

for transformed_name, original_predictor in zip(
    binary_output_names,
    BINARY_PREDICTORS
):

    lineage_rows.append({
        "Transformed_feature": transformed_name,
        "Original_predictor": original_predictor,
        "Preprocessing_block": "binary"
    })


# -----------------------------------------------------------------------------
# 6C. Multi-category block
#
# One original categorical predictor may create several one-hot columns.
#
# We use the fitted encoder's own `categories_` attribute to determine exactly
# how many transformed columns belong to each original predictor.
#
# This avoids parsing feature-name strings or relying on naming conventions.
# -----------------------------------------------------------------------------

fitted_categorical_pipeline = (
    audit_logistic_preprocessor
    .named_transformers_["categorical"]
)

fitted_onehot = (
    fitted_categorical_pipeline
    .named_steps["onehot"]
)

categorical_slice = (
    audit_logistic_preprocessor
    .output_indices_["categorical"]
)

categorical_output_names = (
    logistic_feature_names[
        categorical_slice
    ]
)


categorical_original_predictors = []

for predictor, learned_categories in zip(
    CATEGORICAL_PREDICTORS_ML,
    fitted_onehot.categories_
):

    categorical_original_predictors.extend(
        [predictor] * len(learned_categories)
    )


assert (
    len(categorical_output_names)
    == len(categorical_original_predictors)
), (
    "Categorical transformed-feature count does not match the "
    "one-hot category structure."
)


for transformed_name, original_predictor in zip(
    categorical_output_names,
    categorical_original_predictors
):

    lineage_rows.append({
        "Transformed_feature": transformed_name,
        "Original_predictor": original_predictor,
        "Preprocessing_block": "categorical"
    })


# =============================================================================
# 7. FINAL FEATURE-LINEAGE TABLE
# =============================================================================

feature_lineage = pd.DataFrame(
    lineage_rows
)

feature_lineage.insert(
    0,
    "Feature_index",
    np.arange(
        len(feature_lineage)
    )
)


# =============================================================================
# 8. FEATURE-LINEAGE INTEGRITY CHECKS
# =============================================================================

assert (
    len(feature_lineage)
    == len(logistic_feature_names)
)

assert feature_lineage[
    "Transformed_feature"
].is_unique, (
    "Transformed feature names are not unique."
)

assert (
    set(feature_lineage["Original_predictor"])
    == set(PRIMARY_PREDICTORS)
), (
    "Not all 21 original predictors are represented in the feature lineage."
)


# Every original predictor must contribute at least one transformed column.
features_per_predictor = (
    feature_lineage
    .groupby("Original_predictor")
    .size()
    .rename("N_transformed_features")
    .reindex(PRIMARY_PREDICTORS)
)


assert features_per_predictor.notna().all()
assert (features_per_predictor >= 1).all()


# =============================================================================
# 9. SUMMARIZE THE AUDIT
# =============================================================================

print("=" * 100)
print("DEVELOPMENT PREPROCESSING AND FEATURE-LINEAGE AUDIT")
print("=" * 100)

print("\nDevelopment population")
print("----------------------")
print("Women:", len(X_dev_audit))
print(
    "Original predictors:",
    len(PRIMARY_PREDICTORS)
)

print("\nTransformed feature space")
print("-------------------------")
print(
    "Logistic transformed features:",
    X_dev_logistic_audit.shape[1]
)
print(
    "Tree transformed features:",
    X_dev_tree_audit.shape[1]
)

print(
    "Feature structures identical:",
    np.array_equal(
        logistic_feature_names,
        tree_feature_names
    )
)

print("\nMatrix integrity")
print("----------------")
print(
    "Logistic matrix finite:",
    matrix_is_finite(
        X_dev_logistic_audit
    )
)
print(
    "Tree matrix finite:",
    matrix_is_finite(
        X_dev_tree_audit
    )
)

print("\nFeature lineage")
print("---------------")
print(
    "Lineage rows:",
    len(feature_lineage)
)
print(
    "Unique transformed names:",
    feature_lineage[
        "Transformed_feature"
    ].is_unique
)
print(
    "Original predictors represented:",
    feature_lineage[
        "Original_predictor"
    ].nunique()
)

print("\nTransformed features per original predictor")
print("-------------------------------------------")
print(
    features_per_predictor.to_string()
)

print("\nGovernance")
print("----------")
print("Held-out test data used: No")
print("Audit preprocessors used for model fitting: No")
print("Feature lineage constructed from fitted sklearn objects: Yes")

print(
    "\nSTATUS: DEVELOPMENT PREPROCESSING "
    "AND FEATURE LINEAGE VERIFIED"
)

In [ ]:
# Show the actual 0/1 coding produced by the fitted binary encoder

binary_audit = pd.DataFrame(
    audit_logistic_preprocessor
    .named_transformers_["binary"]
    .transform(
        X_dev_audit[
            BINARY_PREDICTORS
        ]
    ),
    columns=BINARY_PREDICTORS
)

print(
    binary_audit.value_counts(
        subset=["residence"]
    )
)

print("\nFirst 10 encoded rows:")
print(
    binary_audit.head(80)
)

In [ ]:
print("Numeric features:", len(NUMERIC_PREDICTORS))
print("Binary features:", len(BINARY_PREDICTORS))

categorical_feature_count = (
    features_per_predictor.loc[
        CATEGORICAL_PREDICTORS_ML
    ].sum()
)

print(
    "One-hot categorical features:",
    int(categorical_feature_count)
)

print(
    "Total transformed features:",
    len(NUMERIC_PREDICTORS)
    + len(BINARY_PREDICTORS)
    + int(categorical_feature_count)
)

In [ ]:
print(
    feature_lineage.to_string(
        index=False
    )
)

## 2.8 Stratum-preserving PSU-grouped internal cross-validation

Model development uses five-fold internal cross-validation within the development
dataset.

The resampling scheme respects both major design features relevant to the
analysis:

- women from the same primary sampling unit (PSU) remain together; and
- PSUs are allocated to folds within survey strata so that each validation fold
  retains representation from all 27 EDHS strata.

The ANC8 outcome is not used to construct the folds.

Within each survey stratum, development PSUs are reproducibly shuffled using a
prespecified random seed and distributed across the five folds as evenly as
possible.

Each PSU is assigned to exactly one validation fold. Therefore, across the five
iterations:

1. every development PSU serves once as validation data;
2. no PSU appears simultaneously in fitting and validation data;
3. every development woman receives exactly one out-of-fold prediction when
   out-of-fold predictions are later generated;
4. preprocessing is fitted only within the fitting portion of each fold; and
5. the held-out final test set remains completely outside model development.

Outcome prevalence is examined only after fold assignment as a descriptive
diagnostic and is not used to alter the folds.

The same locked fold assignments will be reused for all candidate algorithms
and hyperparameter configurations to ensure fair model comparison.
## cv_splits is the canonical cross-validation object for all downstream model
## tuning, out-of-fold prediction, threshold selection, and sensitivity analyses.
## Do not recreate or replace CV folds later in Notebook 2.

In [ ]:
# =============================================================================
# 2.8 STRATUM-PRESERVING PSU-GROUPED CROSS-VALIDATION
#
# DESIGN
# ------
# - Development data only
# - 5 folds
# - PSU is the indivisible resampling unit
# - Fold allocation occurs within survey stratum
# - ANC8 outcome is NOT used to construct folds
# - One fixed seed governs fold assignment
#
# The resulting fold assignment is intended to be reused unchanged throughout
# model tuning and out-of-fold prediction.
# =============================================================================

N_FOLDS = 5
CV_SEED = 2026

rng_cv = np.random.default_rng(CV_SEED)


# =============================================================================
# 1. BUILD ONE ROW PER DEVELOPMENT PSU WITH ITS SURVEY STRATUM
# =============================================================================

development_psu_frame = (
    train_df[
        ["v021", "v022"]
    ]
    .drop_duplicates()
    .sort_values(
        ["v022", "v021"]
    )
    .reset_index(drop=True)
)

assert development_psu_frame["v021"].is_unique

assert (
    development_psu_frame["v022"].nunique()
    == train_df["v022"].nunique()
    == 27
)


# =============================================================================
# 2. ASSIGN PSUs TO FOLDS WITHIN EACH SURVEY STRATUM
#
# Within each stratum:
#   1. PSU identifiers are shuffled reproducibly.
#   2. A nearly equal number of PSUs is assigned to each of the five folds.
#   3. The fold labels themselves are randomly permuted so that any remainder
#      PSUs are not systematically allocated to the same fold numbers across
#      strata.
#
# ANC8 outcome values are not used.
# =============================================================================

psu_fold_rows = []

for stratum, stratum_psus in development_psu_frame.groupby(
    "v022",
    sort=True
):

    psus = (
        stratum_psus["v021"]
        .to_numpy()
        .copy()
    )

    # Reproducibly shuffle PSUs within this stratum.
    rng_cv.shuffle(psus)

    n_psus_stratum = len(psus)

    # Number of complete sets of five folds plus remainder.
    base_n = n_psus_stratum // N_FOLDS
    remainder = n_psus_stratum % N_FOLDS

    # Start with an equal number of assignments to every fold.
    fold_labels = np.repeat(
        np.arange(N_FOLDS),
        base_n
    )

    # If the stratum cannot divide exactly by five, allocate the remaining
    # PSUs to randomly selected folds rather than always to folds 0, 1, 2...
    if remainder > 0:

        remainder_folds = rng_cv.choice(
            np.arange(N_FOLDS),
            size=remainder,
            replace=False
        )

        fold_labels = np.concatenate(
            [
                fold_labels,
                remainder_folds
            ]
        )

    # Shuffle labels so that fold assignment is unrelated to PSU ordering.
    rng_cv.shuffle(fold_labels)

    assert len(fold_labels) == n_psus_stratum

    for psu, fold_id in zip(
        psus,
        fold_labels
    ):

        psu_fold_rows.append({
            "v021": psu,
            "v022": stratum,
            "cv_fold": int(fold_id)
        })


psu_fold_assignment = pd.DataFrame(
    psu_fold_rows
)
# =============================================================================
# 3. VALIDATE PSU-LEVEL FOLD ASSIGNMENT
# =============================================================================

assert psu_fold_assignment["v021"].is_unique

assert (
    psu_fold_assignment["v021"].nunique()
    == train_df["v021"].nunique()
)

assert set(
    psu_fold_assignment["cv_fold"].unique()
) == set(range(N_FOLDS))


# Every survey stratum must appear in every fold.
strata_by_fold = (
    psu_fold_assignment
    .groupby("cv_fold")["v022"]
    .nunique()
)

assert (
    strata_by_fold == 27
).all(), (
    "At least one CV fold does not contain all 27 survey strata."
)


# =============================================================================
# 4. MERGE LOCKED FOLD ASSIGNMENT BACK TO DEVELOPMENT WOMEN
# =============================================================================

development_cv_index = (
    train_df[
        ["caseid", "v021", "v022"]
    ]
    .merge(
        psu_fold_assignment,
        on=["v021", "v022"],
        how="left",
        validate="many_to_one"
    )
)


assert development_cv_index["cv_fold"].notna().all()

assert (
    development_cv_index["caseid"].nunique()
    == len(train_df)
)

assert development_cv_index["caseid"].is_unique


# =============================================================================
# 5. CREATE REUSABLE CV SPLITS
#
# Each tuple contains:
#   - fit_idx: row positions used for model fitting
#   - val_idx: row positions used for validation
#
# The indices refer to train_df / X_dev / y_dev in their current row order.
# =============================================================================

cv_splits = []

for fold_id in range(N_FOLDS):

    val_mask = (
        development_cv_index["cv_fold"]
        .eq(fold_id)
        .to_numpy()
    )

    val_idx = np.flatnonzero(
        val_mask
    )

    fit_idx = np.flatnonzero(
        ~val_mask
    )

    cv_splits.append(
        (fit_idx, val_idx)
    )


# =============================================================================
# 6. AUDIT EACH FOLD
#
# Outcome prevalence is calculated here only AFTER folds have been locked.
# It is descriptive and must not be used to revise fold assignments.
# =============================================================================

cv_audit_rows = []

for fold_id, (fit_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):

    fit_df = train_df.iloc[
        fit_idx
    ]

    val_df = train_df.iloc[
        val_idx
    ]

    fit_psus = set(
        fit_df["v021"]
    )

    val_psus = set(
        val_df["v021"]
    )

    overlap = (
        fit_psus
        .intersection(val_psus)
    )

    weighted_val_prev = np.average(
        val_df["anc8plus"],
        weights=val_df["survey_weight"]
    )

    weighted_fit_prev = np.average(
        fit_df["anc8plus"],
        weights=fit_df["survey_weight"]
    )

    cv_audit_rows.append({
        "Fold": fold_id,
        "Fit_women": len(fit_df),
        "Validation_women": len(val_df),
        "Fit_PSUs": len(fit_psus),
        "Validation_PSUs": len(val_psus),
        "Validation_strata": val_df["v022"].nunique(),
        "PSU_overlap": len(overlap),
        "Fit_weighted_ANC8_prev": weighted_fit_prev,
        "Validation_weighted_ANC8_prev": weighted_val_prev
    })

    assert len(overlap) == 0

    assert (
        val_df["v022"].nunique()
        == 27
    )


cv_audit = pd.DataFrame(
    cv_audit_rows
)


# =============================================================================
# 7. VERIFY COMPLETE OUT-OF-FOLD COVERAGE
#
# Each development woman must appear exactly once as validation data.
# =============================================================================

validation_counts = np.zeros(
    len(train_df),
    dtype=int
)

for _, val_idx in cv_splits:

    validation_counts[
        val_idx
    ] += 1


assert (
    validation_counts == 1
).all(), (
    "Each development woman must appear in exactly one validation fold."
)


# =============================================================================
# 8. AUDIT PSU DISTRIBUTION ACROSS FOLDS AND STRATA
# =============================================================================

psu_fold_stratum_table = (
    psu_fold_assignment
    .groupby(
        ["v022", "cv_fold"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

assert (
    psu_fold_stratum_table > 0
).all().all()


# =============================================================================
# 9. DISPLAY RESULTS
# =============================================================================

print("=" * 104)
print("STRATUM-PRESERVING PSU-GROUPED 5-FOLD CROSS-VALIDATION")
print("=" * 104)

print("\nDesign")
print("------")
print("Development women:", len(train_df))
print(
    "Development PSUs:",
    train_df["v021"].nunique()
)
print(
    "Survey strata:",
    train_df["v022"].nunique()
)
print("Number of folds:", N_FOLDS)
print("Random seed:", CV_SEED)
print("Outcome used to construct folds: No")

print("\nFold audit")
print("----------")

cv_display = cv_audit.copy()

cv_display[
    "Fit_weighted_ANC8_prev"
] = (
    cv_display[
        "Fit_weighted_ANC8_prev"
    ] * 100
)

cv_display[
    "Validation_weighted_ANC8_prev"
] = (
    cv_display[
        "Validation_weighted_ANC8_prev"
    ] * 100
)

print(
    cv_display.to_string(
        index=False,
        formatters={
            "Fit_weighted_ANC8_prev":
                "{:.2f}%".format,
            "Validation_weighted_ANC8_prev":
                "{:.2f}%".format
        }
    )
)

print("\nIntegrity")
print("---------")
print(
    "Every validation fold contains all 27 strata:",
    bool(
        (
            cv_audit[
                "Validation_strata"
            ] == 27
        ).all()
    )
)

print(
    "PSU overlap in every fold = 0:",
    bool(
        (
            cv_audit[
                "PSU_overlap"
            ] == 0
        ).all()
    )
)

print(
    "Every development woman appears once as validation:",
    bool(
        (
            validation_counts == 1
        ).all()
    )
)

print(
    "All development PSUs assigned exactly once:",
    psu_fold_assignment[
        "v021"
    ].is_unique
)

print(
    "\nSTATUS: STRATUM-PRESERVING "
    "PSU-GROUPED CV LOCKED"
)

## 2.9 DHS sampling weights in model development and evaluation

The 2024–25 EDHS uses unequal-probability sampling. DHS sampling weights
therefore provide information about each sampled woman's relative
representation in the survey target population.

Survey weighting and clustering address different aspects of the study design:

- PSU grouping is used to prevent observations from the same sampled cluster
  being divided across model-development partitions;
- survey weights address unequal representation of sampled women.

Prediction-model guidance requires these analytical choices to be explicitly
reported but does not prescribe one universal weighting strategy for
machine-learning models fitted to complex survey data.

### Primary weighting strategy

The primary analysis incorporates DHS sampling weights in two distinct ways.

**Population-weighted evaluation**

Performance measures are calculated using the relative DHS sampling weights.
This includes development out-of-fold performance and final held-out
performance.

Because weighted performance measures depend on relative rather than absolute
weight magnitude, rescaling all survey weights by the same constant does not
change these estimates.

**Model fitting**

For algorithms that support observation-level weights, model fitting uses DHS
weights normalized to have mean 1 in the development population.

Mean-1 normalization preserves the relative sampling weights while avoiding
dependence of regularization or optimization on the arbitrary numerical scale
in which DHS weights are stored.

During cross-validation, the fitting weights will additionally be normalized
within each fitting fold before being passed to the estimator. This keeps their
mean equal to 1 across model-development iterations.

### Class imbalance

ANC8+ represents approximately one fifth of the weighted development
population. The natural outcome distribution is retained.

No outcome-class rebalancing, oversampling, undersampling, SMOTE, or
algorithm-specific positive-class weighting is used in the primary analysis.
Performance measures appropriate for probability prediction and a less-common
positive outcome are used instead.

### Sensitivity analysis

A prespecified sensitivity analysis will compare:

1. survey-weighted model fitting; and
2. unweighted model fitting,

while evaluating both approaches using the same survey-weighted performance
measures and the same locked cross-validation and test partitions.

This separates the effect of weighting during model estimation from the use of
survey weights to estimate population-relevant predictive performance.

Sampling weights, PSU identifiers, and survey strata are design variables and
are never included as model predictors.

In [ ]:
# =============================================================================
# 2.9 PREPARE DEVELOPMENT OUTCOME, SURVEY DESIGN VARIABLES, AND WEIGHTS
#
# IMPORTANT DISTINCTION
# ---------------------
# survey_weight_dev:
#     Relative DHS sampling weight used for population-weighted evaluation.
#
# fit_weight_dev:
#     Same relative weights normalized to mean 1 for model fitting.
#
# Neither weight is included in the predictor matrix.
# =============================================================================


# =============================================================================
# 1. CANONICAL DEVELOPMENT OBJECTS
# =============================================================================

X_dev = train_df[
    PRIMARY_PREDICTORS
].copy()

y_dev = train_df[
    "anc8plus"
].astype(int).copy()

groups_dev = train_df[
    "v021"
].copy()

strata_dev = train_df[
    "v022"
].copy()


# =============================================================================
# 2. DHS SURVEY WEIGHT FOR EVALUATION
#
# DHS stores v005 with an implied factor of 1,000,000.
#
# Dividing by 1,000,000 restores the conventional relative weight scale.
# Relative weighting is what matters for weighted performance metrics.
# =============================================================================

survey_weight_dev = (
    train_df["v005"]
    .astype(float)
    / 1_000_000
)


# =============================================================================
# 3. MEAN-1 WEIGHT FOR MODEL FITTING
#
# Dividing by the development-sample mean preserves all relative weight ratios
# but places the average observation weight at 1.
#
# This is useful because absolute sample-weight magnitude can interact with
# estimator regularization or optimization parameters.
# =============================================================================

fit_weight_dev = (
    survey_weight_dev
    / survey_weight_dev.mean()
)


# =============================================================================
# 4. BASIC INTEGRITY CHECKS
# =============================================================================

n_dev = len(train_df)

assert len(X_dev) == n_dev
assert len(y_dev) == n_dev
assert len(survey_weight_dev) == n_dev
assert len(fit_weight_dev) == n_dev
assert len(groups_dev) == n_dev
assert len(strata_dev) == n_dev

assert y_dev.isin([0, 1]).all()

assert np.isfinite(
    survey_weight_dev
).all()

assert np.isfinite(
    fit_weight_dev
).all()

assert (
    survey_weight_dev > 0
).all()

assert (
    fit_weight_dev > 0
).all()

assert np.isclose(
    fit_weight_dev.mean(),
    1.0
)


# =============================================================================
# 5. VERIFY THAT NORMALIZATION PRESERVES RELATIVE WEIGHTS
#
# Every fit weight should equal its survey weight multiplied by the same
# constant.
# =============================================================================

weight_ratio = (
    fit_weight_dev
    / survey_weight_dev
)

assert np.allclose(
    weight_ratio,
    weight_ratio.iloc[0]
)


# =============================================================================
# 6. DEVELOPMENT OUTCOME DISTRIBUTION
# =============================================================================

unweighted_dev_prev = (
    y_dev.mean()
)

weighted_dev_prev = np.average(
    y_dev,
    weights=survey_weight_dev
)


# =============================================================================
# 7. CLASS-BALANCE ACCOUNTING
#
# This is descriptive only.
# No class rebalancing is performed.
# =============================================================================

n_positive = int(
    y_dev.sum()
)

n_negative = int(
    (y_dev == 0).sum()
)

positive_to_negative_ratio = (
    n_positive / n_negative
)


# =============================================================================
# 8. DISPLAY
# =============================================================================

print("=" * 100)
print("DEVELOPMENT WEIGHTING AND CLASS-BALANCE GOVERNANCE")
print("=" * 100)

print("\nDevelopment objects")
print("-------------------")
print("Predictor rows:", len(X_dev))
print("Outcome rows:", len(y_dev))
print("PSU rows:", len(groups_dev))
print("Stratum rows:", len(strata_dev))

print("\nOutcome distribution")
print("--------------------")
print("ANC8+ women:", n_positive)
print("ANC8<8 women:", n_negative)
print(
    "Unweighted ANC8+ prevalence:",
    f"{unweighted_dev_prev * 100:.2f}%"
)
print(
    "Survey-weighted ANC8+ prevalence:",
    f"{weighted_dev_prev * 100:.2f}%"
)

print("\nSurvey weight used for evaluation")
print("---------------------------------")
print(
    survey_weight_dev.describe()
)

print("\nMean-1 weight used for model fitting")
print("------------------------------------")
print(
    fit_weight_dev.describe()
)

print(
    "\nMean fitting weight:",
    f"{fit_weight_dev.mean():.6f}"
)

print("\nClass-imbalance handling")
print("------------------------")
print("Oversampling: No")
print("Undersampling: No")
print("SMOTE: No")
print("Class weighting / scale_pos_weight: No")
print(
    "Primary strategy:",
    "natural outcome distribution + appropriate performance metrics"
)

print("\nDesign-variable governance")
print("--------------------------")
print("Survey weight included in X: No")
print("PSU included in X: No")
print("Survey stratum included in X: No")

print(
    "\nSTATUS: DEVELOPMENT WEIGHTING "
    "AND CLASS-BALANCE STRATEGY LOCKED"
)

## 2.10 Prespecified model-performance framework

Model performance measures are specified before fitting or comparing models.
This avoids selecting evaluation measures because they happen to favour a
particular model.

The primary ANC8 models generate predicted probabilities rather than only
binary classifications. Evaluation therefore emphasizes probability-based
performance before threshold-dependent classification measures.

### 1. Discrimination: AUROC

The **Area Under the Receiver Operating Characteristic Curve (AUROC)** measures
how well the model ranks women who attained ANC8 above women who did not.

Conceptually, AUROC can be interpreted as the probability that a randomly
selected ANC8+ woman receives a higher predicted probability than a randomly
selected ANC8<8 woman.

- 0.50 indicates no useful discrimination.
- 1.00 indicates perfect discrimination.
- Higher values are better.

Conventional values such as 0.70 or 0.80 may help describe performance, but
they are not universal scientific cut-offs.

### 2. Positive-class ranking: Average Precision

**Average Precision (AP)** summarizes performance across the precision-recall
relationship.

It is particularly informative when the positive outcome is relatively
uncommon, as ANC8+ represents about 19% of the survey-weighted analytic
population.

Higher AP is better, but unlike AUROC there is no universal value such as
0.70 that defines a good model. Its natural no-skill reference is related to
the prevalence of the positive outcome.

For this study, the weighted no-skill reference is approximately 0.19.
For the development population, the survey-weighted ANC8+ prevalence is
19.89%. Therefore, approximately 0.199 provides the natural prevalence-based
reference for Average Precision during development evaluation. The corresponding
reference in the held-out test set will be calculated from that test population
when final evaluation is performed.

### 3. Probabilistic accuracy: Brier score

The **Brier score** measures the average squared difference between predicted
probabilities and observed outcomes.

A prediction close to the observed outcome receives a small error, while an
overconfident incorrect prediction receives a larger error.

- 0 represents perfect probabilistic prediction.
- Lower values are better.

The Brier score will be interpreted relative to outcome prevalence and simple
reference predictions rather than using an arbitrary universal cut-off.

### 4. Calibration

Calibration evaluates agreement between predicted ANC8 probabilities and
observed ANC8 frequencies.

Calibration will be assessed graphically and numerically rather than reduced to
a single pass/fail statistic.

The primary calibration assessment will include:

- a calibration plot comparing predicted and observed probabilities;
- calibration-in-the-large (CITL), for which the ideal value is 0;
- calibration slope, for which the ideal value is 1.

CITL evaluates whether predictions are systematically too high or too low
overall. A positive CITL indicates that observed outcome frequency is higher
than predicted on average, whereas a negative CITL indicates overall
overprediction.

The calibration slope evaluates the spread of predicted risks. A slope below 1
is commonly consistent with predictions that are too extreme, whereas a slope
above 1 indicates predictions that are not sufficiently dispersed.

For diagnostic purposes, a joint logistic recalibration model may also be
reported, but its intercept will be clearly distinguished from CITL because
these are not the same quantity.

Calibration calculations and plots used for primary evaluation will incorporate
the DHS survey weights.

### 5. Threshold-based performance

A probability threshold is required only when predicted probabilities are
converted into binary predictions of ANC8+ versus ANC8<8.

After a threshold has been selected using development data only, secondary
classification measures will include:

- **Sensitivity (recall):** proportion of true ANC8+ women correctly identified.
- **Specificity:** proportion of true ANC8<8 women correctly identified.
- **Positive predictive value (precision):** proportion predicted ANC8+ who
  actually attained ANC8.
- **Negative predictive value:** proportion predicted ANC8<8 who actually did
  not attain ANC8.
- **F1 score:** harmonic mean of precision and recall.

F1 is a machine-learning classification measure and is unrelated to the
F-statistic used in analysis of variance.

Threshold-dependent metrics are secondary because their values depend on the
chosen decision threshold.

### Threshold governance

The final test set will not be used to select a classification threshold.

Any operational threshold will be derived from training/cross-validation
predictions, documented, locked, and subsequently applied unchanged to the
held-out test set.

### Weighting

Primary development and held-out evaluation metrics will use DHS sampling
weights.

A prespecified sensitivity analysis will compare survey-weighted versus
unweighted model fitting while retaining survey-weighted evaluation for both
approaches. The same locked cross-validation and held-out test partitions will
be used.

In [ ]:
PRIMARY_METRICS = {
    "AUROC":
        "Higher is better; discrimination/ranking",

    "Average Precision":
        "Higher is better; interpret relative to outcome prevalence",

    "Brier score":
        "Lower is better; overall probabilistic accuracy",

    "Calibration-in-the-large":
        "Ideal = 0; overall underprediction or overprediction",

    "Calibration slope":
        "Ideal = 1; spread/extremeness of predicted probabilities",

    "Calibration plot":
        "Predicted probabilities should agree with observed outcome frequency"
}
THRESHOLD_METRICS = {
    "Sensitivity": "Higher = more true ANC8+ cases identified",
    "Specificity": "Higher = more true ANC8<8 cases identified",
    "PPV / Precision": "Higher = more positive predictions are truly ANC8+",
    "NPV": "Higher = more negative predictions are truly ANC8<8",
    "F1": "Balances precision and sensitivity"
}

print(
    "\nPrimary evaluation weighted by DHS sampling weights: Yes"
)
print(
    "Weighted vs unweighted model-fitting sensitivity prespecified: Yes"
)
print(
    "Threshold selected using final test set: No"
)
print(
    "True CITL distinguished from joint recalibration intercept: Yes"
)

## 2.11 Candidate model-development architecture

Three supervised binary-classification algorithms are prespecified for
prediction of ANC8 attainment:

1. L2-regularized logistic regression;
2. Random Forest;
3. XGBoost.

The algorithms represent different levels of functional flexibility while using
the same 21 prespecified predictor constructs and the same governed development
population.

### Logistic regression

L2-regularized logistic regression serves as the statistical benchmark model.

It models the log-odds of ANC8 attainment as an additive function of the
transformed predictors. Numeric/count predictors therefore enter linearly on
the log-odds scale unless represented otherwise, whereas categorical predictors
enter through their encoded indicator variables.

Regularization is used to control coefficient magnitude and improve predictive
stability in the expanded feature space. The regularization strength will be
selected using the locked development cross-validation folds.

The logistic model is included primarily as a transparent predictive benchmark.
Its fitted coefficients will not automatically be interpreted as conventional
epidemiologic effect estimates because the study objective is prediction, the
model is regularized, and categorical predictors are represented using the
governed predictive encoding defined earlier.

### Random Forest

Random Forest combines predictions from multiple decision trees fitted to
resampled data and predictor subsets.

It can represent nonlinear predictor-outcome relationships and interactions
without requiring those forms to be specified in advance.

Tree complexity and ensemble size will be controlled through a limited,
prespecified hyperparameter search performed within the locked development
cross-validation folds.

### XGBoost

XGBoost is a gradient-boosted tree algorithm in which decision trees are built
sequentially, with each new tree attempting to improve the current predictive
model.

Like Random Forest, it can represent nonlinearities and interactions without
requiring their functional forms to be specified in advance, but it uses a
different sequential boosting strategy.

A limited, prespecified hyperparameter search will control model complexity,
learning rate, and regularization.

### Common model-development principles

For all three candidate algorithms:

- the same 21 frozen substantive predictors are used;
- the held-out test set is not used for model development or hyperparameter
  selection;
- preprocessing remains inside each model pipeline;
- preprocessing parameters are learned separately within each cross-validation
  fitting fold;
- the same locked five-fold, stratum-preserving PSU-grouped cross-validation
  assignments are used for all candidate models;
- PSU identifiers, survey strata, and DHS sampling weights are design variables,
  not predictors;
- primary model fitting uses mean-1 normalized DHS sampling weights;
- fitting weights are renormalized to mean 1 within each cross-validation
  fitting fold;
- development performance is evaluated using DHS survey weights;
- no oversampling, undersampling, SMOTE, class weighting, or `scale_pos_weight`
  is used in the primary analysis;
- model comparison is based on the prespecified performance framework defined
  in Section 2.10;
- model explainability is treated as a downstream scientific analysis requiring
  preserved feature lineage rather than as a criterion for altering predictor
  definitions during model fitting.

All hyperparameter selection occurs within the development dataset. The final
held-out test set remains untouched until the model-development procedure is
locked.

In [ ]:
# =============================================================================
# 2.11 PRESPECIFY MODEL SET
# =============================================================================

MODEL_PLAN = {
    "Logistic regression": "Interpretable regularized baseline",
    "Random Forest": "Nonlinear ensemble tree model",
    "XGBoost": "Gradient-boosted tree model"
}

print("=" * 88)
print("PRESPECIFIED MODEL SET")
print("=" * 88)

for model, role in MODEL_PLAN.items():
    print(f"{model}: {role}")

print("\nNumber of candidate algorithms:", len(MODEL_PLAN))
print("Final test set used for tuning: No")

## 2.12 Prespecified hyperparameter tuning strategy

Hyperparameters are model settings that control model complexity, regularization,
and learning behaviour but are not learned directly as ordinary model
coefficients or tree splits.

Hyperparameter tuning is conducted exclusively within the development dataset.

The same locked five-fold, stratum-preserving PSU-grouped cross-validation
assignments defined in Section 2.8 are used for every candidate configuration
of every algorithm.

The final held-out test set is not used to select hyperparameters.

### General tuning principles

For each candidate hyperparameter configuration:

1. the model pipeline is fitted separately in each cross-validation fitting
   fold;
2. preprocessing is learned only from that fitting fold;
3. all women from the same PSU remain together;
4. each fitting fold uses DHS sampling weights renormalized to mean 1 within
   that fold;
5. predicted probabilities are generated only for the corresponding validation
   fold;
6. after all five folds, every development woman has exactly one out-of-fold
   predicted probability;
7. development performance is calculated from these out-of-fold probabilities
   using the DHS survey weights.

No oversampling, undersampling, SMOTE, class weighting, or `scale_pos_weight`
is used during primary tuning.

### Logistic regression

The tuned parameter is the inverse L2-regularization strength (`C`).

Smaller `C` values impose stronger regularization, while larger values allow
larger fitted coefficients.

The prespecified values are:

- 0.01
- 0.1
- 1
- 10

### Random Forest

The prespecified search varies:

- number of trees (`n_estimators`);
- maximum tree depth (`max_depth`);
- minimum observations required to split a node (`min_samples_split`);
- minimum observations in a terminal leaf (`min_samples_leaf`);
- number of transformed features considered at each split (`max_features`).

The search is intentionally modest because the objective is not to exhaustively
optimize a very large hyperparameter space, but to compare a limited set of
scientifically reasonable complexity settings reproducibly.

### XGBoost

The prespecified search varies:

- number of boosting trees (`n_estimators`);
- learning rate (`learning_rate`);
- maximum tree depth (`max_depth`);
- minimum child weight (`min_child_weight`);
- row subsampling (`subsample`);
- transformed-feature subsampling (`colsample_bytree`).

No positive-class weighting is used.

### Hyperparameter-selection rule

Survey-weighted Average Precision calculated from the complete set of
development out-of-fold predictions is the primary hyperparameter-selection
criterion.

For each algorithm, the configuration with the highest survey-weighted
out-of-fold Average Precision is selected.

Survey-weighted AUROC and Brier score are retained as complementary development
performance measures but are not used to change the primary selection rule
after results are observed.

If two candidate configurations have numerically identical Average Precision
at full computational precision, the configuration with the lower
survey-weighted Brier score is preferred. If still tied, the configuration with
the higher survey-weighted AUROC is preferred.

The selected hyperparameters are then locked before any held-out test
predictions are generated.

Hyperparameter selection identifies the preferred configuration *within* each
algorithm. It does not by itself determine which algorithm is the final best
model; the three tuned algorithms will subsequently be evaluated on the
untouched held-out test set using the full prespecified performance framework.

In [ ]:
# =============================================================================
# 2.12 PRESPECIFIED HYPERPARAMETER SEARCH SPACES AND SELECTION GOVERNANCE
#
# No model is fitted in this cell.
# No held-out test data are used.
#
# These grids will later be evaluated using the locked `cv_splits` object from
# Section 2.8.
# =============================================================================

from sklearn.model_selection import ParameterGrid


# =============================================================================
# 1. LOGISTIC REGRESSION
#
# C is the inverse regularization strength:
#   smaller C -> stronger L2 regularization
#   larger C  -> weaker L2 regularization
# =============================================================================

PARAM_GRID_LOGISTIC = {
    "model__C": [
        0.01,
        0.1,
        1.0,
        10.0
    ]
}


# =============================================================================
# 2. RANDOM FOREST
# =============================================================================

PARAM_GRID_RF = {
    "model__n_estimators": [
        300,
        600
    ],

    "model__max_depth": [
        None,
        6,
        12
    ],

    "model__min_samples_split": [
        2,
        10
    ],

    "model__min_samples_leaf": [
        1,
        5
    ],

    "model__max_features": [
        "sqrt",
        0.5
    ]
}


# =============================================================================
# 3. XGBOOST
#
# No scale_pos_weight is included because class weighting is not part of the
# primary analysis.
# =============================================================================

PARAM_GRID_XGB = {
    "model__n_estimators": [
        200,
        400
    ],

    "model__learning_rate": [
        0.03,
        0.1
    ],

    "model__max_depth": [
        3,
        6
    ],

    "model__min_child_weight": [
        1,
        5
    ],

    "model__subsample": [
        0.8,
        1.0
    ],

    "model__colsample_bytree": [
        0.8,
        1.0
    ]
}


# =============================================================================
# 4. COUNT SEARCH COMBINATIONS
# =============================================================================

N_LOGISTIC_CONFIGS = len(
    list(
        ParameterGrid(
            PARAM_GRID_LOGISTIC
        )
    )
)

N_RF_CONFIGS = len(
    list(
        ParameterGrid(
            PARAM_GRID_RF
        )
    )
)

N_XGB_CONFIGS = len(
    list(
        ParameterGrid(
            PARAM_GRID_XGB
        )
    )
)


# =============================================================================
# 5. GOVERNED HYPERPARAMETER-SELECTION RULE
#
# Primary:
#   maximize survey-weighted out-of-fold Average Precision
#
# Tie-break 1:
#   lower survey-weighted Brier score
#
# Tie-break 2:
#   higher survey-weighted AUROC
#
# IMPORTANT:
# These criteria apply only to hyperparameter selection WITHIN each algorithm.
# Final comparison among algorithms occurs later on the held-out test set.
# =============================================================================

HYPERPARAMETER_SELECTION_RULE = {
    "primary_metric":
        "weighted_AP",

    "primary_direction":
        "maximize",

    "tie_break_1":
        "weighted_Brier",

    "tie_break_1_direction":
        "minimize",

    "tie_break_2":
        "weighted_AUROC",

    "tie_break_2_direction":
        "maximize"
}


# =============================================================================
# 6. GOVERNANCE CHECKS
# =============================================================================

assert N_LOGISTIC_CONFIGS == 4
assert N_RF_CONFIGS == 48
assert N_XGB_CONFIGS == 64

assert len(cv_splits) == 5

assert np.isclose(
    fit_weight_dev.mean(),
    1.0
)

assert len(survey_weight_dev) == len(X_dev)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 100)
print("HYPERPARAMETER SEARCH AND SELECTION STRATEGY REGISTERED")
print("=" * 100)

print("\nSearch-space size")
print("-----------------")
print(
    "Logistic regression configurations:",
    N_LOGISTIC_CONFIGS
)
print(
    "Random Forest configurations:",
    N_RF_CONFIGS
)
print(
    "XGBoost configurations:",
    N_XGB_CONFIGS
)

print("\nCross-validation governance")
print("---------------------------")
print(
    "Locked CV folds:",
    len(cv_splits)
)
print(
    "Stratum-preserving PSU-grouped folds:",
    "Yes"
)
print(
    "Same folds used for every candidate:",
    "Yes"
)

print("\nWeight governance")
print("-----------------")
print(
    "Model-fitting weights:",
    "mean-1 DHS weights within each fitting fold"
)
print(
    "Development evaluation weights:",
    "DHS survey weights"
)
print(
    "Class weighting:",
    "No"
)

print("\nHyperparameter-selection rule")
print("-----------------------------")
print(
    "Primary:",
    "maximize survey-weighted OOF Average Precision"
)
print(
    "Tie-break 1:",
    "lower survey-weighted OOF Brier score"
)
print(
    "Tie-break 2:",
    "higher survey-weighted OOF AUROC"
)

print("\nHeld-out test set used here: No")
print("Models fitted here: No")

print(
    "\nSTATUS: HYPERPARAMETER SEARCH "
    "AND SELECTION STRATEGY LOCKED"
)

## 2.13 Governed candidate-model pipelines

Each candidate algorithm is combined with its prespecified preprocessing
procedure in a single scikit-learn pipeline.

Embedding preprocessing inside the model pipeline is essential because every
cross-validation fitting fold must learn its own imputation, scaling, and
categorical-encoding parameters. No preprocessing fitted to the complete
development dataset in Section 2.7 is reused for model fitting.

### Logistic regression pipeline

The logistic-regression pipeline uses:

- median imputation of numeric/count predictors;
- standardization of numeric/count predictors;
- strict prespecified 0/1 encoding of binary predictors;
- most-frequent imputation of genuine missing categorical values;
- complete one-hot encoding of multi-category predictors;
- L2-regularized logistic regression.

The inverse regularization parameter (`C`) is not fixed here because it will be
selected from the prespecified grid in Section 2.12.

### Random Forest pipeline

The Random Forest pipeline uses:

- median imputation of numeric/count predictors without standardization;
- the same strict binary encoding;
- the same categorical one-hot representation;
- a Random Forest classifier.

Hyperparameters governing tree number and complexity are left to the
prespecified Section 2.12 search grid.

### XGBoost pipeline

XGBoost uses the same tree-oriented preprocessing representation as Random
Forest.

Its fixed objective is binary probabilistic classification. Parameters
controlling boosting complexity and learning behaviour are supplied later from
the prespecified Section 2.12 search grid.

### Fixed algorithm settings

A common random seed of 2026 is used for stochastic algorithms.

No algorithm uses:

- `class_weight`;
- `scale_pos_weight`;
- oversampling;
- undersampling; or
- synthetic observations.

DHS fitting weights are not embedded inside the pipeline specification because
they vary by cross-validation fitting fold. They will be supplied explicitly
through `model__sample_weight` when each pipeline is fitted.

The held-out test set is not accessed in this section, and no model is fitted.

In [ ]:
# =============================================================================
# 2.13 DEFINE GOVERNED CANDIDATE-MODEL PIPELINES
#
# IMPORTANT
# ---------
# - Pipelines are DEFINED only.
# - No model is fitted here.
# - No held-out test data are accessed.
# - Hyperparameters listed in Section 2.12 remain to be tuned.
# - Survey fitting weights will be supplied later at .fit() time.
# =============================================================================

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =============================================================================
# 1. COMMON MODEL RANDOM SEED
# =============================================================================

MODEL_SEED = 2026


# =============================================================================
# 2. LOGISTIC REGRESSION PIPELINE
#
# C is intentionally left at its estimator default here.
# During tuning, every candidate value from PARAM_GRID_LOGISTIC will be set
# explicitly before fitting.
# =============================================================================

pipeline_logistic = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor_logistic)
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=5000,
                class_weight=None,
                random_state=MODEL_SEED
            )
        )
    ]
)


# =============================================================================
# 3. RANDOM FOREST PIPELINE
#
# Complexity-related settings such as n_estimators, max_depth,
# min_samples_split, min_samples_leaf and max_features will be supplied from
# PARAM_GRID_RF during tuning.
#
# Fixed defaults used as part of the algorithm definition are written
# explicitly where useful for reproducibility.
# =============================================================================

pipeline_rf = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor_tree)
        ),
        (
            "model",
            RandomForestClassifier(
                criterion="gini",
                bootstrap=True,
                class_weight=None,
                random_state=MODEL_SEED,
                n_jobs=-1
            )
        )
    ]
)


# =============================================================================
# 4. XGBOOST PIPELINE
#
# tree_method="hist" provides an efficient CPU implementation for this
# structured tabular dataset.
#
# No scale_pos_weight is used because positive-class weighting is not part of
# the primary analysis.
# =============================================================================

pipeline_xgb = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor_tree)
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=MODEL_SEED,
                n_jobs=-1,
                verbosity=0
            )
        )
    ]
)


# =============================================================================
# 5. PIPELINE-STRUCTURE AUDIT
# =============================================================================

candidate_pipelines = {
    "Logistic regression": pipeline_logistic,
    "Random Forest": pipeline_rf,
    "XGBoost": pipeline_xgb
}


for model_name, pipeline in candidate_pipelines.items():

    # Every candidate must contain exactly the governed two-stage structure:
    # preprocessing followed by the estimator.
    assert list(
        pipeline.named_steps.keys()
    ) == [
        "preprocessor",
        "model"
    ], (
        f"Unexpected pipeline structure for {model_name}."
    )


# Logistic must use the logistic preprocessor architecture.
assert isinstance(
    pipeline_logistic.named_steps["model"],
    LogisticRegression
)


# Tree algorithms must use their intended classifier classes.
assert isinstance(
    pipeline_rf.named_steps["model"],
    RandomForestClassifier
)

assert isinstance(
    pipeline_xgb.named_steps["model"],
    XGBClassifier
)


# No class weighting in primary models.
assert (
    pipeline_logistic
    .named_steps["model"]
    .get_params()["class_weight"]
    is None
)

assert (
    pipeline_rf
    .named_steps["model"]
    .get_params()["class_weight"]
    is None
)

assert (
    "scale_pos_weight"
    not in PARAM_GRID_XGB
)


# =============================================================================
# 6. VERIFY THAT REGISTERED GRID PARAMETERS EXIST IN THEIR PIPELINES
#
# This catches spelling errors such as a grid parameter that cannot actually
# be set on the corresponding pipeline.
# =============================================================================

for parameter_name in PARAM_GRID_LOGISTIC:

    assert (
        parameter_name
        in pipeline_logistic.get_params()
    ), (
        f"Unknown logistic grid parameter: {parameter_name}"
    )


for parameter_name in PARAM_GRID_RF:

    assert (
        parameter_name
        in pipeline_rf.get_params()
    ), (
        f"Unknown Random Forest grid parameter: {parameter_name}"
    )


for parameter_name in PARAM_GRID_XGB:

    assert (
        parameter_name
        in pipeline_xgb.get_params()
    ), (
        f"Unknown XGBoost grid parameter: {parameter_name}"
    )


# =============================================================================
# 7. DISPLAY GOVERNANCE SUMMARY
# =============================================================================

print("=" * 100)
print("GOVERNED CANDIDATE-MODEL PIPELINES DEFINED")
print("=" * 100)

print("\nCandidate algorithms")
print("--------------------")
print("1. L2-regularized logistic regression")
print("2. Random Forest")
print("3. XGBoost")

print("\nPreprocessing architecture")
print("--------------------------")
print(
    "Logistic regression:",
    "logistic preprocessing + L2 logistic model"
)
print(
    "Random Forest:",
    "tree preprocessing + Random Forest"
)
print(
    "XGBoost:",
    "tree preprocessing + gradient-boosted trees"
)

print("\nReproducibility")
print("---------------")
print("Model random seed:", MODEL_SEED)
print(
    "Registered tuning parameters all recognized:",
    "Yes"
)

print("\nClass-imbalance handling")
print("------------------------")
print("Logistic class_weight: None")
print("Random Forest class_weight: None")
print("XGBoost scale_pos_weight tuning: No")

print("\nWeight handling")
print("---------------")
print(
    "Sampling weights embedded in pipeline:",
    "No"
)
print(
    "Weights supplied during fitting:",
    "Yes, through model__sample_weight"
)

print("\nGovernance")
print("----------")
print("Models fitted here: No")
print("Held-out test set accessed here: No")

print(
    "\nSTATUS: CANDIDATE-MODEL "
    "PIPELINES DEFINED AND VERIFIED"
)

## 2.14 Cross-validation implementation and pipeline smoke test

Before hyperparameter tuning, the locked development workflow is audited to
confirm that the objects created in Sections 2.8–2.13 operate together as
intended.

For each of the five locked cross-validation folds, this audit verifies that:

- fitting and validation observations are defined by the canonical `cv_splits`
  object;
- women from the same PSU never appear in both portions of a fold;
- all 27 survey strata remain represented in each validation fold;
- fitting, outcome, and weight vectors remain row-aligned;
- DHS model-fitting weights are renormalized to mean 1 within each fitting fold;
- validation observations retain their ordinary DHS survey weights for later
  evaluation.

A single temporary logistic-regression pipeline is then fitted on the first
cross-validation fitting fold as a smoke test. This confirms that:

- preprocessing can be fitted using only the fitting fold;
- the governed sample weights are accepted by the estimator;
- the pipeline produces finite probabilities for the corresponding validation
  fold;
- the fitted preprocessing structure still produces the expected transformed
  feature space.

The temporary smoke-test model is not used for hyperparameter selection,
performance reporting, threshold selection, final model fitting, or
explainability.

The held-out final test set is not accessed.

In [ ]:
# =============================================================================
# 2.14 CROSS-VALIDATION IMPLEMENTATION AND PIPELINE SMOKE TEST
#
# PURPOSE
# -------
# Verify that the locked objects from Sections 2.8-2.13 work together before
# launching the full hyperparameter searches.
#
# IMPORTANT
# ---------
# - Uses development data only.
# - Uses the canonical `cv_splits` object.
# - Does NOT recreate folds.
# - Does NOT tune hyperparameters.
# - Does NOT calculate model-performance metrics.
# - Does NOT access the held-out final test set.
# =============================================================================


# =============================================================================
# 1. AUDIT ALL FIVE LOCKED CROSS-VALIDATION FOLDS
# =============================================================================

cv_implementation_rows = []

for fold_number, (fit_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):

    # -------------------------------------------------------------------------
    # Fold-specific development objects
    # -------------------------------------------------------------------------

    X_fit = X_dev.iloc[fit_idx]
    y_fit = y_dev.iloc[fit_idx]

    X_val = X_dev.iloc[val_idx]
    y_val = y_dev.iloc[val_idx]

    fit_psus = set(
        groups_dev.iloc[fit_idx]
    )

    val_psus = set(
        groups_dev.iloc[val_idx]
    )

    fit_strata = set(
        strata_dev.iloc[fit_idx]
    )

    val_strata = set(
        strata_dev.iloc[val_idx]
    )


    # -------------------------------------------------------------------------
    # Fitting weights
    #
    # `fit_weight_dev` already preserves the DHS relative weights and has mean
    # 1 in the complete development population.
    #
    # We renormalize again WITHIN each fitting fold so that every estimator fit
    # receives weights whose mean is exactly 1.
    # -------------------------------------------------------------------------

    fold_fit_weight = (
        fit_weight_dev
        .iloc[fit_idx]
        .astype(float)
        .copy()
    )

    fold_fit_weight = (
        fold_fit_weight
        / fold_fit_weight.mean()
    )


    # Validation weights are NOT mean-normalized.
    # They retain the ordinary DHS relative scale for weighted evaluation.
    fold_val_survey_weight = (
        survey_weight_dev
        .iloc[val_idx]
        .astype(float)
        .copy()
    )


    # -------------------------------------------------------------------------
    # Integrity checks
    # -------------------------------------------------------------------------

    assert len(X_fit) == len(y_fit) == len(fold_fit_weight)

    assert (
        len(X_val)
        == len(y_val)
        == len(fold_val_survey_weight)
    )

    assert len(
        fit_psus.intersection(
            val_psus
        )
    ) == 0

    assert len(val_strata) == 27

    assert np.isclose(
        fold_fit_weight.mean(),
        1.0
    )

    assert (
        fold_fit_weight > 0
    ).all()

    assert (
        fold_val_survey_weight > 0
    ).all()


    # -------------------------------------------------------------------------
    # Store audit information
    # -------------------------------------------------------------------------

    cv_implementation_rows.append({
        "Fold": fold_number,

        "Fit_women":
            len(fit_idx),

        "Validation_women":
            len(val_idx),

        "Fit_PSUs":
            len(fit_psus),

        "Validation_PSUs":
            len(val_psus),

        "Validation_strata":
            len(val_strata),

        "PSU_overlap":
            len(
                fit_psus.intersection(
                    val_psus
                )
            ),

        "Fit_weight_mean":
            fold_fit_weight.mean(),

        "Validation_weight_mean":
            fold_val_survey_weight.mean()
    })


cv_implementation_audit = pd.DataFrame(
    cv_implementation_rows
)


# =============================================================================
# 2. COMPLETE-COVERAGE CHECK
#
# Every development row should occur exactly once in validation across the five
# locked folds.
# =============================================================================

validation_coverage = np.zeros(
    len(X_dev),
    dtype=int
)

for _, val_idx in cv_splits:

    validation_coverage[
        val_idx
    ] += 1


assert (
    validation_coverage == 1
).all()


# =============================================================================
# 3. PIPELINE SMOKE TEST USING FOLD 1 ONLY
#
# This is NOT tuning.
#
# We fit one temporary logistic-regression clone solely to verify:
#   - pipeline preprocessing;
#   - fold-specific fitting weights;
#   - prediction mechanics;
#   - transformed feature count.
#
# C = 1.0 is used only as a neutral smoke-test setting.
# It is NOT selected or retained from this audit.
# =============================================================================

smoke_fit_idx, smoke_val_idx = (
    cv_splits[0]
)

smoke_fit_weight = (
    fit_weight_dev
    .iloc[smoke_fit_idx]
    .astype(float)
    .copy()
)

smoke_fit_weight = (
    smoke_fit_weight
    / smoke_fit_weight.mean()
)


smoke_model = clone(
    pipeline_logistic
)

smoke_model.set_params(
    model__C=1.0
)


smoke_model.fit(
    X_dev.iloc[smoke_fit_idx],
    y_dev.iloc[smoke_fit_idx],
    model__sample_weight=smoke_fit_weight
)


smoke_prob = smoke_model.predict_proba(
    X_dev.iloc[smoke_val_idx]
)[:, 1]


# =============================================================================
# 4. SMOKE-TEST INTEGRITY CHECKS
# =============================================================================

assert len(
    smoke_prob
) == len(
    smoke_val_idx
)

assert np.isfinite(
    smoke_prob
).all()

assert (
    (smoke_prob >= 0)
    & (smoke_prob <= 1)
).all()


# Retrieve fitted feature names directly from the preprocessing component.
smoke_feature_names = (
    smoke_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


assert len(
    smoke_feature_names
) == len(
    feature_lineage
)

assert np.array_equal(
    smoke_feature_names,
    feature_lineage[
        "Transformed_feature"
    ].to_numpy()
)


# =============================================================================
# 5. DISCARD THE TEMPORARY MODEL
#
# Explicitly remove the temporary fitted objects so there is no possibility of
# accidentally reusing them downstream.
# =============================================================================

del smoke_model
del smoke_prob
del smoke_feature_names
del smoke_fit_weight


# =============================================================================
# 6. DISPLAY AUDIT
# =============================================================================

print("=" * 104)
print("LOCKED CROSS-VALIDATION IMPLEMENTATION AUDIT")
print("=" * 104)

display_audit = (
    cv_implementation_audit
    .copy()
)

print(
    display_audit.to_string(
        index=False,
        formatters={
            "Fit_weight_mean":
                "{:.6f}".format,

            "Validation_weight_mean":
                "{:.6f}".format
        }
    )
)


print("\nIntegrity")
print("---------")

print(
    "All validation folds contain 27 strata:",
    bool(
        (
            cv_implementation_audit[
                "Validation_strata"
            ] == 27
        ).all()
    )
)

print(
    "PSU overlap = 0 in every fold:",
    bool(
        (
            cv_implementation_audit[
                "PSU_overlap"
            ] == 0
        ).all()
    )
)

print(
    "Mean fitting weight = 1 in every fold:",
    bool(
        np.allclose(
            cv_implementation_audit[
                "Fit_weight_mean"
            ],
            1.0
        )
    )
)

print(
    "Every development woman validated exactly once:",
    bool(
        (
            validation_coverage == 1
        ).all()
    )
)


print("\nPipeline smoke test")
print("-------------------")
print(
    "Temporary Fold-1 logistic pipeline fitted successfully:",
    "Yes"
)
print(
    "Fold-specific model__sample_weight accepted:",
    "Yes"
)
print(
    "Expected transformed feature structure recovered:",
    "Yes"
)
print(
    "Temporary fitted audit model retained:",
    "No"
)


print("\nGovernance")
print("----------")
print("Hyperparameter tuning performed: No")
print("Performance metrics calculated: No")
print("Held-out final test set accessed: No")

print(
    "\nSTATUS: CROSS-VALIDATION AND "
    "PIPELINE IMPLEMENTATION VERIFIED"
)

## 2.15 Logistic regression tuning and out-of-fold prediction

L2-regularized logistic regression is tuned first as the prespecified
statistical benchmark model.

For each candidate value of the inverse regularization parameter (`C`):

1. the same locked five cross-validation folds are used;
2. preprocessing is fitted only within the fitting portion of each fold;
3. DHS fitting weights are renormalized to mean 1 within that fitting fold;
4. the model is fitted using those fold-specific weights;
5. predicted probabilities are generated for the corresponding validation fold;
6. after all five folds, each development woman has exactly one out-of-fold
   predicted probability.

Survey-weighted Average Precision is the primary hyperparameter-selection
criterion.

Survey-weighted AUROC and Brier score are calculated as complementary
development measures.

The configuration with the highest weighted Average Precision is selected.
The prespecified tie-breaking rule from Section 2.12 is applied only if required.

The final held-out test set is not accessed.

### Hyperparameter-selection criterion

Because ANC8+ is the less common outcome class, **survey-weighted Average
Precision (AP)** is prespecified as the primary criterion for selecting
hyperparameters during cross-validation.

Higher weighted AP indicates better concentration of true ANC8+ observations
among women assigned higher predicted probabilities.

Weighted AUROC and weighted Brier score are also calculated for each candidate
configuration as complementary measures, but they are not used opportunistically
to choose the hyperparameter setting.

If two settings have essentially similar weighted AP, preference will be given
to the simpler/more strongly regularized model, while considering the
complementary metrics.

In [ ]:
# =============================================================================
# 2.15 LOGISTIC REGRESSION:
#      SURVEY-WEIGHTED, STRATUM-PRESERVING PSU-GROUPED CV TUNING
#
# PRIMARY SELECTION METRIC:
#   Survey-weighted out-of-fold Average Precision
#
# IMPORTANT
# ---------
# - Uses the locked `cv_splits` object.
# - Uses mean-1 fitting weights within each fitting fold.
# - Uses DHS survey weights for OOF performance evaluation.
# - Does not access the final held-out test set.
# =============================================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)


# =============================================================================
# 1. STORAGE OBJECTS
# =============================================================================

logistic_cv_results = []

logistic_oof_predictions = {}


# =============================================================================
# 2. EVALUATE EACH PRESPECIFIED C VALUE
# =============================================================================

for params in ParameterGrid(
    PARAM_GRID_LOGISTIC
):

    # Complete OOF vector for this candidate configuration.
    oof_prob = np.full(
        len(X_dev),
        np.nan,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Locked cross-validation folds
    # -------------------------------------------------------------------------

    for fit_idx, val_idx in cv_splits:

        # ---------------------------------------------------------------------
        # Fold-specific mean-1 fitting weights
        # ---------------------------------------------------------------------

        fold_fit_weight = (
            fit_weight_dev
            .iloc[fit_idx]
            .astype(float)
            .copy()
        )

        fold_fit_weight = (
            fold_fit_weight
            / fold_fit_weight.mean()
        )

        assert np.isclose(
            fold_fit_weight.mean(),
            1.0
        )


        # ---------------------------------------------------------------------
        # Fresh clone so every fold learns preprocessing from its own fit data.
        # ---------------------------------------------------------------------

        model = clone(
            pipeline_logistic
        )

        model.set_params(
            **params
        )


        # ---------------------------------------------------------------------
        # Fit
        # ---------------------------------------------------------------------

        model.fit(
            X_dev.iloc[fit_idx],
            y_dev.iloc[fit_idx],
            model__sample_weight=fold_fit_weight
        )


        # ---------------------------------------------------------------------
        # Predict only the validation fold
        # ---------------------------------------------------------------------

        oof_prob[
            val_idx
        ] = model.predict_proba(
            X_dev.iloc[val_idx]
        )[:, 1]


    # -------------------------------------------------------------------------
    # Every development woman must now have one finite OOF probability.
    # -------------------------------------------------------------------------

    assert np.isfinite(
        oof_prob
    ).all()

    assert (
        (oof_prob >= 0)
        & (oof_prob <= 1)
    ).all()


    # -------------------------------------------------------------------------
    # Survey-weighted OOF performance
    #
    # IMPORTANT:
    # Evaluation uses survey_weight_dev, not fitting weights.
    # -------------------------------------------------------------------------

    weighted_ap = average_precision_score(
        y_dev,
        oof_prob,
        sample_weight=survey_weight_dev
    )

    weighted_auc = roc_auc_score(
        y_dev,
        oof_prob,
        sample_weight=survey_weight_dev
    )

    weighted_brier = brier_score_loss(
        y_dev,
        oof_prob,
        sample_weight=survey_weight_dev
    )


    # -------------------------------------------------------------------------
    # Save result
    # -------------------------------------------------------------------------

    logistic_cv_results.append({
        **params,
        "weighted_AP":
            weighted_ap,
        "weighted_AUROC":
            weighted_auc,
        "weighted_Brier":
            weighted_brier
    })


    # Save the complete OOF prediction vector for this exact candidate.
    logistic_oof_predictions[
        tuple(
            sorted(
                params.items()
            )
        )
    ] = oof_prob.copy()


# =============================================================================
# 3. CONVERT TO RESULTS TABLE
# =============================================================================

logistic_cv_results = pd.DataFrame(
    logistic_cv_results
)


# =============================================================================
# 4. APPLY THE PRESPECIFIED SELECTION RULE
#
# Primary:
#   highest weighted AP
#
# Tie-break 1:
#   lowest weighted Brier
#
# Tie-break 2:
#   highest weighted AUROC
#
# Sorting uses full floating-point precision.
# =============================================================================

logistic_cv_results = (
    logistic_cv_results
    .sort_values(
        by=[
            "weighted_AP",
            "weighted_Brier",
            "weighted_AUROC"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 5. LOCK THE SELECTED LOGISTIC CONFIGURATION
# =============================================================================

best_logistic_row = (
    logistic_cv_results
    .iloc[0]
)

BEST_LOGISTIC_PARAMS = {
    "model__C":
        float(
            best_logistic_row[
                "model__C"
            ]
        )
}

BEST_LOGISTIC_C = (
    BEST_LOGISTIC_PARAMS[
        "model__C"
    ]
)


# Retrieve the exact OOF vector corresponding to the selected configuration.
best_logistic_key = tuple(
    sorted(
        BEST_LOGISTIC_PARAMS.items()
    )
)

oof_logistic = (
    logistic_oof_predictions[
        best_logistic_key
    ]
    .copy()
)


# =============================================================================
# 6. FINAL INTEGRITY CHECKS
# =============================================================================

assert len(
    oof_logistic
) == len(
    X_dev
)

assert np.isfinite(
    oof_logistic
).all()

assert BEST_LOGISTIC_C in (
    PARAM_GRID_LOGISTIC[
        "model__C"
    ]
)


# =============================================================================
# 7. DISPLAY RESULTS
# =============================================================================

print("=" * 100)
print(
    "LOGISTIC REGRESSION: "
    "LOCKED-CV HYPERPARAMETER TUNING"
)
print("=" * 100)

print(
    logistic_cv_results.to_string(
        index=False,
        formatters={
            "weighted_AP":
                "{:.4f}".format,
            "weighted_AUROC":
                "{:.4f}".format,
            "weighted_Brier":
                "{:.4f}".format
        }
    )
)

print("\nSelected logistic configuration")
print("--------------------------------")
print(
    "C:",
    BEST_LOGISTIC_C
)

print(
    "Selection criterion:",
    "maximum survey-weighted OOF Average Precision"
)

print(
    "Tie-breaking rule available:",
    "Brier -> AUROC"
)

print("\nOOF governance")
print("--------------")
print(
    "OOF predictions available for all development women:",
    bool(
        np.isfinite(
            oof_logistic
        ).all()
    )
)

print(
    "Locked CV folds used:",
    len(cv_splits)
)

print(
    "Held-out final test set used:",
    "No"
)

print(
    "\nSTATUS: LOGISTIC REGRESSION "
    "HYPERPARAMETERS SELECTED AND LOCKED"
)

## 2.16 Random Forest tuning and out-of-fold prediction

Random Forest hyperparameters are evaluated using the same locked
stratum-preserving PSU-grouped cross-validation folds used for logistic
regression.

For every candidate configuration:

1. a fresh copy of the governed Random Forest pipeline is created;
2. preprocessing is learned using only the cross-validation fitting fold;
3. DHS fitting weights are normalized to mean 1 within that fitting fold;
4. the model is fitted using those weights;
5. predicted probabilities are generated only for the corresponding validation
   fold;
6. the five validation predictions are combined into one complete out-of-fold
   probability vector for the development population.

Survey-weighted Average Precision is the primary hyperparameter-selection
criterion.

Survey-weighted AUROC and Brier score are retained as complementary measures.

The prespecified selection and tie-breaking rules defined in Section 2.12 are
applied without modification.

The original hyperparameter dictionary for each candidate is preserved
explicitly so that values such as `max_depth=None` cannot be altered through
tabular type conversion.

The held-out final test set is not accessed.

In [ ]:
# =============================================================================
# 2.16 RANDOM FOREST:
#      SURVEY-WEIGHTED, STRATUM-PRESERVING PSU-GROUPED CV TUNING
#
# 48 candidate configurations × 5 locked folds.
#
# PRIMARY SELECTION METRIC:
#   Survey-weighted out-of-fold Average Precision.
#
# No held-out test data are used.
# =============================================================================


# =============================================================================
# 1. REUSABLE CANDIDATE-EVALUATION FUNCTION
#
# This helper performs exactly the governed development procedure:
#
#   candidate pipeline
#          ↓
#   five locked CV folds
#          ↓
#   fold-specific mean-1 fitting weights
#          ↓
#   one OOF probability per development woman
#          ↓
#   survey-weighted AP, AUROC and Brier score
#
# We will reuse this same helper for XGBoost in Section 2.17 so that the two
# tree algorithms cannot accidentally use different CV or weighting logic.
# =============================================================================

def evaluate_candidate_oof(
    pipeline,
    params,
    X,
    y,
    cv_splits_locked,
    fitting_weights,
    evaluation_weights
):
    """
    Fit one candidate model configuration across the locked CV folds and
    return its complete out-of-fold probability vector plus survey-weighted
    performance measures.

    Parameters
    ----------
    pipeline
        Unfitted sklearn pipeline.

    params
        Hyperparameter dictionary applied to the pipeline.

    X, y
        Development predictors and binary ANC8 outcome.

    cv_splits_locked
        Prespecified list of (fit_idx, val_idx) arrays.

    fitting_weights
        DHS relative weights used for model fitting. These are renormalized
        to mean 1 separately inside every fitting fold.

    evaluation_weights
        DHS survey weights used to evaluate the combined OOF predictions.
        These are not renormalized fold by fold.

    Returns
    -------
    dict
        OOF probabilities and survey-weighted AP, AUROC and Brier score.
    """

    oof_prob = np.full(
        len(X),
        np.nan,
        dtype=float
    )

    validation_count = np.zeros(
        len(X),
        dtype=int
    )


    # -------------------------------------------------------------------------
    # Locked cross-validation
    # -------------------------------------------------------------------------

    for fit_idx, val_idx in cv_splits_locked:

        # Fold-specific fitting weights.
        fold_fit_weight = (
            fitting_weights
            .iloc[fit_idx]
            .astype(float)
            .copy()
        )

        fold_fit_weight = (
            fold_fit_weight
            / fold_fit_weight.mean()
        )

        assert np.isclose(
            fold_fit_weight.mean(),
            1.0
        )


        # Fresh pipeline for every fold.
        model = clone(
            pipeline
        )

        model.set_params(
            **params
        )


        # Fit preprocessing + model only on this fitting fold.
        model.fit(
            X.iloc[fit_idx],
            y.iloc[fit_idx],
            model__sample_weight=fold_fit_weight
        )


        # Generate probabilities only for the validation fold.
        fold_prob = model.predict_proba(
            X.iloc[val_idx]
        )[:, 1]


        assert np.isfinite(
            fold_prob
        ).all()

        assert (
            (fold_prob >= 0)
            & (fold_prob <= 1)
        ).all()


        oof_prob[
            val_idx
        ] = fold_prob

        validation_count[
            val_idx
        ] += 1


    # -------------------------------------------------------------------------
    # Every woman must have exactly one OOF prediction.
    # -------------------------------------------------------------------------

    assert (
        validation_count == 1
    ).all()

    assert np.isfinite(
        oof_prob
    ).all()


    # -------------------------------------------------------------------------
    # Survey-weighted development performance.
    # -------------------------------------------------------------------------

    weighted_ap = average_precision_score(
        y,
        oof_prob,
        sample_weight=evaluation_weights
    )

    weighted_auc = roc_auc_score(
        y,
        oof_prob,
        sample_weight=evaluation_weights
    )

    weighted_brier = brier_score_loss(
        y,
        oof_prob,
        sample_weight=evaluation_weights
    )


    return {
        "oof_prob":
            oof_prob,

        "weighted_AP":
            weighted_ap,

        "weighted_AUROC":
            weighted_auc,

        "weighted_Brier":
            weighted_brier
    }


# =============================================================================
# 2. EVALUATE ALL 48 RANDOM FOREST CONFIGURATIONS
# =============================================================================

rf_result_rows = []

rf_oof_predictions = {}


for configuration_number, params in enumerate(
    ParameterGrid(
        PARAM_GRID_RF
    ),
    start=1
):

    # Preserve the exact original Python parameter dictionary.
    # This matters particularly for max_depth=None.
    params_exact = dict(
        params
    )


    candidate_result = evaluate_candidate_oof(
        pipeline=pipeline_rf,
        params=params_exact,
        X=X_dev,
        y=y_dev,
        cv_splits_locked=cv_splits,
        fitting_weights=fit_weight_dev,
        evaluation_weights=survey_weight_dev
    )


    # Stable configuration identifier.
    config_id = (
        f"RF_{configuration_number:02d}"
    )


    # Store the exact hyperparameter dictionary separately from the display
    # columns so pandas cannot alter None into NaN for later refitting.
    rf_result_rows.append({
        "config_id":
            config_id,

        "params_exact":
            params_exact,

        **params_exact,

        "weighted_AP":
            candidate_result[
                "weighted_AP"
            ],

        "weighted_AUROC":
            candidate_result[
                "weighted_AUROC"
            ],

        "weighted_Brier":
            candidate_result[
                "weighted_Brier"
            ]
    })


    rf_oof_predictions[
        config_id
    ] = (
        candidate_result[
            "oof_prob"
        ].copy()
    )


# =============================================================================
# 3. RESULTS TABLE
# =============================================================================

rf_cv_results = pd.DataFrame(
    rf_result_rows
)


assert len(
    rf_cv_results
) == N_RF_CONFIGS == 48


# =============================================================================
# 4. APPLY THE LOCKED SELECTION RULE
# =============================================================================

rf_cv_results = (
    rf_cv_results
    .sort_values(
        by=[
            "weighted_AP",
            "weighted_Brier",
            "weighted_AUROC"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 5. LOCK THE SELECTED RANDOM FOREST CONFIGURATION
#
# IMPORTANT:
# Retrieve the exact dictionary rather than rebuilding it from DataFrame
# columns. This preserves Python None correctly.
# =============================================================================

best_rf_row = (
    rf_cv_results
    .iloc[0]
)

BEST_RF_CONFIG_ID = (
    best_rf_row[
        "config_id"
    ]
)

BEST_RF_PARAMS = dict(
    best_rf_row[
        "params_exact"
    ]
)


oof_rf = (
    rf_oof_predictions[
        BEST_RF_CONFIG_ID
    ]
    .copy()
)


# =============================================================================
# 6. INTEGRITY CHECKS
# =============================================================================

assert set(
    BEST_RF_PARAMS.keys()
) == set(
    PARAM_GRID_RF.keys()
)

assert len(
    oof_rf
) == len(
    X_dev
)

assert np.isfinite(
    oof_rf
).all()


# =============================================================================
# 7. DISPLAY TOP 10 CONFIGURATIONS
#
# params_exact is intentionally excluded from the display because it is an
# internal governance object rather than a manuscript-facing column.
# =============================================================================

rf_display_columns = (
    ["config_id"]
    + list(
        PARAM_GRID_RF.keys()
    )
    + [
        "weighted_AP",
        "weighted_AUROC",
        "weighted_Brier"
    ]
)


print("=" * 108)
print(
    "RANDOM FOREST: "
    "LOCKED-CV HYPERPARAMETER TUNING"
)
print("=" * 108)

print(
    rf_cv_results[
        rf_display_columns
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "weighted_AP":
                "{:.4f}".format,

            "weighted_AUROC":
                "{:.4f}".format,

            "weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nSelected Random Forest configuration")
print("------------------------------------")

print(
    "Configuration ID:",
    BEST_RF_CONFIG_ID
)

for parameter_name, parameter_value in (
    BEST_RF_PARAMS.items()
):

    print(
        f"{parameter_name}: "
        f"{parameter_value}"
    )


print(
    "\nSelection criterion:",
    "maximum survey-weighted OOF Average Precision"
)

print(
    "Tie-breaking rule available:",
    "Brier -> AUROC"
)


print("\nOOF governance")
print("--------------")

print(
    "OOF predictions available for all development women:",
    bool(
        np.isfinite(
            oof_rf
        ).all()
    )
)

print(
    "Locked CV folds used:",
    len(cv_splits)
)

print(
    "Held-out final test set used:",
    "No"
)


print(
    "\nSTATUS: RANDOM FOREST "
    "HYPERPARAMETERS SELECTED AND LOCKED"
)

## 2.17 XGBoost tuning and out-of-fold prediction

XGBoost hyperparameters are evaluated using the same locked
stratum-preserving PSU-grouped cross-validation folds used for logistic
regression and Random Forest.

For every candidate configuration:

1. a fresh copy of the governed XGBoost pipeline is created;
2. preprocessing is learned only from the fitting fold;
3. DHS fitting weights are normalized to mean 1 within that fitting fold;
4. the model is fitted using those weights;
5. predicted probabilities are generated only for the corresponding validation
   fold;
6. the five validation predictions are combined into one complete out-of-fold
   probability vector for the development population.

Survey-weighted Average Precision is the primary hyperparameter-selection
criterion.

Survey-weighted AUROC and Brier score are retained as complementary measures.

The prespecified selection and tie-breaking rules defined in Section 2.12 are
applied without modification.

No `scale_pos_weight` or other class weighting is used.

The original hyperparameter dictionary for each candidate is preserved
explicitly to avoid later reconstruction from DataFrame values.

The held-out final test set is not accessed.

In [ ]:
# =============================================================================
# 2.17 XGBOOST:
#      SURVEY-WEIGHTED, STRATUM-PRESERVING PSU-GROUPED CV TUNING
#
# 64 candidate configurations × 5 locked folds.
#
# PRIMARY SELECTION METRIC:
#   Survey-weighted out-of-fold Average Precision.
#
# Reuses the governed `evaluate_candidate_oof()` helper from Section 2.16.
# =============================================================================


# =============================================================================
# 1. STORAGE
# =============================================================================

xgb_result_rows = []

xgb_oof_predictions = {}


# =============================================================================
# 2. EVALUATE ALL 64 XGBOOST CONFIGURATIONS
# =============================================================================

for configuration_number, params in enumerate(
    ParameterGrid(
        PARAM_GRID_XGB
    ),
    start=1
):

    # Preserve exact Python parameter values.
    params_exact = dict(
        params
    )


    candidate_result = evaluate_candidate_oof(
        pipeline=pipeline_xgb,
        params=params_exact,
        X=X_dev,
        y=y_dev,
        cv_splits_locked=cv_splits,
        fitting_weights=fit_weight_dev,
        evaluation_weights=survey_weight_dev
    )


    config_id = (
        f"XGB_{configuration_number:02d}"
    )


    xgb_result_rows.append({
        "config_id":
            config_id,

        "params_exact":
            params_exact,

        **params_exact,

        "weighted_AP":
            candidate_result[
                "weighted_AP"
            ],

        "weighted_AUROC":
            candidate_result[
                "weighted_AUROC"
            ],

        "weighted_Brier":
            candidate_result[
                "weighted_Brier"
            ]
    })


    xgb_oof_predictions[
        config_id
    ] = (
        candidate_result[
            "oof_prob"
        ].copy()
    )


# =============================================================================
# 3. RESULTS TABLE
# =============================================================================

xgb_cv_results = pd.DataFrame(
    xgb_result_rows
)


assert len(
    xgb_cv_results
) == N_XGB_CONFIGS == 64


# =============================================================================
# 4. APPLY THE LOCKED SELECTION RULE
# =============================================================================

xgb_cv_results = (
    xgb_cv_results
    .sort_values(
        by=[
            "weighted_AP",
            "weighted_Brier",
            "weighted_AUROC"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 5. LOCK THE SELECTED XGBOOST CONFIGURATION
# =============================================================================

best_xgb_row = (
    xgb_cv_results
    .iloc[0]
)

BEST_XGB_CONFIG_ID = (
    best_xgb_row[
        "config_id"
    ]
)

BEST_XGB_PARAMS = dict(
    best_xgb_row[
        "params_exact"
    ]
)


oof_xgb = (
    xgb_oof_predictions[
        BEST_XGB_CONFIG_ID
    ]
    .copy()
)


# =============================================================================
# 6. INTEGRITY CHECKS
# =============================================================================

assert set(
    BEST_XGB_PARAMS.keys()
) == set(
    PARAM_GRID_XGB.keys()
)

assert len(
    oof_xgb
) == len(
    X_dev
)

assert np.isfinite(
    oof_xgb
).all()


# =============================================================================
# 7. DISPLAY TOP 10 CONFIGURATIONS
# =============================================================================

xgb_display_columns = (
    ["config_id"]
    + list(
        PARAM_GRID_XGB.keys()
    )
    + [
        "weighted_AP",
        "weighted_AUROC",
        "weighted_Brier"
    ]
)


print("=" * 112)
print(
    "XGBOOST: "
    "LOCKED-CV HYPERPARAMETER TUNING"
)
print("=" * 112)

print(
    xgb_cv_results[
        xgb_display_columns
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "weighted_AP":
                "{:.4f}".format,

            "weighted_AUROC":
                "{:.4f}".format,

            "weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nSelected XGBoost configuration")
print("--------------------------------")

print(
    "Configuration ID:",
    BEST_XGB_CONFIG_ID
)

for parameter_name, parameter_value in (
    BEST_XGB_PARAMS.items()
):

    print(
        f"{parameter_name}: "
        f"{parameter_value}"
    )


print(
    "\nSelection criterion:",
    "maximum survey-weighted OOF Average Precision"
)

print(
    "Tie-breaking rule available:",
    "Brier -> AUROC"
)


print("\nOOF governance")
print("--------------")

print(
    "OOF predictions available for all development women:",
    bool(
        np.isfinite(
            oof_xgb
        ).all()
    )
)

print(
    "Locked CV folds used:",
    len(cv_splits)
)

print(
    "Held-out final test set used:",
    "No"
)


print(
    "\nSTATUS: XGBOOST "
    "HYPERPARAMETERS SELECTED AND LOCKED"
)

XGBoost achieved weighted AP of 0.4897, slightly higher than logistic regression but lower than Random Forest. However, its AUROC and Brier score were inferior to logistic regression. Overall, none of the nonlinear models showed a consistent predictive advantage over the regularized logistic baseline during internal validation.

## 2.18 Development-stage comparison and transition to held-out evaluation

Three prespecified candidate algorithms have now been developed using:

- the same development population;
- the same 21 frozen predictors;
- governed algorithm-specific preprocessing;
- the same locked stratum-preserving PSU-grouped five-fold cross-validation;
- fold-specific mean-1 DHS fitting weights;
- survey-weighted out-of-fold evaluation; and
- prespecified hyperparameter-selection rules.

The tuned models show modest differences in development out-of-fold
performance.

Random Forest achieved the highest survey-weighted Average Precision and the
lowest weighted Brier score.

Logistic regression achieved the highest weighted AUROC.

XGBoost showed intermediate performance across the three measures.

These development results are used to describe internal model-development
performance, not to declare a final overall winning algorithm.

Average Precision was prespecified as the primary criterion for selecting
hyperparameters *within* each algorithm. No separate algorithm-level rule was
prespecified for eliminating candidate algorithms based on development
cross-validation results.

Therefore, all three tuned algorithms proceed unchanged to the untouched
held-out test set.

The held-out evaluation will compare discrimination, positive-class ranking,
probabilistic accuracy, and calibration using the performance framework
prespecified in Section 2.10.

No further hyperparameter tuning will be performed after held-out test
predictions are generated.

In [ ]:
# =============================================================================
# 2.18 DEVELOPMENT OOF MODEL COMPARISON
#
# This cell compares the three already-selected algorithm configurations.
#
# IMPORTANT
# ---------
# - No model is fitted here.
# - No hyperparameter is changed here.
# - No held-out test data are accessed.
# =============================================================================


# =============================================================================
# 1. FUNCTION FOR GOVERNED OOF PERFORMANCE
# =============================================================================

def weighted_oof_metrics(
    y_true,
    probability,
    weight
):
    """
    Calculate the prespecified survey-weighted development performance metrics.
    """

    return {
        "Weighted_AP":
            average_precision_score(
                y_true,
                probability,
                sample_weight=weight
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_true,
                probability,
                sample_weight=weight
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_true,
                probability,
                sample_weight=weight
            )
    }


# =============================================================================
# 2. CALCULATE METRICS FROM THE LOCKED OOF PREDICTIONS
# =============================================================================

development_model_comparison = pd.DataFrame([
    {
        "Model":
            "Logistic regression",

        **weighted_oof_metrics(
            y_dev,
            oof_logistic,
            survey_weight_dev
        )
    },

    {
        "Model":
            "Random Forest",

        **weighted_oof_metrics(
            y_dev,
            oof_rf,
            survey_weight_dev
        )
    },

    {
        "Model":
            "XGBoost",

        **weighted_oof_metrics(
            y_dev,
            oof_xgb,
            survey_weight_dev
        )
    }
])


# =============================================================================
# 3. IDENTIFY METRIC-SPECIFIC LEADERS
#
# These are descriptive only.
# They do NOT define an overall winning algorithm.
# =============================================================================

best_ap_model = (
    development_model_comparison
    .loc[
        development_model_comparison[
            "Weighted_AP"
        ].idxmax(),
        "Model"
    ]
)

best_auc_model = (
    development_model_comparison
    .loc[
        development_model_comparison[
            "Weighted_AUROC"
        ].idxmax(),
        "Model"
    ]
)

best_brier_model = (
    development_model_comparison
    .loc[
        development_model_comparison[
            "Weighted_Brier"
        ].idxmin(),
        "Model"
    ]
)


# =============================================================================
# 4. INTEGRITY CHECKS
# =============================================================================

assert len(
    development_model_comparison
) == 3

assert set(
    development_model_comparison[
        "Model"
    ]
) == {
    "Logistic regression",
    "Random Forest",
    "XGBoost"
}

assert np.isfinite(
    development_model_comparison[
        [
            "Weighted_AP",
            "Weighted_AUROC",
            "Weighted_Brier"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 96)
print("DEVELOPMENT OUT-OF-FOLD MODEL COMPARISON")
print("=" * 96)

print(
    development_model_comparison
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nMetric-specific leaders")
print("-----------------------")

print(
    "Highest weighted AP:",
    best_ap_model
)

print(
    "Highest weighted AUROC:",
    best_auc_model
)

print(
    "Lowest weighted Brier:",
    best_brier_model
)


print("\nInterpretation governance")
print("-------------------------")

print(
    "Overall algorithm winner declared from development CV:",
    "No"
)

print(
    "All three tuned algorithms proceed to held-out evaluation:",
    "Yes"
)

print(
    "Hyperparameters changed after tuning:",
    "No"
)

print(
    "Held-out final test set accessed:",
    "No"
)

print(
    "\nSTATUS: DEVELOPMENT MODEL COMPARISON "
    "COMPLETE; ALL THREE MODELS ADVANCE"
)

## 2.19 Final refitting on the full development dataset

After hyperparameter tuning is completed and locked, each candidate pipeline is
refitted once using the complete development dataset.

The selected configuration for each algorithm is fixed before this refitting:

- logistic regression uses the selected `C`;
- Random Forest uses the selected tree and ensemble settings;
- XGBoost uses the selected boosting settings.

For each algorithm:

1. a fresh clone of the governed pipeline is created;
2. the locked hyperparameters are applied;
3. preprocessing is learned from the complete development dataset only;
4. the model is fitted using the full-development DHS fitting weights normalized
   to mean 1;
5. no information from the held-out test set is used during fitting.

The held-out test set remains outside all preprocessing, tuning, fitting, and
model-selection procedures.

Once these full-development models are fitted, their parameters and
preprocessing states are treated as locked. No hyperparameter, predictor,
preprocessing rule, or class-balance strategy will be changed in response to
test-set results.

In [ ]:
# =============================================================================
# 2.19 REFIT ALL THREE LOCKED MODELS ON THE FULL DEVELOPMENT DATASET
#
# IMPORTANT
# ---------
# - Hyperparameters are already selected and locked.
# - Preprocessing is re-learned from the complete development dataset.
# - Full-development fitting weights have mean 1.
# - The held-out test set is not accessed.
# =============================================================================


# =============================================================================
# 1. FINAL DEVELOPMENT FITTING WEIGHTS
#
# `fit_weight_dev` already has mean 1 across the complete development set.
# Re-normalize defensively so this section is self-checking and reproducible.
# =============================================================================

final_fit_weight_dev = (
    fit_weight_dev
    .astype(float)
    .copy()
)

final_fit_weight_dev = (
    final_fit_weight_dev
    / final_fit_weight_dev.mean()
)

assert np.isclose(
    final_fit_weight_dev.mean(),
    1.0
)

assert (
    final_fit_weight_dev > 0
).all()

assert len(
    final_fit_weight_dev
) == len(
    X_dev
)


# =============================================================================
# 2. LOGISTIC REGRESSION
# =============================================================================

final_logistic = clone(
    pipeline_logistic
)

final_logistic.set_params(
    **BEST_LOGISTIC_PARAMS
)

final_logistic.fit(
    X_dev,
    y_dev,
    model__sample_weight=final_fit_weight_dev
)


# =============================================================================
# 3. RANDOM FOREST
# =============================================================================

final_rf = clone(
    pipeline_rf
)

final_rf.set_params(
    **BEST_RF_PARAMS
)

final_rf.fit(
    X_dev,
    y_dev,
    model__sample_weight=final_fit_weight_dev
)


# =============================================================================
# 4. XGBOOST
# =============================================================================

final_xgb = clone(
    pipeline_xgb
)

final_xgb.set_params(
    **BEST_XGB_PARAMS
)

final_xgb.fit(
    X_dev,
    y_dev,
    model__sample_weight=final_fit_weight_dev
)


# =============================================================================
# 5. VERIFY THAT THE LOCKED HYPERPARAMETERS WERE APPLIED
# =============================================================================

assert np.isclose(
    final_logistic
    .named_steps["model"]
    .get_params()["C"],
    BEST_LOGISTIC_PARAMS["model__C"]
)

for parameter_name, parameter_value in BEST_RF_PARAMS.items():

    estimator_parameter = (
        parameter_name
        .replace(
            "model__",
            ""
        )
    )

    assert (
        final_rf
        .named_steps["model"]
        .get_params()[estimator_parameter]
        == parameter_value
    )


for parameter_name, parameter_value in BEST_XGB_PARAMS.items():

    estimator_parameter = (
        parameter_name
        .replace(
            "model__",
            ""
        )
    )

    assert (
        final_xgb
        .named_steps["model"]
        .get_params()[estimator_parameter]
        == parameter_value
    )


# =============================================================================
# 6. VERIFY FITTED PREPROCESSING STRUCTURE
#
# Each fully refitted pipeline should recover the same governed transformed
# feature structure defined earlier.
# =============================================================================

final_logistic_features = (
    final_logistic
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

final_rf_features = (
    final_rf
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

final_xgb_features = (
    final_xgb
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


assert np.array_equal(
    final_logistic_features,
    feature_lineage[
        "Transformed_feature"
    ].to_numpy()
)

assert np.array_equal(
    final_rf_features,
    feature_lineage[
        "Transformed_feature"
    ].to_numpy()
)

assert np.array_equal(
    final_xgb_features,
    feature_lineage[
        "Transformed_feature"
    ].to_numpy()
)


# =============================================================================
# 7. GOVERNED MODEL REGISTRY
#
# This becomes the canonical model container for downstream prediction.
# =============================================================================

final_models = {
    "Logistic regression":
        final_logistic,

    "Random Forest":
        final_rf,

    "XGBoost":
        final_xgb
}


# =============================================================================
# 8. DISPLAY
# =============================================================================

print("=" * 100)
print("FINAL LOCKED MODELS REFITTED ON FULL DEVELOPMENT DATASET")
print("=" * 100)

print("\nDevelopment fitting")
print("-------------------")
print("Development women:", len(X_dev))
print(
    "Mean final fitting weight:",
    f"{final_fit_weight_dev.mean():.6f}"
)

print("\nLocked configurations")
print("---------------------")
print(
    "Logistic regression:",
    BEST_LOGISTIC_PARAMS
)
print(
    "Random Forest:",
    BEST_RF_PARAMS
)
print(
    "XGBoost:",
    BEST_XGB_PARAMS
)

print("\nFeature structure")
print("-----------------")
print(
    "Logistic transformed features:",
    len(final_logistic_features)
)
print(
    "Random Forest transformed features:",
    len(final_rf_features)
)
print(
    "XGBoost transformed features:",
    len(final_xgb_features)
)

print("\nGovernance")
print("----------")
print("Hyperparameters changed after tuning: No")
print("Development rows used for final fitting:", len(X_dev))
print("Held-out test rows used for fitting: 0")
print("Held-out test set accessed here: No")

print(
    "\nSTATUS: FINAL DEVELOPMENT MODELS "
    "REFITTED AND LOCKED"
)

## 2.20 Locked held-out test prediction and integrity audit

Model development is now complete.

The three tuned pipelines have been refitted using the complete development
dataset, and their preprocessing rules and hyperparameters are locked.

This section opens the previously untouched held-out test set for prediction.

To preserve the separation between model development and final evaluation, the
sequence is explicit:

1. the 21 predictor columns are taken from the held-out test set;
2. the three locked models generate predicted ANC8 probabilities;
3. prediction integrity is verified;
4. only after the predictions have been generated are the observed ANC8
   outcomes attached for subsequent evaluation.

No model is fitted, updated, recalibrated, or otherwise modified using the test
set.

The test-set survey weights are retained for subsequent population-weighted
evaluation but are not used to generate individual predictions.

The resulting probability vectors constitute the locked held-out predictions
for the three baseline models. Subsequent test-set results may be evaluated and
reported, but they must not be used to alter the fitted models or their
hyperparameters.

In [ ]:
# =============================================================================
# 2.20 GENERATE AND LOCK HELD-OUT TEST PREDICTIONS
#
# IMPORTANT
# ---------
# Model development is finished.
#
# Sequence:
#   1. prepare test predictors;
#   2. verify test/development separation;
#   3. generate predictions from already-fitted models;
#   4. audit predictions;
#   5. only then attach observed test outcomes for later evaluation.
#
# NO .fit() call is permitted in this section.
# =============================================================================


# =============================================================================
# 1. PREPARE HELD-OUT TEST PREDICTORS
# =============================================================================

X_test_final = (
    test_df[
        PRIMARY_PREDICTORS
    ]
    .copy()
)


# DHS relative sampling weights for later TEST EVALUATION.
#
# These weights are NOT used to generate predictions.
survey_weight_test = (
    test_df["v005"]
    .astype(float)
    / 1_000_000
)


# Test design variables are retained only for later audit / uncertainty
# estimation. They are not model predictors.
groups_test = (
    test_df["v021"]
    .copy()
)

strata_test = (
    test_df["v022"]
    .copy()
)


# =============================================================================
# 2. TEST-SET STRUCTURAL INTEGRITY
# =============================================================================

assert len(
    X_test_final
) == len(
    test_df
) == 1375

assert list(
    X_test_final.columns
) == list(
    PRIMARY_PREDICTORS
)

assert len(
    survey_weight_test
) == len(
    X_test_final
)

assert np.isfinite(
    survey_weight_test
).all()

assert (
    survey_weight_test > 0
).all()


# -------------------------------------------------------------------------
# Development and test women must remain completely disjoint.
# -------------------------------------------------------------------------

development_caseids = set(
    train_df["caseid"]
)

test_caseids = set(
    test_df["caseid"]
)

assert len(
    development_caseids.intersection(
        test_caseids
    )
) == 0


# Their union must reconstruct the complete resolved baseline analysis cohort.
assert (
    development_caseids
    | test_caseids
) == set(
    baseline_df["caseid"]
)


# -------------------------------------------------------------------------
# Check that the expected survey-design structure is retained.
# -------------------------------------------------------------------------

assert groups_test.nunique() == 159

assert strata_test.nunique() == 27


# =============================================================================
# 3. GENERATE PREDICTIONS
#
# Critically, no test outcomes are referenced before this point.
#
# `final_models` contains the already-fitted pipelines from Section 2.19.
# =============================================================================

test_probability_by_model = {}


for model_name, fitted_model in (
    final_models.items()
):

    probability = (
        fitted_model
        .predict_proba(
            X_test_final
        )[:, 1]
    )


    # -------------------------------------------------------------------------
    # Immediate prediction checks
    # -------------------------------------------------------------------------

    assert len(
        probability
    ) == len(
        X_test_final
    )

    assert np.isfinite(
        probability
    ).all()

    assert (
        (probability >= 0)
        & (probability <= 1)
    ).all()


    test_probability_by_model[
        model_name
    ] = probability.copy()


# =============================================================================
# 4. CREATE LOCKED PREDICTION TABLE
#
# At this point the table contains:
#   - case identifier
#   - survey-design information
#   - survey weight
#   - model predictions
#
# It STILL does not require the observed outcome to generate predictions.
# =============================================================================

test_predictions = pd.DataFrame({
    "caseid":
        test_df["caseid"].to_numpy(),

    "v021":
        groups_test.to_numpy(),

    "v022":
        strata_test.to_numpy(),

    "survey_weight":
        survey_weight_test.to_numpy(),

    "prob_logistic":
        test_probability_by_model[
            "Logistic regression"
        ],

    "prob_random_forest":
        test_probability_by_model[
            "Random Forest"
        ],

    "prob_xgboost":
        test_probability_by_model[
            "XGBoost"
        ]
})


probability_cols = [
    "prob_logistic",
    "prob_random_forest",
    "prob_xgboost"
]


# =============================================================================
# 5. LOCK PREDICTIONS BEFORE OUTCOME-BASED EVALUATION
#
# A copy is retained as the canonical held-out probability artifact in memory.
# No model modification is permitted after this point.
# =============================================================================

test_predictions_locked = (
    test_predictions
    .copy(deep=True)
)


# =============================================================================
# 6. ONLY NOW ATTACH THE OBSERVED TEST OUTCOME
#
# The outcome was not needed to generate any of the three predictions.
# From this point onward, the test set is formally opened for evaluation.
# =============================================================================

y_test_final = (
    test_df["anc8plus"]
    .astype(int)
    .copy()
)


assert len(
    y_test_final
) == len(
    test_predictions_locked
)

assert y_test_final.isin(
    [0, 1]
).all()


test_predictions_locked[
    "anc8plus_observed"
] = (
    y_test_final.to_numpy()
)


# =============================================================================
# 7. FINAL PREDICTION-TABLE INTEGRITY
# =============================================================================

assert test_predictions_locked[
    "caseid"
].is_unique

assert test_predictions_locked[
    probability_cols
].notna().all().all()

assert np.isfinite(
    test_predictions_locked[
        probability_cols
    ].to_numpy()
).all()


for column in probability_cols:

    assert test_predictions_locked[
        column
    ].between(
        0,
        1
    ).all()


# Observed weighted prevalence is descriptive test-set context.
# It is calculated only AFTER predictions are locked.
weighted_test_prevalence = np.average(
    y_test_final,
    weights=survey_weight_test
)


# =============================================================================
# 8. DISPLAY
# =============================================================================

print("=" * 104)
print("LOCKED HELD-OUT TEST-PREDICTION INTEGRITY AUDIT")
print("=" * 104)

print("\nHeld-out cohort")
print("---------------")
print(
    "Test women:",
    len(test_predictions_locked)
)
print(
    "Test PSUs:",
    groups_test.nunique()
)
print(
    "Test strata:",
    strata_test.nunique()
)

print(
    "Development/test case-ID overlap:",
    len(
        development_caseids.intersection(
            test_caseids
        )
    )
)

print(
    "Observed ANC8+ women:",
    int(
        y_test_final.sum()
    )
)

print(
    "Unweighted test ANC8+ prevalence:",
    f"{y_test_final.mean() * 100:.2f}%"
)

print(
    "Survey-weighted test ANC8+ prevalence:",
    f"{weighted_test_prevalence * 100:.2f}%"
)


print("\nPrediction ranges")
print("-----------------")

for column in probability_cols:

    print(
        f"{column}: "
        f"min={test_predictions_locked[column].min():.4f}, "
        f"median={test_predictions_locked[column].median():.4f}, "
        f"max={test_predictions_locked[column].max():.4f}"
    )


print("\nPrediction integrity")
print("--------------------")

print(
    "Prediction available for every test woman:",
    "Yes"
)

print(
    "All predictions finite:",
    "Yes"
)

print(
    "All probabilities between 0 and 1:",
    "Yes"
)

print(
    "Predictions generated before outcome-based evaluation:",
    "Yes"
)


print("\nGovernance")
print("----------")

print(
    "Additional model fitting on test data:",
    "No"
)

print(
    "Test-set preprocessing fitted:",
    "No"
)

print(
    "Hyperparameters changed after test prediction:",
    "No"
)

print(
    "Classification threshold applied:",
    "No"
)

print(
    "\nSTATUS: HELD-OUT TEST PREDICTIONS "
    "GENERATED, AUDITED, AND LOCKED"
)

## 2.21 Final held-out probability-based performance

The three locked baseline models are now evaluated using the held-out test
predictions generated in Section 2.20.

No model is fitted, tuned, recalibrated, or otherwise modified in this section.

Primary probability-based performance is evaluated using the DHS sampling
weights and the prespecified measures defined in Section 2.10:

- **Average Precision (AP):** summarizes positive-class ranking performance;
- **AUROC:** summarizes discrimination between women who attained ANC8 and
  those who did not;
- **Brier score:** summarizes overall squared error of predicted probabilities.

Higher AP and AUROC indicate better performance. Lower Brier score indicates
better probabilistic accuracy.

### Average Precision reference

For a non-informative ranking, the natural prevalence-based reference for
Average Precision corresponds to the positive-outcome prevalence in the
population being evaluated.

The survey-weighted ANC8+ prevalence in the held-out test population is
therefore reported alongside AP as its test-specific reference. The development
AP reference is not reused because the development and test populations have
different weighted ANC8 prevalence.

### Interpretation

These are final held-out performance estimates for the baseline prediction
horizon.

Metric differences among algorithms are described rather than used for further
model modification.

Uncertainty intervals and calibration are assessed separately in subsequent
sections.

In [ ]:
# =============================================================================
# 2.21 FINAL HELD-OUT PROBABILITY-BASED PERFORMANCE
#
# IMPORTANT
# ---------
# - Uses ONLY the locked prediction artifact from Section 2.20.
# - No model fitting occurs here.
# - No predictions are regenerated.
# - No hyperparameters are changed.
# =============================================================================


# =============================================================================
# 1. CANONICAL HELD-OUT EVALUATION OBJECTS
# =============================================================================

y_test_eval = (
    test_predictions_locked[
        "anc8plus_observed"
    ]
    .astype(int)
    .to_numpy()
)

w_test_eval = (
    test_predictions_locked[
        "survey_weight"
    ]
    .astype(float)
    .to_numpy()
)


assert len(
    y_test_eval
) == len(
    w_test_eval
) == len(
    test_predictions_locked
)

assert np.isfinite(
    w_test_eval
).all()

assert (
    w_test_eval > 0
).all()

assert np.isin(
    y_test_eval,
    [0, 1]
).all()


# =============================================================================
# 2. TEST-SPECIFIC PREVALENCE REFERENCE
#
# For AP, the prevalence of the positive outcome provides the natural
# prevalence-based no-skill reference.
# =============================================================================

test_weighted_prevalence = np.average(
    y_test_eval,
    weights=w_test_eval
)


# =============================================================================
# 3. LOCKED MODEL-PREDICTION COLUMNS
# =============================================================================

heldout_probability_columns = {
    "Logistic regression":
        "prob_logistic",

    "Random Forest":
        "prob_random_forest",

    "XGBoost":
        "prob_xgboost"
}


# =============================================================================
# 4. CALCULATE PRESPECIFIED HELD-OUT METRICS
# =============================================================================

heldout_performance_rows = []


for model_name, probability_column in (
    heldout_probability_columns.items()
):

    probability = (
        test_predictions_locked[
            probability_column
        ]
        .astype(float)
        .to_numpy()
    )


    assert np.isfinite(
        probability
    ).all()

    assert (
        (probability >= 0)
        & (probability <= 1)
    ).all()


    weighted_ap = average_precision_score(
        y_test_eval,
        probability,
        sample_weight=w_test_eval
    )

    weighted_auc = roc_auc_score(
        y_test_eval,
        probability,
        sample_weight=w_test_eval
    )

    weighted_brier = brier_score_loss(
        y_test_eval,
        probability,
        sample_weight=w_test_eval
    )


    heldout_performance_rows.append({
        "Model":
            model_name,

        "Weighted_AP":
            weighted_ap,

        "AP_prevalence_reference":
            test_weighted_prevalence,

        "Weighted_AUROC":
            weighted_auc,

        "Weighted_Brier":
            weighted_brier
    })


# =============================================================================
# 5. CANONICAL FINAL PERFORMANCE TABLE
# =============================================================================

heldout_performance = pd.DataFrame(
    heldout_performance_rows
)


assert len(
    heldout_performance
) == 3

assert np.isfinite(
    heldout_performance[
        [
            "Weighted_AP",
            "AP_prevalence_reference",
            "Weighted_AUROC",
            "Weighted_Brier"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 6. METRIC-SPECIFIC LEADERS
#
# Descriptive only.
# No model modification follows from these labels.
# =============================================================================

heldout_best_ap = (
    heldout_performance
    .loc[
        heldout_performance[
            "Weighted_AP"
        ].idxmax(),
        "Model"
    ]
)

heldout_best_auc = (
    heldout_performance
    .loc[
        heldout_performance[
            "Weighted_AUROC"
        ].idxmax(),
        "Model"
    ]
)

heldout_best_brier = (
    heldout_performance
    .loc[
        heldout_performance[
            "Weighted_Brier"
        ].idxmin(),
        "Model"
    ]
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 104)
print("FINAL HELD-OUT PROBABILITY-BASED PERFORMANCE")
print("=" * 104)

print(
    heldout_performance
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "AP_prevalence_reference":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nHeld-out outcome context")
print("------------------------")

print(
    "Survey-weighted ANC8+ prevalence / AP reference:",
    f"{test_weighted_prevalence:.4f}",
    f"({test_weighted_prevalence * 100:.2f}%)"
)


print("\nMetric-specific leaders")
print("-----------------------")

print(
    "Highest weighted AP:",
    heldout_best_ap
)

print(
    "Highest weighted AUROC:",
    heldout_best_auc
)

print(
    "Lowest weighted Brier:",
    heldout_best_brier
)


print("\nEvaluation governance")
print("---------------------")

print(
    "Predictions regenerated here:",
    "No"
)

print(
    "Model fitting performed here:",
    "No"
)

print(
    "Hyperparameters changed:",
    "No"
)

print(
    "Test results used for model updating:",
    "No"
)

print(
    "\nSTATUS: FINAL HELD-OUT "
    "PROBABILITY PERFORMANCE ESTIMATED"
)

## 2.22 Numerical calibration on the held-out test set

Calibration evaluates agreement between predicted ANC8 probabilities and
observed ANC8 frequencies in the held-out population.

Three complementary quantities are reported.

### Calibration-in-the-large (CITL)

CITL assesses systematic underprediction or overprediction while fixing the
calibration slope at 1.

It is estimated using a weighted logistic calibration model in which the logit
of the predicted probability is included as an offset.

The ideal CITL value is 0.

- CITL > 0 indicates overall underprediction.
- CITL < 0 indicates overall overprediction.

### Calibration slope

The calibration slope evaluates the spread of predicted risks.

It is estimated from a weighted logistic recalibration model containing both an
intercept and the logit of the predicted probability.

The ideal slope is 1.

- slope < 1 generally indicates predictions that are too extreme;
- slope > 1 indicates predictions that are not sufficiently dispersed.

### Joint recalibration intercept

The intercept estimated jointly with the calibration slope is retained as a
diagnostic quantity.

It is explicitly labelled the **joint recalibration intercept** and is not
interpreted as CITL.

Survey-weighted observed prevalence and mean predicted probability are also
reported.

The test set is used only for evaluation. No model recalibration or updating is
performed.

In [ ]:
# =============================================================================
# 2.22 HELD-OUT NUMERICAL CALIBRATION
#
# CITL:
#   slope fixed at 1 using the predicted log-odds as an OFFSET.
#
# Calibration slope:
#   estimated jointly with an intercept.
#
# The joint recalibration intercept is NOT CITL.
# =============================================================================

import statsmodels.api as sm


# =============================================================================
# 1. CALIBRATION FUNCTION
# =============================================================================

def weighted_calibration_summary(
    y,
    probability,
    weights
):
    """
    Calculate survey-weighted calibration diagnostics.

    Returns:
        - observed weighted prevalence
        - weighted mean predicted probability
        - calibration-in-the-large (CITL)
        - joint recalibration intercept
        - calibration slope
    """

    # -------------------------------------------------------------------------
    # Convert inputs to NumPy arrays.
    # -------------------------------------------------------------------------

    y = np.asarray(
        y,
        dtype=float
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Basic integrity checks.
    # -------------------------------------------------------------------------

    assert (
        len(y)
        == len(probability)
        == len(weights)
    )

    assert np.isin(
        y,
        [0, 1]
    ).all()

    assert np.isfinite(
        probability
    ).all()

    assert np.isfinite(
        weights
    ).all()

    assert (
        weights > 0
    ).all()


    # -------------------------------------------------------------------------
    # Normalize weights to mean 1 for stable model fitting.
    #
    # Multiplying all weights by one common constant does not change the
    # estimated calibration coefficients.
    # -------------------------------------------------------------------------

    calibration_weights = (
        weights
        / weights.mean()
    )


    # -------------------------------------------------------------------------
    # Predicted probabilities must be converted to log-odds.
    #
    # Clipping prevents log(0) or log(1).
    # -------------------------------------------------------------------------

    p = np.clip(
        probability,
        1e-6,
        1 - 1e-6
    )

    logit_p = np.log(
        p / (1 - p)
    )


    # =========================================================================
    # 2. CALIBRATION-IN-THE-LARGE
    #
    # logit[P(Y=1)] = alpha + offset(logit(predicted probability))
    #
    # The coefficient of the predicted log-odds is fixed at 1.
    # Only alpha is estimated.
    #
    # Ideal alpha = 0.
    # =========================================================================

    citl_design = np.ones(
        (len(y), 1),
        dtype=float
    )

    citl_model = sm.GLM(
        y,
        citl_design,
        family=sm.families.Binomial(),
        offset=logit_p,
        freq_weights=calibration_weights
    ).fit()

    citl = float(
        citl_model.params[0]
    )


    # =========================================================================
    # 3. JOINT RECALIBRATION MODEL
    #
    # logit[P(Y=1)] =
    #     alpha + beta * logit(predicted probability)
    #
    # beta = calibration slope.
    #
    # alpha here is the JOINT RECALIBRATION INTERCEPT, not CITL.
    # =========================================================================

    joint_design = pd.DataFrame({
        "constant":
            np.ones(
                len(y),
                dtype=float
            ),

        "logit_prediction":
            logit_p
    })


    joint_model = sm.GLM(
        y,
        joint_design,
        family=sm.families.Binomial(),
        freq_weights=calibration_weights
    ).fit()


    joint_intercept = float(
        joint_model.params[
            "constant"
        ]
    )

    calibration_slope = float(
        joint_model.params[
            "logit_prediction"
        ]
    )


    # =========================================================================
    # 4. DESCRIPTIVE CALIBRATION QUANTITIES
    # =========================================================================

    observed_weighted_prevalence = np.average(
        y,
        weights=weights
    )

    mean_predicted_probability = np.average(
        probability,
        weights=weights
    )


    return {
        "Observed_weighted_prevalence":
            observed_weighted_prevalence,

        "Mean_predicted_probability":
            mean_predicted_probability,

        "CITL":
            citl,

        "Joint_recalibration_intercept":
            joint_intercept,

        "Calibration_slope":
            calibration_slope
    }


# =============================================================================
# 5. APPLY TO THE THREE LOCKED MODELS
# =============================================================================

calibration_rows = []


for model_name, probability_column in (
    heldout_probability_columns.items()
):

    result = weighted_calibration_summary(
        y=y_test_eval,

        probability=(
            test_predictions_locked[
                probability_column
            ].to_numpy()
        ),

        weights=w_test_eval
    )


    calibration_rows.append({
        "Model":
            model_name,

        **result
    })


heldout_calibration = pd.DataFrame(
    calibration_rows
)


# =============================================================================
# 6. INTEGRITY CHECKS
# =============================================================================

assert len(
    heldout_calibration
) == 3

assert np.isfinite(
    heldout_calibration[
        [
            "Observed_weighted_prevalence",
            "Mean_predicted_probability",
            "CITL",
            "Joint_recalibration_intercept",
            "Calibration_slope"
        ]
    ].to_numpy()
).all()


assert np.allclose(
    heldout_calibration[
        "Observed_weighted_prevalence"
    ],
    test_weighted_prevalence
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 124)
print("FINAL HELD-OUT NUMERICAL CALIBRATION")
print("=" * 124)

print(
    heldout_calibration
    .to_string(
        index=False,
        formatters={
            "Observed_weighted_prevalence":
                "{:.4f}".format,

            "Mean_predicted_probability":
                "{:.4f}".format,

            "CITL":
                "{:.4f}".format,

            "Joint_recalibration_intercept":
                "{:.4f}".format,

            "Calibration_slope":
                "{:.4f}".format
        }
    )
)


print("\nIdeal values")
print("------------")
print("CITL: 0")
print("Calibration slope: 1")


print("\nInterpretation")
print("--------------")
print(
    "CITL > 0:",
    "overall underprediction"
)
print(
    "CITL < 0:",
    "overall overprediction"
)
print(
    "Slope < 1:",
    "predictions generally too extreme"
)
print(
    "Slope > 1:",
    "predictions generally not sufficiently dispersed"
)


print("\nGovernance")
print("----------")
print(
    "Joint recalibration intercept labelled as CITL:",
    "No"
)
print(
    "Models recalibrated or updated:",
    "No"
)
print(
    "Test predictions changed:",
    "No"
)

print(
    "\nSTATUS: HELD-OUT NUMERICAL "
    "CALIBRATION ASSESSED"
)

### Calibration summary

All three models showed some degree of miscalibration on the held-out test set.

The observed weighted ANC8 prevalence was 19.23%, while mean predicted
probabilities ranged from 15.76% to 15.98%, indicating overall underprediction.

Logistic regression had a calibration intercept close to zero but a slope below
one (0.79), suggesting somewhat over-dispersed predictions. Random Forest had a
positive intercept (0.43) and slope slightly above one (1.11), while XGBoost
showed intermediate calibration with intercept 0.10 and slope 0.88.

No recalibration was performed using the test set.

### 2.22.1 Survey-weighted calibration plot

Numerical calibration measures are complemented by a graphical assessment.

For each model, held-out observations are ordered by their predicted ANC8
probability and divided into approximately ten groups containing similar total
DHS survey weight.

This produces population-weighted risk groups rather than groups containing
exactly the same unweighted number of women.

Within each group:

- the survey-weighted mean predicted probability is calculated;
- the survey-weighted observed ANC8 proportion is calculated.

A perfectly calibrated model would lie on the 45-degree reference line where
the observed probability equals the predicted probability.

The grouped curves are descriptive calibration diagnostics. They do not
recalibrate or modify the prediction models.

In [ ]:
# =============================================================================
# 2.22.1 SURVEY-WEIGHTED GROUPED CALIBRATION PLOT
#
# Each model is divided into approximately 10 groups with similar cumulative
# survey weight.
#
# Within each group:
#   x = weighted mean predicted probability
#   y = weighted observed ANC8 proportion
#
# No model updating or recalibration occurs.
# =============================================================================

import matplotlib.pyplot as plt


# =============================================================================
# 1. FUNCTION TO ASSIGN SURVEY-WEIGHTED RISK GROUPS
# =============================================================================

def weighted_risk_groups(
    probability,
    weights,
    n_groups=10
):
    """
    Assign observations to approximately equal-survey-weight groups after
    ordering them by predicted probability.

    Groups run from:
        1 = lowest predicted ANC8 probability
        ...
        10 = highest predicted ANC8 probability
    """

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    assert len(
        probability
    ) == len(
        weights
    )

    assert (
        weights > 0
    ).all()


    # Stable ordering by predicted probability.
    order = np.argsort(
        probability,
        kind="mergesort"
    )


    ordered_weights = (
        weights[
            order
        ]
    )


    # Midpoint of each observation's contribution to cumulative survey weight.
    cumulative_weight_midpoint = (
        np.cumsum(
            ordered_weights
        )
        - 0.5 * ordered_weights
    )


    cumulative_fraction = (
        cumulative_weight_midpoint
        / ordered_weights.sum()
    )


    # Convert cumulative survey-weight position into groups 1,...,n_groups.
    ordered_group = np.floor(
        cumulative_fraction
        * n_groups
    ).astype(int) + 1


    ordered_group = np.clip(
        ordered_group,
        1,
        n_groups
    )


    # Return groups in the original row order.
    group = np.empty(
        len(probability),
        dtype=int
    )

    group[
        order
    ] = ordered_group


    return group


# =============================================================================
# 2. BUILD MODEL-SPECIFIC CALIBRATION TABLE
# =============================================================================

calibration_plot_rows = []


for model_name, probability_column in (
    heldout_probability_columns.items()
):

    probability = (
        test_predictions_locked[
            probability_column
        ]
        .astype(float)
        .to_numpy()
    )


    risk_group = weighted_risk_groups(
        probability=probability,
        weights=w_test_eval,
        n_groups=10
    )


    for group_number in range(
        1,
        11
    ):

        group_mask = (
            risk_group
            == group_number
        )


        assert group_mask.sum() > 0


        group_weights = (
            w_test_eval[
                group_mask
            ]
        )


        weighted_mean_prediction = np.average(
            probability[
                group_mask
            ],
            weights=group_weights
        )


        weighted_observed_probability = np.average(
            y_test_eval[
                group_mask
            ],
            weights=group_weights
        )


        calibration_plot_rows.append({
            "Model":
                model_name,

            "Risk_group":
                group_number,

            "Unweighted_n":
                int(
                    group_mask.sum()
                ),

            "Survey_weight_sum":
                float(
                    group_weights.sum()
                ),

            "Mean_predicted_probability":
                weighted_mean_prediction,

            "Observed_probability":
                weighted_observed_probability
        })


heldout_calibration_groups = pd.DataFrame(
    calibration_plot_rows
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert len(
    heldout_calibration_groups
) == 30

assert (
    heldout_calibration_groups
    .groupby(
        "Model"
    )
    .size()
    .eq(10)
    .all()
)

assert np.isfinite(
    heldout_calibration_groups[
        [
            "Mean_predicted_probability",
            "Observed_probability"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 4. DISPLAY GROUPED CALIBRATION VALUES
# =============================================================================

print("=" * 104)
print("SURVEY-WEIGHTED GROUPED CALIBRATION TABLE")
print("=" * 104)

print(
    heldout_calibration_groups
    .to_string(
        index=False,
        formatters={
            "Survey_weight_sum":
                "{:.2f}".format,

            "Mean_predicted_probability":
                "{:.4f}".format,

            "Observed_probability":
                "{:.4f}".format
        }
    )
)


# =============================================================================
# 5. CALIBRATION PLOT
#
# One common plot is used so the three models can be compared directly.
#
# No manually specified colors are used.
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)


# Perfect-calibration reference.
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


for model_name in (
    heldout_probability_columns.keys()
):

    model_plot = (
        heldout_calibration_groups[
            heldout_calibration_groups[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "Mean_predicted_probability"
        )
    )


    ax.plot(
        model_plot[
            "Mean_predicted_probability"
        ],
        model_plot[
            "Observed_probability"
        ],
        marker="o",
        label=model_name
    )


ax.set_xlabel(
    "Survey-weighted mean predicted ANC8 probability"
)

ax.set_ylabel(
    "Survey-weighted observed ANC8 proportion"
)

ax.set_title(
    "Held-out calibration of ANC8 prediction models"
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1
)

ax.legend()

plt.tight_layout()
plt.show()


print(
    "\nSTATUS: SURVEY-WEIGHTED "
    "CALIBRATION CURVES GENERATED"
)

In [ ]:
# =============================================================================
# 2.22.1B SURVEY-WEIGHTED SMOOTH CALIBRATION CURVES
#
# Publication-oriented calibration plot:
#
#   dashed diagonal = perfect calibration reference
#   points          = weighted grouped observed probabilities
#   smooth curves   = weighted flexible calibration functions
#
# The smooth curves are estimated using a binomial GLM with a cubic spline
# function of the model's predicted log-odds.
#
# No recalibration is applied to the prediction models.
# =============================================================================

import matplotlib.pyplot as plt
import statsmodels.api as sm

from patsy import dmatrix


# =============================================================================
# 1. FUNCTION TO ESTIMATE A WEIGHTED SMOOTH CALIBRATION CURVE
# =============================================================================

def weighted_smooth_calibration_curve(
    y,
    probability,
    weights,
    spline_df=4,
    grid_size=200
):
    """
    Estimate a flexible survey-weighted calibration curve.

    The observed binary outcome is regressed on a cubic spline of the
    model-predicted log-odds using a weighted binomial GLM.

    Parameters
    ----------
    y : array-like
        Observed binary outcome.

    probability : array-like
        Locked predicted probabilities.

    weights : array-like
        DHS survey weights.

    spline_df : int
        Degrees of freedom controlling spline flexibility.
        A modest value of 4 is used to avoid an excessively wiggly curve.

    grid_size : int
        Number of points used to draw the smooth calibration curve.

    Returns
    -------
    prediction_grid
        Predicted-probability values on the x-axis.

    calibrated_observed_grid
        Smoothed estimated observed ANC8 probability.
    """

    y = np.asarray(
        y,
        dtype=float
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Avoid infinite log-odds.
    # -------------------------------------------------------------------------

    p = np.clip(
        probability,
        1e-6,
        1 - 1e-6
    )

    logit_p = np.log(
        p / (1 - p)
    )


    # -------------------------------------------------------------------------
    # Normalize weights to mean 1 for numerical stability.
    # Relative DHS weighting is preserved.
    # -------------------------------------------------------------------------

    calibration_weights = (
        weights
        / weights.mean()
    )


    # -------------------------------------------------------------------------
    # Cubic spline basis of predicted log-odds.
    #
    # include_intercept=False because statsmodels adds the intercept.
    # -------------------------------------------------------------------------

    spline_basis = dmatrix(
        f"bs(x, df={spline_df}, degree=3, include_intercept=False)",
        {"x": logit_p},
        return_type="dataframe"
    )


    # -------------------------------------------------------------------------
    # Weighted binomial calibration model.
    # -------------------------------------------------------------------------

    smooth_model = sm.GLM(
        y,
        spline_basis,
        family=sm.families.Binomial(),
        freq_weights=calibration_weights
    ).fit()


    # -------------------------------------------------------------------------
    # Prediction grid.
    #
    # Use the range actually represented by this model's held-out predictions.
    # We do not extrapolate calibration to probabilities never produced by the
    # model.
    # -------------------------------------------------------------------------

    prediction_grid = np.linspace(
        p.min(),
        p.max(),
        grid_size
    )

    grid_logit = np.log(
        prediction_grid
        / (1 - prediction_grid)
    )


    # IMPORTANT:
    # Use the same spline specification for the prediction grid.
    grid_basis = dmatrix(
        f"bs(x, df={spline_df}, degree=3, include_intercept=False)",
        {"x": grid_logit},
        return_type="dataframe"
    )


    calibrated_observed_grid = (
        smooth_model.predict(
            grid_basis
        )
    )


    return (
        prediction_grid,
        np.asarray(
            calibrated_observed_grid
        )
    )


# =============================================================================
# 2. DRAW THE CALIBRATION FIGURE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)


# -----------------------------------------------------------------------------
# Perfect-calibration reference.
#
# THIS LINE SHOULD REMAIN STRAIGHT.
# -----------------------------------------------------------------------------

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


# =============================================================================
# 3. MODEL-SPECIFIC SMOOTH CURVES + GROUPED POINTS
# =============================================================================

for model_name, probability_column in (
    heldout_probability_columns.items()
):

    probability = (
        test_predictions_locked[
            probability_column
        ]
        .astype(float)
        .to_numpy()
    )


    # -------------------------------------------------------------------------
    # Smooth survey-weighted calibration curve.
    # -------------------------------------------------------------------------

    prediction_grid, observed_grid = (
        weighted_smooth_calibration_curve(
            y=y_test_eval,
            probability=probability,
            weights=w_test_eval,
            spline_df=4
        )
    )


    ax.plot(
        prediction_grid,
        observed_grid,
        linewidth=2,
        label=model_name
    )


    # -------------------------------------------------------------------------
    # Add the previously calculated survey-weighted grouped calibration points.
    #
    # These points show the empirical grouped data underlying the smooth curve.
    # -------------------------------------------------------------------------

    model_groups = (
        heldout_calibration_groups[
            heldout_calibration_groups[
                "Model"
            ] == model_name
        ]
    )


    ax.scatter(
        model_groups[
            "Mean_predicted_probability"
        ],
        model_groups[
            "Observed_probability"
        ],
        s=35
    )


# =============================================================================
# 4. PRESENTATION
# =============================================================================

ax.set_xlabel(
    "Predicted probability of ANC8+"
)

ax.set_ylabel(
    "Observed probability of ANC8+"
)

ax.set_title(
    "Held-out calibration of ANC8 prediction models"
)


# Zoom to the clinically/data-relevant probability range rather than forcing
# most of the figure to display an empty 0-1 region.
max_display = min(
    1.0,
    max(
        test_predictions_locked[
            probability_cols
        ].max()
    ) + 0.05
)

ax.set_xlim(
    0,
    max_display
)

ax.set_ylim(
    0,
    max_display
)

ax.legend()

plt.tight_layout()
plt.show()


print(
    "STATUS: WEIGHTED SMOOTH CALIBRATION "
    "CURVES GENERATED"
)

### Calibration plot interpretation

The three models showed broadly reasonable calibration across much of the
predicted-risk range, particularly at lower predicted probabilities.

However, several mid-range risk groups lay above the line of perfect
calibration, indicating underprediction of ANC8 attainment in those groups.

Calibration became more variable at higher predicted probabilities, where fewer
observations were available.

No model showed uniformly superior calibration across the full probability
range, although logistic regression and XGBoost generally tracked the reference
line reasonably closely.

## 2.23 Development-derived classification thresholds

The primary model outputs are predicted probabilities of ANC8 attainment.

Binary classification requires an operating threshold, but no externally
established clinical or policy threshold exists for this prediction task.
Therefore, threshold-based classification measures are secondary to the
probability-based measures reported earlier.

For descriptive classification performance, a separate threshold is selected
for each algorithm using its already-locked development out-of-fold
predictions.

The operating point is chosen by maximizing the survey-weighted Youden index:

**Youden J = sensitivity + specificity - 1**

Threshold selection follows these principles:

- only development out-of-fold predictions are used;
- DHS survey weights are used when estimating sensitivity and specificity;
- each algorithm receives its own threshold;
- the already-generated held-out test predictions play no role in threshold
  selection;
- no threshold is adjusted after inspecting held-out performance;
- the selected values are descriptive operating points and are not interpreted
  as clinical or policy cut-offs.

The threshold rule was defined before held-out classification performance is
examined. The resulting thresholds are locked and subsequently applied
unchanged to the held-out probabilities.

In [ ]:
# =============================================================================
# 2.23 SELECT MODEL-SPECIFIC THRESHOLDS FROM LOCKED DEVELOPMENT OOF PREDICTIONS
#
# Youden J = sensitivity + specificity - 1
#
# IMPORTANT
# ---------
# - OOF probabilities are NOT regenerated.
# - Uses the locked development OOF vectors from Sections 2.15-2.17.
# - Uses survey weights for threshold evaluation.
# - Does not use held-out test outcomes or probabilities to select thresholds.
# =============================================================================

from sklearn.metrics import roc_curve


# =============================================================================
# 1. LOCKED DEVELOPMENT OOF PREDICTIONS
# =============================================================================

development_oof_probabilities = {
    "Logistic regression":
        oof_logistic,

    "Random Forest":
        oof_rf,

    "XGBoost":
        oof_xgb
}


# =============================================================================
# 2. INTEGRITY CHECKS
# =============================================================================

for model_name, probability in (
    development_oof_probabilities.items()
):

    probability = np.asarray(
        probability,
        dtype=float
    )

    assert len(
        probability
    ) == len(
        y_dev
    )

    assert np.isfinite(
        probability
    ).all()

    assert (
        (probability >= 0)
        & (probability <= 1)
    ).all()


# =============================================================================
# 3. SURVEY-WEIGHTED YOUDEN THRESHOLD FUNCTION
# =============================================================================

def weighted_youden_threshold(
    y,
    probability,
    weights
):
    """
    Select the probability threshold maximizing the survey-weighted Youden J.

    Youden J = sensitivity + specificity - 1
             = true-positive rate - false-positive rate

    Returns
    -------
    threshold
        Selected operating threshold.

    sensitivity
        Survey-weighted sensitivity at the selected threshold.

    specificity
        Survey-weighted specificity at the selected threshold.

    youden_j
        Survey-weighted Youden index at the selected threshold.
    """

    y = np.asarray(
        y,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Weighted ROC operating points.
    # -------------------------------------------------------------------------

    fpr, tpr, thresholds = roc_curve(
        y,
        probability,
        sample_weight=weights
    )


    specificity = (
        1.0
        - fpr
    )

    youden_j = (
        tpr
        + specificity
        - 1.0
    )


    # -------------------------------------------------------------------------
    # Exclude the non-finite threshold sometimes returned as the first
    # ROC threshold.
    # -------------------------------------------------------------------------

    valid = np.isfinite(
        thresholds
    )

    candidate_table = pd.DataFrame({
        "Threshold":
            thresholds[
                valid
            ],

        "Sensitivity":
            tpr[
                valid
            ],

        "Specificity":
            specificity[
                valid
            ],

        "Youden_J":
            youden_j[
                valid
            ]
    })


    # -------------------------------------------------------------------------
    # Deterministic selection rule:
    #
    # 1. highest Youden J
    # 2. if exactly tied, higher sensitivity
    # 3. if still tied, higher specificity
    # 4. if still tied, higher threshold
    #
    # The tie rules only resolve exact numerical equality in J.
    # -------------------------------------------------------------------------

    candidate_table = (
        candidate_table
        .sort_values(
            by=[
                "Youden_J",
                "Sensitivity",
                "Specificity",
                "Threshold"
            ],
            ascending=[
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    selected = (
        candidate_table
        .iloc[0]
    )


    return {
        "Threshold":
            float(
                selected[
                    "Threshold"
                ]
            ),

        "Development_sensitivity":
            float(
                selected[
                    "Sensitivity"
                ]
            ),

        "Development_specificity":
            float(
                selected[
                    "Specificity"
                ]
            ),

        "Development_Youden_J":
            float(
                selected[
                    "Youden_J"
                ]
            )
    }


# =============================================================================
# 4. SELECT THRESHOLD FOR EACH LOCKED ALGORITHM
# =============================================================================

threshold_rows = []


for model_name, probability in (
    development_oof_probabilities.items()
):

    result = weighted_youden_threshold(
        y=y_dev,
        probability=probability,
        weights=survey_weight_dev
    )


    threshold_rows.append({
        "Model":
            model_name,

        **result
    })


development_thresholds = pd.DataFrame(
    threshold_rows
)


# =============================================================================
# 5. CANONICAL LOCKED THRESHOLD DICTIONARY
# =============================================================================

LOCKED_THRESHOLDS = {
    row["Model"]:
        float(
            row["Threshold"]
        )

    for _, row in (
        development_thresholds
        .iterrows()
    )
}


# =============================================================================
# 6. INTEGRITY
# =============================================================================

assert set(
    LOCKED_THRESHOLDS.keys()
) == {
    "Logistic regression",
    "Random Forest",
    "XGBoost"
}

assert development_thresholds[
    "Threshold"
].between(
    0,
    1
).all()

assert development_thresholds[
    "Development_sensitivity"
].between(
    0,
    1
).all()

assert development_thresholds[
    "Development_specificity"
].between(
    0,
    1
).all()


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 104)
print("DEVELOPMENT-DERIVED SURVEY-WEIGHTED YOUDEN THRESHOLDS")
print("=" * 104)

print(
    development_thresholds
    .to_string(
        index=False,
        formatters={
            "Threshold":
                "{:.4f}".format,

            "Development_sensitivity":
                "{:.3f}".format,

            "Development_specificity":
                "{:.3f}".format,

            "Development_Youden_J":
                "{:.3f}".format
        }
    )
)


print("\nGovernance")
print("----------")

print(
    "Threshold source:",
    "locked development OOF predictions"
)

print(
    "OOF predictions regenerated:",
    "No"
)

print(
    "Survey weights used:",
    "Yes"
)

print(
    "Held-out test data used for threshold selection:",
    "No"
)

print(
    "Thresholds interpreted as clinical/policy cut-offs:",
    "No"
)

print(
    "\nSTATUS: DEVELOPMENT-DERIVED "
    "CLASSIFICATION THRESHOLDS LOCKED"
)

## 2.24 Final threshold-based performance on the held-out test set

The model-specific thresholds selected exclusively from development
out-of-fold predictions are now applied unchanged to the locked held-out test
probabilities.

No threshold is selected, adjusted, or optimized using held-out test outcomes.

For each algorithm, survey-weighted classification performance is summarized
using:

- **Sensitivity:** proportion of ANC8+ women classified as ANC8+;
- **Specificity:** proportion of ANC8<8 women classified as ANC8<8;
- **Positive predictive value (PPV / precision):** proportion of women
  classified ANC8+ who actually attained ANC8;
- **Negative predictive value (NPV):** proportion classified ANC8<8 who
  actually did not attain ANC8;
- **F1 score:** harmonic mean of sensitivity and PPV.

The survey-weighted proportion classified as ANC8+ is also reported to describe
the practical operating behavior of each threshold.

These threshold-dependent measures are secondary to the probability-based and
calibration measures because they depend on the selected operating point.

The development-derived thresholds are descriptive model operating points, not
clinical or policy cut-offs.

In [ ]:
# =============================================================================
# 2.24 FINAL HELD-OUT THRESHOLD-BASED PERFORMANCE
#
# IMPORTANT
# ---------
# - Uses LOCKED development-derived thresholds.
# - Uses LOCKED held-out probabilities.
# - Thresholds are NOT re-estimated from test outcomes.
# - No model fitting occurs.
# =============================================================================

from sklearn.metrics import confusion_matrix


# =============================================================================
# 1. SURVEY-WEIGHTED THRESHOLD METRICS
# =============================================================================

def weighted_threshold_metrics(
    y,
    probability,
    weights,
    threshold
):
    """
    Calculate survey-weighted binary-classification performance at one
    prespecified probability threshold.

    Returns:
        sensitivity
        specificity
        PPV
        NPV
        F1
        weighted predicted-positive proportion
    """

    y = np.asarray(
        y,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Convert probability to binary classification using the LOCKED threshold.
    # -------------------------------------------------------------------------

    predicted_class = (
        probability >= threshold
    ).astype(int)


    # -------------------------------------------------------------------------
    # Weighted confusion matrix.
    #
    # Rows    = observed class
    # Columns = predicted class
    # -------------------------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y,
        predicted_class,
        sample_weight=weights,
        labels=[0, 1]
    ).ravel()


    # -------------------------------------------------------------------------
    # Survey-weighted classification metrics.
    # -------------------------------------------------------------------------

    sensitivity = (
        tp / (tp + fn)
    )

    specificity = (
        tn / (tn + fp)
    )

    ppv = (
        tp / (tp + fp)
    )

    npv = (
        tn / (tn + fn)
    )

    f1 = (
        2 * ppv * sensitivity
        / (ppv + sensitivity)
    )


    # Survey-weighted proportion of women classified as ANC8+.
    predicted_positive_rate = np.average(
        predicted_class,
        weights=weights
    )


    return {
        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "PPV":
            ppv,

        "NPV":
            npv,

        "F1":
            f1,

        "Predicted_positive_rate":
            predicted_positive_rate
    }


# =============================================================================
# 2. APPLY EACH LOCKED THRESHOLD TO THE HELD-OUT TEST PREDICTIONS
# =============================================================================

threshold_test_rows = []


for model_name, probability_column in (
    heldout_probability_columns.items()
):

    threshold = (
        LOCKED_THRESHOLDS[
            model_name
        ]
    )

    probability = (
        test_predictions_locked[
            probability_column
        ]
        .astype(float)
        .to_numpy()
    )


    result = weighted_threshold_metrics(
        y=y_test_eval,
        probability=probability,
        weights=w_test_eval,
        threshold=threshold
    )


    threshold_test_rows.append({
        "Model":
            model_name,

        "Threshold":
            threshold,

        **result
    })


heldout_threshold_performance = pd.DataFrame(
    threshold_test_rows
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert len(
    heldout_threshold_performance
) == 3

assert set(
    heldout_threshold_performance[
        "Model"
    ]
) == set(
    LOCKED_THRESHOLDS.keys()
)

for column in [
    "Threshold",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
    "Predicted_positive_rate"
]:

    assert heldout_threshold_performance[
        column
    ].between(
        0,
        1
    ).all()


# =============================================================================
# 4. METRIC-SPECIFIC LEADERS
#
# These are descriptive only.
# =============================================================================

best_sensitivity_model = (
    heldout_threshold_performance
    .loc[
        heldout_threshold_performance[
            "Sensitivity"
        ].idxmax(),
        "Model"
    ]
)

best_specificity_model = (
    heldout_threshold_performance
    .loc[
        heldout_threshold_performance[
            "Specificity"
        ].idxmax(),
        "Model"
    ]
)

best_ppv_model = (
    heldout_threshold_performance
    .loc[
        heldout_threshold_performance[
            "PPV"
        ].idxmax(),
        "Model"
    ]
)

best_npv_model = (
    heldout_threshold_performance
    .loc[
        heldout_threshold_performance[
            "NPV"
        ].idxmax(),
        "Model"
    ]
)

best_f1_model = (
    heldout_threshold_performance
    .loc[
        heldout_threshold_performance[
            "F1"
        ].idxmax(),
        "Model"
    ]
)


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 112)
print("FINAL HELD-OUT THRESHOLD-BASED PERFORMANCE")
print("=" * 112)

print(
    heldout_threshold_performance
    .to_string(
        index=False,
        formatters={
            "Threshold":
                "{:.4f}".format,

            "Sensitivity":
                "{:.3f}".format,

            "Specificity":
                "{:.3f}".format,

            "PPV":
                "{:.3f}".format,

            "NPV":
                "{:.3f}".format,

            "F1":
                "{:.3f}".format,

            "Predicted_positive_rate":
                "{:.3f}".format
        }
    )
)


print("\nMetric-specific leaders")
print("-----------------------")

print(
    "Highest sensitivity:",
    best_sensitivity_model
)

print(
    "Highest specificity:",
    best_specificity_model
)

print(
    "Highest PPV:",
    best_ppv_model
)

print(
    "Highest NPV:",
    best_npv_model
)

print(
    "Highest F1:",
    best_f1_model
)


print("\nGovernance")
print("----------")

print(
    "Threshold source:",
    "development OOF predictions"
)

print(
    "Threshold re-estimated on test set:",
    "No"
)

print(
    "Test predictions regenerated:",
    "No"
)

print(
    "Model fitting performed:",
    "No"
)

print(
    "Thresholds interpreted as clinical/policy cut-offs:",
    "No"
)

print(
    "\nSTATUS: FINAL HELD-OUT "
    "THRESHOLD PERFORMANCE ESTIMATED"
)

## 2.25 Consolidated held-out performance of baseline ANC8 models

The final held-out performance of the three baseline ANC8 prediction models is
summarized in a single table.

The table combines:

- probability-based performance:
  - Average Precision;
  - AUROC;
  - Brier score;
- calibration:
  - calibration-in-the-large (CITL);
  - calibration slope;
- threshold-based classification performance:
  - sensitivity;
  - specificity;
  - positive predictive value;
  - negative predictive value;
  - F1 score.

The model-specific thresholds were selected exclusively from development
out-of-fold predictions and were applied unchanged to the held-out test set.

All held-out performance estimates use DHS survey weights.

No further fitting, recalibration, hyperparameter tuning, or threshold
modification is performed in constructing this summary.

The table is descriptive. No single overall model-selection score was
prespecified across all performance domains.

In [ ]:
# =============================================================================
# 2.25 CONSOLIDATED FINAL HELD-OUT PERFORMANCE
#
# Combines:
#   - probability-based performance
#   - numerical calibration
#   - threshold-based classification performance
#
# No new fitting or prediction occurs here.
# =============================================================================


# =============================================================================
# 1. MERGE THE THREE GOVERNED FINAL RESULT OBJECTS
# =============================================================================

final_baseline_performance = (
    heldout_performance
    .merge(
        heldout_calibration[
            [
                "Model",
                "CITL",
                "Calibration_slope"
            ]
        ],
        on="Model",
        how="left",
        validate="one_to_one"
    )
    .merge(
        heldout_threshold_performance[
            [
                "Model",
                "Threshold",
                "Sensitivity",
                "Specificity",
                "PPV",
                "NPV",
                "F1",
                "Predicted_positive_rate"
            ]
        ],
        on="Model",
        how="left",
        validate="one_to_one"
    )
)


# =============================================================================
# 2. SELECT FINAL DISPLAY COLUMNS
# =============================================================================

final_baseline_performance = (
    final_baseline_performance[
        [
            "Model",
            "Weighted_AP",
            "Weighted_AUROC",
            "Weighted_Brier",
            "CITL",
            "Calibration_slope",
            "Threshold",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1",
            "Predicted_positive_rate"
        ]
    ]
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert len(
    final_baseline_performance
) == 3

assert set(
    final_baseline_performance[
        "Model"
    ]
) == {
    "Logistic regression",
    "Random Forest",
    "XGBoost"
}

assert final_baseline_performance[
    "Model"
].is_unique

assert final_baseline_performance[
    [
        "Weighted_AP",
        "Weighted_AUROC",
        "Weighted_Brier",
        "CITL",
        "Calibration_slope",
        "Threshold",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
        "F1",
        "Predicted_positive_rate"
    ]
].notna().all().all()


# =============================================================================
# 4. DESCRIPTIVE METRIC LEADERS
#
# These labels summarize individual metrics only.
# They do NOT create a post-hoc composite winner score.
# =============================================================================

metric_leaders = {
    "Highest AP":
        final_baseline_performance.loc[
            final_baseline_performance[
                "Weighted_AP"
            ].idxmax(),
            "Model"
        ],

    "Highest AUROC":
        final_baseline_performance.loc[
            final_baseline_performance[
                "Weighted_AUROC"
            ].idxmax(),
            "Model"
        ],

    "Lowest Brier":
        final_baseline_performance.loc[
            final_baseline_performance[
                "Weighted_Brier"
            ].idxmin(),
            "Model"
        ],

    "CITL closest to 0":
        final_baseline_performance.loc[
            final_baseline_performance[
                "CITL"
            ].abs().idxmin(),
            "Model"
        ],

    "Slope closest to 1":
        final_baseline_performance.loc[
            (
                final_baseline_performance[
                    "Calibration_slope"
                ] - 1
            ).abs().idxmin(),
            "Model"
        ],

    "Highest sensitivity":
        final_baseline_performance.loc[
            final_baseline_performance[
                "Sensitivity"
            ].idxmax(),
            "Model"
        ],

    "Highest specificity":
        final_baseline_performance.loc[
            final_baseline_performance[
                "Specificity"
            ].idxmax(),
            "Model"
        ],

    "Highest PPV":
        final_baseline_performance.loc[
            final_baseline_performance[
                "PPV"
            ].idxmax(),
            "Model"
        ],

    "Highest NPV":
        final_baseline_performance.loc[
            final_baseline_performance[
                "NPV"
            ].idxmax(),
            "Model"
        ],

    "Highest F1":
        final_baseline_performance.loc[
            final_baseline_performance[
                "F1"
            ].idxmax(),
            "Model"
        ]
}


# =============================================================================
# 5. DISPLAY FINAL TABLE
# =============================================================================

print("=" * 150)
print("CONSOLIDATED FINAL HELD-OUT PERFORMANCE: BASELINE ANC8 MODELS")
print("=" * 150)

print(
    final_baseline_performance
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format,

            "CITL":
                "{:.4f}".format,

            "Calibration_slope":
                "{:.4f}".format,

            "Threshold":
                "{:.4f}".format,

            "Sensitivity":
                "{:.3f}".format,

            "Specificity":
                "{:.3f}".format,

            "PPV":
                "{:.3f}".format,

            "NPV":
                "{:.3f}".format,

            "F1":
                "{:.3f}".format,

            "Predicted_positive_rate":
                "{:.3f}".format
        }
    )
)


print("\nMetric-specific leaders")
print("-----------------------")

for metric_name, model_name in (
    metric_leaders.items()
):

    print(
        f"{metric_name}: {model_name}"
    )


print("\nGovernance")
print("----------")

print(
    "Composite overall winner score created:",
    "No"
)

print(
    "Further fitting performed:",
    "No"
)

print(
    "Further recalibration performed:",
    "No"
)

print(
    "Thresholds modified using test data:",
    "No"
)

print(
    "\nSTATUS: CONSOLIDATED FINAL "
    "BASELINE PERFORMANCE TABLE COMPLETE"
)

XGBoost showed the strongest overall held-out performance, with the highest Average Precision, lowest Brier score, smallest calibration-in-the-large deviation, highest sensitivity and NPV, while maintaining AUROC comparable to logistic regression.

### 2.25.1 Held-out survey-design audit for uncertainty estimation

Before estimating uncertainty around held-out model performance, the survey
design structure of the test population is examined.

The final test set contains women clustered within primary sampling units
(PSUs) and PSUs nested within survey strata.

A cluster bootstrap should preserve PSU-level dependence. Where feasible,
resampling PSUs within strata also preserves the stratified survey structure.

However, stratified PSU bootstrap resampling requires sufficient PSU
representation within each held-out stratum. A stratum represented by only one
test PSU cannot contribute between-PSU resampling variation within that stratum.

Therefore, before selecting the bootstrap procedure, this section audits:

- the number of held-out PSUs per survey stratum;
- the number of women per stratum;
- whether any held-out strata contain only one PSU;
- the minimum and maximum number of PSUs represented per stratum.

No bootstrap is performed in this section and no model is refitted.

In [ ]:
# =============================================================================
# 2.25.1 HELD-OUT SURVEY-DESIGN AUDIT FOR UNCERTAINTY ESTIMATION
#
# PURPOSE
# -------
# Decide whether PSU resampling can be performed within survey strata or whether
# singleton held-out strata require an alternative cluster-bootstrap strategy.
#
# No bootstrap is performed yet.
# =============================================================================


# =============================================================================
# 1. BUILD TEST DESIGN AUDIT TABLE
# =============================================================================

test_design_audit = (
    test_predictions_locked
    .groupby(
        "v022",
        as_index=False
    )
    .agg(
        Women=(
            "caseid",
            "size"
        ),

        PSUs=(
            "v021",
            "nunique"
        ),

        Survey_weight_sum=(
            "survey_weight",
            "sum"
        )
    )
    .sort_values(
        "v022"
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 2. IDENTIFY SINGLETON AND LOW-PSU STRATA
# =============================================================================

singleton_test_strata = (
    test_design_audit[
        test_design_audit[
            "PSUs"
        ] == 1
    ]
    .copy()
)

two_psu_test_strata = (
    test_design_audit[
        test_design_audit[
            "PSUs"
        ] == 2
    ]
    .copy()
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert test_design_audit[
    "Women"
].sum() == len(
    test_predictions_locked
)

assert test_design_audit[
    "PSUs"
].sum() == groups_test.nunique()

assert len(
    test_design_audit
) == strata_test.nunique() == 27


# =============================================================================
# 4. DISPLAY
# =============================================================================

print("=" * 104)
print("HELD-OUT SURVEY-DESIGN AUDIT FOR UNCERTAINTY ESTIMATION")
print("=" * 104)

print(
    test_design_audit.to_string(
        index=False,
        formatters={
            "Survey_weight_sum":
                "{:.2f}".format
        }
    )
)


print("\nDesign summary")
print("--------------")

print(
    "Held-out women:",
    len(test_predictions_locked)
)

print(
    "Held-out PSUs:",
    groups_test.nunique()
)

print(
    "Held-out strata:",
    strata_test.nunique()
)

print(
    "Minimum PSUs per stratum:",
    int(
        test_design_audit[
            "PSUs"
        ].min()
    )
)

print(
    "Maximum PSUs per stratum:",
    int(
        test_design_audit[
            "PSUs"
        ].max()
    )
)

print(
    "Singleton strata:",
    len(
        singleton_test_strata
    )
)

print(
    "Strata with exactly two PSUs:",
    len(
        two_psu_test_strata
    )
)


if len(
    singleton_test_strata
) > 0:

    print(
        "\nSingleton stratum IDs:",
        singleton_test_strata[
            "v022"
        ].tolist()
    )


print("\nGovernance")
print("----------")
print(
    "Bootstrap performed here:",
    "No"
)
print(
    "Models refitted:",
    "No"
)

print(
    "\nSTATUS: HELD-OUT DESIGN "
    "STRUCTURE AUDITED"
)

### 2.25.2 Stratified PSU bootstrap uncertainty for held-out probability performance

Uncertainty around held-out predictive performance is estimated using a
stratified cluster bootstrap.

The held-out population contains 159 PSUs distributed across all 27 survey
strata, with at least two PSUs represented in every stratum. Therefore, PSUs
can be resampled with replacement **within their original survey strata**.

For each bootstrap replicate:

1. within each survey stratum, the same number of PSUs as originally observed
   in that held-out stratum is sampled with replacement;
2. all women belonging to each selected PSU are retained together;
3. the original DHS survey weights are retained;
4. the already-locked predictions from Logistic regression, Random Forest and
   XGBoost are evaluated on exactly the same resampled population.

Models are not refitted and predictions are not regenerated.

The procedure estimates empirical 95% percentile confidence intervals for the
three primary probability-based performance measures:

- survey-weighted Average Precision;
- survey-weighted AUROC;
- survey-weighted Brier score.

Because the same resampled PSUs are used for every algorithm, the bootstrap
replicates can subsequently be used for paired comparisons between models.

In [ ]:
# =============================================================================
# 2.25.2 STRATIFIED PSU BOOTSTRAP OF HELD-OUT PROBABILITY PERFORMANCE
#
# Resampling unit:
#     PSU
#
# Resampling structure:
#     PSUs sampled WITH REPLACEMENT WITHIN each survey stratum.
#
# Important:
#     - models are NOT refitted;
#     - predictions are NOT regenerated;
#     - the same bootstrap sample is used for all three algorithms;
#     - original DHS survey weights remain attached to women;
#     - repeated selection of a PSU gives that PSU corresponding multiplicity.
# =============================================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)


# =============================================================================
# 1. BOOTSTRAP SETTINGS
# =============================================================================

N_BOOT_BASELINE = 2000
BOOTSTRAP_SEED_BASELINE = 2026

rng_baseline = np.random.default_rng(
    BOOTSTRAP_SEED_BASELINE
)


# =============================================================================
# 2. BUILD ONE GOVERNED HELD-OUT EVALUATION DATAFRAME
#
# Everything comes from the already-locked test prediction object.
# =============================================================================

baseline_boot_df = (
    test_predictions_locked[
        [
            "caseid",
            "v021",
            "v022",
            "survey_weight",
            "anc8plus_observed",
            "prob_logistic",
            "prob_random_forest",
            "prob_xgboost"
        ]
    ]
    .copy()
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert len(
    baseline_boot_df
) == 1375

assert baseline_boot_df[
    "caseid"
].is_unique

assert baseline_boot_df[
    "anc8plus_observed"
].isin(
    [0, 1]
).all()

assert (
    baseline_boot_df[
        "survey_weight"
    ] > 0
).all()

assert baseline_boot_df[
    "v022"
].nunique() == 27

assert baseline_boot_df[
    "v021"
].nunique() == 159


# =============================================================================
# 4. CREATE STRATUM-SPECIFIC PSU LISTS
#
# We explicitly treat PSU membership within each stratum.
# =============================================================================

stratum_psus = {}

for stratum, stratum_df in (
    baseline_boot_df
    .groupby(
        "v022",
        sort=True
    )
):

    psus = (
        stratum_df[
            "v021"
        ]
        .drop_duplicates()
        .to_numpy()
    )

    # Our design audit established at least two PSUs per held-out stratum.
    assert len(psus) >= 2

    stratum_psus[
        stratum
    ] = psus


# =============================================================================
# 5. MAP EACH STRATUM-PSU COMBINATION TO ITS WOMEN
#
# Using a (stratum, PSU) key avoids assuming that PSU identifiers alone must
# be globally unique.
# =============================================================================

stratum_psu_rows = {}

for (
    stratum,
    psu
), group in baseline_boot_df.groupby(
    [
        "v022",
        "v021"
    ],
    sort=False
):

    stratum_psu_rows[
        (
            stratum,
            psu
        )
    ] = group.index.to_numpy()


# =============================================================================
# 6. FUNCTION FOR THE THREE PRIMARY WEIGHTED PERFORMANCE METRICS
# =============================================================================

def bootstrap_probability_metrics(
    y,
    probability,
    weights
):
    """
    Calculate survey-weighted probability-based prediction metrics.
    """

    y = np.asarray(
        y,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )


    weighted_ap = average_precision_score(
        y,
        probability,
        sample_weight=weights
    )


    weighted_auroc = roc_auc_score(
        y,
        probability,
        sample_weight=weights
    )


    weighted_brier = brier_score_loss(
        y,
        probability,
        sample_weight=weights
    )


    return {
        "AP":
            weighted_ap,

        "AUROC":
            weighted_auroc,

        "Brier":
            weighted_brier
    }


# =============================================================================
# 7. LOCKED MODEL-PROBABILITY COLUMNS
# =============================================================================

bootstrap_probability_columns = {
    "Logistic regression":
        "prob_logistic",

    "Random Forest":
        "prob_random_forest",

    "XGBoost":
        "prob_xgboost"
}


# =============================================================================
# 8. RUN STRATIFIED PSU BOOTSTRAP
# =============================================================================

bootstrap_rows = []

retained_bootstrap_ids = []


for bootstrap_id in range(
    1,
    N_BOOT_BASELINE + 1
):

    sampled_row_indices = []


    # -------------------------------------------------------------------------
    # Resample PSUs independently WITHIN every survey stratum.
    #
    # If a stratum originally contributes k PSUs to the held-out population,
    # exactly k PSU selections are made in that bootstrap replicate.
    # -------------------------------------------------------------------------

    for stratum, psus in (
        stratum_psus.items()
    ):

        sampled_psus = rng_baseline.choice(
            psus,
            size=len(psus),
            replace=True
        )


        for sampled_psu in sampled_psus:

            sampled_row_indices.extend(
                stratum_psu_rows[
                    (
                        stratum,
                        sampled_psu
                    )
                ]
            )


    # -------------------------------------------------------------------------
    # Repeated PSU selection intentionally produces repeated women.
    #
    # This is equivalent to giving the selected PSU corresponding bootstrap
    # multiplicity.
    # -------------------------------------------------------------------------

    boot_sample = baseline_boot_df.loc[
        sampled_row_indices
    ]


    y_boot = (
        boot_sample[
            "anc8plus_observed"
        ]
        .astype(int)
        .to_numpy()
    )

    w_boot = (
        boot_sample[
            "survey_weight"
        ]
        .astype(float)
        .to_numpy()
    )


    # -------------------------------------------------------------------------
    # AUROC requires both outcome classes.
    #
    # With this sample size this should almost always hold, but we check rather
    # than assume.
    # -------------------------------------------------------------------------

    if np.unique(
        y_boot
    ).size < 2:

        continue


    retained_bootstrap_ids.append(
        bootstrap_id
    )


    # -------------------------------------------------------------------------
    # Evaluate ALL models on the SAME resampled population.
    # -------------------------------------------------------------------------

    for model_name, probability_column in (
        bootstrap_probability_columns.items()
    ):

        probability_boot = (
            boot_sample[
                probability_column
            ]
            .astype(float)
            .to_numpy()
        )


        metrics = bootstrap_probability_metrics(
            y=y_boot,
            probability=probability_boot,
            weights=w_boot
        )


        bootstrap_rows.append({
            "Bootstrap":
                bootstrap_id,

            "Model":
                model_name,

            "AP":
                metrics[
                    "AP"
                ],

            "AUROC":
                metrics[
                    "AUROC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ]
        })


baseline_bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# =============================================================================
# 9. BOOTSTRAP INTEGRITY
# =============================================================================

n_retained = (
    baseline_bootstrap_results[
        "Bootstrap"
    ]
    .nunique()
)


assert n_retained > 0


# Every retained replicate must contain all three algorithms.
assert (
    baseline_bootstrap_results
    .groupby(
        "Bootstrap"
    )[
        "Model"
    ]
    .nunique()
    .eq(3)
    .all()
)


assert np.isfinite(
    baseline_bootstrap_results[
        [
            "AP",
            "AUROC",
            "Brier"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 10. CONVERT BOOTSTRAP DISTRIBUTIONS TO 95% PERCENTILE INTERVALS
# =============================================================================

baseline_bootstrap_ci_rows = []


for model_name, model_boot in (
    baseline_bootstrap_results
    .groupby(
        "Model"
    )
):

    for metric in [
        "AP",
        "AUROC",
        "Brier"
    ]:

        values = (
            model_boot[
                metric
            ]
            .to_numpy()
        )


        ci_low, ci_high = np.percentile(
            values,
            [
                2.5,
                97.5
            ]
        )


        baseline_bootstrap_ci_rows.append({
            "Model":
                model_name,

            "Metric":
                metric,

            "CI_low":
                ci_low,

            "CI_high":
                ci_high
        })


baseline_bootstrap_ci = pd.DataFrame(
    baseline_bootstrap_ci_rows
)


# =============================================================================
# 11. ADD THE ORIGINAL HELD-OUT POINT ESTIMATES
# =============================================================================

point_estimate_long = (
    heldout_performance[
        [
            "Model",
            "Weighted_AP",
            "Weighted_AUROC",
            "Weighted_Brier"
        ]
    ]
    .rename(
        columns={
            "Weighted_AP":
                "AP",

            "Weighted_AUROC":
                "AUROC",

            "Weighted_Brier":
                "Brier"
        }
    )
    .melt(
        id_vars="Model",
        var_name="Metric",
        value_name="Point_estimate"
    )
)


baseline_bootstrap_summary = (
    point_estimate_long
    .merge(
        baseline_bootstrap_ci,
        on=[
            "Model",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


# =============================================================================
# 12. DISPLAY
# =============================================================================

print("=" * 108)
print("STRATIFIED PSU BOOTSTRAP 95% CIs: HELD-OUT BASELINE PERFORMANCE")
print("=" * 108)

print(
    baseline_bootstrap_summary
    .to_string(
        index=False,
        formatters={
            "Point_estimate":
                "{:.4f}".format,

            "CI_low":
                "{:.4f}".format,

            "CI_high":
                "{:.4f}".format
        }
    )
)


print("\nBootstrap design")
print("----------------")

print(
    "Resampling unit:",
    "PSU"
)

print(
    "Resampling within survey strata:",
    "Yes"
)

print(
    "Survey strata represented:",
    len(
        stratum_psus
    )
)

print(
    "Bootstrap replicates requested:",
    N_BOOT_BASELINE
)

print(
    "Bootstrap replicates retained:",
    n_retained
)

print(
    "Same resampled population used for all algorithms:",
    "Yes"
)

print(
    "Models refitted:",
    "No"
)

print(
    "Predictions regenerated:",
    "No"
)

print(
    "\nSTATUS: STRATIFIED PSU BOOTSTRAP "
    "UNCERTAINTY ESTIMATED"
)

### 2.25.3 Paired bootstrap uncertainty in model-performance differences

The preceding confidence intervals describe uncertainty around the performance
of each model separately. Overlap between separate confidence intervals,
however, does not directly quantify uncertainty in the difference between two
models evaluated on the same women.

Because Logistic regression, Random Forest, and XGBoost were evaluated on the
same held-out observations, model comparisons are paired.

The stratified PSU bootstrap from Section 2.25.2 preserved this pairing by
evaluating all three algorithms on exactly the same resampled population in
each bootstrap replicate.

XGBoost is compared with Logistic regression and Random Forest because it
showed the strongest overall held-out point-estimate performance in the
preceding descriptive assessment.

Differences are oriented so that positive values consistently favor XGBoost:

- **AP advantage:** XGBoost AP minus comparator AP;
- **AUROC advantage:** XGBoost AUROC minus comparator AUROC;
- **Brier advantage:** comparator Brier minus XGBoost Brier, because lower
  Brier score indicates better probability accuracy.

For each paired difference, a 95% empirical percentile bootstrap confidence
interval is estimated.

These comparisons quantify sampling uncertainty in the observed performance
differences. They do not refit models, alter predictions, or create a new
model-selection rule.

In [ ]:
# =============================================================================
# 2.25.3 PAIRED STRATIFIED-PSU BOOTSTRAP MODEL COMPARISONS
#
# PURPOSE
# -------
# Quantify uncertainty around XGBoost's held-out performance differences
# relative to:
#
#   1. Logistic regression
#   2. Random Forest
#
# All comparisons use the SAME bootstrap replicate for both algorithms.
#
# Difference orientation:
#   AP advantage     > 0  -> favors XGBoost
#   AUROC advantage  > 0  -> favors XGBoost
#   Brier advantage  > 0  -> favors XGBoost
#
# For Brier:
#   comparator Brier - XGBoost Brier
# because LOWER Brier is better.
# =============================================================================


# =============================================================================
# 1. RESHAPE BOOTSTRAP RESULTS TO ONE ROW PER BOOTSTRAP REPLICATE
# =============================================================================

bootstrap_wide = (
    baseline_bootstrap_results
    .pivot(
        index="Bootstrap",
        columns="Model",
        values=[
            "AP",
            "AUROC",
            "Brier"
        ]
    )
    .sort_index()
)


# =============================================================================
# 2. INTEGRITY CHECKS
# =============================================================================

assert len(
    bootstrap_wide
) == n_retained

assert bootstrap_wide.notna().all().all()

assert set(
    bootstrap_wide.columns.get_level_values(
        "Model"
    )
) == {
    "Logistic regression",
    "Random Forest",
    "XGBoost"
}


# =============================================================================
# 3. DEFINE THE TWO PAIRED COMPARISONS
# =============================================================================

comparators = [
    "Logistic regression",
    "Random Forest"
]


# =============================================================================
# 4. ORIGINAL HELD-OUT POINT ESTIMATES
#
# These are used only to report the observed paired performance differences.
# =============================================================================

heldout_metric_lookup = (
    heldout_performance
    .set_index(
        "Model"
    )
)


# =============================================================================
# 5. CALCULATE PAIRED BOOTSTRAP DIFFERENCES
# =============================================================================

paired_difference_rows = []

paired_bootstrap_distributions = []


for comparator in comparators:

    # -------------------------------------------------------------------------
    # Average Precision:
    # XGBoost minus comparator.
    # Positive = XGBoost better.
    # -------------------------------------------------------------------------

    ap_boot_difference = (
        bootstrap_wide[
            (
                "AP",
                "XGBoost"
            )
        ]
        -
        bootstrap_wide[
            (
                "AP",
                comparator
            )
        ]
    )


    ap_point_difference = (
        heldout_metric_lookup.loc[
            "XGBoost",
            "Weighted_AP"
        ]
        -
        heldout_metric_lookup.loc[
            comparator,
            "Weighted_AP"
        ]
    )


    # -------------------------------------------------------------------------
    # AUROC:
    # XGBoost minus comparator.
    # Positive = XGBoost better.
    # -------------------------------------------------------------------------

    auroc_boot_difference = (
        bootstrap_wide[
            (
                "AUROC",
                "XGBoost"
            )
        ]
        -
        bootstrap_wide[
            (
                "AUROC",
                comparator
            )
        ]
    )


    auroc_point_difference = (
        heldout_metric_lookup.loc[
            "XGBoost",
            "Weighted_AUROC"
        ]
        -
        heldout_metric_lookup.loc[
            comparator,
            "Weighted_AUROC"
        ]
    )


    # -------------------------------------------------------------------------
    # Brier:
    # comparator minus XGBoost.
    #
    # This orientation means positive values again favor XGBoost because
    # lower Brier score represents better probability accuracy.
    # -------------------------------------------------------------------------

    brier_boot_difference = (
        bootstrap_wide[
            (
                "Brier",
                comparator
            )
        ]
        -
        bootstrap_wide[
            (
                "Brier",
                "XGBoost"
            )
        ]
    )


    brier_point_difference = (
        heldout_metric_lookup.loc[
            comparator,
            "Weighted_Brier"
        ]
        -
        heldout_metric_lookup.loc[
            "XGBoost",
            "Weighted_Brier"
        ]
    )


    # -------------------------------------------------------------------------
    # Store the bootstrap distributions for possible later diagnostics/export.
    # -------------------------------------------------------------------------

    comparator_distribution = pd.DataFrame({
        "Bootstrap":
            bootstrap_wide.index,

        "Comparator":
            comparator,

        "AP_advantage_XGBoost":
            ap_boot_difference.to_numpy(),

        "AUROC_advantage_XGBoost":
            auroc_boot_difference.to_numpy(),

        "Brier_advantage_XGBoost":
            brier_boot_difference.to_numpy()
    })


    paired_bootstrap_distributions.append(
        comparator_distribution
    )


    # -------------------------------------------------------------------------
    # Convert each paired bootstrap distribution to a 95% percentile interval.
    # -------------------------------------------------------------------------

    for (
        metric,
        point_difference,
        bootstrap_difference
    ) in [

        (
            "AP",
            ap_point_difference,
            ap_boot_difference
        ),

        (
            "AUROC",
            auroc_point_difference,
            auroc_boot_difference
        ),

        (
            "Brier",
            brier_point_difference,
            brier_boot_difference
        )
    ]:

        ci_low, ci_high = np.percentile(
            bootstrap_difference,
            [
                2.5,
                97.5
            ]
        )


        paired_difference_rows.append({
            "Comparison":
                f"XGBoost vs {comparator}",

            "Metric":
                metric,

            "XGBoost_advantage":
                float(
                    point_difference
                ),

            "CI_low":
                float(
                    ci_low
                ),

            "CI_high":
                float(
                    ci_high
                ),

            # This is a descriptive indicator only:
            # whether the empirical 95% interval excludes zero.
            "CI_excludes_zero":
                bool(
                    (ci_low > 0)
                    or (ci_high < 0)
                )
        })


# =============================================================================
# 6. CANONICAL OUTPUT OBJECTS
# =============================================================================

paired_bootstrap_distributions = pd.concat(
    paired_bootstrap_distributions,
    ignore_index=True
)


paired_model_comparisons = pd.DataFrame(
    paired_difference_rows
)


# =============================================================================
# 7. INTEGRITY CHECKS
# =============================================================================

assert len(
    paired_model_comparisons
) == 6

assert len(
    paired_bootstrap_distributions
) == (
    n_retained
    * 2
)

assert np.isfinite(
    paired_model_comparisons[
        [
            "XGBoost_advantage",
            "CI_low",
            "CI_high"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 8. DISPLAY
# =============================================================================

print("=" * 118)
print("PAIRED STRATIFIED-PSU BOOTSTRAP COMPARISONS: XGBOOST ADVANTAGE")
print("=" * 118)

print(
    paired_model_comparisons
    .to_string(
        index=False,
        formatters={
            "XGBoost_advantage":
                "{:+.4f}".format,

            "CI_low":
                "{:+.4f}".format,

            "CI_high":
                "{:+.4f}".format
        }
    )
)


print("\nDifference orientation")
print("----------------------")

print(
    "AP:",
    "XGBoost - comparator"
)

print(
    "AUROC:",
    "XGBoost - comparator"
)

print(
    "Brier:",
    "comparator - XGBoost"
)

print(
    "Positive difference:",
    "favors XGBoost"
)


print("\nBootstrap governance")
print("--------------------")

print(
    "Paired bootstrap:",
    "Yes"
)

print(
    "Resampling unit:",
    "PSU"
)

print(
    "Resampling within strata:",
    "Yes"
)

print(
    "Replicates:",
    n_retained
)

print(
    "Models refitted:",
    "No"
)

print(
    "Predictions regenerated:",
    "No"
)

print(
    "\nSTATUS: PAIRED MODEL-COMPARISON "
    "UNCERTAINTY ESTIMATED"
)

## 2.26 Governance of the ANC-entry prediction extension

The primary baseline ANC8 models use information available before ANC-process
information is known.

A secondary **ANC-entry prediction horizon** is now evaluated to answer a
different prediction question:

> Among women who initiated ANC, does information available at the beginning of
> ANC improve prediction of eventual attainment of eight or more ANC contacts?

### Distinct prediction time

Timing of the first ANC contact is not available before ANC initiation.
Therefore, it cannot be added to the baseline predictor set without changing
the prediction time.

The two prediction horizons are consequently distinct:

- **Baseline horizon:** prediction from background and pre-ANC information;
- **ANC-entry horizon:** prediction after ANC has started and timing of the
  first ANC contact is known.

### Governed predictor amendment

The frozen 21-predictor baseline artifact remains unchanged.

Any ANC-entry predictor must:

1. be available at the ANC-entry prediction time;
2. have a clear clinical and scientific rationale;
3. be assessed for direct or indirect outcome leakage;
4. have its source coding, denominator, and missingness documented;
5. be derived separately from the frozen baseline predictors;
6. be linked using `caseid`, never row order.

### Candidate ANC-entry predictor

The prespecified candidate is **early ANC initiation**, based on timing of the
first ANC contact.

The source variables, observed coding, denominator, and special values will be
ascertained directly from the raw EDHS IR file before the construct is created.

### Incremental predictive-value objective

The purpose of this secondary analysis is not simply to examine whether early
ANC initiation is associated with ANC8 attainment.

The objective is to determine whether adding information available at ANC
entry improves prediction beyond the original 21-predictor baseline model.

Any baseline-versus-ANC-entry comparison must therefore be conducted on the
same eligible ANC-entry analytic population and using the same locked
development/test partition wherever possible.

No ANC-entry model will be fitted until variable ascertainment and cohort
definition are complete.

## 2.27 Ascertainment of first-ANC timing variables

Before constructing the ANC-entry predictor, the EDHS source variables for
timing of the first ANC contact are examined directly from the raw IR file.

The candidate variables are:

- `m13_1`: timing of the first ANC contact expressed in months;
- `m13a_1`: original first-ANC timing response retaining whether timing was
  reported in weeks or months;
- `m14_1`: number of ANC contacts, used here to establish the ANC-attendance
  denominator.

This audit determines:

1. the observed coding of `m13_1` and `m13a_1`;
2. whether timing information is structurally absent among women with no ANC;
3. whether women who initiated ANC generally have timing recorded;
4. the frequency and location of special or unknown timing codes;
5. whether the raw timing data can be linked one-to-one with the frozen
   baseline analytic population.

No ANC-entry predictor is created in this section.

The frozen baseline dataset and its 21 predictors remain unchanged.

In [ ]:
# =============================================================================
# 2.27 ASCERTAIN FIRST-ANC TIMING SOURCE VARIABLES
#
# PURPOSE
# -------
# Inspect the raw EDHS variables used to describe timing of the first ANC
# contact before constructing any ANC-entry predictor.
#
# Candidate variables
# -------------------
# m13_1  : first ANC timing expressed in months
# m13a_1 : original first ANC timing response, retaining week/month coding
# m14_1  : number of ANC contacts
#
# IMPORTANT
# ---------
# This cell AUDITS source data only.
# It does NOT create early_anc.
# =============================================================================


# =============================================================================
# 1. READ ONLY THE REQUIRED RAW VARIABLES
# =============================================================================

RAW_IR_PATH = PROJECT_DIR / "ETIR8AFL.dta"

anc_timing_raw = pd.read_stata(
    RAW_IR_PATH,
    columns=[
        "caseid",
        "m13_1",
        "m13a_1",
        "m14_1"
    ],
    convert_categoricals=False
)


# =============================================================================
# 2. RAW-FILE INTEGRITY
# =============================================================================

assert anc_timing_raw[
    "caseid"
].is_unique

assert len(
    anc_timing_raw
) == 21395


# =============================================================================
# 3. LINK TO THE CURRENT FROZEN BASELINE ANALYTIC POPULATION
#
# Link by caseid only.
# Row order is never used.
# =============================================================================

anc_timing_audit = (
    baseline_df[
        ["caseid"]
    ]
    .merge(
        anc_timing_raw,
        on="caseid",
        how="left",
        validate="one_to_one"
    )
)


assert len(
    anc_timing_audit
) == len(
    baseline_df
)

assert anc_timing_audit[
    "caseid"
].is_unique


# =============================================================================
# 4. DEFINE ANC-ATTENDANCE STATUS FOR AUDIT PURPOSES ONLY
#
# This is NOT a predictor.
#
# m14_1 = 0   -> no ANC
# m14_1 >= 1  -> ANC initiated
# =============================================================================

assert anc_timing_audit[
    "m14_1"
].notna().all()

assert (
    anc_timing_audit[
        "m14_1"
    ] != 98
).all()


anc_timing_audit[
    "anc_attendance_status"
] = np.where(
    anc_timing_audit[
        "m14_1"
    ].eq(0),
    "No ANC",
    "ANC initiated"
)


# =============================================================================
# 5. CLASSIFY m13a_1 RECORDING STATUS
#
# This does NOT decode timing into early/late ANC.
#
# 998 = timing reported as unknown
# NaN = no recorded timing
# all other observed values = timing recorded
# =============================================================================

anc_timing_audit[
    "m13a_recording_status"
] = np.select(
    [
        anc_timing_audit[
            "m13a_1"
        ].eq(998),

        anc_timing_audit[
            "m13a_1"
        ].isna()
    ],
    [
        "Unknown timing (998)",
        "No recorded timing"
    ],
    default="Timing recorded"
)


# =============================================================================
# 6. SOURCE-CODE DISTRIBUTIONS
# =============================================================================

m13_distribution = (
    anc_timing_audit[
        "m13_1"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


m13a_distribution = (
    anc_timing_audit[
        "m13a_1"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


# =============================================================================
# 7. DENOMINATOR AUDIT
#
# Cross-tab ANC attendance against availability of original timing information.
# =============================================================================

timing_denominator_audit = pd.crosstab(
    anc_timing_audit[
        "anc_attendance_status"
    ],
    anc_timing_audit[
        "m13a_recording_status"
    ],
    dropna=False
)


# =============================================================================
# 8. KEY COUNTS
# =============================================================================

n_total = len(
    anc_timing_audit
)

n_no_anc = int(
    anc_timing_audit[
        "m14_1"
    ].eq(0).sum()
)

n_anc_initiated = int(
    anc_timing_audit[
        "m14_1"
    ].ge(1).sum()
)

n_timing_recorded = int(
    anc_timing_audit[
        "m13a_recording_status"
    ].eq(
        "Timing recorded"
    ).sum()
)

n_timing_unknown = int(
    anc_timing_audit[
        "m13a_recording_status"
    ].eq(
        "Unknown timing (998)"
    ).sum()
)

n_timing_missing = int(
    anc_timing_audit[
        "m13a_recording_status"
    ].eq(
        "No recorded timing"
    ).sum()
)


# =============================================================================
# 9. STRUCTURAL CONSISTENCY CHECKS
# =============================================================================

# Women with no ANC should not have first-ANC timing recorded.
assert not (
    anc_timing_audit[
        "m14_1"
    ].eq(0)
    &
    anc_timing_audit[
        "m13a_1"
    ].notna()
).any()


# Among ANC initiators, m13a_1 should either contain timing or the explicit
# unknown code 998.
assert not (
    anc_timing_audit[
        "m14_1"
    ].ge(1)
    &
    anc_timing_audit[
        "m13a_1"
    ].isna()
).any()


# m13_1 and m13a_1 should have the same structural missingness pattern.
assert (
    anc_timing_audit[
        "m13_1"
    ].isna()
    ==
    anc_timing_audit[
        "m13a_1"
    ].isna()
).all()


# =============================================================================
# 10. DISPLAY
# =============================================================================

print("=" * 100)
print("FIRST-ANC TIMING SOURCE-VARIABLE AUDIT")
print("=" * 100)

print("\nAnalytic population")
print("-------------------")
print(
    "Baseline women:",
    n_total
)
print(
    "No ANC:",
    n_no_anc
)
print(
    "ANC initiated:",
    n_anc_initiated
)


print("\nObserved coding: m13_1")
print("-----------------------")
print(
    m13_distribution
)


print("\nObserved coding: m13a_1")
print("------------------------")
print(
    m13a_distribution
)


print("\nANC attendance × m13a_1 recording status")
print("-----------------------------------------")
print(
    timing_denominator_audit
)


print("\nTiming-data accounting")
print("----------------------")
print(
    "Timing recorded:",
    n_timing_recorded
)
print(
    "Unknown timing (998):",
    n_timing_unknown
)
print(
    "Structurally absent / no ANC:",
    n_timing_missing
)


print("\nIntegrity")
print("---------")
print(
    "Raw caseid unique:",
    "Yes"
)
print(
    "Merged by caseid one-to-one:",
    "Yes"
)
print(
    "No-ANC women with timing recorded:",
    0
)
print(
    "ANC initiators with structurally missing m13a_1:",
    0
)
print(
    "early_anc constructed:",
    "No"
)

print(
    "\nSTATUS: FIRST-ANC TIMING SOURCE "
    "VARIABLES ASCERTAINED"
)

### 2.27.1 Audit of first-ANC timing around the early-initiation boundary

The variable `m13a_1` preserves the original unit in which timing of the first
ANC contact was reported.

Observed codes indicate two timing units:

- values in the `100` series represent timing reported in **weeks**;
- values in the `200` series represent timing reported in **months**;
- `998` represents unknown timing.

Before constructing the ANC-entry predictor, the original reporting units and
values are decoded and examined around the intended first-trimester boundary.

Particular attention is given to:

- 12 versus 13 weeks;
- 3 versus 4 months;
- any unusual or implausible timing values.

This step is descriptive only. No early-ANC indicator is created.

In [ ]:
# =============================================================================
# 2.27.1 AUDIT WEEK/MONTH REPORTING AROUND THE EARLY-ANC BOUNDARY
#
# PURPOSE
# -------
# Decode the ORIGINAL reporting unit in m13a_1 before defining early ANC.
#
# Coding structure observed in EDHS:
#
#   100-series = reported in weeks
#   200-series = reported in months
#   998        = unknown timing
#
# No early_anc variable is created here.
# =============================================================================


# =============================================================================
# 1. RESTRICT TO WOMEN WITH KNOWN FIRST-ANC TIMING
# =============================================================================

timing_known = (
    anc_timing_audit[
        anc_timing_audit[
            "m13a_1"
        ].notna()
        &
        anc_timing_audit[
            "m13a_1"
        ].ne(998)
    ]
    .copy()
)


assert len(
    timing_known
) == 5001


# =============================================================================
# 2. VERIFY THAT ALL KNOWN CODES BELONG TO EXPECTED WEEK/MONTH SERIES
# =============================================================================

expected_known_code = (
    timing_known[
        "m13a_1"
    ].between(
        100,
        199
    )
    |
    timing_known[
        "m13a_1"
    ].between(
        200,
        299
    )
)

assert expected_known_code.all()


# =============================================================================
# 3. DECODE ORIGINAL REPORTING UNIT
# =============================================================================

timing_known[
    "timing_unit"
] = np.where(
    timing_known[
        "m13a_1"
    ].between(
        100,
        199
    ),
    "Weeks",
    "Months"
)


# =============================================================================
# 4. DECODE THE ORIGINAL NUMERIC VALUE
#
# Examples:
#   112 -> 12 weeks
#   203 -> 3 months
# =============================================================================

timing_known[
    "timing_value"
] = np.where(
    timing_known[
        "timing_unit"
    ].eq(
        "Weeks"
    ),
    timing_known[
        "m13a_1"
    ] - 100,
    timing_known[
        "m13a_1"
    ] - 200
)


# =============================================================================
# 5. REPORTING-UNIT COUNTS
# =============================================================================

timing_unit_counts = (
    timing_known[
        "timing_unit"
    ]
    .value_counts()
)


# =============================================================================
# 6. COMPLETE VALUE DISTRIBUTIONS BY ORIGINAL REPORTING UNIT
# =============================================================================

week_value_distribution = (
    timing_known.loc[
        timing_known[
            "timing_unit"
        ].eq(
            "Weeks"
        ),
        "timing_value"
    ]
    .value_counts()
    .sort_index()
)


month_value_distribution = (
    timing_known.loc[
        timing_known[
            "timing_unit"
        ].eq(
            "Months"
        ),
        "timing_value"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 7. EXPLICIT AUDIT AROUND THE INTENDED BOUNDARY
#
# We inspect rather than classify.
#
# Week boundary:
#   12 weeks
#   13 weeks
#
# Month boundary:
#   3 months
#   4 months
# =============================================================================

boundary_audit = pd.DataFrame({
    "Boundary_value": [
        "12 weeks",
        "13 weeks",
        "3 months",
        "4 months"
    ],

    "Count": [
        int(
            (
                timing_known[
                    "timing_unit"
                ].eq("Weeks")
                &
                timing_known[
                    "timing_value"
                ].eq(12)
            ).sum()
        ),

        int(
            (
                timing_known[
                    "timing_unit"
                ].eq("Weeks")
                &
                timing_known[
                    "timing_value"
                ].eq(13)
            ).sum()
        ),

        int(
            (
                timing_known[
                    "timing_unit"
                ].eq("Months")
                &
                timing_known[
                    "timing_value"
                ].eq(3)
            ).sum()
        ),

        int(
            (
                timing_known[
                    "timing_unit"
                ].eq("Months")
                &
                timing_known[
                    "timing_value"
                ].eq(4)
            ).sum()
        )
    ]
})


# =============================================================================
# 8. IDENTIFY EXTREME / UNUSUAL VALUES FOR MANUAL REVIEW
#
# This does NOT automatically classify them as errors.
# =============================================================================

unusual_timing_records = (
    timing_known.loc[
        (
            timing_known[
                "timing_unit"
            ].eq(
                "Weeks"
            )
            &
            timing_known[
                "timing_value"
            ].gt(20)
        )
        |
        (
            timing_known[
                "timing_unit"
            ].eq(
                "Months"
            )
            &
            timing_known[
                "timing_value"
            ].gt(9)
        ),
        [
            "caseid",
            "m13_1",
            "m13a_1",
            "m14_1",
            "timing_unit",
            "timing_value"
        ]
    ]
    .copy()
)


# =============================================================================
# 9. INTEGRITY CHECKS
# =============================================================================

assert (
    timing_unit_counts.sum()
    == 5001
)

assert (
    week_value_distribution.sum()
    +
    month_value_distribution.sum()
    == 5001
)


# =============================================================================
# 10. DISPLAY
# =============================================================================

print("=" * 100)
print("FIRST-ANC TIMING BOUNDARY AUDIT")
print("=" * 100)


print("\nOriginal reporting unit")
print("-----------------------")
print(
    timing_unit_counts
)


print("\nWeek-reported values")
print("--------------------")
print(
    week_value_distribution
)


print("\nMonth-reported values")
print("---------------------")
print(
    month_value_distribution
)


print("\nBoundary-specific counts")
print("------------------------")
print(
    boundary_audit.to_string(
        index=False
    )
)


print("\nUnusual timing records requiring review")
print("---------------------------------------")

if len(
    unusual_timing_records
) == 0:

    print(
        "None"
    )

else:

    print(
        unusual_timing_records.to_string(
            index=False
        )
    )


print("\nGovernance")
print("----------")

print(
    "Known timing records:",
    len(
        timing_known
    )
)

print(
    "Unknown timing records excluded from this audit:",
    n_timing_unknown
)

print(
    "early_anc constructed:",
    "No"
)

print(
    "\nSTATUS: FIRST-ANC TIMING "
    "BOUNDARY AUDITED"
)

## 2.28 Construction of the ANC-entry timing predictor

Early ANC initiation is defined as initiation of antenatal care during the
first trimester.

Because EDHS records first-contact timing in either weeks or months, the
definition is operationalized using the original reporting unit:

- timing reported in weeks: **12 weeks or earlier = early ANC**;
- timing reported in months: **3 months or earlier = early ANC**;
- later timing = not early ANC;
- unknown timing (`m13a_1 = 998`) remains unresolved;
- women with no ANC are structurally outside the ANC-entry prediction
  population.

The three-month threshold is used as the operational month-based equivalent of
the first-trimester boundary because most respondents reported timing in
completed months rather than exact weeks.

One woman reported first ANC at 40 weeks (`m13a_1 = 140`), corresponding to
approximately nine months in `m13_1`. This value is unusual but unambiguously
falls outside the early-ANC definition and is retained as late initiation.

The new predictor is constructed separately from the frozen 21-predictor
baseline artifact.

In [ ]:
# =============================================================================
# 2.28 CONSTRUCT EARLY ANC INITIATION
#
# 1 = first ANC within first trimester
# 0 = first ANC later than first trimester
# NaN = unknown timing or structurally unavailable because no ANC occurred
# =============================================================================

anc_entry_timing = (
    anc_timing_audit[
        [
            "caseid",
            "m13_1",
            "m13a_1",
            "m14_1"
        ]
    ]
    .copy()
)

# Start unresolved.
anc_entry_timing["early_anc"] = np.nan


# =============================================================================
# 1. KNOWN WEEK-REPORTED TIMING
# =============================================================================

week_mask = (
    anc_entry_timing["m13a_1"].between(100, 199)
)

week_value = (
    anc_entry_timing["m13a_1"] - 100
)

anc_entry_timing.loc[
    week_mask,
    "early_anc"
] = (
    week_value[week_mask] <= 12
).astype(float)


# =============================================================================
# 2. KNOWN MONTH-REPORTED TIMING
# =============================================================================

month_mask = (
    anc_entry_timing["m13a_1"].between(200, 299)
)

month_value = (
    anc_entry_timing["m13a_1"] - 200
)

anc_entry_timing.loc[
    month_mask,
    "early_anc"
] = (
    month_value[month_mask] <= 3
).astype(float)


# =============================================================================
# 3. LEAVE UNKNOWN TIMING AND NO-ANC WOMEN UNRESOLVED
#
# m13a_1 = 998 -> unknown timing
# NaN          -> no ANC / structurally unavailable
# =============================================================================

assert anc_entry_timing.loc[
    anc_entry_timing["m13a_1"].eq(998),
    "early_anc"
].isna().all()

assert anc_entry_timing.loc[
    anc_entry_timing["m14_1"].eq(0),
    "early_anc"
].isna().all()


# =============================================================================
# 4. INTEGRITY CHECKS
# =============================================================================

assert anc_entry_timing[
    "caseid"
].is_unique

assert set(
    anc_entry_timing[
        "early_anc"
    ]
    .dropna()
    .unique()
) <= {0.0, 1.0}

# Every known first-ANC timing must resolve to early or late.
assert anc_entry_timing.loc[
    anc_entry_timing["m13a_1"].notna()
    &
    anc_entry_timing["m13a_1"].ne(998),
    "early_anc"
].notna().all()


# =============================================================================
# 5. ACCOUNTING
# =============================================================================

early_anc_distribution = (
    anc_entry_timing[
        "early_anc"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

n_early = int(
    anc_entry_timing[
        "early_anc"
    ].eq(1).sum()
)

n_late = int(
    anc_entry_timing[
        "early_anc"
    ].eq(0).sum()
)

n_unresolved = int(
    anc_entry_timing[
        "early_anc"
    ].isna().sum()
)


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 96)
print("EARLY ANC INITIATION CONSTRUCT")
print("=" * 96)

print(
    "1 = Early ANC: first contact <=12 weeks or <=3 reported months"
)

print(
    "0 = Later ANC"
)

print(
    "NaN = no ANC or unknown first-contact timing"
)

print("\nDistribution")
print("------------")

print(
    early_anc_distribution
)

print("\nAccounting")
print("----------")

print(
    "Early ANC:",
    n_early
)

print(
    "Later ANC:",
    n_late
)

print(
    "Unresolved / structurally unavailable:",
    n_unresolved
)

print(
    "Total:",
    len(
        anc_entry_timing
    )
)

print("\nGovernance")
print("----------")

print(
    "Frozen baseline predictors modified:",
    "No"
)

print(
    "40-week record retained as late ANC:",
    "Yes"
)

print(
    "\nSTATUS: EARLY ANC PREDICTOR CONSTRUCTED"
)

## 2.29 Define the ANC-entry analytic population

The ANC-entry prediction horizon is restricted to women for whom first-ANC
timing is available and therefore meaningful.

Eligibility requires:

- a resolved ANC8 outcome;
- initiation of ANC;
- resolved first-contact timing and therefore a resolved `early_anc` value.

Women with no ANC are structurally outside this prediction horizon because
first-contact timing does not exist for them.

Women with unknown timing (`m13a_1 = 998`) are excluded because the ANC-entry
predictor cannot be resolved.

The ANC-entry population is therefore defined independently from the frozen
baseline artifact and linked using `caseid`.

For assessment of incremental predictive value, both model specifications will
later be fitted and evaluated within this identical ANC-entry population:

1. **ANC-entry baseline specification:** the original 21 predictors;
2. **ANC-entry extended specification:** the same 21 predictors plus
   `early_anc`.

Keeping the comparison population identical ensures that any performance
difference is attributable to the additional ANC-entry information rather than
to a change in analytic denominator.

In [ ]:
# =============================================================================
# 2.29 DEFINE THE ANC-ENTRY ANALYTIC POPULATION
#
# Eligibility:
#   - resolved ANC8 outcome
#   - ANC initiated
#   - known first-ANC timing
#   - resolved early_anc
#
# IMPORTANT
# ---------
# The frozen baseline artifact is NOT modified.
# ANC-entry data are created as a separate analytic object.
# =============================================================================


# =============================================================================
# 1. RETAIN ONLY THE ANC-ENTRY FEATURE NEEDED FOR MERGE
# =============================================================================

anc_entry_feature = (
    anc_entry_timing[
        [
            "caseid",
            "early_anc"
        ]
    ]
    .copy()
)


assert anc_entry_feature[
    "caseid"
].is_unique


# =============================================================================
# 2. MERGE EARLY_ANC INTO THE RESOLVED BASELINE ANALYTIC POPULATION
#
# baseline_df already represents the resolved ANC8 population used for
# primary modelling.
#
# Merge strictly by caseid.
# =============================================================================

anc_entry_full = (
    baseline_df
    .merge(
        anc_entry_feature,
        on="caseid",
        how="left",
        validate="one_to_one"
    )
)


# =============================================================================
# 3. DEFINE ANC-ENTRY ELIGIBILITY
#
# A resolved early_anc value implies:
#   - ANC occurred;
#   - first-contact timing was known.
# =============================================================================

anc_entry_eligible = (
    anc_entry_full[
        anc_entry_full[
            "early_anc"
        ].notna()
    ]
    .copy()
)


# =============================================================================
# 4. INTEGRITY CHECKS
# =============================================================================

assert anc_entry_full[
    "caseid"
].is_unique

assert anc_entry_eligible[
    "caseid"
].is_unique


# Expected resolved early-ANC population from Section 2.28.
assert len(
    anc_entry_eligible
) == (
    anc_entry_full[
        "early_anc"
    ].notna().sum()
)


assert anc_entry_eligible[
    "anc8plus"
].notna().all()

assert anc_entry_eligible[
    "early_anc"
].isin(
    [0.0, 1.0]
).all()


# All original 21 predictors remain available.
assert all(
    predictor in anc_entry_eligible.columns
    for predictor in PRIMARY_PREDICTORS
)


# =============================================================================
# 5. ACCOUNT FOR EXCLUSIONS
# =============================================================================

n_baseline_resolved = len(
    baseline_df
)

n_anc_entry = len(
    anc_entry_eligible
)

n_excluded_entry = (
    n_baseline_resolved
    - n_anc_entry
)


# =============================================================================
# 6. ANC8 OUTCOME CONTEXT WITHIN THE NEW PREDICTION HORIZON
# =============================================================================

anc_entry_weighted_prevalence = np.average(
    anc_entry_eligible[
        "anc8plus"
    ].astype(int),
    weights=(
        anc_entry_eligible[
            "v005"
        ]
        / 1_000_000
    )
)

anc_entry_unweighted_prevalence = (
    anc_entry_eligible[
        "anc8plus"
    ].mean()
)


# =============================================================================
# 7. EARLY-ANC DISTRIBUTION WITHIN THE ELIGIBLE POPULATION
# =============================================================================

anc_entry_early_distribution = (
    anc_entry_eligible[
        "early_anc"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 8. DISPLAY
# =============================================================================

print("=" * 104)
print("ANC-ENTRY ANALYTIC POPULATION")
print("=" * 104)

print("\nPopulation accounting")
print("---------------------")

print(
    "Resolved baseline ANC8 population:",
    n_baseline_resolved
)

print(
    "ANC-entry eligible population:",
    n_anc_entry
)

print(
    "Excluded from ANC-entry horizon:",
    n_excluded_entry
)


print("\nEligibility basis")
print("-----------------")

print(
    "Resolved ANC8 outcome:",
    "Yes"
)

print(
    "ANC initiated:",
    "Yes"
)

print(
    "First-ANC timing resolved:",
    "Yes"
)

print(
    "early_anc resolved:",
    "Yes"
)


print("\nANC8 outcome context")
print("--------------------")

print(
    "ANC8+ count:",
    int(
        anc_entry_eligible[
            "anc8plus"
        ].sum()
    )
)

print(
    "Unweighted ANC8+ prevalence:",
    f"{anc_entry_unweighted_prevalence:.2%}"
)

print(
    "Survey-weighted ANC8+ prevalence:",
    f"{anc_entry_weighted_prevalence:.2%}"
)


print("\nEarly ANC distribution")
print("----------------------")

print(
    anc_entry_early_distribution
)


print("\nGovernance")
print("----------")

print(
    "Original frozen baseline artifact modified:",
    "No"
)

print(
    "ANC-entry data linked by caseid:",
    "Yes"
)

print(
    "Different denominator for baseline vs extended ANC-entry models:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY ANALYTIC "
    "POPULATION DEFINED"
)

## 2.30 Preserve the original development/test assignment

The ANC-entry analysis inherits the development/test assignment created for the
primary baseline analysis.

No new random split is generated.

The ANC-entry analytic cohort is restricted by `caseid` from the already
defined baseline development and held-out test populations.

This preserves comparability because:

- every woman retains her original development/test status;
- PSUs remain entirely within one partition;
- the ANC-entry comparison is not influenced by a new random split;
- both the 21-predictor baseline specification and the 22-predictor extended
  specification use exactly the same development and held-out observations.

The inherited split therefore provides a paired evaluation framework for
incremental predictive value.## 2.30 Preserve the original development/test assignment

The ANC-entry analysis inherits the original PSU-aware train/test assignment
from the baseline analysis.

No new random split is created.

This preserves comparability because:

- women retain the same development/test status assigned previously;
- PSUs remain entirely within one partition;
- the final ANC-entry comparison is not affected by a new random partition.

Within the ANC-entry-eligible population, both the 21-predictor baseline model
and the 22-predictor extended model will therefore use the same development and
test observations.

In [ ]:
# =============================================================================
# 2.30 INHERIT THE ORIGINAL DEVELOPMENT / HELD-OUT TEST ASSIGNMENT
#
# IMPORTANT
# ---------
# No new split is created.
# ANC-entry women inherit their original partition by caseid.
# =============================================================================


# =============================================================================
# 1. ORIGINAL PARTITION IDENTIFIERS
# =============================================================================

development_caseids = set(
    train_df[
        "caseid"
    ]
)

test_caseids = set(
    test_df[
        "caseid"
    ]
)


# Original baseline partitions must remain disjoint.
assert development_caseids.isdisjoint(
    test_caseids
)


# =============================================================================
# 2. RESTRICT ANC-ENTRY POPULATION TO THE ORIGINAL PARTITIONS
# =============================================================================

anc_entry_train = (
    anc_entry_eligible[
        anc_entry_eligible[
            "caseid"
        ].isin(
            development_caseids
        )
    ]
    .copy()
)

anc_entry_test = (
    anc_entry_eligible[
        anc_entry_eligible[
            "caseid"
        ].isin(
            test_caseids
        )
    ]
    .copy()
)


# =============================================================================
# 3. INTEGRITY CHECKS
# =============================================================================

assert anc_entry_train[
    "caseid"
].is_unique

assert anc_entry_test[
    "caseid"
].is_unique


assert set(
    anc_entry_train[
        "caseid"
    ]
).isdisjoint(
    set(
        anc_entry_test[
            "caseid"
        ]
    )
)


# Every ANC-entry-eligible woman must belong to exactly one inherited partition.
assert (
    set(
        anc_entry_train[
            "caseid"
        ]
    )
    |
    set(
        anc_entry_test[
            "caseid"
        ]
    )
) == set(
    anc_entry_eligible[
        "caseid"
    ]
)


assert (
    len(
        anc_entry_train
    )
    +
    len(
        anc_entry_test
    )
) == len(
    anc_entry_eligible
)


# =============================================================================
# 4. PSU GOVERNANCE
# =============================================================================

anc_entry_train_psus = set(
    anc_entry_train[
        "v021"
    ]
)

anc_entry_test_psus = set(
    anc_entry_test[
        "v021"
    ]
)


assert anc_entry_train_psus.isdisjoint(
    anc_entry_test_psus
)


# =============================================================================
# 5. OUTCOME CONTEXT BY PARTITION
# =============================================================================

def partition_summary(
    df,
    label
):
    """
    Return basic weighted/unweighted ANC8 context for one inherited partition.
    """

    weights = (
        df[
            "v005"
        ]
        .astype(float)
        / 1_000_000
    )

    outcome = (
        df[
            "anc8plus"
        ]
        .astype(int)
    )

    return {
        "Partition":
            label,

        "Women":
            len(df),

        "PSUs":
            df[
                "v021"
            ].nunique(),

        "Strata":
            df[
                "v022"
            ].nunique(),

        "ANC8_positive":
            int(
                outcome.sum()
            ),

        "Unweighted_prevalence":
            outcome.mean(),

        "Weighted_prevalence":
            np.average(
                outcome,
                weights=weights
            )
    }


anc_entry_partition_summary = pd.DataFrame([
    partition_summary(
        anc_entry_train,
        "Development"
    ),

    partition_summary(
        anc_entry_test,
        "Held-out test"
    )
])


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 108)
print("ANC-ENTRY INHERITED DEVELOPMENT / TEST PARTITION")
print("=" * 108)

print(
    anc_entry_partition_summary
    .to_string(
        index=False,
        formatters={
            "Unweighted_prevalence":
                "{:.2%}".format,

            "Weighted_prevalence":
                "{:.2%}".format
        }
    )
)


print("\nIntegrity")
print("---------")

print(
    "New random split created:",
    "No"
)

print(
    "Development/test caseid overlap:",
    0
)

print(
    "Development/test PSU overlap:",
    0
)

print(
    "All ANC-entry women assigned exactly once:",
    "Yes"
)

print(
    "Original partition membership preserved:",
    "Yes"
)

print(
    "\nSTATUS: ANC-ENTRY ORIGINAL "
    "PARTITION PRESERVED"
)

## 2.31 Paired ANC-entry model specifications and cross-validation

The ANC-entry analysis compares two predictor specifications within the same
analytic population and inherited development/test partition.

### Model A: ANC-entry baseline specification

Uses the original 21 frozen predictors.

### Model B: ANC-entry extended specification

Uses the same 21 predictors plus:

- `early_anc`: 0 = later ANC initiation; 1 = early ANC initiation.

Because both specifications are fitted and evaluated on exactly the same
ANC-entry-eligible women, differences in predictive performance reflect the
additional ANC-entry information rather than a change in denominator.

### Internal validation

The original development/test partition is retained unchanged.

However, the internal development cross-validation folds are recreated because
restricting to ANC-entry-eligible women changes the composition of the
development population.

Five-fold cross-validation is constructed using:

- entire PSUs as indivisible units;
- assignment performed independently within survey strata;
- no ANC8 outcome information in fold assignment;
- each development woman appearing exactly once in validation;
- no PSU appearing in both fitting and validation portions of a fold.

The resulting fold object is locked and must be reused for all paired
ANC-entry model tuning and out-of-fold prediction.

The held-out ANC-entry test set remains untouched.

In [ ]:
# =============================================================================
# 2.31 DEFINE PAIRED ANC-ENTRY SPECIFICATIONS AND LOCKED STRATIFIED PSU CV
#
# IMPORTANT
# ---------
# - No new development/test split.
# - CV is recreated ONLY within anc_entry_train.
# - Whole PSUs remain together.
# - PSU assignment is performed within survey strata.
# - Outcome is NOT used to construct folds.
# =============================================================================

ENTRY_N_FOLDS = 5
ENTRY_CV_SEED = 2026


# =============================================================================
# 1. PAIRED PREDICTOR SPECIFICATIONS
# =============================================================================

ANC_ENTRY_BASELINE_PREDICTORS = (
    PRIMARY_PREDICTORS.copy()
)

ANC_ENTRY_EXTENDED_PREDICTORS = (
    PRIMARY_PREDICTORS
    + ["early_anc"]
)


assert len(
    ANC_ENTRY_BASELINE_PREDICTORS
) == 21

assert len(
    ANC_ENTRY_EXTENDED_PREDICTORS
) == 22

assert (
    ANC_ENTRY_EXTENDED_PREDICTORS[:-1]
    == ANC_ENTRY_BASELINE_PREDICTORS
)

assert (
    ANC_ENTRY_EXTENDED_PREDICTORS[-1]
    == "early_anc"
)


# =============================================================================
# 2. DEVELOPMENT OBJECTS
# =============================================================================

X_entry_base = (
    anc_entry_train[
        ANC_ENTRY_BASELINE_PREDICTORS
    ]
    .copy()
)

X_entry_ext = (
    anc_entry_train[
        ANC_ENTRY_EXTENDED_PREDICTORS
    ]
    .copy()
)

y_entry = (
    anc_entry_train[
        "anc8plus"
    ]
    .astype(int)
    .copy()
)

groups_entry = (
    anc_entry_train[
        "v021"
    ]
    .copy()
)

strata_entry = (
    anc_entry_train[
        "v022"
    ]
    .copy()
)


# =============================================================================
# 3. BASIC DEVELOPMENT INTEGRITY
# =============================================================================

assert len(
    X_entry_base
) == len(
    X_entry_ext
) == len(
    y_entry
) == len(
    groups_entry
) == len(
    strata_entry
) == 4017

assert X_entry_base.columns.tolist() == (
    ANC_ENTRY_BASELINE_PREDICTORS
)

assert X_entry_ext.columns.tolist() == (
    ANC_ENTRY_EXTENDED_PREDICTORS
)

assert X_entry_ext[
    "early_anc"
].isin(
    [0.0, 1.0]
).all()


# =============================================================================
# 4. BUILD ONE PSU-LEVEL DEVELOPMENT FRAME
#
# Each PSU must belong to exactly one survey stratum.
# Outcome is intentionally absent.
# =============================================================================

entry_development_psu_frame = (
    anc_entry_train[
        [
            "v021",
            "v022"
        ]
    ]
    .drop_duplicates()
    .copy()
)


assert (
    entry_development_psu_frame[
        "v021"
    ]
    .is_unique
)


# =============================================================================
# 5. AUDIT PSU AVAILABILITY BY STRATUM
# =============================================================================

entry_psus_per_stratum = (
    entry_development_psu_frame
    .groupby(
        "v022"
    )[
        "v021"
    ]
    .nunique()
    .sort_index()
)


minimum_entry_psus_per_stratum = int(
    entry_psus_per_stratum.min()
)


# To place every stratum in every fold, each stratum needs >= 5 development PSUs.
assert minimum_entry_psus_per_stratum >= ENTRY_N_FOLDS, (
    "At least one ANC-entry development stratum has fewer than "
    f"{ENTRY_N_FOLDS} PSUs. Fold construction requires review."
)


# =============================================================================
# 6. ASSIGN WHOLE PSUs TO 5 FOLDS WITHIN EACH STRATUM
#
# For each stratum:
#   1. shuffle PSUs reproducibly;
#   2. create approximately equal numbers of PSUs per fold;
#   3. randomly rotate which fold receives remainder PSU(s).
#
# No outcome information is used.
# =============================================================================

rng_entry_cv = np.random.default_rng(
    ENTRY_CV_SEED
)


entry_psu_fold_rows = []


for stratum, stratum_psu_data in (
    entry_development_psu_frame
    .groupby(
        "v022",
        sort=True
    )
):

    psus = (
        stratum_psu_data[
            "v021"
        ]
        .to_numpy()
        .copy()
    )


    # Randomize PSU order reproducibly.
    rng_entry_cv.shuffle(
        psus
    )


    n_psus = len(
        psus
    )


    # Base number of PSUs allocated to every fold.
    base_size = (
        n_psus
        // ENTRY_N_FOLDS
    )


    # Remaining PSUs after equal allocation.
    remainder = (
        n_psus
        % ENTRY_N_FOLDS
    )


    fold_sizes = np.full(
        ENTRY_N_FOLDS,
        base_size,
        dtype=int
    )


    # Randomize which folds receive remainder PSU(s).
    remainder_fold_order = (
        rng_entry_cv.permutation(
            ENTRY_N_FOLDS
        )
    )


    if remainder > 0:

        fold_sizes[
            remainder_fold_order[
                :remainder
            ]
        ] += 1


    assert fold_sizes.sum() == n_psus
    assert fold_sizes.min() >= 1


    start = 0


    for fold_zero_based, fold_size in enumerate(
        fold_sizes
    ):

        fold_psus = (
            psus[
                start:
                start + fold_size
            ]
        )


        for psu in fold_psus:

            entry_psu_fold_rows.append({
                "v021":
                    psu,

                "v022":
                    stratum,

                "fold":
                    fold_zero_based + 1
            })


        start += fold_size


# =============================================================================
# 7. CANONICAL PSU-FOLD ASSIGNMENT
# =============================================================================

entry_psu_fold_assignment = pd.DataFrame(
    entry_psu_fold_rows
)


assert len(
    entry_psu_fold_assignment
) == len(
    entry_development_psu_frame
)

assert entry_psu_fold_assignment[
    "v021"
].is_unique

assert entry_psu_fold_assignment[
    "fold"
].between(
    1,
    ENTRY_N_FOLDS
).all()


# =============================================================================
# 8. MAP FOLD ASSIGNMENT BACK TO DEVELOPMENT WOMEN
# =============================================================================

entry_development_cv_index = (
    anc_entry_train[
        [
            "caseid",
            "v021",
            "v022"
        ]
    ]
    .merge(
        entry_psu_fold_assignment,
        on=[
            "v021",
            "v022"
        ],
        how="left",
        validate="many_to_one"
    )
)


assert entry_development_cv_index[
    "fold"
].notna().all()

assert entry_development_cv_index[
    "caseid"
].is_unique


# =============================================================================
# 9. CONSTRUCT CANONICAL CV SPLITS
#
# These positional indices correspond exactly to anc_entry_train / X_entry_*.
# =============================================================================

entry_cv_splits = []


for fold in range(
    1,
    ENTRY_N_FOLDS + 1
):

    validation_mask = (
        entry_development_cv_index[
            "fold"
        ].eq(
            fold
        )
        .to_numpy()
    )


    val_idx = np.flatnonzero(
        validation_mask
    )

    fit_idx = np.flatnonzero(
        ~validation_mask
    )


    entry_cv_splits.append(
        (
            fit_idx,
            val_idx
        )
    )


# =============================================================================
# 10. AUDIT EACH FOLD
# =============================================================================

entry_cv_audit_rows = []

validation_counts = np.zeros(
    len(
        anc_entry_train
    ),
    dtype=int
)


for fold_number, (
    fit_idx,
    val_idx
) in enumerate(
    entry_cv_splits,
    start=1
):

    validation_counts[
        val_idx
    ] += 1


    fit_psus = set(
        groups_entry.iloc[
            fit_idx
        ]
    )

    val_psus = set(
        groups_entry.iloc[
            val_idx
        ]
    )


    fit_strata = set(
        strata_entry.iloc[
            fit_idx
        ]
    )

    val_strata = set(
        strata_entry.iloc[
            val_idx
        ]
    )


    assert fit_psus.isdisjoint(
        val_psus
    )

    assert len(
        val_strata
    ) == 27


    entry_cv_audit_rows.append({
        "Fold":
            fold_number,

        "Fit_women":
            len(
                fit_idx
            ),

        "Validation_women":
            len(
                val_idx
            ),

        "Fit_PSUs":
            len(
                fit_psus
            ),

        "Validation_PSUs":
            len(
                val_psus
            ),

        "Validation_strata":
            len(
                val_strata
            ),

        "PSU_overlap":
            len(
                fit_psus
                & val_psus
            ),

        "Fit_weighted_ANC8_prevalence":
            np.average(
                y_entry.iloc[
                    fit_idx
                ],
                weights=(
                    anc_entry_train[
                        "v005"
                    ]
                    .iloc[
                        fit_idx
                    ]
                    .astype(float)
                )
            ),

        "Validation_weighted_ANC8_prevalence":
            np.average(
                y_entry.iloc[
                    val_idx
                ],
                weights=(
                    anc_entry_train[
                        "v005"
                    ]
                    .iloc[
                        val_idx
                    ]
                    .astype(float)
                )
            )
    })


entry_cv_audit = pd.DataFrame(
    entry_cv_audit_rows
)


# =============================================================================
# 11. GLOBAL CV INTEGRITY
# =============================================================================

assert (
    validation_counts
    == 1
).all()


assert (
    entry_psu_fold_assignment[
        "v021"
    ]
    .nunique()
    == anc_entry_train[
        "v021"
    ]
    .nunique()
)


assert (
    entry_cv_audit[
        "PSU_overlap"
    ] == 0
).all()


assert (
    entry_cv_audit[
        "Validation_strata"
    ] == 27
).all()


# =============================================================================
# 12. DISPLAY
# =============================================================================

print("=" * 116)
print("ANC-ENTRY PAIRED SPECIFICATIONS AND LOCKED DEVELOPMENT CV")
print("=" * 116)

print(
    "Baseline specification predictors:",
    len(
        ANC_ENTRY_BASELINE_PREDICTORS
    )
)

print(
    "Extended specification predictors:",
    len(
        ANC_ENTRY_EXTENDED_PREDICTORS
    )
)

print(
    "Only added predictor:",
    "early_anc"
)


print("\nDevelopment structure")
print("---------------------")

print(
    "Development women:",
    len(
        anc_entry_train
    )
)

print(
    "Development PSUs:",
    groups_entry.nunique()
)

print(
    "Development strata:",
    strata_entry.nunique()
)

print(
    "Minimum PSUs per stratum:",
    minimum_entry_psus_per_stratum
)


print("\nCross-validation audit")
print("----------------------")

print(
    entry_cv_audit.to_string(
        index=False,
        formatters={
            "Fit_weighted_ANC8_prevalence":
                "{:.2%}".format,

            "Validation_weighted_ANC8_prevalence":
                "{:.2%}".format
        }
    )
)


print("\nIntegrity")
print("---------")

print(
    "CV folds:",
    len(
        entry_cv_splits
    )
)

print(
    "Outcome used to assign folds:",
    "No"
)

print(
    "Every development woman validates exactly once:",
    "Yes"
)

print(
    "Every development PSU assigned exactly once:",
    "Yes"
)

print(
    "All 27 strata represented in every validation fold:",
    "Yes"
)

print(
    "Held-out test women used in CV:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY PAIRED SPECIFICATIONS "
    "AND DEVELOPMENT CV LOCKED"
)

## 2.32 Preprocessing extension for the ANC-entry model

The ANC-entry analysis uses the same preprocessing rules established for the
primary 21-predictor baseline models.

The only additional predictor is:

- `early_anc`: 0 = later ANC initiation; 1 = early ANC initiation.

Because `early_anc` is already a resolved numeric binary variable, it is passed
through unchanged. It is not one-hot encoded and is not treated as a
continuous measurement.

Two paired preprocessing specifications are defined:

1. **ANC-entry baseline preprocessing:** the original 21-predictor
   preprocessing;
2. **ANC-entry extended preprocessing:** the same preprocessing plus
   `early_anc`.

For Logistic regression:

- continuous numeric predictors retain median imputation and standardization;
- governed binary predictors retain their fixed 0/1 encoding;
- multicategory predictors retain full one-hot encoding;
- `early_anc` is retained directly as 0/1.

For tree-based models:

- continuous numeric predictors retain median imputation without scaling;
- all other preprocessing rules are unchanged;
- `early_anc` is retained directly as 0/1.

No preprocessing is fitted in this section and the held-out ANC-entry test set
is not inspected.

In [ ]:
# =============================================================================
# 2.32 DEFINE ANC-ENTRY PREPROCESSING EXTENSION
#
# PURPOSE
# -------
# Create paired preprocessing specifications:
#
#   Baseline: 21 governed predictors
#   Extended: same 21 predictors + early_anc
#
# No preprocessing is fitted here.
# =============================================================================

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer


# =============================================================================
# 1. DIRECT PASS-THROUGH TRANSFORMER FOR early_anc
#
# feature_names_out="one-to-one" ensures that the feature name can later be
# recovered reliably for lineage / explainability auditing.
# =============================================================================

early_anc_transformer = FunctionTransformer(
    validate=False,
    feature_names_out="one-to-one"
)


# =============================================================================
# 2. ANC-ENTRY BASELINE PREPROCESSORS
#
# These are fresh clones of the already-governed baseline preprocessing
# specifications.
# =============================================================================

preprocessor_entry_baseline_logistic = clone(
    preprocessor_logistic
)

preprocessor_entry_baseline_tree = clone(
    preprocessor_tree
)


# =============================================================================
# 3. ANC-ENTRY EXTENDED PREPROCESSOR: LOGISTIC REGRESSION
#
# Original 21 predictors retain exactly the same preprocessing rules.
# early_anc is added only as a direct 0/1 feature.
# =============================================================================

preprocessor_entry_extended_logistic = ColumnTransformer(
    transformers=[
        (
            "numeric",
            clone(numeric_logistic),
            NUMERIC_PREDICTORS
        ),

        (
            "binary",
            clone(binary_encoder),
            BINARY_PREDICTORS
        ),

        (
            "categorical",
            clone(categorical_pipeline),
            CATEGORICAL_PREDICTORS_ML
        ),

        (
            "early_anc",
            clone(early_anc_transformer),
            ["early_anc"]
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# =============================================================================
# 4. ANC-ENTRY EXTENDED PREPROCESSOR: TREE MODELS
# =============================================================================

preprocessor_entry_extended_tree = ColumnTransformer(
    transformers=[
        (
            "numeric",
            clone(numeric_tree),
            NUMERIC_PREDICTORS
        ),

        (
            "binary",
            clone(binary_encoder),
            BINARY_PREDICTORS
        ),

        (
            "categorical",
            clone(categorical_pipeline),
            CATEGORICAL_PREDICTORS_ML
        ),

        (
            "early_anc",
            clone(early_anc_transformer),
            ["early_anc"]
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# =============================================================================
# 5. STRUCTURAL CHECKS
# =============================================================================

assert len(
    ANC_ENTRY_BASELINE_PREDICTORS
) == 21

assert len(
    ANC_ENTRY_EXTENDED_PREDICTORS
) == 22


# The only new predictor must be early_anc.
assert set(
    ANC_ENTRY_EXTENDED_PREDICTORS
) - set(
    ANC_ENTRY_BASELINE_PREDICTORS
) == {
    "early_anc"
}


# early_anc must be completely resolved in development.
assert X_entry_ext[
    "early_anc"
].notna().all()

assert set(
    X_entry_ext[
        "early_anc"
    ].unique()
) <= {
    0.0,
    1.0
}


# =============================================================================
# 6. PREPROCESSOR GOVERNANCE AUDIT
#
# We inspect configuration only.
# Nothing is fitted.
# =============================================================================

entry_preprocessing_audit = pd.DataFrame(
    [
        {
            "Specification":
                "Baseline logistic",

            "Predictors":
                21,

            "Continuous_numeric":
                len(NUMERIC_PREDICTORS),

            "Governed_binary":
                len(BINARY_PREDICTORS),

            "Multicategory":
                len(CATEGORICAL_PREDICTORS_ML),

            "early_anc":
                "Not included",

            "Scaling":
                "Numeric only"
        },

        {
            "Specification":
                "Extended logistic",

            "Predictors":
                22,

            "Continuous_numeric":
                len(NUMERIC_PREDICTORS),

            "Governed_binary":
                len(BINARY_PREDICTORS),

            "Multicategory":
                len(CATEGORICAL_PREDICTORS_ML),

            "early_anc":
                "Direct 0/1",

            "Scaling":
                "Numeric only"
        },

        {
            "Specification":
                "Baseline tree",

            "Predictors":
                21,

            "Continuous_numeric":
                len(NUMERIC_PREDICTORS),

            "Governed_binary":
                len(BINARY_PREDICTORS),

            "Multicategory":
                len(CATEGORICAL_PREDICTORS_ML),

            "early_anc":
                "Not included",

            "Scaling":
                "None"
        },

        {
            "Specification":
                "Extended tree",

            "Predictors":
                22,

            "Continuous_numeric":
                len(NUMERIC_PREDICTORS),

            "Governed_binary":
                len(BINARY_PREDICTORS),

            "Multicategory":
                len(CATEGORICAL_PREDICTORS_ML),

            "early_anc":
                "Direct 0/1",

            "Scaling":
                "None"
        }
    ]
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 108)
print("ANC-ENTRY PREPROCESSING EXTENSION DEFINED")
print("=" * 108)

print(
    entry_preprocessing_audit.to_string(
        index=False
    )
)


print("\nPredictor architecture")
print("----------------------")

print(
    "Baseline predictors:",
    len(
        ANC_ENTRY_BASELINE_PREDICTORS
    )
)

print(
    "Extended predictors:",
    len(
        ANC_ENTRY_EXTENDED_PREDICTORS
    )
)

print(
    "Only added predictor:",
    "early_anc"
)

print(
    "early_anc coding:",
    "0 = later ANC, 1 = early ANC"
)

print(
    "early_anc one-hot encoded:",
    "No"
)

print(
    "early_anc standardized:",
    "No"
)


print("\nGovernance")
print("----------")

print(
    "Original 21-predictor preprocessing changed:",
    "No"
)

print(
    "Preprocessors fitted here:",
    "No"
)

print(
    "Held-out test data inspected:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY PREPROCESSING "
    "SPECIFICATIONS DEFINED"
)

## 2.33 ANC-entry weighting strategy and paired model pipelines

The ANC-entry analysis retains the same weighting principles used in the
primary baseline analysis.

Two weight representations are used for different purposes:

- **survey evaluation weight:** `v005 / 1,000,000`, retaining the DHS relative
  sampling weights for estimating population-relevant predictive performance;
- **model-fitting weight:** the survey weight normalized to mean 1.

Within every cross-validation fitting fold, the fitting weights are
renormalized again to mean 1 before being passed to the estimator. This avoids
allowing the arbitrary absolute scale of sampling weights to influence
regularization or numerical optimization.

Survey weights, PSU identifiers, and strata are never included as predictors.

For each algorithm, two otherwise comparable pipelines are defined:

1. **baseline specification:** the original 21 predictors;
2. **extended specification:** the same 21 predictors plus `early_anc`.

The three model families remain:

- L2-regularized Logistic regression;
- Random Forest;
- XGBoost.

The same prespecified hyperparameter search spaces used in the primary analysis
will be reused.

No model is fitted in this section and the ANC-entry held-out test set remains
untouched.

In [ ]:
# =============================================================================
# 2.33 ANC-ENTRY WEIGHTS AND PAIRED MODEL PIPELINES
#
# Defines:
#   - survey evaluation weights
#   - mean-1 model-fitting weights
#   - paired baseline / extended pipelines for all three algorithms
#
# No fitting occurs here.
# =============================================================================


# =============================================================================
# 1. DEVELOPMENT SURVEY WEIGHTS
# =============================================================================

survey_weight_entry_dev = (
    anc_entry_train[
        "v005"
    ]
    .astype(float)
    / 1_000_000
)


assert len(
    survey_weight_entry_dev
) == len(
    anc_entry_train
)

assert (
    survey_weight_entry_dev > 0
).all()


# =============================================================================
# 2. GLOBAL MEAN-1 FITTING WEIGHTS
#
# These are the canonical development fitting weights.
#
# IMPORTANT:
# During CV, the FIT portion of these weights must again be normalized to
# mean 1 inside each fold.
# =============================================================================

fit_weight_entry_dev = (
    survey_weight_entry_dev
    / survey_weight_entry_dev.mean()
)


assert np.isclose(
    fit_weight_entry_dev.mean(),
    1.0
)


# =============================================================================
# 3. LOGISTIC REGRESSION PIPELINES
# =============================================================================

pipeline_entry_logistic_baseline = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_baseline_logistic
        )
    ),

    (
        "model",
        LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            class_weight=None,
            random_state=MODEL_SEED
        )
    )
])


pipeline_entry_logistic_extended = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_extended_logistic
        )
    ),

    (
        "model",
        LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            class_weight=None,
            random_state=MODEL_SEED
        )
    )
])


# =============================================================================
# 4. RANDOM FOREST PIPELINES
# =============================================================================

pipeline_entry_rf_baseline = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_baseline_tree
        )
    ),

    (
        "model",
        RandomForestClassifier(
            criterion="gini",
            bootstrap=True,
            class_weight=None,
            random_state=MODEL_SEED,
            n_jobs=-1
        )
    )
])


pipeline_entry_rf_extended = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_extended_tree
        )
    ),

    (
        "model",
        RandomForestClassifier(
            criterion="gini",
            bootstrap=True,
            class_weight=None,
            random_state=MODEL_SEED,
            n_jobs=-1
        )
    )
])


# =============================================================================
# 5. XGBOOST PIPELINES
# =============================================================================

pipeline_entry_xgb_baseline = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_baseline_tree
        )
    ),

    (
        "model",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=MODEL_SEED,
            n_jobs=-1,
            verbosity=0
        )
    )
])


pipeline_entry_xgb_extended = Pipeline([
    (
        "preprocessor",
        clone(
            preprocessor_entry_extended_tree
        )
    ),

    (
        "model",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=MODEL_SEED,
            n_jobs=-1,
            verbosity=0
        )
    )
])


# =============================================================================
# 6. CANONICAL PIPELINE REGISTRY
# =============================================================================

entry_candidate_pipelines = {
    "Logistic regression": {
        "Baseline":
            pipeline_entry_logistic_baseline,

        "Extended":
            pipeline_entry_logistic_extended
    },

    "Random Forest": {
        "Baseline":
            pipeline_entry_rf_baseline,

        "Extended":
            pipeline_entry_rf_extended
    },

    "XGBoost": {
        "Baseline":
            pipeline_entry_xgb_baseline,

        "Extended":
            pipeline_entry_xgb_extended
    }
}


# =============================================================================
# 7. HYPERPARAMETER GRIDS
#
# Reuse the exact prespecified grids from the primary model analysis.
# =============================================================================

ENTRY_PARAM_GRIDS = {
    "Logistic regression":
        PARAM_GRID_LOGISTIC,

    "Random Forest":
        PARAM_GRID_RF,

    "XGBoost":
        PARAM_GRID_XGB
}


# =============================================================================
# 8. GOVERNANCE / INTEGRITY CHECKS
# =============================================================================

assert set(
    entry_candidate_pipelines.keys()
) == {
    "Logistic regression",
    "Random Forest",
    "XGBoost"
}


for algorithm, specifications in (
    entry_candidate_pipelines.items()
):

    assert set(
        specifications.keys()
    ) == {
        "Baseline",
        "Extended"
    }


# No class rebalancing.
assert (
    pipeline_entry_logistic_baseline[
        "model"
    ].class_weight
    is None
)

assert (
    pipeline_entry_logistic_extended[
        "model"
    ].class_weight
    is None
)

assert (
    pipeline_entry_rf_baseline[
        "model"
    ].class_weight
    is None
)

assert (
    pipeline_entry_rf_extended[
        "model"
    ].class_weight
    is None
)


# XGBoost must not use class-rebalancing weights.
#
# Depending on the installed XGBoost version, the unused/default
# scale_pos_weight parameter may appear as either None or 1.
# Both indicate that no class reweighting has been requested.

for model in [
    pipeline_entry_xgb_baseline["model"],
    pipeline_entry_xgb_extended["model"]
]:

    scale_pos_weight = model.get_params().get(
        "scale_pos_weight",
        None
    )

    assert scale_pos_weight in [
        None,
        1,
        1.0
    ]

# Survey-design variables must not enter either predictor specification.
for forbidden_variable in [
    "v005",
    "v021",
    "v022"
]:

    assert forbidden_variable not in (
        ANC_ENTRY_BASELINE_PREDICTORS
    )

    assert forbidden_variable not in (
        ANC_ENTRY_EXTENDED_PREDICTORS
    )


# =============================================================================
# 9. DISPLAY
# =============================================================================

print("=" * 112)
print("ANC-ENTRY WEIGHTS AND PAIRED MODEL PIPELINES DEFINED")
print("=" * 112)


print("\nDevelopment weighting")
print("---------------------")

print(
    "Development women:",
    len(
        anc_entry_train
    )
)

print(
    "Survey-weight mean:",
    f"{survey_weight_entry_dev.mean():.6f}"
)

print(
    "Mean-1 fitting-weight mean:",
    f"{fit_weight_entry_dev.mean():.6f}"
)

print(
    "Fold-specific fitting-weight renormalization required:",
    "Yes"
)


print("\nPaired model architecture")
print("-------------------------")

for algorithm in (
    entry_candidate_pipelines
):

    print(
        f"{algorithm}: baseline + extended"
    )


print(
    "\nBaseline predictors:",
    len(
        ANC_ENTRY_BASELINE_PREDICTORS
    )
)

print(
    "Extended predictors:",
    len(
        ANC_ENTRY_EXTENDED_PREDICTORS
    )
)

print(
    "Only additional predictor:",
    "early_anc"
)


print("\nHyperparameter search sizes")
print("---------------------------")

print(
    "Logistic regression:",
    len(
        list(
            ParameterGrid(
                PARAM_GRID_LOGISTIC
            )
        )
    )
)

print(
    "Random Forest:",
    len(
        list(
            ParameterGrid(
                PARAM_GRID_RF
            )
        )
    )
)

print(
    "XGBoost:",
    len(
        list(
            ParameterGrid(
                PARAM_GRID_XGB
            )
        )
    )
)


print("\nGovernance")
print("----------")

print(
    "Class balancing introduced:",
    "No"
)

print(
    "Survey design variables used as predictors:",
    "No"
)

print(
    "Models fitted here:",
    "No"
)

print(
    "Held-out ANC-entry test accessed:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY WEIGHTS AND "
    "PAIRED PIPELINES LOCKED"
)

## 2.34 Paired logistic-regression tuning at ANC entry

Logistic regression is the first model family evaluated for incremental
predictive value at the ANC-entry horizon.

Two specifications are tuned independently:

- **Baseline:** the original 21 predictors;
- **Extended:** the same 21 predictors plus `early_anc`.

Both specifications use:

- the same 4,017 development women;
- the same locked five-fold stratified PSU cross-validation structure;
- the same survey-weighted evaluation framework;
- fold-specific mean-1 fitting weights;
- the same prespecified regularization grid:
  `C = 0.01, 0.1, 1.0, 10.0`.

For every candidate configuration, one out-of-fold probability is generated
for every development woman.

Hyperparameter selection uses the same rule as the primary analysis:

1. maximize survey-weighted pooled out-of-fold Average Precision;
2. if exactly tied, minimize survey-weighted Brier score;
3. if still tied, maximize survey-weighted AUROC.

The baseline and extended specifications are tuned separately so that the
incremental comparison does not force both specifications to share a
regularization strength that may be inappropriate after adding `early_anc`.

The held-out ANC-entry test set is not used.

In [ ]:
# =============================================================================
# 2.34 CANONICAL ANC-ENTRY OOF TUNING ENGINE
#      + PAIRED LOGISTIC-REGRESSION TUNING
#
# This function will later be reused WITHOUT modification for:
#   - Random Forest
#   - XGBoost
#
# Governance
# ----------
# - same locked entry_cv_splits for every candidate;
# - fold fitting weights renormalized to mean 1;
# - survey weights used for pooled OOF evaluation;
# - no held-out test access;
# - one OOF probability per development woman;
# - selection rule: AP -> Brier -> AUROC.
# =============================================================================


# =============================================================================
# 1. CANONICAL ANC-ENTRY CANDIDATE-EVALUATION FUNCTION
# =============================================================================

def evaluate_entry_candidate_oof(
    pipeline,
    params,
    X,
    y,
    cv_splits_locked,
    fitting_weights,
    evaluation_weights
):
    """
    Evaluate one ANC-entry model configuration by locked cross-validation.

    Parameters
    ----------
    pipeline
        Unfitted sklearn-compatible pipeline.

    params
        Hyperparameter dictionary passed through set_params().

    X
        Predictor dataframe for one specification.

    y
        Binary ANC8 outcome.

    cv_splits_locked
        Canonical ANC-entry (fit_idx, val_idx) folds created in Section 2.31.

    fitting_weights
        Development model-fitting weights.

    evaluation_weights
        DHS survey weights used for pooled OOF metric estimation.

    Returns
    -------
    dict containing:
        Weighted_AP
        Weighted_AUROC
        Weighted_Brier
        OOF_probability
    """

    # -------------------------------------------------------------------------
    # Allocate exactly one OOF probability per development woman.
    # -------------------------------------------------------------------------

    oof_probability = np.full(
        len(X),
        np.nan,
        dtype=float
    )

    validation_count = np.zeros(
        len(X),
        dtype=int
    )


    # =========================================================================
    # LOCKED CROSS-VALIDATION
    # =========================================================================

    for fit_idx, val_idx in cv_splits_locked:

        # ---------------------------------------------------------------------
        # Fresh clone prevents information carrying between folds.
        # ---------------------------------------------------------------------

        model = clone(
            pipeline
        )

        model.set_params(
            **params
        )


        # ---------------------------------------------------------------------
        # Fold-specific fitting weights.
        #
        # Renormalize the FIT fold to mean 1. This is important because the
        # absolute scale of sample weights can affect regularization /
        # optimization even though relative survey weighting is unchanged.
        # ---------------------------------------------------------------------

        fold_fit_weight = (
            fitting_weights.iloc[
                fit_idx
            ]
            .astype(float)
            .copy()
        )

        fold_fit_weight = (
            fold_fit_weight
            / fold_fit_weight.mean()
        )


        assert np.isclose(
            fold_fit_weight.mean(),
            1.0
        )


        # ---------------------------------------------------------------------
        # Fit using only the fitting portion of this fold.
        # ---------------------------------------------------------------------

        model.fit(
            X.iloc[
                fit_idx
            ],
            y.iloc[
                fit_idx
            ],
            model__sample_weight=fold_fit_weight
        )


        # ---------------------------------------------------------------------
        # Generate probability for the untouched validation portion.
        # ---------------------------------------------------------------------

        fold_probability = (
            model.predict_proba(
                X.iloc[
                    val_idx
                ]
            )[:, 1]
        )


        assert np.isfinite(
            fold_probability
        ).all()

        assert (
            (fold_probability >= 0)
            &
            (fold_probability <= 1)
        ).all()


        oof_probability[
            val_idx
        ] = fold_probability

        validation_count[
            val_idx
        ] += 1


    # =========================================================================
    # OOF INTEGRITY
    # =========================================================================

    assert (
        validation_count == 1
    ).all()

    assert np.isfinite(
        oof_probability
    ).all()


    # =========================================================================
    # POOLED SURVEY-WEIGHTED OOF PERFORMANCE
    #
    # Evaluation uses the ORIGINAL survey-weight scale.
    # =========================================================================

    weighted_ap = average_precision_score(
        y,
        oof_probability,
        sample_weight=evaluation_weights
    )

    weighted_auroc = roc_auc_score(
        y,
        oof_probability,
        sample_weight=evaluation_weights
    )

    weighted_brier = brier_score_loss(
        y,
        oof_probability,
        sample_weight=evaluation_weights
    )


    return {
        "Weighted_AP":
            float(
                weighted_ap
            ),

        "Weighted_AUROC":
            float(
                weighted_auroc
            ),

        "Weighted_Brier":
            float(
                weighted_brier
            ),

        "OOF_probability":
            oof_probability
    }


# =============================================================================
# 2. GENERIC GRID-TUNING WRAPPER
#
# This wrapper uses the SAME selection rule for every ANC-entry algorithm and
# specification.
# =============================================================================

def tune_entry_specification(
    algorithm_name,
    specification_name,
    pipeline,
    param_grid,
    X,
    y,
    cv_splits_locked,
    fitting_weights,
    evaluation_weights
):
    """
    Tune one algorithm/specification using pooled weighted OOF performance.

    Selection hierarchy:
        1. highest Weighted_AP
        2. lowest Weighted_Brier
        3. highest Weighted_AUROC
    """

    candidate_rows = []

    candidate_oof = {}


    # -------------------------------------------------------------------------
    # ParameterGrid preserves the complete prespecified candidate set.
    # -------------------------------------------------------------------------

    parameter_candidates = list(
        ParameterGrid(
            param_grid
        )
    )


    for configuration_number, params in enumerate(
        parameter_candidates,
        start=1
    ):

        result = evaluate_entry_candidate_oof(
            pipeline=pipeline,
            params=params,
            X=X,
            y=y,
            cv_splits_locked=cv_splits_locked,
            fitting_weights=fitting_weights,
            evaluation_weights=evaluation_weights
        )


        config_id = (
            f"{algorithm_name}_{specification_name}_"
            f"{configuration_number:02d}"
        )


        candidate_rows.append({
            "Config_ID":
                config_id,

            "Algorithm":
                algorithm_name,

            "Specification":
                specification_name,

            # Keep an exact Python dictionary so values such as None are not
            # converted to NaN by dataframe display/storage.
            "params_exact":
                params.copy(),

            "Weighted_AP":
                result[
                    "Weighted_AP"
                ],

            "Weighted_AUROC":
                result[
                    "Weighted_AUROC"
                ],

            "Weighted_Brier":
                result[
                    "Weighted_Brier"
                ]
        })


        candidate_oof[
            config_id
        ] = result[
            "OOF_probability"
        ].copy()


    results = pd.DataFrame(
        candidate_rows
    )


    # -------------------------------------------------------------------------
    # Prespecified deterministic selection rule.
    # -------------------------------------------------------------------------

    results = (
        results
        .sort_values(
            by=[
                "Weighted_AP",
                "Weighted_Brier",
                "Weighted_AUROC"
            ],
            ascending=[
                False,
                True,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    selected_row = (
        results.iloc[0]
    )


    selected_config_id = (
        selected_row[
            "Config_ID"
        ]
    )


    selected_params = (
        selected_row[
            "params_exact"
        ].copy()
    )


    selected_oof = (
        candidate_oof[
            selected_config_id
        ]
        .copy()
    )


    # -------------------------------------------------------------------------
    # Final integrity.
    # -------------------------------------------------------------------------

    assert len(
        selected_oof
    ) == len(
        y
    )

    assert np.isfinite(
        selected_oof
    ).all()


    return {
        "results":
            results,

        "selected_config_id":
            selected_config_id,

        "selected_params":
            selected_params,

        "selected_oof":
            selected_oof
    }


# =============================================================================
# 3. TUNE ANC-ENTRY LOGISTIC BASELINE SPECIFICATION
# =============================================================================

entry_logistic_baseline_tuning = tune_entry_specification(
    algorithm_name="LOG",
    specification_name="BASE",
    pipeline=pipeline_entry_logistic_baseline,
    param_grid=PARAM_GRID_LOGISTIC,
    X=X_entry_base,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 4. TUNE ANC-ENTRY LOGISTIC EXTENDED SPECIFICATION
# =============================================================================

entry_logistic_extended_tuning = tune_entry_specification(
    algorithm_name="LOG",
    specification_name="EXT",
    pipeline=pipeline_entry_logistic_extended,
    param_grid=PARAM_GRID_LOGISTIC,
    X=X_entry_ext,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 5. CANONICAL LOGISTIC OUTPUT OBJECTS
# =============================================================================

entry_logistic_baseline_results = (
    entry_logistic_baseline_tuning[
        "results"
    ]
)

entry_logistic_extended_results = (
    entry_logistic_extended_tuning[
        "results"
    ]
)


BEST_ENTRY_LOGISTIC_BASELINE_CONFIG_ID = (
    entry_logistic_baseline_tuning[
        "selected_config_id"
    ]
)

BEST_ENTRY_LOGISTIC_EXTENDED_CONFIG_ID = (
    entry_logistic_extended_tuning[
        "selected_config_id"
    ]
)


BEST_ENTRY_LOGISTIC_BASELINE_PARAMS = (
    entry_logistic_baseline_tuning[
        "selected_params"
    ]
)

BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS = (
    entry_logistic_extended_tuning[
        "selected_params"
    ]
)


oof_entry_logistic_baseline = (
    entry_logistic_baseline_tuning[
        "selected_oof"
    ]
)

oof_entry_logistic_extended = (
    entry_logistic_extended_tuning[
        "selected_oof"
    ]
)


# =============================================================================
# 6. COMBINE RESULTS FOR DISPLAY
# =============================================================================

entry_logistic_cv_results = pd.concat(
    [
        entry_logistic_baseline_results,
        entry_logistic_extended_results
    ],
    ignore_index=True
)


# =============================================================================
# 7. DISPLAY SELECTED LOGISTIC CONFIGURATIONS
# =============================================================================

selected_entry_logistic_rows = pd.DataFrame([
    {
        "Specification":
            "Baseline",

        "Config_ID":
            BEST_ENTRY_LOGISTIC_BASELINE_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_LOGISTIC_BASELINE_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_logistic_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_logistic_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_logistic_baseline,
                sample_weight=survey_weight_entry_dev
            )
    },

    {
        "Specification":
            "Extended",

        "Config_ID":
            BEST_ENTRY_LOGISTIC_EXTENDED_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_logistic_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_logistic_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_logistic_extended,
                sample_weight=survey_weight_entry_dev
            )
    }
])


print("=" * 120)
print("ANC-ENTRY PAIRED LOGISTIC-REGRESSION TUNING")
print("=" * 120)

print(
    selected_entry_logistic_rows
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nSelection governance")
print("--------------------")

print(
    "Same development women:",
    "Yes"
)

print(
    "Same locked CV folds:",
    "Yes"
)

print(
    "Same survey-weighted evaluation:",
    "Yes"
)

print(
    "Baseline and extended tuned independently:",
    "Yes"
)

print(
    "Primary tuning criterion:",
    "Weighted pooled OOF Average Precision"
)

print(
    "Tie-breakers:",
    "Brier, then AUROC"
)

print(
    "Held-out test accessed:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY LOGISTIC "
    "BASELINE AND EXTENDED MODELS TUNED"
)

## 2.35 Paired Random Forest tuning at ANC entry

Random Forest is evaluated using the same paired ANC-entry development
framework.

Two specifications are tuned independently:

- **Baseline:** the original 21 predictors;
- **Extended:** the same 21 predictors plus `early_anc`.

Both specifications use:

- the same 4,017 development women;
- the same locked five-fold stratified PSU cross-validation folds;
- fold-specific mean-1 fitting weights;
- survey-weighted pooled out-of-fold evaluation;
- the same prespecified Random Forest hyperparameter grid used in the primary
  analysis.

Hyperparameter selection follows the locked hierarchy:

1. highest survey-weighted Average Precision;
2. lower Brier score if exactly tied;
3. higher AUROC if still tied.

The held-out ANC-entry test set is not accessed.

In [ ]:
# =============================================================================
# 2.35 PAIRED RANDOM FOREST TUNING AT ANC ENTRY
#
# Reuses the canonical tuning engine defined and audited in Section 2.34.
#
# No new CV structure is created.
# No held-out test information is used.
# =============================================================================


# =============================================================================
# 1. TUNE BASELINE RANDOM FOREST
# =============================================================================

entry_rf_baseline_tuning = tune_entry_specification(
    algorithm_name="RF",
    specification_name="BASE",
    pipeline=pipeline_entry_rf_baseline,
    param_grid=PARAM_GRID_RF,
    X=X_entry_base,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 2. TUNE EXTENDED RANDOM FOREST
# =============================================================================

entry_rf_extended_tuning = tune_entry_specification(
    algorithm_name="RF",
    specification_name="EXT",
    pipeline=pipeline_entry_rf_extended,
    param_grid=PARAM_GRID_RF,
    X=X_entry_ext,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 3. CANONICAL RESULT OBJECTS
# =============================================================================

entry_rf_baseline_results = (
    entry_rf_baseline_tuning[
        "results"
    ]
)

entry_rf_extended_results = (
    entry_rf_extended_tuning[
        "results"
    ]
)


BEST_ENTRY_RF_BASELINE_CONFIG_ID = (
    entry_rf_baseline_tuning[
        "selected_config_id"
    ]
)

BEST_ENTRY_RF_EXTENDED_CONFIG_ID = (
    entry_rf_extended_tuning[
        "selected_config_id"
    ]
)


BEST_ENTRY_RF_BASELINE_PARAMS = (
    entry_rf_baseline_tuning[
        "selected_params"
    ]
)

BEST_ENTRY_RF_EXTENDED_PARAMS = (
    entry_rf_extended_tuning[
        "selected_params"
    ]
)


oof_entry_rf_baseline = (
    entry_rf_baseline_tuning[
        "selected_oof"
    ]
)

oof_entry_rf_extended = (
    entry_rf_extended_tuning[
        "selected_oof"
    ]
)


# =============================================================================
# 4. SUMMARIZE SELECTED CONFIGURATIONS
# =============================================================================

selected_entry_rf_rows = pd.DataFrame([
    {
        "Specification":
            "Baseline",

        "Config_ID":
            BEST_ENTRY_RF_BASELINE_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_RF_BASELINE_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_rf_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_rf_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_rf_baseline,
                sample_weight=survey_weight_entry_dev
            )
    },

    {
        "Specification":
            "Extended",

        "Config_ID":
            BEST_ENTRY_RF_EXTENDED_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_RF_EXTENDED_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_rf_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_rf_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_rf_extended,
                sample_weight=survey_weight_entry_dev
            )
    }
])


# =============================================================================
# 5. INTEGRITY
# =============================================================================

assert len(
    entry_rf_baseline_results
) == 48

assert len(
    entry_rf_extended_results
) == 48

assert len(
    oof_entry_rf_baseline
) == len(
    y_entry
)

assert len(
    oof_entry_rf_extended
) == len(
    y_entry
)

assert np.isfinite(
    oof_entry_rf_baseline
).all()

assert np.isfinite(
    oof_entry_rf_extended
).all()


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 132)
print("ANC-ENTRY PAIRED RANDOM FOREST TUNING")
print("=" * 132)

print(
    selected_entry_rf_rows
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nSelection governance")
print("--------------------")

print(
    "Candidate configurations per specification:",
    48
)

print(
    "Same development women:",
    "Yes"
)

print(
    "Same locked CV folds:",
    "Yes"
)

print(
    "Baseline and extended tuned independently:",
    "Yes"
)

print(
    "Primary tuning criterion:",
    "Weighted pooled OOF Average Precision"
)

print(
    "Held-out test accessed:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY RANDOM FOREST "
    "BASELINE AND EXTENDED MODELS TUNED"
)

## 2.36 Paired XGBoost tuning at ANC entry

XGBoost is evaluated using the same paired ANC-entry development framework
used for Logistic regression and Random Forest.

Two specifications are tuned independently:

- **Baseline:** the original 21 predictors;
- **Extended:** the same 21 predictors plus `early_anc`.

Both specifications use:

- the same 4,017 development women;
- the same locked five-fold stratified PSU cross-validation folds;
- fold-specific mean-1 fitting weights;
- survey-weighted pooled out-of-fold evaluation;
- the same prespecified XGBoost hyperparameter grid used in the primary
  analysis.

Hyperparameter selection follows the locked hierarchy:

1. highest survey-weighted Average Precision;
2. lower Brier score if exactly tied;
3. higher AUROC if still tied.

The held-out ANC-entry test set is not accessed.

In [ ]:
# =============================================================================
# 2.36 PAIRED XGBOOST TUNING AT ANC ENTRY
#
# Reuses the canonical ANC-entry tuning engine from Section 2.34.
#
# No new folds.
# No test access.
# =============================================================================


# =============================================================================
# 1. TUNE BASELINE XGBOOST
# =============================================================================

entry_xgb_baseline_tuning = tune_entry_specification(
    algorithm_name="XGB",
    specification_name="BASE",
    pipeline=pipeline_entry_xgb_baseline,
    param_grid=PARAM_GRID_XGB,
    X=X_entry_base,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 2. TUNE EXTENDED XGBOOST
# =============================================================================

entry_xgb_extended_tuning = tune_entry_specification(
    algorithm_name="XGB",
    specification_name="EXT",
    pipeline=pipeline_entry_xgb_extended,
    param_grid=PARAM_GRID_XGB,
    X=X_entry_ext,
    y=y_entry,
    cv_splits_locked=entry_cv_splits,
    fitting_weights=fit_weight_entry_dev,
    evaluation_weights=survey_weight_entry_dev
)


# =============================================================================
# 3. CANONICAL RESULT OBJECTS
# =============================================================================

entry_xgb_baseline_results = (
    entry_xgb_baseline_tuning[
        "results"
    ]
)

entry_xgb_extended_results = (
    entry_xgb_extended_tuning[
        "results"
    ]
)


BEST_ENTRY_XGB_BASELINE_CONFIG_ID = (
    entry_xgb_baseline_tuning[
        "selected_config_id"
    ]
)

BEST_ENTRY_XGB_EXTENDED_CONFIG_ID = (
    entry_xgb_extended_tuning[
        "selected_config_id"
    ]
)


BEST_ENTRY_XGB_BASELINE_PARAMS = (
    entry_xgb_baseline_tuning[
        "selected_params"
    ]
)

BEST_ENTRY_XGB_EXTENDED_PARAMS = (
    entry_xgb_extended_tuning[
        "selected_params"
    ]
)


oof_entry_xgb_baseline = (
    entry_xgb_baseline_tuning[
        "selected_oof"
    ]
)

oof_entry_xgb_extended = (
    entry_xgb_extended_tuning[
        "selected_oof"
    ]
)


# =============================================================================
# 4. SUMMARIZE SELECTED CONFIGURATIONS
# =============================================================================

selected_entry_xgb_rows = pd.DataFrame([
    {
        "Specification":
            "Baseline",

        "Config_ID":
            BEST_ENTRY_XGB_BASELINE_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_XGB_BASELINE_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_xgb_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_xgb_baseline,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_xgb_baseline,
                sample_weight=survey_weight_entry_dev
            )
    },

    {
        "Specification":
            "Extended",

        "Config_ID":
            BEST_ENTRY_XGB_EXTENDED_CONFIG_ID,

        "Selected_params":
            BEST_ENTRY_XGB_EXTENDED_PARAMS,

        "Weighted_AP":
            average_precision_score(
                y_entry,
                oof_entry_xgb_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_AUROC":
            roc_auc_score(
                y_entry,
                oof_entry_xgb_extended,
                sample_weight=survey_weight_entry_dev
            ),

        "Weighted_Brier":
            brier_score_loss(
                y_entry,
                oof_entry_xgb_extended,
                sample_weight=survey_weight_entry_dev
            )
    }
])


# =============================================================================
# 5. INTEGRITY
# =============================================================================

assert len(
    entry_xgb_baseline_results
) == 64

assert len(
    entry_xgb_extended_results
) == 64

assert len(
    oof_entry_xgb_baseline
) == len(
    y_entry
)

assert len(
    oof_entry_xgb_extended
) == len(
    y_entry
)

assert np.isfinite(
    oof_entry_xgb_baseline
).all()

assert np.isfinite(
    oof_entry_xgb_extended
).all()


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 132)
print("ANC-ENTRY PAIRED XGBOOST TUNING")
print("=" * 132)

print(
    selected_entry_xgb_rows
    .to_string(
        index=False,
        formatters={
            "Weighted_AP":
                "{:.4f}".format,

            "Weighted_AUROC":
                "{:.4f}".format,

            "Weighted_Brier":
                "{:.4f}".format
        }
    )
)


print("\nSelection governance")
print("--------------------")

print(
    "Candidate configurations per specification:",
    64
)

print(
    "Same development women:",
    "Yes"
)

print(
    "Same locked CV folds:",
    "Yes"
)

print(
    "Baseline and extended tuned independently:",
    "Yes"
)

print(
    "Primary tuning criterion:",
    "Weighted pooled OOF Average Precision"
)

print(
    "Held-out test accessed:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY XGBOOST "
    "BASELINE AND EXTENDED MODELS TUNED"
)

## 2.37 Development out-of-fold incremental predictive value of early ANC

The selected baseline and extended ANC-entry models are now compared using
their locked development out-of-fold predictions.

For each algorithm, both specifications were evaluated on exactly the same:

- 4,017 development women;
- five stratified PSU cross-validation folds;
- survey-weighted evaluation framework.

Incremental predictive value is summarized as:

- change in weighted Average Precision;
- change in weighted AUROC;
- change in weighted Brier score.

For AP and AUROC, positive differences favor the extended model.

For Brier score, improvement is reported as:

**baseline Brier minus extended Brier**

so that positive values consistently indicate improved performance after
adding `early_anc`.

These are development out-of-fold comparisons only. The held-out ANC-entry
test set remains untouched.

In [ ]:
# =============================================================================
# 2.37 DEVELOPMENT OOF INCREMENTAL VALUE OF early_anc
# =============================================================================


# =============================================================================
# 1. COLLECT LOCKED SELECTED OOF PREDICTIONS
# =============================================================================

entry_selected_oof = {
    "Logistic regression": {
        "Baseline":
            oof_entry_logistic_baseline,

        "Extended":
            oof_entry_logistic_extended
    },

    "Random Forest": {
        "Baseline":
            oof_entry_rf_baseline,

        "Extended":
            oof_entry_rf_extended
    },

    "XGBoost": {
        "Baseline":
            oof_entry_xgb_baseline,

        "Extended":
            oof_entry_xgb_extended
    }
}


# =============================================================================
# 2. CALCULATE PAIRED DEVELOPMENT OOF PERFORMANCE
# =============================================================================

entry_oof_comparison_rows = []


for algorithm, predictions in (
    entry_selected_oof.items()
):

    baseline_probability = np.asarray(
        predictions[
            "Baseline"
        ],
        dtype=float
    )

    extended_probability = np.asarray(
        predictions[
            "Extended"
        ],
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Integrity
    # -------------------------------------------------------------------------

    assert len(
        baseline_probability
    ) == len(
        extended_probability
    ) == len(
        y_entry
    )

    assert np.isfinite(
        baseline_probability
    ).all()

    assert np.isfinite(
        extended_probability
    ).all()


    # -------------------------------------------------------------------------
    # Baseline performance
    # -------------------------------------------------------------------------

    baseline_ap = average_precision_score(
        y_entry,
        baseline_probability,
        sample_weight=survey_weight_entry_dev
    )

    baseline_auroc = roc_auc_score(
        y_entry,
        baseline_probability,
        sample_weight=survey_weight_entry_dev
    )

    baseline_brier = brier_score_loss(
        y_entry,
        baseline_probability,
        sample_weight=survey_weight_entry_dev
    )


    # -------------------------------------------------------------------------
    # Extended performance
    # -------------------------------------------------------------------------

    extended_ap = average_precision_score(
        y_entry,
        extended_probability,
        sample_weight=survey_weight_entry_dev
    )

    extended_auroc = roc_auc_score(
        y_entry,
        extended_probability,
        sample_weight=survey_weight_entry_dev
    )

    extended_brier = brier_score_loss(
        y_entry,
        extended_probability,
        sample_weight=survey_weight_entry_dev
    )


    # -------------------------------------------------------------------------
    # Paired incremental value
    #
    # Positive values always favor the extended specification.
    # -------------------------------------------------------------------------

    delta_ap = (
        extended_ap
        - baseline_ap
    )

    delta_auroc = (
        extended_auroc
        - baseline_auroc
    )

    brier_improvement = (
        baseline_brier
        - extended_brier
    )


    entry_oof_comparison_rows.append({
        "Algorithm":
            algorithm,

        "Baseline_AP":
            baseline_ap,

        "Extended_AP":
            extended_ap,

        "Delta_AP":
            delta_ap,

        "Baseline_AUROC":
            baseline_auroc,

        "Extended_AUROC":
            extended_auroc,

        "Delta_AUROC":
            delta_auroc,

        "Baseline_Brier":
            baseline_brier,

        "Extended_Brier":
            extended_brier,

        "Brier_improvement":
            brier_improvement
    })


entry_oof_incremental_value = pd.DataFrame(
    entry_oof_comparison_rows
)


# =============================================================================
# 3. INTEGRITY
# =============================================================================

assert len(
    entry_oof_incremental_value
) == 3

assert (
    entry_oof_incremental_value[
        "Delta_AP"
    ] > 0
).all()

assert (
    entry_oof_incremental_value[
        "Delta_AUROC"
    ] > 0
).all()

assert (
    entry_oof_incremental_value[
        "Brier_improvement"
    ] > 0
).all()


# =============================================================================
# 4. IDENTIFY DEVELOPMENT METRIC LEADERS
# =============================================================================

best_extended_ap_model = (
    entry_oof_incremental_value
    .loc[
        entry_oof_incremental_value[
            "Extended_AP"
        ].idxmax(),
        "Algorithm"
    ]
)

best_extended_auroc_model = (
    entry_oof_incremental_value
    .loc[
        entry_oof_incremental_value[
            "Extended_AUROC"
        ].idxmax(),
        "Algorithm"
    ]
)

best_extended_brier_model = (
    entry_oof_incremental_value
    .loc[
        entry_oof_incremental_value[
            "Extended_Brier"
        ].idxmin(),
        "Algorithm"
    ]
)


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 150)
print("ANC-ENTRY DEVELOPMENT OOF INCREMENTAL VALUE OF early_anc")
print("=" * 150)

print(
    entry_oof_incremental_value
    .to_string(
        index=False,
        formatters={
            "Baseline_AP":
                "{:.4f}".format,

            "Extended_AP":
                "{:.4f}".format,

            "Delta_AP":
                "{:+.4f}".format,

            "Baseline_AUROC":
                "{:.4f}".format,

            "Extended_AUROC":
                "{:.4f}".format,

            "Delta_AUROC":
                "{:+.4f}".format,

            "Baseline_Brier":
                "{:.4f}".format,

            "Extended_Brier":
                "{:.4f}".format,

            "Brier_improvement":
                "{:+.4f}".format
        }
    )
)


print("\nExtended-model metric leaders")
print("-----------------------------")

print(
    "Highest AP:",
    best_extended_ap_model
)

print(
    "Highest AUROC:",
    best_extended_auroc_model
)

print(
    "Lowest Brier:",
    best_extended_brier_model
)


print("\nGovernance")
print("----------")

print(
    "Same women used for baseline/extended comparison:",
    "Yes"
)

print(
    "Same folds used:",
    "Yes"
)

print(
    "OOF predictions regenerated:",
    "No"
)

print(
    "Held-out test accessed:",
    "No"
)

print(
    "\nSTATUS: DEVELOPMENT OOF "
    "INCREMENTAL VALUE SUMMARIZED"
)

## 2.38 Final refit of ANC-entry baseline and extended models

The selected ANC-entry configurations are now refitted on the complete
development population of 4,017 women.

Six final models are fitted:

- Logistic regression, 21-predictor baseline;
- Logistic regression, 22-predictor extended;
- Random Forest, 21-predictor baseline;
- Random Forest, 22-predictor extended;
- XGBoost, 21-predictor baseline;
- XGBoost, 22-predictor extended.

For each model:

- the hyperparameters selected from development out-of-fold tuning are locked;
- the relevant governed preprocessing pipeline is refitted using all
  development observations;
- model-fitting survey weights are normalized to mean 1;
- no class rebalancing is introduced.

The held-out ANC-entry test population remains untouched.

After this step, no further model fitting or hyperparameter tuning is permitted
before final held-out evaluation.

In [ ]:
# =============================================================================
# 2.38 FINAL REFIT OF ALL SIX ANC-ENTRY MODELS
#
# Development only.
#
# After this cell:
#   - hyperparameters are locked;
#   - preprocessing is fitted on all 4,017 development women;
#   - six final model objects exist;
#   - held-out ANC-entry test data remain unopened for prediction.
# =============================================================================


# =============================================================================
# 1. FINAL DEVELOPMENT FITTING WEIGHTS
#
# Normalize again defensively to mean 1.
# =============================================================================

final_fit_weight_entry = (
    fit_weight_entry_dev
    / fit_weight_entry_dev.mean()
)


assert np.isclose(
    final_fit_weight_entry.mean(),
    1.0
)


# =============================================================================
# 2. HELPER TO REFIT ONE LOCKED MODEL
# =============================================================================

def fit_final_entry_model(
    pipeline,
    selected_params,
    X,
    y,
    fitting_weights
):
    """
    Clone one governed pipeline, apply its locked selected hyperparameters,
    and refit it on the complete ANC-entry development population.
    """

    model = clone(
        pipeline
    )

    model.set_params(
        **selected_params
    )

    model.fit(
        X,
        y,
        model__sample_weight=fitting_weights
    )

    return model


# =============================================================================
# 3. LOGISTIC REGRESSION
# =============================================================================

final_entry_logistic_baseline = fit_final_entry_model(
    pipeline=pipeline_entry_logistic_baseline,
    selected_params=BEST_ENTRY_LOGISTIC_BASELINE_PARAMS,
    X=X_entry_base,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


final_entry_logistic_extended = fit_final_entry_model(
    pipeline=pipeline_entry_logistic_extended,
    selected_params=BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS,
    X=X_entry_ext,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


# =============================================================================
# 4. RANDOM FOREST
# =============================================================================

final_entry_rf_baseline = fit_final_entry_model(
    pipeline=pipeline_entry_rf_baseline,
    selected_params=BEST_ENTRY_RF_BASELINE_PARAMS,
    X=X_entry_base,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


final_entry_rf_extended = fit_final_entry_model(
    pipeline=pipeline_entry_rf_extended,
    selected_params=BEST_ENTRY_RF_EXTENDED_PARAMS,
    X=X_entry_ext,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


# =============================================================================
# 5. XGBOOST
# =============================================================================

final_entry_xgb_baseline = fit_final_entry_model(
    pipeline=pipeline_entry_xgb_baseline,
    selected_params=BEST_ENTRY_XGB_BASELINE_PARAMS,
    X=X_entry_base,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


final_entry_xgb_extended = fit_final_entry_model(
    pipeline=pipeline_entry_xgb_extended,
    selected_params=BEST_ENTRY_XGB_EXTENDED_PARAMS,
    X=X_entry_ext,
    y=y_entry,
    fitting_weights=final_fit_weight_entry
)


# =============================================================================
# 6. CANONICAL FINAL MODEL REGISTRY
# =============================================================================

final_entry_models = {
    "Logistic regression": {
        "Baseline":
            final_entry_logistic_baseline,

        "Extended":
            final_entry_logistic_extended
    },

    "Random Forest": {
        "Baseline":
            final_entry_rf_baseline,

        "Extended":
            final_entry_rf_extended
    },

    "XGBoost": {
        "Baseline":
            final_entry_xgb_baseline,

        "Extended":
            final_entry_xgb_extended
    }
}


# =============================================================================
# 7. AUDIT LOCKED HYPERPARAMETERS
# =============================================================================

final_entry_selected_params = {
    ("Logistic regression", "Baseline"):
        BEST_ENTRY_LOGISTIC_BASELINE_PARAMS,

    ("Logistic regression", "Extended"):
        BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS,

    ("Random Forest", "Baseline"):
        BEST_ENTRY_RF_BASELINE_PARAMS,

    ("Random Forest", "Extended"):
        BEST_ENTRY_RF_EXTENDED_PARAMS,

    ("XGBoost", "Baseline"):
        BEST_ENTRY_XGB_BASELINE_PARAMS,

    ("XGBoost", "Extended"):
        BEST_ENTRY_XGB_EXTENDED_PARAMS
}


for (
    algorithm,
    specification
), expected_params in (
    final_entry_selected_params.items()
):

    fitted_model = (
        final_entry_models[
            algorithm
        ][
            specification
        ]
    )

    fitted_params = (
        fitted_model
        .get_params()
    )

    for parameter_name, expected_value in (
        expected_params.items()
    ):

        assert (
            fitted_params[
                parameter_name
            ]
            == expected_value
        )


# =============================================================================
# 8. AUDIT TRANSFORMED FEATURE ARCHITECTURE
#
# Baseline should reproduce the original 78 transformed features.
# Extended should add exactly one feature: early_anc.
# =============================================================================

entry_feature_audit_rows = []


for algorithm, models in (
    final_entry_models.items()
):

    for specification, fitted_model in (
        models.items()
    ):

        feature_names = (
            fitted_model[
                "preprocessor"
            ]
            .get_feature_names_out()
        )

        n_transformed = len(
            feature_names
        )


        entry_feature_audit_rows.append({
            "Algorithm":
                algorithm,

            "Specification":
                specification,

            "Transformed_features":
                n_transformed,

            "early_anc_present":
                any(
                    "early_anc"
                    in feature_name
                    for feature_name
                    in feature_names
                )
        })


entry_final_feature_audit = pd.DataFrame(
    entry_feature_audit_rows
)


# Baseline = original 78 transformed features.
assert (
    entry_final_feature_audit.loc[
        entry_final_feature_audit[
            "Specification"
        ].eq(
            "Baseline"
        ),
        "Transformed_features"
    ]
    == 78
).all()


# Extended = same 78 + one direct early_anc feature.
assert (
    entry_final_feature_audit.loc[
        entry_final_feature_audit[
            "Specification"
        ].eq(
            "Extended"
        ),
        "Transformed_features"
    ]
    == 79
).all()


assert not (
    entry_final_feature_audit.loc[
        entry_final_feature_audit[
            "Specification"
        ].eq(
            "Baseline"
        ),
        "early_anc_present"
    ]
).any()


assert (
    entry_final_feature_audit.loc[
        entry_final_feature_audit[
            "Specification"
        ].eq(
            "Extended"
        ),
        "early_anc_present"
    ]
).all()


# =============================================================================
# 9. DISPLAY
# =============================================================================

print("=" * 124)
print("FINAL ANC-ENTRY MODELS REFITTED ON COMPLETE DEVELOPMENT POPULATION")
print("=" * 124)

print(
    "Development women:",
    len(
        anc_entry_train
    )
)

print(
    "Final fitting-weight mean:",
    f"{final_fit_weight_entry.mean():.6f}"
)


print("\nSelected configurations")
print("-----------------------")

print(
    "Logistic baseline:",
    BEST_ENTRY_LOGISTIC_BASELINE_PARAMS
)

print(
    "Logistic extended:",
    BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS
)

print(
    "RF baseline:",
    BEST_ENTRY_RF_BASELINE_PARAMS
)

print(
    "RF extended:",
    BEST_ENTRY_RF_EXTENDED_PARAMS
)

print(
    "XGBoost baseline:",
    BEST_ENTRY_XGB_BASELINE_PARAMS
)

print(
    "XGBoost extended:",
    BEST_ENTRY_XGB_EXTENDED_PARAMS
)


print("\nTransformed feature audit")
print("-------------------------")

print(
    entry_final_feature_audit.to_string(
        index=False
    )
)


print("\nGovernance")
print("----------")

print(
    "Final models fitted:",
    6
)

print(
    "Hyperparameters changed after tuning:",
    "No"
)

print(
    "Class rebalancing introduced:",
    "No"
)

print(
    "Held-out ANC-entry test accessed:",
    "No"
)

print(
    "Further model development permitted before test evaluation:",
    "No"
)

print(
    "\nSTATUS: SIX FINAL ANC-ENTRY MODELS "
    "LOCKED FOR HELD-OUT EVALUATION"
)

## 2.38.1 Locked held-out prediction for the ANC-entry horizon

Model development is complete.

The six final ANC-entry models are now applied to the inherited held-out test
population.

Two predictor matrices are prepared:

- the 21-predictor baseline specification;
- the 22-predictor extended specification containing `early_anc`.

All six predicted-probability vectors are generated before the held-out ANC8
outcome is accessed.

The probability table is then locked. Only after prediction is complete is the
observed ANC8 outcome attached for evaluation.

From this point onward:

- no model may be refitted;
- no hyperparameter may be changed;
- no preprocessing rule may be altered;
- held-out results may not feed back into model development.

The held-out population is used strictly for final evaluation.

In [ ]:
# =============================================================================
# 2.39 LOCKED ANC-ENTRY HELD-OUT PREDICTION
#
# CRITICAL BOUNDARY
# -----------------
# 1. Prepare test predictors.
# 2. Generate six probabilities WITHOUT using the test outcome.
# 3. Lock predictions.
# 4. Only then attach the observed ANC8 outcome.
#
# No fitting occurs here.
# =============================================================================


# =============================================================================
# 1. PREPARE HELD-OUT PREDICTOR MATRICES
# =============================================================================

X_entry_test_base = (
    anc_entry_test[
        ANC_ENTRY_BASELINE_PREDICTORS
    ]
    .copy()
)

X_entry_test_ext = (
    anc_entry_test[
        ANC_ENTRY_EXTENDED_PREDICTORS
    ]
    .copy()
)


# =============================================================================
# 2. HELD-OUT DESIGN VARIABLES
#
# These are retained for evaluation and later design-aware uncertainty.
# They are NOT predictors.
# =============================================================================

survey_weight_entry_test = (
    anc_entry_test[
        "v005"
    ]
    .astype(float)
    / 1_000_000
)

groups_entry_test = (
    anc_entry_test[
        "v021"
    ]
    .copy()
)

strata_entry_test = (
    anc_entry_test[
        "v022"
    ]
    .copy()
)


# =============================================================================
# 3. PRE-PREDICTION INTEGRITY
# =============================================================================

assert len(
    X_entry_test_base
) == 984

assert len(
    X_entry_test_ext
) == 984

assert X_entry_test_base.columns.tolist() == (
    ANC_ENTRY_BASELINE_PREDICTORS
)

assert X_entry_test_ext.columns.tolist() == (
    ANC_ENTRY_EXTENDED_PREDICTORS
)

assert X_entry_test_ext[
    "early_anc"
].isin(
    [0.0, 1.0]
).all()

assert (
    survey_weight_entry_test > 0
).all()


# Development and held-out women remain disjoint.
assert set(
    anc_entry_train[
        "caseid"
    ]
).isdisjoint(
    set(
        anc_entry_test[
            "caseid"
        ]
    )
)


# Development and held-out PSUs remain disjoint.
assert set(
    anc_entry_train[
        "v021"
    ]
).isdisjoint(
    set(
        anc_entry_test[
            "v021"
        ]
    )
)


# =============================================================================
# 4. GENERATE SIX HELD-OUT PROBABILITY VECTORS
#
# IMPORTANT:
# The outcome is not referenced anywhere in this section.
# =============================================================================

prob_entry_logistic_baseline = (
    final_entry_logistic_baseline
    .predict_proba(
        X_entry_test_base
    )[:, 1]
)

prob_entry_logistic_extended = (
    final_entry_logistic_extended
    .predict_proba(
        X_entry_test_ext
    )[:, 1]
)


prob_entry_rf_baseline = (
    final_entry_rf_baseline
    .predict_proba(
        X_entry_test_base
    )[:, 1]
)

prob_entry_rf_extended = (
    final_entry_rf_extended
    .predict_proba(
        X_entry_test_ext
    )[:, 1]
)


prob_entry_xgb_baseline = (
    final_entry_xgb_baseline
    .predict_proba(
        X_entry_test_base
    )[:, 1]
)

prob_entry_xgb_extended = (
    final_entry_xgb_extended
    .predict_proba(
        X_entry_test_ext
    )[:, 1]
)


# =============================================================================
# 5. PROBABILITY INTEGRITY
# =============================================================================

entry_probability_vectors = {
    "prob_logistic_baseline":
        prob_entry_logistic_baseline,

    "prob_logistic_extended":
        prob_entry_logistic_extended,

    "prob_rf_baseline":
        prob_entry_rf_baseline,

    "prob_rf_extended":
        prob_entry_rf_extended,

    "prob_xgb_baseline":
        prob_entry_xgb_baseline,

    "prob_xgb_extended":
        prob_entry_xgb_extended
}


for probability_name, probability in (
    entry_probability_vectors.items()
):

    assert len(
        probability
    ) == 984

    assert np.isfinite(
        probability
    ).all()

    assert (
        (probability >= 0)
        &
        (probability <= 1)
    ).all()


# =============================================================================
# 6. BUILD BLINDED HELD-OUT PREDICTION TABLE
#
# Still NO outcome here.
# =============================================================================

entry_test_predictions_blinded = pd.DataFrame({
    "caseid":
        anc_entry_test[
            "caseid"
        ].to_numpy(),

    "v021":
        groups_entry_test.to_numpy(),

    "v022":
        strata_entry_test.to_numpy(),

    "survey_weight":
        survey_weight_entry_test.to_numpy(),

    "early_anc":
        anc_entry_test[
            "early_anc"
        ].to_numpy(),

    "prob_logistic_baseline":
        prob_entry_logistic_baseline,

    "prob_logistic_extended":
        prob_entry_logistic_extended,

    "prob_rf_baseline":
        prob_entry_rf_baseline,

    "prob_rf_extended":
        prob_entry_rf_extended,

    "prob_xgb_baseline":
        prob_entry_xgb_baseline,

    "prob_xgb_extended":
        prob_entry_xgb_extended
})


assert entry_test_predictions_blinded[
    "caseid"
].is_unique


# =============================================================================
# 7. LOCK THE PREDICTIONS BEFORE OPENING THE OUTCOME
# =============================================================================

entry_test_probabilities_locked = (
    entry_test_predictions_blinded
    .copy(
        deep=True
    )
)


# =============================================================================
# 8. NOW OPEN THE HELD-OUT OUTCOME
#
# This is the first formal use of the ANC-entry held-out outcome.
# =============================================================================

y_entry_test = (
    anc_entry_test[
        "anc8plus"
    ]
    .astype(int)
    .copy()
)


# Create the final locked evaluation table.
entry_test_predictions_locked = (
    entry_test_probabilities_locked
    .copy(
        deep=True
    )
)


entry_test_predictions_locked[
    "anc8plus_observed"
] = (
    y_entry_test.to_numpy()
)


# =============================================================================
# 9. HELD-OUT OUTCOME CONTEXT
# =============================================================================

entry_test_weighted_prevalence = np.average(
    y_entry_test,
    weights=survey_weight_entry_test
)

entry_test_unweighted_prevalence = (
    y_entry_test.mean()
)


# =============================================================================
# 10. PREDICTION-RANGE AUDIT
# =============================================================================

entry_prediction_range_rows = []


for probability_name, probability in (
    entry_probability_vectors.items()
):

    entry_prediction_range_rows.append({
        "Probability":
            probability_name,

        "Minimum":
            float(
                np.min(
                    probability
                )
            ),

        "Median":
            float(
                np.median(
                    probability
                )
            ),

        "Maximum":
            float(
                np.max(
                    probability
                )
            )
    })


entry_prediction_ranges = pd.DataFrame(
    entry_prediction_range_rows
)


# =============================================================================
# 11. FINAL INTEGRITY
# =============================================================================

assert len(
    entry_test_predictions_locked
) == 984

assert entry_test_predictions_locked[
    "anc8plus_observed"
].isin(
    [0, 1]
).all()

assert (
    entry_test_predictions_locked[
        "caseid"
    ].to_numpy()
    ==
    anc_entry_test[
        "caseid"
    ].to_numpy()
).all()


# =============================================================================
# 12. DISPLAY
# =============================================================================

print("=" * 126)
print("LOCKED ANC-ENTRY HELD-OUT PREDICTIONS")
print("=" * 126)

print("\nHeld-out population")
print("-------------------")

print(
    "Women:",
    len(
        entry_test_predictions_locked
    )
)

print(
    "PSUs:",
    groups_entry_test.nunique()
)

print(
    "Strata:",
    strata_entry_test.nunique()
)

print(
    "Development/test PSU overlap:",
    0
)


print("\nObserved ANC8 outcome context")
print("-----------------------------")

print(
    "ANC8+ count:",
    int(
        y_entry_test.sum()
    )
)

print(
    "Unweighted ANC8+ prevalence:",
    f"{entry_test_unweighted_prevalence:.2%}"
)

print(
    "Survey-weighted ANC8+ prevalence:",
    f"{entry_test_weighted_prevalence:.2%}"
)


print("\nPrediction ranges")
print("-----------------")

print(
    entry_prediction_ranges.to_string(
        index=False,
        formatters={
            "Minimum":
                "{:.4f}".format,

            "Median":
                "{:.4f}".format,

            "Maximum":
                "{:.4f}".format
        }
    )
)


print("\nGovernance")
print("----------")

print(
    "Predictions generated before outcome access:",
    "Yes"
)

print(
    "Six probability vectors locked:",
    "Yes"
)

print(
    "Models fitted here:",
    "No"
)

print(
    "Hyperparameters changed:",
    "No"
)

print(
    "Preprocessing changed:",
    "No"
)

print(
    "Test results may feed back into development:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY HELD-OUT "
    "PREDICTIONS LOCKED AND OUTCOME OPENED"
)

## 2.39 Paired ANC-entry calibration summary

Calibration is evaluated for all six locked ANC-entry models on the same
984-woman held-out test population.

For each model, the following are estimated:

- survey-weighted observed ANC8 prevalence;
- survey-weighted mean predicted probability;
- calibration-in-the-large (CITL);
- joint calibration intercept;
- calibration slope.

**Calibration-in-the-large (CITL)** measures systematic overprediction or
underprediction while fixing the calibration slope at 1. Its ideal value is 0.

CITL is estimated from:

`logit(observed outcome) = CITL + offset(logit(predicted probability))`

Interpretation:

- negative CITL indicates overall overprediction;
- positive CITL indicates overall underprediction.

The **joint calibration intercept and slope** are estimated simultaneously from:

`logit(observed outcome) = intercept + slope × logit(predicted probability)`

Their ideal values are:

- joint intercept = 0;
- calibration slope = 1.

A slope below 1 generally indicates predictions that are too extreme, whereas a
slope above 1 indicates predictions that are not sufficiently spread.

These are held-out calibration point estimates. No recalibration is performed,
and no result is allowed to feed back into model development.

In [ ]:
# =============================================================================
# 2.39 PAIRED ANC-ENTRY CALIBRATION SUMMARY
#
# Correct calibration terminology:
#
#   CITL:
#       intercept-only calibration model with predicted log-odds as OFFSET
#       and slope fixed at 1.
#
#   Joint calibration intercept + slope:
#       both estimated simultaneously.
#
# Ideal:
#   CITL = 0
#   Joint intercept = 0
#   Calibration slope = 1
#
# No recalibration is performed.
# =============================================================================

import statsmodels.api as sm


# =============================================================================
# 1. HELPER: SAFE LOGIT
#
# Exact probabilities of 0 or 1 produce infinite logits.
# The clipping is numerical protection only and does not materially change
# ordinary predicted probabilities.
# =============================================================================

def safe_logit(probability, epsilon=1e-6):

    probability = np.asarray(
        probability,
        dtype=float
    )

    probability_clipped = np.clip(
        probability,
        epsilon,
        1 - epsilon
    )

    return np.log(
        probability_clipped
        /
        (
            1 - probability_clipped
        )
    )


# =============================================================================
# 2. HELPER: CALIBRATION POINT ESTIMATES
# =============================================================================

def calculate_weighted_calibration(
    y,
    probability,
    survey_weight
):
    """
    Calculate weighted held-out calibration summaries.

    Returns
    -------
    Observed_prevalence
    Mean_predicted_probability
    CITL
    Joint_calibration_intercept
    Calibration_slope
    """

    y = np.asarray(
        y,
        dtype=float
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    survey_weight = np.asarray(
        survey_weight,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Integrity
    # -------------------------------------------------------------------------

    assert len(
        y
    ) == len(
        probability
    ) == len(
        survey_weight
    )

    assert np.isfinite(
        probability
    ).all()

    assert np.isfinite(
        survey_weight
    ).all()

    assert (
        survey_weight > 0
    ).all()


    # -------------------------------------------------------------------------
    # Survey-weighted observed prevalence
    # -------------------------------------------------------------------------

    observed_prevalence = np.average(
        y,
        weights=survey_weight
    )


    # -------------------------------------------------------------------------
    # Survey-weighted mean predicted probability
    # -------------------------------------------------------------------------

    mean_predicted_probability = np.average(
        probability,
        weights=survey_weight
    )


    # -------------------------------------------------------------------------
    # Convert probability to predicted log-odds.
    # -------------------------------------------------------------------------

    predicted_logit = safe_logit(
        probability
    )


    # =========================================================================
    # TRUE CALIBRATION-IN-THE-LARGE
    #
    # logit(Y) = alpha + offset(logit(predicted probability))
    #
    # Slope is fixed at 1.
    # =========================================================================

    citl_design = np.ones(
        (
            len(y),
            1
        )
    )

    citl_model = sm.GLM(
        y,
        citl_design,
        family=sm.families.Binomial(),
        offset=predicted_logit,
        freq_weights=survey_weight
    )

    citl_result = citl_model.fit()

    citl = float(
        citl_result.params[
            0
        ]
    )


    # =========================================================================
    # JOINT CALIBRATION INTERCEPT + SLOPE
    #
    # logit(Y) = alpha + beta * logit(predicted probability)
    # =========================================================================

    joint_design = sm.add_constant(
        predicted_logit,
        has_constant="add"
    )

    joint_model = sm.GLM(
        y,
        joint_design,
        family=sm.families.Binomial(),
        freq_weights=survey_weight
    )

    joint_result = joint_model.fit()


    joint_intercept = float(
        joint_result.params[
            0
        ]
    )

    calibration_slope = float(
        joint_result.params[
            1
        ]
    )


    return {
        "Observed_prevalence":
            observed_prevalence,

        "Mean_predicted_probability":
            mean_predicted_probability,

        "CITL":
            citl,

        "Joint_calibration_intercept":
            joint_intercept,

        "Calibration_slope":
            calibration_slope
    }


# =============================================================================
# 3. LOCKED PROBABILITY COLUMN REGISTRY
# =============================================================================

entry_calibration_models = {
    (
        "Logistic regression",
        "Baseline"
    ):
        "prob_logistic_baseline",

    (
        "Logistic regression",
        "Extended"
    ):
        "prob_logistic_extended",

    (
        "Random Forest",
        "Baseline"
    ):
        "prob_rf_baseline",

    (
        "Random Forest",
        "Extended"
    ):
        "prob_rf_extended",

    (
        "XGBoost",
        "Baseline"
    ):
        "prob_xgb_baseline",

    (
        "XGBoost",
        "Extended"
    ):
        "prob_xgb_extended"
}


# =============================================================================
# 4. CALCULATE HELD-OUT CALIBRATION FOR ALL SIX MODELS
# =============================================================================

entry_calibration_rows = []


for (
    algorithm,
    specification
), probability_column in (
    entry_calibration_models.items()
):

    calibration = calculate_weighted_calibration(
        y=entry_test_predictions_locked[
            "anc8plus_observed"
        ],
        probability=entry_test_predictions_locked[
            probability_column
        ],
        survey_weight=entry_test_predictions_locked[
            "survey_weight"
        ]
    )


    entry_calibration_rows.append({
        "Algorithm":
            algorithm,

        "Specification":
            specification,

        "Observed_prevalence":
            calibration[
                "Observed_prevalence"
            ],

        "Mean_predicted_probability":
            calibration[
                "Mean_predicted_probability"
            ],

        "CITL":
            calibration[
                "CITL"
            ],

        "Joint_calibration_intercept":
            calibration[
                "Joint_calibration_intercept"
            ],

        "Calibration_slope":
            calibration[
                "Calibration_slope"
            ]
    })


entry_calibration_summary = pd.DataFrame(
    entry_calibration_rows
)


# =============================================================================
# 5. CALIBRATION DIFFERENCE IN MEAN PROBABILITY
#
# Positive:
#   mean predicted probability exceeds observed prevalence.
#
# Negative:
#   mean predicted probability is below observed prevalence.
# =============================================================================

entry_calibration_summary[
    "Mean_prediction_minus_observed"
] = (
    entry_calibration_summary[
        "Mean_predicted_probability"
    ]
    -
    entry_calibration_summary[
        "Observed_prevalence"
    ]
)


# =============================================================================
# 6. INTEGRITY
# =============================================================================

assert len(
    entry_calibration_summary
) == 6

assert np.isfinite(
    entry_calibration_summary[
        [
            "Observed_prevalence",
            "Mean_predicted_probability",
            "CITL",
            "Joint_calibration_intercept",
            "Calibration_slope"
        ]
    ].to_numpy()
).all()


# Observed prevalence must be identical because all six models are evaluated
# on the same held-out women.
assert np.allclose(
    entry_calibration_summary[
        "Observed_prevalence"
    ],
    entry_calibration_summary[
        "Observed_prevalence"
    ].iloc[0]
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 154)
print("PAIRED ANC-ENTRY HELD-OUT CALIBRATION SUMMARY")
print("=" * 154)

print(
    entry_calibration_summary[
        [
            "Algorithm",
            "Specification",
            "Observed_prevalence",
            "Mean_predicted_probability",
            "Mean_prediction_minus_observed",
            "CITL",
            "Joint_calibration_intercept",
            "Calibration_slope"
        ]
    ]
    .to_string(
        index=False,
        formatters={
            "Observed_prevalence":
                "{:.4f}".format,

            "Mean_predicted_probability":
                "{:.4f}".format,

            "Mean_prediction_minus_observed":
                "{:+.4f}".format,

            "CITL":
                "{:+.4f}".format,

            "Joint_calibration_intercept":
                "{:+.4f}".format,

            "Calibration_slope":
                "{:.4f}".format
        }
    )
)


print("\nInterpretation guide")
print("--------------------")

print(
    "Ideal CITL:",
    "0"
)

print(
    "Ideal joint calibration intercept:",
    "0"
)

print(
    "Ideal calibration slope:",
    "1"
)

print(
    "Negative CITL:",
    "overall overprediction"
)

print(
    "Positive CITL:",
    "overall underprediction"
)


print("\nGovernance")
print("----------")

print(
    "Held-out population:",
    len(
        entry_test_predictions_locked
    )
)

print(
    "Models evaluated:",
    6
)

print(
    "Recalibration performed:",
    "No"
)

print(
    "Models refitted:",
    "No"
)

print(
    "Test results fed back into development:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY HELD-OUT "
    "CALIBRATION SUMMARIZED"
)

### 2.39.1 Paired ANC-entry held-out probability-based performance

The six locked ANC-entry models are evaluated on the same 984-woman held-out
test population.

For each algorithm, the baseline 21-predictor specification is compared with
the extended 22-predictor specification that additionally includes `early_anc`.

Primary probability-based performance measures are:

- **Average Precision (AP)**;
- **AUROC**;
- **Brier score**.

Interpretation:

- higher AP is better;
- higher AUROC is better;
- lower Brier score is better.

For paired incremental comparisons:

- `Delta_AP = Extended AP - Baseline AP`;
- `Delta_AUROC = Extended AUROC - Baseline AUROC`;
- `Brier_improvement = Baseline Brier - Extended Brier`.

Thus, positive values consistently favor the extended specification.

The survey-weighted ANC8 prevalence in the held-out ANC-entry population is
reported as the AP reference level.

These are locked held-out point estimates. Statistical uncertainty and paired
design-aware comparisons are assessed later using stratified PSU bootstrap.

In [ ]:
# =============================================================================
# 2.39.1 PAIRED ANC-ENTRY HELD-OUT PROBABILITY PERFORMANCE
#
# Primary metrics:
#   - Average Precision
#   - AUROC
#   - Brier score
#
# Positive incremental differences favor the extended model.
# =============================================================================


# =============================================================================
# 1. MODEL / PROBABILITY REGISTRY
# =============================================================================

entry_test_probability_models = {
    "Logistic regression": {
        "Baseline":
            "prob_logistic_baseline",

        "Extended":
            "prob_logistic_extended"
    },

    "Random Forest": {
        "Baseline":
            "prob_rf_baseline",

        "Extended":
            "prob_rf_extended"
    },

    "XGBoost": {
        "Baseline":
            "prob_xgb_baseline",

        "Extended":
            "prob_xgb_extended"
    }
}


# =============================================================================
# 2. EVALUATION OBJECTS
# =============================================================================

y_entry_test_eval = (
    entry_test_predictions_locked[
        "anc8plus_observed"
    ]
    .astype(int)
)

w_entry_test_eval = (
    entry_test_predictions_locked[
        "survey_weight"
    ]
    .astype(float)
)


# Weighted event prevalence = AP reference level.
entry_test_ap_reference = np.average(
    y_entry_test_eval,
    weights=w_entry_test_eval
)


# =============================================================================
# 3. CALCULATE HELD-OUT PERFORMANCE
# =============================================================================

entry_test_performance_rows = []


for algorithm, specification_map in (
    entry_test_probability_models.items()
):

    # -------------------------------------------------------------------------
    # Baseline probabilities
    # -------------------------------------------------------------------------

    prob_baseline = (
        entry_test_predictions_locked[
            specification_map[
                "Baseline"
            ]
        ]
        .astype(float)
        .to_numpy()
    )


    # -------------------------------------------------------------------------
    # Extended probabilities
    # -------------------------------------------------------------------------

    prob_extended = (
        entry_test_predictions_locked[
            specification_map[
                "Extended"
            ]
        ]
        .astype(float)
        .to_numpy()
    )


    # -------------------------------------------------------------------------
    # Integrity
    # -------------------------------------------------------------------------

    assert len(
        prob_baseline
    ) == len(
        prob_extended
    ) == len(
        y_entry_test_eval
    )

    assert np.isfinite(
        prob_baseline
    ).all()

    assert np.isfinite(
        prob_extended
    ).all()


    # =========================================================================
    # BASELINE PERFORMANCE
    # =========================================================================

    baseline_ap = average_precision_score(
        y_entry_test_eval,
        prob_baseline,
        sample_weight=w_entry_test_eval
    )

    baseline_auroc = roc_auc_score(
        y_entry_test_eval,
        prob_baseline,
        sample_weight=w_entry_test_eval
    )

    baseline_brier = brier_score_loss(
        y_entry_test_eval,
        prob_baseline,
        sample_weight=w_entry_test_eval
    )


    # =========================================================================
    # EXTENDED PERFORMANCE
    # =========================================================================

    extended_ap = average_precision_score(
        y_entry_test_eval,
        prob_extended,
        sample_weight=w_entry_test_eval
    )

    extended_auroc = roc_auc_score(
        y_entry_test_eval,
        prob_extended,
        sample_weight=w_entry_test_eval
    )

    extended_brier = brier_score_loss(
        y_entry_test_eval,
        prob_extended,
        sample_weight=w_entry_test_eval
    )


    # =========================================================================
    # PAIRED INCREMENTAL VALUE
    #
    # Positive values always favor Extended.
    # =========================================================================

    delta_ap = (
        extended_ap
        - baseline_ap
    )

    delta_auroc = (
        extended_auroc
        - baseline_auroc
    )

    brier_improvement = (
        baseline_brier
        - extended_brier
    )


    entry_test_performance_rows.append({
        "Algorithm":
            algorithm,

        "Baseline_AP":
            baseline_ap,

        "Extended_AP":
            extended_ap,

        "Delta_AP":
            delta_ap,

        "Baseline_AUROC":
            baseline_auroc,

        "Extended_AUROC":
            extended_auroc,

        "Delta_AUROC":
            delta_auroc,

        "Baseline_Brier":
            baseline_brier,

        "Extended_Brier":
            extended_brier,

        "Brier_improvement":
            brier_improvement
    })


entry_test_incremental_performance = pd.DataFrame(
    entry_test_performance_rows
)


# =============================================================================
# 4. LONG-FORM SIX-MODEL PERFORMANCE TABLE
# =============================================================================

entry_test_performance_long_rows = []


for algorithm, specification_map in (
    entry_test_probability_models.items()
):

    for specification, probability_column in (
        specification_map.items()
    ):

        probability = (
            entry_test_predictions_locked[
                probability_column
            ]
            .astype(float)
            .to_numpy()
        )


        entry_test_performance_long_rows.append({
            "Algorithm":
                algorithm,

            "Specification":
                specification,

            "Weighted_AP":
                average_precision_score(
                    y_entry_test_eval,
                    probability,
                    sample_weight=w_entry_test_eval
                ),

            "Weighted_AUROC":
                roc_auc_score(
                    y_entry_test_eval,
                    probability,
                    sample_weight=w_entry_test_eval
                ),

            "Weighted_Brier":
                brier_score_loss(
                    y_entry_test_eval,
                    probability,
                    sample_weight=w_entry_test_eval
                )
        })


entry_test_performance_long = pd.DataFrame(
    entry_test_performance_long_rows
)


# =============================================================================
# 5. IDENTIFY HELD-OUT EXTENDED-MODEL LEADERS
# =============================================================================

extended_only = (
    entry_test_performance_long[
        entry_test_performance_long[
            "Specification"
        ].eq(
            "Extended"
        )
    ]
    .copy()
)


best_entry_extended_ap = (
    extended_only.loc[
        extended_only[
            "Weighted_AP"
        ].idxmax(),
        "Algorithm"
    ]
)

best_entry_extended_auroc = (
    extended_only.loc[
        extended_only[
            "Weighted_AUROC"
        ].idxmax(),
        "Algorithm"
    ]
)

best_entry_extended_brier = (
    extended_only.loc[
        extended_only[
            "Weighted_Brier"
        ].idxmin(),
        "Algorithm"
    ]
)


# =============================================================================
# 6. INTEGRITY
# =============================================================================

assert len(
    entry_test_incremental_performance
) == 3

assert len(
    entry_test_performance_long
) == 6

assert np.isfinite(
    entry_test_incremental_performance[
        [
            "Baseline_AP",
            "Extended_AP",
            "Delta_AP",
            "Baseline_AUROC",
            "Extended_AUROC",
            "Delta_AUROC",
            "Baseline_Brier",
            "Extended_Brier",
            "Brier_improvement"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("=" * 154)
print("PAIRED ANC-ENTRY HELD-OUT PROBABILITY PERFORMANCE")
print("=" * 154)

print(
    "Held-out survey-weighted ANC8 prevalence / AP reference:",
    f"{entry_test_ap_reference:.4f}"
)


print("\nPaired baseline versus extended comparison")
print("------------------------------------------")

print(
    entry_test_incremental_performance
    .to_string(
        index=False,
        formatters={
            "Baseline_AP":
                "{:.4f}".format,

            "Extended_AP":
                "{:.4f}".format,

            "Delta_AP":
                "{:+.4f}".format,

            "Baseline_AUROC":
                "{:.4f}".format,

            "Extended_AUROC":
                "{:.4f}".format,

            "Delta_AUROC":
                "{:+.4f}".format,

            "Baseline_Brier":
                "{:.4f}".format,

            "Extended_Brier":
                "{:.4f}".format,

            "Brier_improvement":
                "{:+.4f}".format
        }
    )
)


print("\nExtended-model held-out leaders")
print("-------------------------------")

print(
    "Highest AP:",
    best_entry_extended_ap
)

print(
    "Highest AUROC:",
    best_entry_extended_auroc
)

print(
    "Lowest Brier:",
    best_entry_extended_brier
)


print("\nGovernance")
print("----------")

print(
    "Held-out women:",
    len(
        y_entry_test_eval
    )
)

print(
    "All models evaluated on same women:",
    "Yes"
)

print(
    "Survey-weighted evaluation:",
    "Yes"
)

print(
    "Models refitted:",
    "No"
)

print(
    "Thresholds used:",
    "No"
)

print(
    "Results fed back into development:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY HELD-OUT "
    "PROBABILITY PERFORMANCE SUMMARIZED"
)

## 2.40 Development-derived operating thresholds for paired ANC-entry models

Threshold-based performance requires converting predicted probabilities into
binary ANC8+ versus ANC8<8 classifications.

Each of the six ANC-entry models receives its own operating threshold because
their probability distributions differ.

Thresholds are derived exclusively from the locked development out-of-fold
predictions using the survey-weighted Youden index:

**Youden J = sensitivity + specificity - 1**

For each model, the threshold maximizing weighted Youden J is selected.

Important governance rules:

- thresholds are derived from development OOF predictions only;
- DHS sampling weights are used;
- baseline and extended specifications are treated separately;
- the held-out test outcomes play no role in threshold selection;
- no model is refitted or recalibrated.

Threshold-based measures are secondary to AP, AUROC, Brier score, and
calibration because their values depend on the selected operating threshold.

In [ ]:
# =============================================================================
# 2.40 DEVELOPMENT-DERIVED OPERATING THRESHOLDS
#
# Thresholds are selected using DEVELOPMENT OOF predictions only.
#
# Criterion:
#   weighted Youden J = sensitivity + specificity - 1
#
# The held-out test set is NOT used here.
# =============================================================================

from sklearn.metrics import roc_curve


# =============================================================================
# 1. HELPER: SELECT SURVEY-WEIGHTED YOUDEN THRESHOLD
# =============================================================================

def select_weighted_youden_threshold(
    y_true,
    probability,
    sample_weight
):
    """
    Select the probability threshold maximizing survey-weighted Youden J.

    Returns
    -------
    threshold
    sensitivity
    specificity
    youden_j
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    sample_weight = np.asarray(
        sample_weight,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Weighted ROC coordinates.
    #
    # drop_intermediate=False retains the complete empirical threshold set.
    # -------------------------------------------------------------------------

    fpr, tpr, thresholds = roc_curve(
        y_true,
        probability,
        sample_weight=sample_weight,
        drop_intermediate=False
    )


    specificity = (
        1.0
        - fpr
    )

    youden_j = (
        tpr
        + specificity
        - 1.0
    )


    # -------------------------------------------------------------------------
    # Remove non-finite threshold values such as the initial infinity returned
    # by some sklearn versions.
    # -------------------------------------------------------------------------

    finite_mask = np.isfinite(
        thresholds
    )

    thresholds_finite = (
        thresholds[
            finite_mask
        ]
    )

    tpr_finite = (
        tpr[
            finite_mask
        ]
    )

    specificity_finite = (
        specificity[
            finite_mask
        ]
    )

    youden_finite = (
        youden_j[
            finite_mask
        ]
    )


    # -------------------------------------------------------------------------
    # Select the maximum Youden J.
    #
    # np.argmax returns the first maximum in ROC threshold order if an exact
    # numerical tie occurs. No held-out information is involved.
    # -------------------------------------------------------------------------

    best_index = int(
        np.argmax(
            youden_finite
        )
    )


    return {
        "Threshold":
            float(
                thresholds_finite[
                    best_index
                ]
            ),

        "Sensitivity":
            float(
                tpr_finite[
                    best_index
                ]
            ),

        "Specificity":
            float(
                specificity_finite[
                    best_index
                ]
            ),

        "Youden_J":
            float(
                youden_finite[
                    best_index
                ]
            )
    }


# =============================================================================
# 2. LOCKED DEVELOPMENT OOF REGISTRY
# =============================================================================

entry_oof_threshold_models = {
    (
        "Logistic regression",
        "Baseline"
    ):
        oof_entry_logistic_baseline,

    (
        "Logistic regression",
        "Extended"
    ):
        oof_entry_logistic_extended,

    (
        "Random Forest",
        "Baseline"
    ):
        oof_entry_rf_baseline,

    (
        "Random Forest",
        "Extended"
    ):
        oof_entry_rf_extended,

    (
        "XGBoost",
        "Baseline"
    ):
        oof_entry_xgb_baseline,

    (
        "XGBoost",
        "Extended"
    ):
        oof_entry_xgb_extended
}


# =============================================================================
# 3. SELECT SIX DEVELOPMENT THRESHOLDS
# =============================================================================

entry_threshold_rows = []

ENTRY_LOCKED_THRESHOLDS = {}


for (
    algorithm,
    specification
), oof_probability in (
    entry_oof_threshold_models.items()
):

    threshold_result = (
        select_weighted_youden_threshold(
            y_true=y_entry,
            probability=oof_probability,
            sample_weight=survey_weight_entry_dev
        )
    )


    ENTRY_LOCKED_THRESHOLDS[
        (
            algorithm,
            specification
        )
    ] = threshold_result[
        "Threshold"
    ]


    entry_threshold_rows.append({
        "Algorithm":
            algorithm,

        "Specification":
            specification,

        "Threshold":
            threshold_result[
                "Threshold"
            ],

        "Development_sensitivity":
            threshold_result[
                "Sensitivity"
            ],

        "Development_specificity":
            threshold_result[
                "Specificity"
            ],

        "Development_Youden_J":
            threshold_result[
                "Youden_J"
            ]
    })


development_entry_thresholds = pd.DataFrame(
    entry_threshold_rows
)


# =============================================================================
# 4. INTEGRITY
# =============================================================================

assert len(
    development_entry_thresholds
) == 6

assert len(
    ENTRY_LOCKED_THRESHOLDS
) == 6

assert (
    development_entry_thresholds[
        "Threshold"
    ]
    .between(
        0,
        1
    )
).all()


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 132)
print("DEVELOPMENT-DERIVED OPERATING THRESHOLDS FOR ANC-ENTRY MODELS")
print("=" * 132)

print(
    development_entry_thresholds
    .to_string(
        index=False,
        formatters={
            "Threshold":
                "{:.4f}".format,

            "Development_sensitivity":
                "{:.4f}".format,

            "Development_specificity":
                "{:.4f}".format,

            "Development_Youden_J":
                "{:.4f}".format
        }
    )
)


print("\nGovernance")
print("----------")

print(
    "Threshold source:",
    "Development OOF predictions"
)

print(
    "Threshold criterion:",
    "Survey-weighted Youden J"
)

print(
    "Model-specific thresholds:",
    "Yes"
)

print(
    "Held-out outcomes used:",
    "No"
)

print(
    "Models refitted:",
    "No"
)

print(
    "\nSTATUS: SIX ANC-ENTRY OPERATING "
    "THRESHOLDS LOCKED FROM DEVELOPMENT"
)

## 2.41 Final threshold-based performance for paired ANC-entry models

The six operating thresholds selected from development out-of-fold predictions
are now applied unchanged to the held-out ANC-entry test population.

No threshold is re-estimated or adjusted using held-out outcomes.

For each baseline and extended specification, the following survey-weighted
classification measures are calculated:

- sensitivity;
- specificity;
- positive predictive value (PPV);
- negative predictive value (NPV);
- F1 score;
- predicted-positive rate.

These threshold-based measures are secondary to the probability-based
performance measures because they depend on the chosen operating threshold.

The purpose is to assess whether adding `early_anc` also improves practical
binary classification performance at the locked development-derived operating
point.

In [ ]:
# =============================================================================
# 2.41 FINAL THRESHOLD-BASED PERFORMANCE ON HELD-OUT ANC-ENTRY TEST DATA
#
# Each model uses the threshold locked in Section 2.40.
#
# No threshold re-estimation occurs here.
# =============================================================================


# =============================================================================
# 1. HELPER: SURVEY-WEIGHTED CLASSIFICATION METRICS
# =============================================================================

def weighted_binary_metrics(
    y_true,
    probability,
    threshold,
    sample_weight
):
    """
    Calculate weighted threshold-based classification performance.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    sample_weight = np.asarray(
        sample_weight,
        dtype=float
    )


    predicted_class = (
        probability >= threshold
    ).astype(int)


    # -------------------------------------------------------------------------
    # Weighted confusion-matrix components
    # -------------------------------------------------------------------------

    tp = np.sum(
        sample_weight[
            (y_true == 1)
            &
            (predicted_class == 1)
        ]
    )

    fn = np.sum(
        sample_weight[
            (y_true == 1)
            &
            (predicted_class == 0)
        ]
    )

    tn = np.sum(
        sample_weight[
            (y_true == 0)
            &
            (predicted_class == 0)
        ]
    )

    fp = np.sum(
        sample_weight[
            (y_true == 0)
            &
            (predicted_class == 1)
        ]
    )


    # -------------------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------------------

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    f1 = (
        2 * ppv * sensitivity
        / (ppv + sensitivity)
        if (
            np.isfinite(ppv)
            and np.isfinite(sensitivity)
            and (ppv + sensitivity) > 0
        )
        else np.nan
    )

    predicted_positive_rate = np.average(
        predicted_class,
        weights=sample_weight
    )


    return {
        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "PPV":
            ppv,

        "NPV":
            npv,

        "F1":
            f1,

        "Predicted_positive_rate":
            predicted_positive_rate
    }


# =============================================================================
# 2. PROBABILITY COLUMN REGISTRY
# =============================================================================

entry_test_threshold_models = {
    (
        "Logistic regression",
        "Baseline"
    ):
        "prob_logistic_baseline",

    (
        "Logistic regression",
        "Extended"
    ):
        "prob_logistic_extended",

    (
        "Random Forest",
        "Baseline"
    ):
        "prob_rf_baseline",

    (
        "Random Forest",
        "Extended"
    ):
        "prob_rf_extended",

    (
        "XGBoost",
        "Baseline"
    ):
        "prob_xgb_baseline",

    (
        "XGBoost",
        "Extended"
    ):
        "prob_xgb_extended"
}


# =============================================================================
# 3. APPLY LOCKED THRESHOLDS TO HELD-OUT TEST DATA
# =============================================================================

entry_test_threshold_rows = []


for (
    algorithm,
    specification
), probability_column in (
    entry_test_threshold_models.items()
):

    threshold = (
        ENTRY_LOCKED_THRESHOLDS[
            (
                algorithm,
                specification
            )
        ]
    )


    metrics = weighted_binary_metrics(
        y_true=entry_test_predictions_locked[
            "anc8plus_observed"
        ],
        probability=entry_test_predictions_locked[
            probability_column
        ],
        threshold=threshold,
        sample_weight=entry_test_predictions_locked[
            "survey_weight"
        ]
    )


    entry_test_threshold_rows.append({
        "Algorithm":
            algorithm,

        "Specification":
            specification,

        "Threshold":
            threshold,

        "Sensitivity":
            metrics[
                "Sensitivity"
            ],

        "Specificity":
            metrics[
                "Specificity"
            ],

        "PPV":
            metrics[
                "PPV"
            ],

        "NPV":
            metrics[
                "NPV"
            ],

        "F1":
            metrics[
                "F1"
            ],

        "Predicted_positive_rate":
            metrics[
                "Predicted_positive_rate"
            ]
    })


entry_test_threshold_performance = pd.DataFrame(
    entry_test_threshold_rows
)


# =============================================================================
# 4. INTEGRITY
# =============================================================================

assert len(
    entry_test_threshold_performance
) == 6

assert np.isfinite(
    entry_test_threshold_performance[
        [
            "Threshold",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1",
            "Predicted_positive_rate"
        ]
    ].to_numpy()
).all()


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 156)
print("FINAL HELD-OUT THRESHOLD-BASED PERFORMANCE FOR ANC-ENTRY MODELS")
print("=" * 156)

print(
    entry_test_threshold_performance
    .to_string(
        index=False,
        formatters={
            "Threshold":
                "{:.4f}".format,

            "Sensitivity":
                "{:.4f}".format,

            "Specificity":
                "{:.4f}".format,

            "PPV":
                "{:.4f}".format,

            "NPV":
                "{:.4f}".format,

            "F1":
                "{:.4f}".format,

            "Predicted_positive_rate":
                "{:.4f}".format
        }
    )
)


print("\nGovernance")
print("----------")

print(
    "Threshold source:",
    "Development OOF predictions"
)

print(
    "Thresholds changed on test set:",
    "No"
)

print(
    "Survey-weighted evaluation:",
    "Yes"
)

print(
    "Held-out women:",
    len(
        entry_test_predictions_locked
    )
)

print(
    "Models refitted:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY HELD-OUT "
    "THRESHOLD PERFORMANCE SUMMARIZED"
)

## 2.42 Consolidated incremental value of early ANC initiation

The held-out incremental value of adding `early_anc` is consolidated across
probability-based performance, calibration, and threshold-based classification.

All comparisons are paired within algorithm because baseline and extended
models are evaluated on the same 984 held-out women.

For probability-based measures:

- `Delta_AP = Extended - Baseline`;
- `Delta_AUROC = Extended - Baseline`;
- `Brier_improvement = Baseline - Extended`.

Thus, positive values indicate improvement.

Calibration is summarized as movement toward the ideal:

- `CITL_improvement = |Baseline CITL| - |Extended CITL|`;
- `Slope_improvement = |Baseline slope - 1| - |Extended slope - 1|`.

Positive values therefore indicate calibration closer to the ideal value.

For threshold-based measures:

- sensitivity, specificity, PPV, NPV, and F1 changes are calculated as
  `Extended - Baseline`.

This table contains held-out point estimates only. Statistical uncertainty in
the primary probability-based incremental differences is assessed subsequently
using paired stratified-PSU bootstrap resampling.

In [ ]:
# =============================================================================
# 2.42 CONSOLIDATED INCREMENTAL VALUE OF early_anc
#
# Combines held-out:
#   - probability performance
#   - calibration
#   - threshold-based performance
#
# IMPORTANT:
# This is a POINT-ESTIMATE summary only.
# Statistical uncertainty is handled in Section 2.43.
# =============================================================================


# =============================================================================
# 1. START WITH PRIMARY PROBABILITY-BASED DIFFERENCES
# =============================================================================

entry_incremental_summary = (
    entry_test_incremental_performance[
        [
            "Algorithm",
            "Delta_AP",
            "Delta_AUROC",
            "Brier_improvement"
        ]
    ]
    .copy()
)


# =============================================================================
# 2. CALIBRATION: BASELINE VERSUS EXTENDED
# =============================================================================

calibration_baseline = (
    entry_calibration_summary[
        entry_calibration_summary[
            "Specification"
        ].eq(
            "Baseline"
        )
    ]
    [
        [
            "Algorithm",
            "CITL",
            "Calibration_slope"
        ]
    ]
    .rename(
        columns={
            "CITL":
                "Baseline_CITL",

            "Calibration_slope":
                "Baseline_slope"
        }
    )
)


calibration_extended = (
    entry_calibration_summary[
        entry_calibration_summary[
            "Specification"
        ].eq(
            "Extended"
        )
    ]
    [
        [
            "Algorithm",
            "CITL",
            "Calibration_slope"
        ]
    ]
    .rename(
        columns={
            "CITL":
                "Extended_CITL",

            "Calibration_slope":
                "Extended_slope"
        }
    )
)


calibration_pair = calibration_baseline.merge(
    calibration_extended,
    on="Algorithm",
    how="inner",
    validate="one_to_one"
)


# Positive = closer to ideal CITL of 0.
calibration_pair[
    "CITL_improvement"
] = (
    calibration_pair[
        "Baseline_CITL"
    ].abs()
    -
    calibration_pair[
        "Extended_CITL"
    ].abs()
)


# Positive = closer to ideal slope of 1.
calibration_pair[
    "Slope_improvement"
] = (
    (
        calibration_pair[
            "Baseline_slope"
        ]
        - 1.0
    ).abs()
    -
    (
        calibration_pair[
            "Extended_slope"
        ]
        - 1.0
    ).abs()
)


# =============================================================================
# 3. THRESHOLD-BASED METRICS: BASELINE VERSUS EXTENDED
# =============================================================================

threshold_baseline = (
    entry_test_threshold_performance[
        entry_test_threshold_performance[
            "Specification"
        ].eq(
            "Baseline"
        )
    ]
    [
        [
            "Algorithm",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1"
        ]
    ]
    .rename(
        columns={
            "Sensitivity":
                "Baseline_sensitivity",

            "Specificity":
                "Baseline_specificity",

            "PPV":
                "Baseline_PPV",

            "NPV":
                "Baseline_NPV",

            "F1":
                "Baseline_F1"
        }
    )
)


threshold_extended = (
    entry_test_threshold_performance[
        entry_test_threshold_performance[
            "Specification"
        ].eq(
            "Extended"
        )
    ]
    [
        [
            "Algorithm",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1"
        ]
    ]
    .rename(
        columns={
            "Sensitivity":
                "Extended_sensitivity",

            "Specificity":
                "Extended_specificity",

            "PPV":
                "Extended_PPV",

            "NPV":
                "Extended_NPV",

            "F1":
                "Extended_F1"
        }
    )
)


threshold_pair = threshold_baseline.merge(
    threshold_extended,
    on="Algorithm",
    how="inner",
    validate="one_to_one"
)


threshold_pair[
    "Delta_sensitivity"
] = (
    threshold_pair[
        "Extended_sensitivity"
    ]
    -
    threshold_pair[
        "Baseline_sensitivity"
    ]
)

threshold_pair[
    "Delta_specificity"
] = (
    threshold_pair[
        "Extended_specificity"
    ]
    -
    threshold_pair[
        "Baseline_specificity"
    ]
)

threshold_pair[
    "Delta_PPV"
] = (
    threshold_pair[
        "Extended_PPV"
    ]
    -
    threshold_pair[
        "Baseline_PPV"
    ]
)

threshold_pair[
    "Delta_NPV"
] = (
    threshold_pair[
        "Extended_NPV"
    ]
    -
    threshold_pair[
        "Baseline_NPV"
    ]
)

threshold_pair[
    "Delta_F1"
] = (
    threshold_pair[
        "Extended_F1"
    ]
    -
    threshold_pair[
        "Baseline_F1"
    ]
)


# =============================================================================
# 4. MERGE ALL INCREMENTAL RESULTS
# =============================================================================

entry_incremental_summary = (
    entry_incremental_summary
    .merge(
        calibration_pair[
            [
                "Algorithm",
                "CITL_improvement",
                "Slope_improvement"
            ]
        ],
        on="Algorithm",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        threshold_pair[
            [
                "Algorithm",
                "Delta_sensitivity",
                "Delta_specificity",
                "Delta_PPV",
                "Delta_NPV",
                "Delta_F1"
            ]
        ],
        on="Algorithm",
        how="inner",
        validate="one_to_one"
    )
)


# =============================================================================
# 5. INTEGRITY
# =============================================================================

assert len(
    entry_incremental_summary
) == 3

assert entry_incremental_summary[
    "Algorithm"
].is_unique

assert np.isfinite(
    entry_incremental_summary.drop(
        columns="Algorithm"
    ).to_numpy()
).all()


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 174)
print("CONSOLIDATED HELD-OUT INCREMENTAL VALUE OF early_anc")
print("=" * 174)

print(
    entry_incremental_summary
    .to_string(
        index=False,
        formatters={
            "Delta_AP":
                "{:+.4f}".format,

            "Delta_AUROC":
                "{:+.4f}".format,

            "Brier_improvement":
                "{:+.4f}".format,

            "CITL_improvement":
                "{:+.4f}".format,

            "Slope_improvement":
                "{:+.4f}".format,

            "Delta_sensitivity":
                "{:+.4f}".format,

            "Delta_specificity":
                "{:+.4f}".format,

            "Delta_PPV":
                "{:+.4f}".format,

            "Delta_NPV":
                "{:+.4f}".format,

            "Delta_F1":
                "{:+.4f}".format
        }
    )
)


print("\nDirection convention")
print("--------------------")

print(
    "Positive Delta_AP / Delta_AUROC:",
    "Extended better"
)

print(
    "Positive Brier_improvement:",
    "Extended better"
)

print(
    "Positive CITL_improvement:",
    "Extended closer to CITL = 0"
)

print(
    "Positive Slope_improvement:",
    "Extended closer to slope = 1"
)

print(
    "Positive threshold-metric deltas:",
    "Extended higher"
)


print("\nInference status")
print("----------------")

print(
    "Point estimates summarized:",
    "Yes"
)

print(
    "Confidence intervals calculated here:",
    "No"
)

print(
    "Statistical superiority claimed:",
    "No"
)

print(
    "\nSTATUS: HELD-OUT INCREMENTAL "
    "POINT ESTIMATES CONSOLIDATED"
)

### 2.43.1 Held-out survey-design audit for paired bootstrap

Uncertainty in ANC-entry held-out performance will be estimated using a paired
stratified PSU bootstrap.

The bootstrap resamples primary sampling units (PSUs), rather than individual
women, because EDHS observations within a PSU are not statistically
independent.

Within each survey stratum:

- PSUs will be sampled with replacement;
- the number of sampled PSU draws will equal the number of observed held-out
  PSUs in that stratum;
- all women belonging to each selected PSU will be retained;
- repeated PSU draws will contribute repeated copies of their women.

The same bootstrap replicate will be used simultaneously for the baseline and
extended predictions from all three algorithms. This preserves pairing for
incremental-performance comparisons.

Before resampling, the held-out design is audited for:

- number of women;
- number of PSUs;
- number of strata;
- PSU count within each stratum;
- presence of singleton strata.

No model fitting or prediction occurs in this section.

In [ ]:
# =============================================================================
# 2.43.1 HELD-OUT SURVEY-DESIGN AUDIT FOR PAIRED STRATIFIED-PSU BOOTSTRAP
# =============================================================================


# =============================================================================
# 1. ONE ROW PER HELD-OUT PSU
# =============================================================================

entry_test_psu_frame = (
    entry_test_predictions_locked[
        [
            "v021",
            "v022"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# Each PSU must belong to exactly one survey stratum.
assert (
    entry_test_psu_frame
    .groupby(
        "v021"
    )[
        "v022"
    ]
    .nunique()
    .max()
    == 1
)


# =============================================================================
# 2. PSU COUNTS WITHIN STRATA
# =============================================================================

entry_test_stratum_design = (
    entry_test_predictions_locked
    .groupby(
        "v022",
        observed=True
    )
    .agg(
        Women=(
            "caseid",
            "size"
        ),

        PSUs=(
            "v021",
            "nunique"
        ),

        Survey_weight_sum=(
            "survey_weight",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "v022"
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 3. DESIGN SUMMARY
# =============================================================================

n_entry_test_women = len(
    entry_test_predictions_locked
)

n_entry_test_psus = (
    entry_test_predictions_locked[
        "v021"
    ]
    .nunique()
)

n_entry_test_strata = (
    entry_test_predictions_locked[
        "v022"
    ]
    .nunique()
)

min_psus_per_entry_test_stratum = int(
    entry_test_stratum_design[
        "PSUs"
    ].min()
)

max_psus_per_entry_test_stratum = int(
    entry_test_stratum_design[
        "PSUs"
    ].max()
)

n_singleton_entry_test_strata = int(
    (
        entry_test_stratum_design[
            "PSUs"
        ]
        == 1
    ).sum()
)

n_two_psu_entry_test_strata = int(
    (
        entry_test_stratum_design[
            "PSUs"
        ]
        == 2
    ).sum()
)


# =============================================================================
# 4. INTEGRITY
# =============================================================================

assert n_entry_test_women == 984

assert n_entry_test_psus == 157

assert n_entry_test_strata == 27

assert (
    entry_test_stratum_design[
        "Women"
    ].sum()
    == 984
)

assert (
    entry_test_stratum_design[
        "PSUs"
    ].sum()
    == 157
)


# Stratified PSU bootstrap requires at least two observed PSUs per stratum
# for meaningful within-stratum resampling.
assert (
    min_psus_per_entry_test_stratum
    >= 2
)


# =============================================================================
# 5. DISPLAY
# =============================================================================

print("=" * 126)
print("ANC-ENTRY HELD-OUT SURVEY-DESIGN AUDIT FOR PAIRED BOOTSTRAP")
print("=" * 126)

print("\nOverall structure")
print("-----------------")

print(
    "Held-out women:",
    n_entry_test_women
)

print(
    "Held-out PSUs:",
    n_entry_test_psus
)

print(
    "Held-out strata:",
    n_entry_test_strata
)

print(
    "Minimum PSUs per stratum:",
    min_psus_per_entry_test_stratum
)

print(
    "Maximum PSUs per stratum:",
    max_psus_per_entry_test_stratum
)

print(
    "Singleton strata:",
    n_singleton_entry_test_strata
)

print(
    "Strata with exactly 2 PSUs:",
    n_two_psu_entry_test_strata
)


print("\nStratum-level audit")
print("-------------------")

print(
    entry_test_stratum_design.to_string(
        index=False,
        formatters={
            "Survey_weight_sum":
                "{:.2f}".format
        }
    )
)


print("\nBootstrap readiness")
print("-------------------")

print(
    "Within-stratum PSU resampling supported:",
    "Yes"
)

print(
    "Individual-woman bootstrap planned:",
    "No"
)

print(
    "Same replicate will compare baseline and extended models:",
    "Yes"
)

print(
    "Models will be refitted during bootstrap:",
    "No"
)

print(
    "\nSTATUS: ANC-ENTRY HELD-OUT DESIGN "
    "SUPPORTS STRATIFIED PSU BOOTSTRAP"
)

### 2.43.2 Paired stratified-PSU bootstrap confidence intervals

Uncertainty in held-out ANC-entry predictive performance is estimated using
paired stratified PSU bootstrap resampling.

Within each survey stratum, PSUs are sampled with replacement while preserving
the observed number of PSUs in that stratum. All women belonging to each sampled
PSU are retained together.

For every bootstrap replicate:

- the same resampled observations are used for all six locked models;
- no model is refitted;
- no prediction is regenerated;
- original survey weights are retained;
- weighted Average Precision, AUROC, and Brier score are recalculated.

Paired incremental differences are calculated within the same replicate:

- `Delta_AP = Extended AP - Baseline AP`;
- `Delta_AUROC = Extended AUROC - Baseline AUROC`;
- `Brier_improvement = Baseline Brier - Extended Brier`.

Positive values therefore favor the extended model.

A total of 2,000 bootstrap replicates are generated using seed 2026.
Empirical 95% percentile confidence intervals are reported.

If a paired-difference confidence interval excludes 0, the held-out data provide
clear evidence of a difference in that metric between the baseline and extended
specifications.

In [ ]:
# =============================================================================
# 2.43.2 PAIRED STRATIFIED-PSU BOOTSTRAP FOR ANC-ENTRY PERFORMANCE
#
# Resampling unit:
#   PSU within survey stratum
#
# Replicates:
#   2,000
#
# Seed:
#   2026
#
# Models and predictions remain fixed.
# =============================================================================


# =============================================================================
# 1. BOOTSTRAP SETTINGS
# =============================================================================

ENTRY_BOOTSTRAP_REPLICATES = 2000
ENTRY_BOOTSTRAP_SEED = 2026

rng_entry_bootstrap = np.random.default_rng(
    ENTRY_BOOTSTRAP_SEED
)


# =============================================================================
# 2. PRE-COMPUTE PSU -> ROW-INDEX MAPPING WITHIN STRATUM
#
# This avoids repeatedly searching the dataframe inside every replicate.
# =============================================================================

entry_test_bootstrap_df = (
    entry_test_predictions_locked
    .reset_index(
        drop=True
    )
    .copy()
)


entry_stratum_psus = {}

entry_stratum_psu_rows = {}


for stratum_value, stratum_df in (
    entry_test_bootstrap_df
    .groupby(
        "v022",
        observed=True,
        sort=True
    )
):

    psus = (
        stratum_df[
            "v021"
        ]
        .drop_duplicates()
        .to_numpy()
    )

    entry_stratum_psus[
        stratum_value
    ] = psus


    for psu_value in psus:

        row_indices = (
            stratum_df.index[
                stratum_df[
                    "v021"
                ].eq(
                    psu_value
                )
            ]
            .to_numpy()
        )

        entry_stratum_psu_rows[
            (
                stratum_value,
                psu_value
            )
        ] = row_indices


# =============================================================================
# 3. HELPER: WEIGHTED PROBABILITY METRICS
# =============================================================================

def entry_bootstrap_probability_metrics(
    y_true,
    probability,
    weight
):

    ap = average_precision_score(
        y_true,
        probability,
        sample_weight=weight
    )

    auroc = roc_auc_score(
        y_true,
        probability,
        sample_weight=weight
    )

    brier = brier_score_loss(
        y_true,
        probability,
        sample_weight=weight
    )

    return ap, auroc, brier


# =============================================================================
# 4. MODEL COLUMN REGISTRY
# =============================================================================

entry_bootstrap_model_columns = {
    "Logistic regression": {
        "Baseline":
            "prob_logistic_baseline",

        "Extended":
            "prob_logistic_extended"
    },

    "Random Forest": {
        "Baseline":
            "prob_rf_baseline",

        "Extended":
            "prob_rf_extended"
    },

    "XGBoost": {
        "Baseline":
            "prob_xgb_baseline",

        "Extended":
            "prob_xgb_extended"
    }
}


# =============================================================================
# 5. RUN PAIRED STRATIFIED-PSU BOOTSTRAP
# =============================================================================

entry_bootstrap_metric_rows = []

entry_bootstrap_difference_rows = []

entry_bootstrap_retained = 0


for replicate in range(
    1,
    ENTRY_BOOTSTRAP_REPLICATES + 1
):

    sampled_row_blocks = []


    # -------------------------------------------------------------------------
    # Within every stratum, sample the observed number of PSUs with replacement.
    # -------------------------------------------------------------------------

    for stratum_value, psus in (
        entry_stratum_psus.items()
    ):

        n_psus = len(
            psus
        )

        sampled_psus = (
            rng_entry_bootstrap.choice(
                psus,
                size=n_psus,
                replace=True
            )
        )


        # Each selected PSU contributes all of its women.
        # A PSU selected multiple times contributes repeated copies.
        for sampled_psu in sampled_psus:

            sampled_row_blocks.append(
                entry_stratum_psu_rows[
                    (
                        stratum_value,
                        sampled_psu
                    )
                ]
            )


    sampled_rows = np.concatenate(
        sampled_row_blocks
    )


    bootstrap_df = (
        entry_test_bootstrap_df
        .iloc[
            sampled_rows
        ]
    )


    y_boot = (
        bootstrap_df[
            "anc8plus_observed"
        ]
        .astype(int)
        .to_numpy()
    )

    w_boot = (
        bootstrap_df[
            "survey_weight"
        ]
        .astype(float)
        .to_numpy()
    )


    # -------------------------------------------------------------------------
    # AUROC requires both outcome classes.
    # With this sample size/design, failure should be extremely rare, but
    # explicitly guard against invalid replicates.
    # -------------------------------------------------------------------------

    if np.unique(
        y_boot
    ).size < 2:

        continue


    replicate_metrics = {}


    # =========================================================================
    # PERFORMANCE OF ALL SIX LOCKED MODELS
    # =========================================================================

    for algorithm, specification_map in (
        entry_bootstrap_model_columns.items()
    ):

        replicate_metrics[
            algorithm
        ] = {}


        for specification, probability_column in (
            specification_map.items()
        ):

            probability_boot = (
                bootstrap_df[
                    probability_column
                ]
                .astype(float)
                .to_numpy()
            )


            ap_boot, auroc_boot, brier_boot = (
                entry_bootstrap_probability_metrics(
                    y_true=y_boot,
                    probability=probability_boot,
                    weight=w_boot
                )
            )


            replicate_metrics[
                algorithm
            ][
                specification
            ] = {
                "AP":
                    ap_boot,

                "AUROC":
                    auroc_boot,

                "Brier":
                    brier_boot
            }


            entry_bootstrap_metric_rows.append({
                "Replicate":
                    replicate,

                "Algorithm":
                    algorithm,

                "Specification":
                    specification,

                "AP":
                    ap_boot,

                "AUROC":
                    auroc_boot,

                "Brier":
                    brier_boot
            })


    # =========================================================================
    # PAIRED BASELINE VS EXTENDED DIFFERENCES
    # =========================================================================

    for algorithm in (
        entry_bootstrap_model_columns
    ):

        base = (
            replicate_metrics[
                algorithm
            ][
                "Baseline"
            ]
        )

        ext = (
            replicate_metrics[
                algorithm
            ][
                "Extended"
            ]
        )


        entry_bootstrap_difference_rows.append({
            "Replicate":
                replicate,

            "Algorithm":
                algorithm,

            "Delta_AP":
                ext[
                    "AP"
                ]
                -
                base[
                    "AP"
                ],

            "Delta_AUROC":
                ext[
                    "AUROC"
                ]
                -
                base[
                    "AUROC"
                ],

            "Brier_improvement":
                base[
                    "Brier"
                ]
                -
                ext[
                    "Brier"
                ]
        })


    entry_bootstrap_retained += 1


# =============================================================================
# 6. STORE BOOTSTRAP DISTRIBUTIONS
# =============================================================================

entry_bootstrap_performance_distribution = pd.DataFrame(
    entry_bootstrap_metric_rows
)

entry_bootstrap_difference_distribution = pd.DataFrame(
    entry_bootstrap_difference_rows
)


# =============================================================================
# 7. HELPER: EMPIRICAL 95% PERCENTILE CI
# =============================================================================

def percentile_ci(
    values,
    lower=2.5,
    upper=97.5
):

    values = np.asarray(
        values,
        dtype=float
    )

    return (
        float(
            np.percentile(
                values,
                lower
            )
        ),

        float(
            np.percentile(
                values,
                upper
            )
        )
    )


# =============================================================================
# 8. PERFORMANCE CIs FOR ALL SIX MODELS
# =============================================================================

entry_bootstrap_performance_ci_rows = []


for (
    algorithm,
    specification
), group_df in (
    entry_bootstrap_performance_distribution
    .groupby(
        [
            "Algorithm",
            "Specification"
        ],
        sort=False
    )
):

    for metric in [
        "AP",
        "AUROC",
        "Brier"
    ]:

        lower_ci, upper_ci = percentile_ci(
            group_df[
                metric
            ]
        )


        # Original held-out point estimate.
        point_row = (
            entry_test_performance_long[
                entry_test_performance_long[
                    "Algorithm"
                ].eq(
                    algorithm
                )
                &
                entry_test_performance_long[
                    "Specification"
                ].eq(
                    specification
                )
            ]
            .iloc[0]
        )


        point_estimate = {
            "AP":
                point_row[
                    "Weighted_AP"
                ],

            "AUROC":
                point_row[
                    "Weighted_AUROC"
                ],

            "Brier":
                point_row[
                    "Weighted_Brier"
                ]
        }[
            metric
        ]


        entry_bootstrap_performance_ci_rows.append({
            "Algorithm":
                algorithm,

            "Specification":
                specification,

            "Metric":
                metric,

            "Point_estimate":
                float(
                    point_estimate
                ),

            "CI_lower":
                lower_ci,

            "CI_upper":
                upper_ci
        })


entry_bootstrap_performance_ci = pd.DataFrame(
    entry_bootstrap_performance_ci_rows
)


# =============================================================================
# 9. PAIRED-DIFFERENCE CIs
# =============================================================================

entry_bootstrap_difference_ci_rows = []


for algorithm, group_df in (
    entry_bootstrap_difference_distribution
    .groupby(
        "Algorithm",
        sort=False
    )
):

    point_row = (
        entry_test_incremental_performance[
            entry_test_incremental_performance[
                "Algorithm"
            ].eq(
                algorithm
            )
        ]
        .iloc[0]
    )


    point_estimates = {
        "Delta_AP":
            point_row[
                "Delta_AP"
            ],

        "Delta_AUROC":
            point_row[
                "Delta_AUROC"
            ],

        "Brier_improvement":
            point_row[
                "Brier_improvement"
            ]
    }


    for metric in [
        "Delta_AP",
        "Delta_AUROC",
        "Brier_improvement"
    ]:

        lower_ci, upper_ci = percentile_ci(
            group_df[
                metric
            ]
        )


        entry_bootstrap_difference_ci_rows.append({
            "Algorithm":
                algorithm,

            "Metric":
                metric,

            "Point_estimate":
                float(
                    point_estimates[
                        metric
                    ]
                ),

            "CI_lower":
                lower_ci,

            "CI_upper":
                upper_ci,

            "CI_excludes_zero":
                bool(
                    (
                        lower_ci > 0
                    )
                    or
                    (
                        upper_ci < 0
                    )
                )
        })


entry_bootstrap_difference_ci = pd.DataFrame(
    entry_bootstrap_difference_ci_rows
)


# =============================================================================
# 10. INTEGRITY
# =============================================================================

assert entry_bootstrap_retained == (
    ENTRY_BOOTSTRAP_REPLICATES
)

assert len(
    entry_bootstrap_difference_distribution
) == (
    ENTRY_BOOTSTRAP_REPLICATES
    * 3
)

assert len(
    entry_bootstrap_difference_ci
) == 9

assert len(
    entry_bootstrap_performance_ci
) == 18


# =============================================================================
# 11. DISPLAY PAIRED INCREMENTAL CIs
# =============================================================================

print("=" * 152)
print("PAIRED STRATIFIED-PSU BOOTSTRAP: INCREMENTAL VALUE OF early_anc")
print("=" * 152)

print(
    "Requested replicates:",
    ENTRY_BOOTSTRAP_REPLICATES
)

print(
    "Retained replicates:",
    entry_bootstrap_retained
)

print(
    "Bootstrap seed:",
    ENTRY_BOOTSTRAP_SEED
)


print("\nPaired held-out differences with 95% percentile CIs")
print("----------------------------------------------------")

print(
    entry_bootstrap_difference_ci
    .to_string(
        index=False,
        formatters={
            "Point_estimate":
                "{:+.4f}".format,

            "CI_lower":
                "{:+.4f}".format,

            "CI_upper":
                "{:+.4f}".format
        }
    )
)


print("\nInterpretation")
print("--------------")

print(
    "Delta_AP > 0:",
    "Extended model has higher AP"
)

print(
    "Delta_AUROC > 0:",
    "Extended model has higher AUROC"
)

print(
    "Brier_improvement > 0:",
    "Extended model has lower Brier score"
)

print(
    "CI excludes 0:",
    "Clear held-out evidence of a difference for that metric"
)


print("\nGovernance")
print("----------")

print(
    "Bootstrap unit:",
    "PSU within stratum"
)

print(
    "Same replicate used for baseline and extended:",
    "Yes"
)

print(
    "Survey weights retained:",
    "Yes"
)

print(
    "Models refitted:",
    "No"
)

print(
    "Predictions regenerated:",
    "No"
)

print(
    "\nSTATUS: PAIRED ANC-ENTRY "
    "BOOTSTRAP UNCERTAINTY ESTIMATED"
)

## 2.44 Notebook 3 explainability handoff preparation

Notebook 2 has completed model development and held-out evaluation.

Before final export, the transformed-feature lineage required for subsequent
model explainability is frozen.

The primary full-cohort models already have a governed lineage linking their
78 transformed model features to the original 21 predictor constructs.

The ANC-entry extended models contain one additional predictor, `early_anc`,
and therefore contain 79 transformed features representing 22 original
predictor constructs.

The ANC-entry lineage is derived directly from the fitted preprocessing object,
its output slices, and the fitted one-hot-encoder category structure. Feature
ownership is not inferred by parsing transformed feature names.

This lineage will serve as the authority for grouping transformed features back
to their original predictor constructs in Notebook 3.

In [ ]:
# =============================================================================
# 2.44.1 FREEZE ANC-ENTRY TRANSFORMED-FEATURE LINEAGE
#
# PURPOSE
# -------
# Map every transformed feature in the final ANC-entry extended models
# back to exactly one of the 22 original predictor constructs.
#
# IMPORTANT
# ---------
# Ownership is derived from the fitted ColumnTransformer structure and fitted
# encoder categories, NOT by guessing from transformed feature-name strings.
# =============================================================================


# =============================================================================
# 1. USE ONE FITTED EXTENDED PREPROCESSOR AS THE LINEAGE AUTHORITY
#
# All three extended algorithms use the same predictor/preprocessing
# architecture, so the fitted extended logistic preprocessor is sufficient
# for deriving the structural lineage.
# =============================================================================

entry_lineage_preprocessor = (
    final_entry_logistic_extended[
        "preprocessor"
    ]
)

entry_transformed_feature_names = (
    entry_lineage_preprocessor
    .get_feature_names_out()
)


assert len(
    entry_transformed_feature_names
) == 79


# =============================================================================
# 2. OUTPUT SLICES FROM THE FITTED COLUMNTRANSFORMER
#
# These slices tell us exactly which transformed columns were produced by
# each preprocessing branch.
# =============================================================================

entry_output_slices = (
    entry_lineage_preprocessor
    .output_indices_
)


required_entry_transformers = {
    "numeric",
    "binary",
    "categorical",
    "early_anc"
}


assert required_entry_transformers.issubset(
    set(
        entry_output_slices.keys()
    )
)


# =============================================================================
# 3. BUILD LINEAGE ROWS
# =============================================================================

entry_feature_lineage_rows = []


# -----------------------------------------------------------------------------
# Continuous numeric predictors
# One transformed column per original predictor.
# -----------------------------------------------------------------------------

numeric_slice = (
    entry_output_slices[
        "numeric"
    ]
)

numeric_feature_names = (
    entry_transformed_feature_names[
        numeric_slice
    ]
)


assert len(
    numeric_feature_names
) == len(
    NUMERIC_PREDICTORS
)


for transformed_name, original_predictor in zip(
    numeric_feature_names,
    NUMERIC_PREDICTORS
):

    entry_feature_lineage_rows.append({
        "transformed_feature":
            transformed_name,

        "original_predictor":
            original_predictor,

        "feature_family":
            "numeric"
    })


# -----------------------------------------------------------------------------
# Governed binary predictors
# One transformed 0/1 column per original predictor.
# -----------------------------------------------------------------------------

binary_slice = (
    entry_output_slices[
        "binary"
    ]
)

binary_feature_names = (
    entry_transformed_feature_names[
        binary_slice
    ]
)


assert len(
    binary_feature_names
) == len(
    BINARY_PREDICTORS
)


for transformed_name, original_predictor in zip(
    binary_feature_names,
    BINARY_PREDICTORS
):

    entry_feature_lineage_rows.append({
        "transformed_feature":
            transformed_name,

        "original_predictor":
            original_predictor,

        "feature_family":
            "binary"
    })


# -----------------------------------------------------------------------------
# Multicategory predictors
#
# The fitted OneHotEncoder tells us how many transformed columns belong to
# each original categorical predictor.
# -----------------------------------------------------------------------------

fitted_entry_categorical_pipeline = (
    entry_lineage_preprocessor
    .named_transformers_[
        "categorical"
    ]
)


# Locate the fitted OneHotEncoder without depending on an assumed step name.
fitted_entry_ohe = None

for step_name, step_object in (
    fitted_entry_categorical_pipeline
    .named_steps
    .items()
):

    if isinstance(
        step_object,
        OneHotEncoder
    ):

        fitted_entry_ohe = (
            step_object
        )

        break


assert fitted_entry_ohe is not None, (
    "Could not locate the fitted OneHotEncoder "
    "inside the categorical preprocessing pipeline."
)


categorical_slice = (
    entry_output_slices[
        "categorical"
    ]
)

categorical_feature_names = (
    entry_transformed_feature_names[
        categorical_slice
    ]
)


categorical_cursor = 0


for original_predictor, fitted_categories in zip(
    CATEGORICAL_PREDICTORS_ML,
    fitted_entry_ohe.categories_
):

    n_categories = len(
        fitted_categories
    )

    predictor_feature_names = (
        categorical_feature_names[
            categorical_cursor:
            categorical_cursor + n_categories
        ]
    )


    assert len(
        predictor_feature_names
    ) == n_categories


    for transformed_name in (
        predictor_feature_names
    ):

        entry_feature_lineage_rows.append({
            "transformed_feature":
                transformed_name,

            "original_predictor":
                original_predictor,

            "feature_family":
                "categorical"
        })


    categorical_cursor += (
        n_categories
    )


assert categorical_cursor == len(
    categorical_feature_names
)


# -----------------------------------------------------------------------------
# early_anc
# Exactly one direct 0/1 transformed feature.
# -----------------------------------------------------------------------------

early_anc_slice = (
    entry_output_slices[
        "early_anc"
    ]
)

early_anc_feature_names = (
    entry_transformed_feature_names[
        early_anc_slice
    ]
)


assert len(
    early_anc_feature_names
) == 1


entry_feature_lineage_rows.append({
    "transformed_feature":
        early_anc_feature_names[
            0
        ],

    "original_predictor":
        "early_anc",

    "feature_family":
        "ANC-entry timing"
})


# =============================================================================
# 4. FREEZE LINEAGE DATAFRAME
# =============================================================================

entry_feature_lineage = pd.DataFrame(
    entry_feature_lineage_rows
)


# =============================================================================
# 5. INTEGRITY CHECKS
# =============================================================================

assert len(
    entry_feature_lineage
) == 79


assert entry_feature_lineage[
    "transformed_feature"
].is_unique


assert set(
    entry_feature_lineage[
        "transformed_feature"
    ]
) == set(
    entry_transformed_feature_names
)


assert entry_feature_lineage[
    "original_predictor"
].nunique() == 22


assert set(
    entry_feature_lineage[
        "original_predictor"
    ]
) == set(
    ANC_ENTRY_EXTENDED_PREDICTORS
)


# Every original predictor must own at least one transformed feature.
entry_lineage_counts = (
    entry_feature_lineage
    .groupby(
        "original_predictor"
    )
    .size()
    .rename(
        "Transformed_features"
    )
    .reset_index()
)


assert (
    entry_lineage_counts[
        "Transformed_features"
    ] >= 1
).all()


# early_anc must own exactly one transformed feature.
assert int(
    entry_lineage_counts.loc[
        entry_lineage_counts[
            "original_predictor"
        ].eq(
            "early_anc"
        ),
        "Transformed_features"
    ].iloc[0]
) == 1


# =============================================================================
# 6. DISPLAY
# =============================================================================

print("=" * 126)
print("ANC-ENTRY FEATURE LINEAGE FROZEN FOR NOTEBOOK 3")
print("=" * 126)

print(
    "Original predictor constructs:",
    entry_feature_lineage[
        "original_predictor"
    ].nunique()
)

print(
    "Transformed model features:",
    len(
        entry_feature_lineage
    )
)

print(
    "Duplicate transformed-feature ownership:",
    entry_feature_lineage[
        "transformed_feature"
    ].duplicated().any()
)


print("\nTransformed features per original predictor")
print("-------------------------------------------")

print(
    entry_lineage_counts.to_string(
        index=False
    )
)


print("\nNotebook-3 governance")
print("---------------------")

print(
    "Lineage derived from fitted preprocessing structure:",
    "Yes"
)

print(
    "Feature ownership inferred by parsing names:",
    "No"
)

print(
    "early_anc included:",
    "Yes"
)

print(
    "\nSTATUS: ANC-ENTRY FEATURE LINEAGE "
    "LOCKED FOR EXPLAINABILITY HANDOFF"
)

### 2.44.2 Frozen handoff package for Notebook 3

A self-contained explainability handoff is exported so that Notebook 3 can
begin from frozen fitted models and governed analytic data rather than rerunning
model development.

The handoff contains both prediction horizons:

1. **Full-cohort baseline horizon**
   - 21 original predictors;
   - development and held-out populations;
   - three final fitted model pipelines;
   - locked held-out predictions;
   - 78-feature transformed-feature lineage.

2. **ANC-entry horizon**
   - 21-predictor baseline and 22-predictor extended specifications;
   - development and held-out populations;
   - six final fitted model pipelines;
   - locked held-out predictions;
   - 79-feature extended-model lineage.

Machine-native model/data files are supplemented by human-readable CSV and JSON
metadata.

SHA-256 checksums are generated for every exported artifact so that Notebook 3
can verify that the handoff has not changed.

In [ ]:
# =============================================================================
# 2.44.2 EXPORT FROZEN NOTEBOOK-3 EXPLAINABILITY HANDOFF
#
# PURPOSE
# -------
# Make Notebook 3 independent of Notebook 2 execution state.
#
# Notebook 3 will LOAD these files. It should not reconstruct model development.
# =============================================================================

from pathlib import Path
import hashlib
import json
import platform
import sys
import joblib

import sklearn
import xgboost
import statsmodels


# =============================================================================
# 1. OUTPUT DIRECTORY
# =============================================================================

NOTEBOOK3_HANDOFF_DIR = Path(
    "outputs"
) / "notebook3_explainability_handoff"

NOTEBOOK3_HANDOFF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. VERIFY THAT ALL ESSENTIAL IN-MEMORY OBJECTS EXIST
#
# Fail now rather than silently creating an incomplete handoff.
# =============================================================================

required_handoff_objects = [
    # Primary full-cohort horizon
    "train_df",
    "test_df",
    "final_models",
    "test_predictions_locked",
    "feature_lineage",
    "PRIMARY_PREDICTORS",

    # ANC-entry horizon
    "anc_entry_train",
    "anc_entry_test",
    "final_entry_models",
    "entry_test_predictions_locked",
    "entry_feature_lineage",
    "ANC_ENTRY_BASELINE_PREDICTORS",
    "ANC_ENTRY_EXTENDED_PREDICTORS"
]


missing_handoff_objects = [
    object_name
    for object_name in required_handoff_objects
    if object_name not in globals()
]


assert not missing_handoff_objects, (
    "Missing required Notebook-3 handoff objects: "
    + ", ".join(
        missing_handoff_objects
    )
)


# =============================================================================
# 3. CORE DATASETS
#
# Pickle is used here because categorical/object dtype information is preserved
# more faithfully than in CSV.
# =============================================================================

handoff_data_files = {
    "full_cohort_development.pkl":
        train_df,

    "full_cohort_heldout.pkl":
        test_df,

    "anc_entry_development.pkl":
        anc_entry_train,

    "anc_entry_heldout.pkl":
        anc_entry_test
}


for filename, dataframe in (
    handoff_data_files.items()
):

    dataframe.to_pickle(
        NOTEBOOK3_HANDOFF_DIR
        / filename
    )


# =============================================================================
# 4. FINAL FITTED MODEL PIPELINES
#
# These contain the fitted preprocessing + fitted estimator together.
# =============================================================================

joblib.dump(
    final_models,
    NOTEBOOK3_HANDOFF_DIR
    / "full_cohort_final_models.joblib"
)


joblib.dump(
    final_entry_models,
    NOTEBOOK3_HANDOFF_DIR
    / "anc_entry_final_models.joblib"
)


# =============================================================================
# 5. LOCKED HELD-OUT PREDICTIONS
#
# CSV is deliberately used in addition to model serialization because these
# tables are useful for independent checking and future evaluation plots.
# =============================================================================

test_predictions_locked.to_csv(
    NOTEBOOK3_HANDOFF_DIR
    / "full_cohort_locked_predictions.csv",
    index=False
)


entry_test_predictions_locked.to_csv(
    NOTEBOOK3_HANDOFF_DIR
    / "anc_entry_locked_predictions.csv",
    index=False
)


# =============================================================================
# 6. FEATURE-LINEAGE AUTHORITIES
# =============================================================================

feature_lineage.to_csv(
    NOTEBOOK3_HANDOFF_DIR
    / "full_cohort_feature_lineage.csv",
    index=False
)


entry_feature_lineage.to_csv(
    NOTEBOOK3_HANDOFF_DIR
    / "anc_entry_feature_lineage.csv",
    index=False
)


# =============================================================================
# 7. PREDICTOR ARCHITECTURE
#
# Save explicitly rather than requiring Notebook 3 to recover these lists from
# fitted objects.
# =============================================================================

predictor_architecture = {
    "full_cohort_predictors":
        list(
            PRIMARY_PREDICTORS
        ),

    "anc_entry_baseline_predictors":
        list(
            ANC_ENTRY_BASELINE_PREDICTORS
        ),

    "anc_entry_extended_predictors":
        list(
            ANC_ENTRY_EXTENDED_PREDICTORS
        ),

    "numeric_predictors":
        list(
            NUMERIC_PREDICTORS
        ),

    "binary_predictors":
        list(
            BINARY_PREDICTORS
        ),

    "categorical_predictors":
        list(
            CATEGORICAL_PREDICTORS_ML
        )
}


with open(
    NOTEBOOK3_HANDOFF_DIR
    / "predictor_architecture.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        predictor_architecture,
        file,
        indent=2
    )


# =============================================================================
# 8. SELECTED MODEL CONFIGURATIONS
#
# Keep this metadata outside the serialized models as an audit record.
# =============================================================================

selected_model_metadata = {
    "full_cohort": {
        "Logistic regression":
            BEST_LOGISTIC_PARAMS,

        "Random Forest":
            BEST_RF_PARAMS,

        "XGBoost":
            BEST_XGB_PARAMS
    },

    "anc_entry": {
        "Logistic regression": {
            "Baseline":
                BEST_ENTRY_LOGISTIC_BASELINE_PARAMS,

            "Extended":
                BEST_ENTRY_LOGISTIC_EXTENDED_PARAMS
        },

        "Random Forest": {
            "Baseline":
                BEST_ENTRY_RF_BASELINE_PARAMS,

            "Extended":
                BEST_ENTRY_RF_EXTENDED_PARAMS
        },

        "XGBoost": {
            "Baseline":
                BEST_ENTRY_XGB_BASELINE_PARAMS,

            "Extended":
                BEST_ENTRY_XGB_EXTENDED_PARAMS
        }
    }
}


with open(
    NOTEBOOK3_HANDOFF_DIR
    / "selected_model_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        selected_model_metadata,
        file,
        indent=2,
        default=str
    )


# =============================================================================
# 9. EXPLAINABILITY GOVERNANCE METADATA
#
# This tells Notebook 3 what each horizon means and prevents accidental mixing.
# =============================================================================

handoff_metadata = {
    "handoff_name":
        "ANC8 Notebook 2 to Notebook 3 explainability handoff",

    "primary_outcome":
        "anc8plus",

    "outcome_definition":
        "Attainment of 8 or more ANC contacts",

    "full_cohort": {
        "development_n":
            int(
                len(
                    train_df
                )
            ),

        "heldout_n":
            int(
                len(
                    test_df
                )
            ),

        "original_predictors":
            int(
                len(
                    PRIMARY_PREDICTORS
                )
            ),

        "transformed_features":
            int(
                len(
                    feature_lineage
                )
            ),

        "prediction_horizon":
            "Background/baseline prediction before ANC-process information"
    },

    "anc_entry": {
        "development_n":
            int(
                len(
                    anc_entry_train
                )
            ),

        "heldout_n":
            int(
                len(
                    anc_entry_test
                )
            ),

        "baseline_original_predictors":
            int(
                len(
                    ANC_ENTRY_BASELINE_PREDICTORS
                )
            ),

        "extended_original_predictors":
            int(
                len(
                    ANC_ENTRY_EXTENDED_PREDICTORS
                )
            ),

        "extended_transformed_features":
            int(
                len(
                    entry_feature_lineage
                )
            ),

        "additional_predictor":
            "early_anc",

        "early_anc_definition":
            "1 = first ANC at <=12 weeks or <=3 months; 0 = later initiation",

        "prediction_horizon":
            "Prediction after ANC initiation with first-contact timing available"
    },

    "survey_design": {
        "weight":
            "v005",

        "PSU":
            "v021",

        "stratum":
            "v022"
    },

    "explainability_rule":
        (
            "Interpret fitted prediction models only. "
            "Do not infer causal effects from explanation measures."
        )
}


with open(
    NOTEBOOK3_HANDOFF_DIR
    / "handoff_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        handoff_metadata,
        file,
        indent=2
    )


# =============================================================================
# 10. SOFTWARE ENVIRONMENT
#
# Serialized sklearn/xgboost models should ideally be loaded under compatible
# versions. Therefore versions are frozen explicitly.
# =============================================================================

software_environment = {
    "python":
        sys.version,

    "platform":
        platform.platform(),

    "pandas":
        pd.__version__,

    "numpy":
        np.__version__,

    "scikit_learn":
        sklearn.__version__,

    "xgboost":
        xgboost.__version__,

    "statsmodels":
        statsmodels.__version__,

    "joblib":
        joblib.__version__
}


with open(
    NOTEBOOK3_HANDOFF_DIR
    / "software_environment.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        software_environment,
        file,
        indent=2
    )


# =============================================================================
# 11. HUMAN-READABLE README
# =============================================================================

handoff_readme = """
ANC8 NOTEBOOK 3 EXPLAINABILITY HANDOFF
======================================

Purpose
-------
This directory is the frozen input authority for Notebook 3.

Notebook 3 should load these files rather than rerun model development,
hyperparameter tuning, train/test splitting, or outcome/predictor construction.

Prediction horizons
-------------------
1. Full cohort:
   Original 21-predictor baseline prediction horizon.

2. ANC entry:
   Same 21 predictors among women who initiated ANC and had known timing
   of first ANC, with an extended 22-predictor specification adding early_anc.

Core files
----------
full_cohort_final_models.joblib
    Three fitted primary full-cohort pipelines.

anc_entry_final_models.joblib
    Six fitted ANC-entry pipelines:
    baseline and extended Logistic regression, Random Forest, and XGBoost.

full_cohort_development.pkl
full_cohort_heldout.pkl
anc_entry_development.pkl
anc_entry_heldout.pkl
    Frozen analytic populations for explanation/reference distributions.

full_cohort_locked_predictions.csv
anc_entry_locked_predictions.csv
    Held-out predictions generated before final outcome evaluation.

full_cohort_feature_lineage.csv
anc_entry_feature_lineage.csv
    Authorities for mapping transformed model features back to original
    predictor constructs.

predictor_architecture.json
    Frozen predictor lists.

selected_model_metadata.json
    Final selected hyperparameters.

handoff_metadata.json
    Prediction-horizon and governance metadata.

software_environment.json
    Software versions under which models were fitted.

SHA256_MANIFEST.csv
    Integrity hashes for all files in this directory.

Explainability governance
-------------------------
Explanation analyses describe how fitted models generate predictions.
They do not estimate causal effects.

Original predictor constructs should be the main interpretation level.
One-hot encoded components should not be treated as independent scientific
predictors without explicit justification.
""".strip()


with open(
    NOTEBOOK3_HANDOFF_DIR
    / "README.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        handoff_readme
    )


# =============================================================================
# 12. SHA-256 MANIFEST
#
# Generate after all files above have been written.
# =============================================================================

def sha256_file(
    file_path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


manifest_rows = []


for file_path in sorted(
    NOTEBOOK3_HANDOFF_DIR.iterdir()
):

    if (
        file_path.is_file()
        and file_path.name
        != "SHA256_MANIFEST.csv"
    ):

        manifest_rows.append({
            "file":
                file_path.name,

            "bytes":
                file_path.stat().st_size,

            "sha256":
                sha256_file(
                    file_path
                )
        })


notebook3_handoff_manifest = pd.DataFrame(
    manifest_rows
)


notebook3_handoff_manifest.to_csv(
    NOTEBOOK3_HANDOFF_DIR
    / "SHA256_MANIFEST.csv",
    index=False
)


# =============================================================================
# 13. FINAL AUDIT
# =============================================================================

expected_handoff_files = {
    "full_cohort_development.pkl",
    "full_cohort_heldout.pkl",
    "anc_entry_development.pkl",
    "anc_entry_heldout.pkl",
    "full_cohort_final_models.joblib",
    "anc_entry_final_models.joblib",
    "full_cohort_locked_predictions.csv",
    "anc_entry_locked_predictions.csv",
    "full_cohort_feature_lineage.csv",
    "anc_entry_feature_lineage.csv",
    "predictor_architecture.json",
    "selected_model_metadata.json",
    "handoff_metadata.json",
    "software_environment.json",
    "README.txt",
    "SHA256_MANIFEST.csv"
}


actual_handoff_files = {
    file_path.name
    for file_path in NOTEBOOK3_HANDOFF_DIR.iterdir()
    if file_path.is_file()
}


assert expected_handoff_files.issubset(
    actual_handoff_files
)


print("=" * 126)
print("NOTEBOOK 3 EXPLAINABILITY HANDOFF EXPORTED")
print("=" * 126)

print(
    "Directory:",
    NOTEBOOK3_HANDOFF_DIR.resolve()
)

print(
    "Files exported:",
    len(
        actual_handoff_files
    )
)

print(
    "\nCore model files:"
)

print(
    "  full_cohort_final_models.joblib"
)

print(
    "  anc_entry_final_models.joblib"
)

print(
    "\nFeature lineage:"
)

print(
    "  Full cohort:",
    len(
        feature_lineage
    ),
    "transformed features ->",
    feature_lineage[
        "Original_predictor"
    ].nunique(),
    "constructs"
)

print(
    "  ANC entry:",
    len(
        entry_feature_lineage
    ),
    "transformed features ->",
    entry_feature_lineage[
        "original_predictor"
    ].nunique(),
    "constructs"
)

print(
    "\nIntegrity manifest rows:",
    len(
        notebook3_handoff_manifest
    )
)

print(
    "\nSTATUS: NOTEBOOK 3 HANDOFF "
    "FROZEN AND EXPORTED"
)

In [ ]:
# =============================================================================
# 2.45.1 PUBLICATION-RESULTS OBJECT AUDIT
#
# PURPOSE
# -------
# Verify the final numerical objects that will become manuscript tables.
#
# No calculations are performed here.
# Nothing is modified.
# =============================================================================

publication_object_names = [
    # ---------------------------------------------------------
    # Full-cohort primary analysis
    # ---------------------------------------------------------
    "final_baseline_performance",
    "baseline_bootstrap_ci",
    "paired_model_comparisons",
    "development_thresholds",

    # ---------------------------------------------------------
    # ANC-entry analysis
    # ---------------------------------------------------------
    "entry_test_performance_long",
    "entry_test_incremental_performance",
    "entry_calibration_summary",
    "development_entry_thresholds",
    "entry_test_threshold_performance",
    "entry_incremental_summary",
    "entry_bootstrap_performance_ci",
    "entry_bootstrap_difference_ci",

    # ---------------------------------------------------------
    # Bootstrap raw distributions for reproducibility
    # ---------------------------------------------------------
    "baseline_bootstrap_results",
    "paired_bootstrap_distributions",
    "entry_bootstrap_performance_distribution",
    "entry_bootstrap_difference_distribution"
]


print("=" * 120)
print("PUBLICATION-RESULTS OBJECT AUDIT")
print("=" * 120)


for object_name in publication_object_names:

    print("\n" + "-" * 120)
    print(object_name)
    print("-" * 120)

    if object_name not in globals():

        print("STATUS: NOT FOUND")
        continue


    obj = globals()[object_name]

    print(
        "Type:",
        type(obj).__name__
    )


    if isinstance(
        obj,
        pd.DataFrame
    ):

        print(
            "Shape:",
            obj.shape
        )

        print(
            "Columns:"
        )

        print(
            obj.columns.tolist()
        )

        print(
            "\nFirst rows:"
        )

        print(
            obj.head(
                5
            ).to_string(
                index=False
            )
        )

    elif isinstance(
        obj,
        pd.Series
    ):

        print(
            "Length:",
            len(obj)
        )

        print(
            "Name:",
            obj.name
        )

        print(
            obj.head()
        )

    elif isinstance(
        obj,
        dict
    ):

        print(
            "Dictionary keys:"
        )

        print(
            list(
                obj.keys()
            )[:30]
        )

    else:

        print(
            repr(
                obj
            )[:1000]
        )


print("\n" + "=" * 120)
print("STATUS: PUBLICATION OBJECT AUDIT COMPLETE")
print("=" * 120)

### 2.45.2 Export publication source tables

Final held-out results are reorganized into publication-source tables.

These CSV files are the numerical authority for subsequent Word tables and
figures. No performance metric is recalculated from raw model predictions in
this section.

The publication package contains:

**Main-table sources**

1. Full-cohort held-out model performance and uncertainty.
2. ANC-entry baseline versus extended probability performance and paired
   incremental uncertainty.
3. ANC-entry calibration and threshold-based held-out performance.

**Supplementary-table sources**

1. Full-cohort development-derived operating thresholds.
2. ANC-entry development-derived operating thresholds.
3. Full-cohort pairwise model comparisons.
4. ANC-entry consolidated incremental point estimates.
5. Detailed bootstrap confidence intervals.
6. Held-out survey-design audit.

The complete bootstrap replicate distributions are also exported separately for
reproducibility, but are not intended for direct manuscript presentation.

In [ ]:
# =============================================================================
# 2.45.2 EXPORT PUBLICATION SOURCE TABLES
#
# IMPORTANT
# ---------
# No models are fitted.
# No predictions are regenerated.
# No metrics are recalculated from model outputs.
#
# This section only reorganizes the frozen numerical results into authoritative
# manuscript-source tables.
# =============================================================================

from pathlib import Path
import json


# =============================================================================
# 1. DIRECTORY STRUCTURE
# =============================================================================

PUBLICATION_DIR = (
    Path("outputs")
    / "publication_results"
)

MAIN_TABLE_DIR = (
    PUBLICATION_DIR
    / "main_tables"
)

SUPPLEMENT_DIR = (
    PUBLICATION_DIR
    / "supplementary_tables"
)

REPRODUCIBILITY_DIR = (
    PUBLICATION_DIR
    / "reproducibility"
)


for directory in [
    PUBLICATION_DIR,
    MAIN_TABLE_DIR,
    SUPPLEMENT_DIR,
    REPRODUCIBILITY_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# 2. MAIN TABLE 1
#    FULL-COHORT HELD-OUT PERFORMANCE
#
# Join the locked point estimates with bootstrap confidence intervals for:
#   - AP
#   - AUROC
#   - Brier
#
# Calibration and threshold-based measures remain point estimates.
# =============================================================================

full_cohort_ci_wide = (
    baseline_bootstrap_ci
    .pivot(
        index="Model",
        columns="Metric",
        values=[
            "CI_low",
            "CI_high"
        ]
    )
)


# Flatten MultiIndex columns such as:
# ('CI_low', 'AP') -> AP_CI_low
full_cohort_ci_wide.columns = [
    f"{metric}_{bound}"
    for bound, metric
    in full_cohort_ci_wide.columns
]


full_cohort_ci_wide = (
    full_cohort_ci_wide
    .reset_index()
)


main_table_1_full_cohort = (
    final_baseline_performance
    .merge(
        full_cohort_ci_wide,
        on="Model",
        how="left",
        validate="one_to_one"
    )
)


# Reorder for manuscript readability.
main_table_1_full_cohort = (
    main_table_1_full_cohort[
        [
            "Model",

            "Weighted_AP",
            "AP_CI_low",
            "AP_CI_high",

            "Weighted_AUROC",
            "AUROC_CI_low",
            "AUROC_CI_high",

            "Weighted_Brier",
            "Brier_CI_low",
            "Brier_CI_high",

            "CITL",
            "Calibration_slope",

            "Threshold",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1",
            "Predicted_positive_rate"
        ]
    ]
)


# =============================================================================
# 3. MAIN TABLE 2
#    ANC-ENTRY BASELINE VS EXTENDED PROBABILITY PERFORMANCE
#
# First add 95% CIs to each of the six held-out models.
# =============================================================================

entry_ci_wide = (
    entry_bootstrap_performance_ci
    .pivot(
        index=[
            "Algorithm",
            "Specification"
        ],
        columns="Metric",
        values=[
            "CI_lower",
            "CI_upper"
        ]
    )
)


entry_ci_wide.columns = [
    f"{metric}_{bound}"
    for bound, metric
    in entry_ci_wide.columns
]


entry_ci_wide = (
    entry_ci_wide
    .reset_index()
)


entry_probability_with_ci = (
    entry_test_performance_long
    .merge(
        entry_ci_wide,
        on=[
            "Algorithm",
            "Specification"
        ],
        how="left",
        validate="one_to_one"
    )
)


# -----------------------------------------------------------------------------
# Paired incremental differences and their CIs
# -----------------------------------------------------------------------------

entry_difference_ci_wide = (
    entry_bootstrap_difference_ci
    .pivot(
        index="Algorithm",
        columns="Metric",
        values=[
            "CI_lower",
            "CI_upper"
        ]
    )
)


entry_difference_ci_wide.columns = [
    f"{metric}_{bound}"
    for bound, metric
    in entry_difference_ci_wide.columns
]


entry_difference_ci_wide = (
    entry_difference_ci_wide
    .reset_index()
)


entry_difference_ci_flag = (
    entry_bootstrap_difference_ci
    .pivot(
        index="Algorithm",
        columns="Metric",
        values="CI_excludes_zero"
    )
    .reset_index()
)


entry_difference_ci_flag.columns = [
    (
        column
        if column == "Algorithm"
        else f"{column}_CI_excludes_zero"
    )
    for column in entry_difference_ci_flag.columns
]


entry_incremental_with_ci = (
    entry_test_incremental_performance
    .merge(
        entry_difference_ci_wide,
        on="Algorithm",
        how="left",
        validate="one_to_one"
    )
    .merge(
        entry_difference_ci_flag,
        on="Algorithm",
        how="left",
        validate="one_to_one"
    )
)


# -----------------------------------------------------------------------------
# Create one row per algorithm with baseline, extended, and paired difference.
# -----------------------------------------------------------------------------

entry_baseline_wide = (
    entry_probability_with_ci[
        entry_probability_with_ci[
            "Specification"
        ].eq(
            "Baseline"
        )
    ]
    .drop(
        columns="Specification"
    )
    .rename(
        columns={
            column:
                (
                    column
                    if column == "Algorithm"
                    else f"Baseline_{column}"
                )
            for column
            in entry_probability_with_ci.columns
            if column != "Specification"
        }
    )
)


entry_extended_wide = (
    entry_probability_with_ci[
        entry_probability_with_ci[
            "Specification"
        ].eq(
            "Extended"
        )
    ]
    .drop(
        columns="Specification"
    )
    .rename(
        columns={
            column:
                (
                    column
                    if column == "Algorithm"
                    else f"Extended_{column}"
                )
            for column
            in entry_probability_with_ci.columns
            if column != "Specification"
        }
    )
)


main_table_2_anc_entry_probability = (
    entry_baseline_wide
    .merge(
        entry_extended_wide,
        on="Algorithm",
        validate="one_to_one"
    )
    .merge(
        entry_incremental_with_ci,
        on="Algorithm",
        validate="one_to_one"
    )
)


# =============================================================================
# 4. MAIN TABLE 3
#    ANC-ENTRY CALIBRATION + THRESHOLD PERFORMANCE
# =============================================================================

main_table_3_anc_entry_calibration_threshold = (
    entry_calibration_summary
    .merge(
        entry_test_threshold_performance,
        on=[
            "Algorithm",
            "Specification"
        ],
        how="inner",
        validate="one_to_one"
    )
)


main_table_3_anc_entry_calibration_threshold = (
    main_table_3_anc_entry_calibration_threshold[
        [
            "Algorithm",
            "Specification",

            "Observed_prevalence",
            "Mean_predicted_probability",
            "Mean_prediction_minus_observed",

            "CITL",
            "Joint_calibration_intercept",
            "Calibration_slope",

            "Threshold",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "F1",
            "Predicted_positive_rate"
        ]
    ]
)


# =============================================================================
# 5. SUPPLEMENTARY TABLE S1
#    FULL-COHORT DEVELOPMENT THRESHOLDS
# =============================================================================

supp_table_s1_full_cohort_thresholds = (
    development_thresholds
    .copy()
)


# =============================================================================
# 6. SUPPLEMENTARY TABLE S2
#    ANC-ENTRY DEVELOPMENT THRESHOLDS
# =============================================================================

supp_table_s2_entry_thresholds = (
    development_entry_thresholds
    .copy()
)


# =============================================================================
# 7. SUPPLEMENTARY TABLE S3
#    FULL-COHORT PAIRED MODEL COMPARISONS
# =============================================================================

supp_table_s3_full_cohort_model_comparisons = (
    paired_model_comparisons
    .copy()
)


# =============================================================================
# 8. SUPPLEMENTARY TABLE S4
#    ANC-ENTRY CONSOLIDATED INCREMENTAL POINT ESTIMATES
# =============================================================================

supp_table_s4_entry_incremental = (
    entry_incremental_summary
    .copy()
)


# =============================================================================
# 9. SUPPLEMENTARY TABLE S5
#    ANC-ENTRY MODEL-SPECIFIC BOOTSTRAP CIs
# =============================================================================

supp_table_s5_entry_model_ci = (
    entry_bootstrap_performance_ci
    .copy()
)


# =============================================================================
# 10. SUPPLEMENTARY TABLE S6
#     ANC-ENTRY PAIRED DIFFERENCE BOOTSTRAP CIs
# =============================================================================

supp_table_s6_entry_difference_ci = (
    entry_bootstrap_difference_ci
    .copy()
)


# =============================================================================
# 11. SUPPLEMENTARY TABLE S7
#     ANC-ENTRY HELD-OUT SURVEY DESIGN AUDIT
# =============================================================================

supp_table_s7_entry_design_audit = (
    entry_test_stratum_design
    .copy()
)


# =============================================================================
# 12. EXPORT MAIN-TABLE SOURCE CSVs
# =============================================================================

main_table_files = {
    "Table_1_full_cohort_heldout_performance.csv":
        main_table_1_full_cohort,

    "Table_2_anc_entry_probability_performance.csv":
        main_table_2_anc_entry_probability,

    "Table_3_anc_entry_calibration_threshold_performance.csv":
        main_table_3_anc_entry_calibration_threshold
}


for filename, dataframe in (
    main_table_files.items()
):

    dataframe.to_csv(
        MAIN_TABLE_DIR
        / filename,
        index=False
    )


# =============================================================================
# 13. EXPORT SUPPLEMENTARY SOURCE CSVs
# =============================================================================

supplementary_table_files = {
    "Table_S1_full_cohort_development_thresholds.csv":
        supp_table_s1_full_cohort_thresholds,

    "Table_S2_anc_entry_development_thresholds.csv":
        supp_table_s2_entry_thresholds,

    "Table_S3_full_cohort_model_comparisons.csv":
        supp_table_s3_full_cohort_model_comparisons,

    "Table_S4_anc_entry_incremental_point_estimates.csv":
        supp_table_s4_entry_incremental,

    "Table_S5_anc_entry_model_bootstrap_CI.csv":
        supp_table_s5_entry_model_ci,

    "Table_S6_anc_entry_paired_difference_bootstrap_CI.csv":
        supp_table_s6_entry_difference_ci,

    "Table_S7_anc_entry_heldout_design_audit.csv":
        supp_table_s7_entry_design_audit
}


for filename, dataframe in (
    supplementary_table_files.items()
):

    dataframe.to_csv(
        SUPPLEMENT_DIR
        / filename,
        index=False
    )


# =============================================================================
# 14. EXPORT RAW REPRODUCIBILITY DISTRIBUTIONS
#
# These files are not manuscript tables.
# They preserve the complete 2,000-replicate results.
# =============================================================================

reproducibility_files = {
    "full_cohort_bootstrap_performance_distribution.csv":
        baseline_bootstrap_results,

    "full_cohort_paired_model_bootstrap_distribution.csv":
        paired_bootstrap_distributions,

    "anc_entry_bootstrap_performance_distribution.csv":
        entry_bootstrap_performance_distribution,

    "anc_entry_paired_difference_bootstrap_distribution.csv":
        entry_bootstrap_difference_distribution
}


for filename, dataframe in (
    reproducibility_files.items()
):

    dataframe.to_csv(
        REPRODUCIBILITY_DIR
        / filename,
        index=False
    )


# =============================================================================
# 15. PUBLICATION-SOURCE README
# =============================================================================

publication_readme = """
ANC8 PUBLICATION RESULTS PACKAGE
================================

Purpose
-------
This directory contains the frozen numerical sources used to construct
manuscript tables and evaluation figures.

The CSV files are the numerical authority.
Word tables and figures should be generated from these frozen sources.

Main tables
-----------
Table 1
    Full-cohort held-out predictive performance, bootstrap confidence
    intervals, calibration, and secondary threshold measures.

Table 2
    ANC-entry baseline versus extended probability performance, including
    paired incremental differences and stratified-PSU bootstrap uncertainty.

Table 3
    ANC-entry calibration and held-out threshold-based performance.

Supplementary tables
--------------------
S1  Full-cohort development-derived thresholds.
S2  ANC-entry development-derived thresholds.
S3  Full-cohort pairwise model comparisons.
S4  ANC-entry consolidated incremental point estimates.
S5  ANC-entry model-specific bootstrap confidence intervals.
S6  ANC-entry paired-difference bootstrap confidence intervals.
S7  ANC-entry held-out survey-design audit.

Reproducibility
---------------
The reproducibility directory contains all retained bootstrap replicate
distributions underlying the reported percentile confidence intervals.

Interpretation
--------------
Primary probability-based metrics are Average Precision, AUROC, and Brier
score.

Calibration and threshold-based metrics are complementary evaluations.

For ANC-entry paired differences:
- positive Delta_AP favors the extended model;
- positive Delta_AUROC favors the extended model;
- positive Brier_improvement means lower Brier score for the extended model.

No causal interpretation is intended.
""".strip()


with open(
    PUBLICATION_DIR
    / "README_publication_results.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        publication_readme
    )


# =============================================================================
# 16. NUMERICAL INTEGRITY CHECKS
# =============================================================================

assert main_table_1_full_cohort.shape[0] == 3

assert main_table_2_anc_entry_probability.shape[0] == 3

assert main_table_3_anc_entry_calibration_threshold.shape[0] == 6

assert supp_table_s1_full_cohort_thresholds.shape[0] == 3

assert supp_table_s2_entry_thresholds.shape[0] == 6

assert supp_table_s3_full_cohort_model_comparisons.shape[0] == 6

assert supp_table_s4_entry_incremental.shape[0] == 3

assert supp_table_s5_entry_model_ci.shape[0] == 18

assert supp_table_s6_entry_difference_ci.shape[0] == 9

assert supp_table_s7_entry_design_audit.shape[0] == 27


# Every ANC-entry paired bootstrap CI currently includes zero.
assert not (
    supp_table_s6_entry_difference_ci[
        "CI_excludes_zero"
    ]
).any()


# =============================================================================
# 17. EXPORT INVENTORY
# =============================================================================

publication_inventory = []


for category, directory in [
    (
        "Main table source",
        MAIN_TABLE_DIR
    ),
    (
        "Supplementary table source",
        SUPPLEMENT_DIR
    ),
    (
        "Reproducibility",
        REPRODUCIBILITY_DIR
    )
]:

    for file_path in sorted(
        directory.glob(
            "*.csv"
        )
    ):

        publication_inventory.append({
            "Category":
                category,

            "File":
                file_path.name,

            "Bytes":
                file_path.stat().st_size
        })


publication_inventory = pd.DataFrame(
    publication_inventory
)


publication_inventory.to_csv(
    PUBLICATION_DIR
    / "publication_source_inventory.csv",
    index=False
)


# =============================================================================
# 18. DISPLAY
# =============================================================================

print("=" * 126)
print("PUBLICATION SOURCE TABLES EXPORTED")
print("=" * 126)

print(
    "Publication directory:",
    PUBLICATION_DIR.resolve()
)

print(
    "\nMain-table source files:",
    len(
        main_table_files
    )
)

print(
    "Supplementary-table source files:",
    len(
        supplementary_table_files
    )
)

print(
    "Raw reproducibility files:",
    len(
        reproducibility_files
    )
)


print("\nMain tables")
print("-----------")

for filename in (
    main_table_files
):

    print(
        " ",
        filename
    )


print("\nSupplementary tables")
print("--------------------")

for filename in (
    supplementary_table_files
):

    print(
        " ",
        filename
    )


print("\nReproducibility files")
print("---------------------")

for filename in (
    reproducibility_files
):

    print(
        " ",
        filename
    )


print(
    "\nSTATUS: PUBLICATION NUMERICAL "
    "SOURCES FROZEN AND EXPORTED"
)

### 2.45.3 Publication-quality Word tables

Two manuscript-ready Word documents are generated from the frozen publication
source tables.

The Word documents are presentation layers only. The CSV files exported in the
previous section remain the numerical authority.

Formatting principles include:

- manuscript-style table captions;
- consistent typography and decimal precision;
- landscape orientation for wide tables;
- controlled cell margins and paragraph spacing;
- repeated header rows across pages;
- subtle header shading;
- clear horizontal and vertical borders;
- merged panel headings where appropriate;
- prevention of unnecessary row splitting;
- explanatory notes defining abbreviations, direction of improvement, and
  bootstrap confidence intervals.

No numerical results are recalculated in this section.

In [ ]:
# =============================================================================
# 2.45.3 GENERATE PUBLICATION-QUALITY WORD TABLES
#
# Outputs
# -------
# outputs/publication_results/word_tables/
#     ANC8_Main_Tables.docx
#     ANC8_Supplementary_Tables.docx
#
# Numerical authority remains the frozen CSV files.
# =============================================================================

from pathlib import Path
import json
import pandas as pd

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import (
    WD_TABLE_ALIGNMENT,
    WD_CELL_VERTICAL_ALIGNMENT
)
from docx.enum.section import WD_SECTION
from docx.enum.style import WD_STYLE_TYPE
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# =============================================================================
# 1. PATHS
# =============================================================================

PUBLICATION_DIR = Path("outputs") / "publication_results"

MAIN_TABLE_DIR = PUBLICATION_DIR / "main_tables"
SUPPLEMENT_DIR = PUBLICATION_DIR / "supplementary_tables"
WORD_DIR = PUBLICATION_DIR / "word_tables"

WORD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MAIN_DOCX = WORD_DIR / "ANC8_Main_Tables.docx"
SUPP_DOCX = WORD_DIR / "ANC8_Supplementary_Tables.docx"


# =============================================================================
# 2. LOAD FROZEN SOURCE TABLES
# =============================================================================

t1 = pd.read_csv(
    MAIN_TABLE_DIR
    / "Table_1_full_cohort_heldout_performance.csv"
)

t2 = pd.read_csv(
    MAIN_TABLE_DIR
    / "Table_2_anc_entry_probability_performance.csv"
)

t3 = pd.read_csv(
    MAIN_TABLE_DIR
    / "Table_3_anc_entry_calibration_threshold_performance.csv"
)

s1 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S1_full_cohort_development_thresholds.csv"
)

s2 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S2_anc_entry_development_thresholds.csv"
)

s3 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S3_full_cohort_model_comparisons.csv"
)

s4 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S4_anc_entry_incremental_point_estimates.csv"
)

s5 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S5_anc_entry_model_bootstrap_CI.csv"
)

s6 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S6_anc_entry_paired_difference_bootstrap_CI.csv"
)

s7 = pd.read_csv(
    SUPPLEMENT_DIR
    / "Table_S7_anc_entry_heldout_design_audit.csv"
)


# =============================================================================
# 3. GENERAL DOCUMENT HELPERS
# =============================================================================

def set_cell_shading(cell, fill):
    """
    Apply background shading to a Word table cell.
    """

    tc_pr = cell._tc.get_or_add_tcPr()

    shd = tc_pr.find(
        qn("w:shd")
    )

    if shd is None:
        shd = OxmlElement("w:shd")
        tc_pr.append(shd)

    shd.set(
        qn("w:fill"),
        fill
    )


def set_cell_margins(
    cell,
    top=70,
    start=80,
    bottom=70,
    end=80
):
    """
    Set cell padding in twentieths of a point.
    """

    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()

    tc_mar = tc_pr.first_child_found_in(
        "w:tcMar"
    )

    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)

    for margin_name, margin_value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end
    }.items():

        node = tc_mar.find(
            qn(f"w:{margin_name}")
        )

        if node is None:
            node = OxmlElement(
                f"w:{margin_name}"
            )
            tc_mar.append(node)

        node.set(
            qn("w:w"),
            str(margin_value)
        )

        node.set(
            qn("w:type"),
            "dxa"
        )


def set_cell_border(
    cell,
    top=None,
    bottom=None,
    left=None,
    right=None
):
    """
    Apply selected borders to a Word cell.

    Border format:
        {
            "val": "single",
            "sz": "6",
            "color": "B7B7B7"
        }
    """

    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()

    tc_borders = tc_pr.first_child_found_in(
        "w:tcBorders"
    )

    if tc_borders is None:
        tc_borders = OxmlElement(
            "w:tcBorders"
        )
        tc_pr.append(tc_borders)

    for edge_name, edge_definition in {
        "top": top,
        "bottom": bottom,
        "left": left,
        "right": right
    }.items():

        if edge_definition is None:
            continue

        tag = f"w:{edge_name}"

        edge = tc_borders.find(
            qn(tag)
        )

        if edge is None:
            edge = OxmlElement(tag)
            tc_borders.append(edge)

        for attribute, value in (
            edge_definition.items()
        ):

            edge.set(
                qn(f"w:{attribute}"),
                str(value)
            )


def set_repeat_table_header(row):
    """
    Repeat a table header row when the table spans pages.
    """

    tr_pr = row._tr.get_or_add_trPr()

    tbl_header = OxmlElement(
        "w:tblHeader"
    )

    tbl_header.set(
        qn("w:val"),
        "true"
    )

    tr_pr.append(
        tbl_header
    )


def prevent_row_split(row):
    """
    Keep a row together across pages where possible.
    """

    tr_pr = row._tr.get_or_add_trPr()

    cant_split = OxmlElement(
        "w:cantSplit"
    )

    tr_pr.append(
        cant_split
    )


def set_repeat_header_and_rows(table):
    """
    Repeat first row and prevent data rows from splitting.
    """

    if len(table.rows) > 0:
        set_repeat_table_header(
            table.rows[0]
        )

    for row in table.rows:
        prevent_row_split(row)


def format_cell_text(
    cell,
    bold=False,
    size=8.5,
    align=None,
    color="000000"
):
    """
    Standardize typography inside one cell.
    """

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_margins(
        cell
    )

    for paragraph in cell.paragraphs:

        paragraph.paragraph_format.space_before = Pt(0)
        paragraph.paragraph_format.space_after = Pt(0)
        paragraph.paragraph_format.line_spacing = 1.0

        if align is not None:
            paragraph.alignment = align

        for run in paragraph.runs:

            run.font.name = "Arial"
            run.font.size = Pt(size)
            run.font.bold = bold
            run.font.color.rgb = RGBColor.from_string(
                color
            )


def apply_professional_table_style(
    table,
    header_rows=1,
    stub_columns=1,
    font_size=8.5
):
    """
    Apply a clean journal-style table design:
      - subtle header shading
      - strong top/bottom borders
      - lighter internal grid
      - bold stub column
      - centered numeric cells
    """

    table.alignment = (
        WD_TABLE_ALIGNMENT.CENTER
    )

    table.autofit = True

    dark_border = {
        "val": "single",
        "sz": "10",
        "color": "5B6573"
    }

    light_border = {
        "val": "single",
        "sz": "4",
        "color": "D8DCE2"
    }


    n_rows = len(
        table.rows
    )

    n_cols = len(
        table.columns
    )


    for r_idx, row in enumerate(
        table.rows
    ):

        prevent_row_split(row)

        for c_idx, cell in enumerate(
            row.cells
        ):

            is_header = (
                r_idx < header_rows
            )

            is_stub = (
                c_idx < stub_columns
                and not is_header
            )


            if is_header:

                set_cell_shading(
                    cell,
                    "E9EDF2"
                )

                format_cell_text(
                    cell,
                    bold=True,
                    size=font_size,
                    align=WD_ALIGN_PARAGRAPH.CENTER,
                    color="1F2933"
                )

            elif is_stub:

                format_cell_text(
                    cell,
                    bold=True,
                    size=font_size,
                    align=WD_ALIGN_PARAGRAPH.LEFT
                )

            else:

                format_cell_text(
                    cell,
                    bold=False,
                    size=font_size,
                    align=WD_ALIGN_PARAGRAPH.CENTER
                )


            # Light internal borders.
            set_cell_border(
                cell,
                top=light_border,
                bottom=light_border,
                left=light_border,
                right=light_border
            )


            # Strong outer top border.
            if r_idx == 0:

                set_cell_border(
                    cell,
                    top=dark_border
                )


            # Strong outer bottom border.
            if r_idx == (
                n_rows - 1
            ):

                set_cell_border(
                    cell,
                    bottom=dark_border
                )


    # Repeat the true table header.
    if n_rows > 0:

        set_repeat_table_header(
            table.rows[0]
        )


def configure_document_styles(doc):
    """
    Configure global document typography.
    """

    normal = doc.styles["Normal"]

    normal.font.name = "Arial"
    normal.font.size = Pt(10)

    normal.paragraph_format.space_after = Pt(6)
    normal.paragraph_format.line_spacing = 1.05


    # Caption style.
    if "Table Caption" not in [
        style.name
        for style in doc.styles
    ]:

        caption_style = doc.styles.add_style(
            "Table Caption",
            WD_STYLE_TYPE.PARAGRAPH
        )

    else:

        caption_style = doc.styles[
            "Table Caption"
        ]


    caption_style.font.name = "Arial"
    caption_style.font.size = Pt(10)
    caption_style.font.bold = True

    caption_style.paragraph_format.space_before = Pt(6)
    caption_style.paragraph_format.space_after = Pt(5)


    # Table-note style.
    if "Table Note" not in [
        style.name
        for style in doc.styles
    ]:

        note_style = doc.styles.add_style(
            "Table Note",
            WD_STYLE_TYPE.PARAGRAPH
        )

    else:

        note_style = doc.styles[
            "Table Note"
        ]


    note_style.font.name = "Arial"
    note_style.font.size = Pt(8.5)

    note_style.paragraph_format.space_before = Pt(3)
    note_style.paragraph_format.space_after = Pt(8)


def add_page_number(paragraph):
    """
    Insert PAGE field in footer.
    """

    paragraph.alignment = (
        WD_ALIGN_PARAGRAPH.CENTER
    )

    run = paragraph.add_run()

    fld_char1 = OxmlElement(
        "w:fldChar"
    )

    fld_char1.set(
        qn("w:fldCharType"),
        "begin"
    )


    instr_text = OxmlElement(
        "w:instrText"
    )

    instr_text.set(
        qn("xml:space"),
        "preserve"
    )

    instr_text.text = " PAGE "


    fld_char2 = OxmlElement(
        "w:fldChar"
    )

    fld_char2.set(
        qn("w:fldCharType"),
        "end"
    )


    run._r.append(
        fld_char1
    )

    run._r.append(
        instr_text
    )

    run._r.append(
        fld_char2
    )


def setup_section(
    section,
    landscape=False
):
    """
    Configure journal-style margins and page orientation.
    """

    if landscape:

        section.orientation = 1

        section.page_width = Inches(
            11.69
        )

        section.page_height = Inches(
            8.27
        )

    else:

        section.orientation = 0

        section.page_width = Inches(
            8.27
        )

        section.page_height = Inches(
            11.69
        )


    section.top_margin = Inches(
        0.65
    )

    section.bottom_margin = Inches(
        0.65
    )

    section.left_margin = Inches(
        0.65
    )

    section.right_margin = Inches(
        0.65
    )


    add_page_number(
        section.footer.paragraphs[0]
    )


def add_table_caption(
    doc,
    table_number,
    title
):
    """
    Add publication-style caption above table.
    """

    paragraph = doc.add_paragraph(
        style="Table Caption"
    )

    paragraph.paragraph_format.keep_with_next = True

    run = paragraph.add_run(
        f"Table {table_number}. "
    )

    run.bold = True

    paragraph.add_run(
        title
    )


def add_table_note(
    doc,
    text
):
    """
    Add note immediately below a table.
    """

    paragraph = doc.add_paragraph(
        style="Table Note"
    )

    paragraph.paragraph_format.keep_together = True

    run = paragraph.add_run(
        "Note. "
    )

    run.bold = True

    paragraph.add_run(
        text
    )


def add_document_title(
    doc,
    title,
    subtitle=None
):
    """
    Clean title page heading.
    """

    p = doc.add_paragraph()

    p.alignment = (
        WD_ALIGN_PARAGRAPH.CENTER
    )

    p.paragraph_format.space_before = Pt(10)
    p.paragraph_format.space_after = Pt(4)

    run = p.add_run(
        title
    )

    run.font.name = "Arial"
    run.font.size = Pt(16)
    run.font.bold = True


    if subtitle:

        p2 = doc.add_paragraph()

        p2.alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )

        p2.paragraph_format.space_after = Pt(12)

        run2 = p2.add_run(
            subtitle
        )

        run2.font.name = "Arial"
        run2.font.size = Pt(10)
        run2.font.italic = True


def fmt4(value):
    """
    Four-decimal numeric formatter.
    """

    if pd.isna(value):
        return "—"

    return f"{float(value):.4f}"


def fmt3(value):
    """
    Three-decimal numeric formatter.
    """

    if pd.isna(value):
        return "—"

    return f"{float(value):.3f}"


def ci_text(
    estimate,
    low,
    high,
    digits=3
):
    """
    Format point estimate with 95% CI.
    """

    if digits == 4:

        return (
            f"{float(estimate):.4f} "
            f"({float(low):.4f}–{float(high):.4f})"
        )

    return (
        f"{float(estimate):.3f} "
        f"({float(low):.3f}–{float(high):.3f})"
    )


def signed_ci_text(
    estimate,
    low,
    high
):
    """
    Paired difference with explicit sign.
    """

    return (
        f"{float(estimate):+.3f} "
        f"({float(low):+.3f} to {float(high):+.3f})"
    )


def add_dataframe_table(
    doc,
    dataframe,
    display_columns,
    column_labels,
    header_rows=1,
    stub_columns=1,
    font_size=8.5
):
    """
    Generic controlled dataframe -> Word table helper.
    """

    table = doc.add_table(
        rows=1,
        cols=len(
            display_columns
        )
    )

    table.allow_autofit = True


    # Header.
    for c_idx, column_label in enumerate(
        column_labels
    ):

        table.rows[0].cells[
            c_idx
        ].text = str(
            column_label
        )


    # Body.
    for _, row in dataframe.iterrows():

        cells = table.add_row().cells

        for c_idx, column in enumerate(
            display_columns
        ):

            value = row[
                column
            ]

            cells[c_idx].text = (
                "—"
                if pd.isna(
                    value
                )
                else str(
                    value
                )
            )


    apply_professional_table_style(
        table,
        header_rows=header_rows,
        stub_columns=stub_columns,
        font_size=font_size
    )

    return table


# =============================================================================
# 4. MAIN DOCUMENT
# =============================================================================

main_doc = Document()

configure_document_styles(
    main_doc
)

setup_section(
    main_doc.sections[0],
    landscape=True
)

add_document_title(
    main_doc,
    "ANC8 Prediction Analysis",
    "Manuscript Main Tables"
)


# =============================================================================
# MAIN TABLE 1
# Full-cohort held-out performance
# =============================================================================

table1_display = pd.DataFrame({
    "Model":
        t1["Model"],

    "AP (95% CI)":
        [
            ci_text(
                r["Weighted_AP"],
                r["AP_CI_low"],
                r["AP_CI_high"]
            )
            for _, r in t1.iterrows()
        ],

    "AUROC (95% CI)":
        [
            ci_text(
                r["Weighted_AUROC"],
                r["AUROC_CI_low"],
                r["AUROC_CI_high"]
            )
            for _, r in t1.iterrows()
        ],

    "Brier score (95% CI)":
        [
            ci_text(
                r["Weighted_Brier"],
                r["Brier_CI_low"],
                r["Brier_CI_high"]
            )
            for _, r in t1.iterrows()
        ],

    "CITL":
        [
            fmt3(
                value
            )
            for value
            in t1["CITL"]
        ],

    "Calibration slope":
        [
            fmt3(
                value
            )
            for value
            in t1[
                "Calibration_slope"
            ]
        ]
})


add_table_caption(
    main_doc,
    "1",
    (
        "Held-out performance of the full-cohort ANC8 prediction models."
    )
)


add_dataframe_table(
    main_doc,
    table1_display,
    display_columns=list(
        table1_display.columns
    ),
    column_labels=list(
        table1_display.columns
    ),
    stub_columns=1,
    font_size=9
)


add_table_note(
    main_doc,
    (
        "AP = Average Precision; AUROC = area under the receiver operating "
        "characteristic curve; CITL = calibration-in-the-large. "
        "Confidence intervals are empirical 95% percentile intervals from "
        "2,000 stratified PSU bootstrap replicates. Higher AP and AUROC are "
        "better; lower Brier score is better. Ideal CITL = 0 and ideal "
        "calibration slope = 1."
    )
)


# =============================================================================
# MAIN TABLE 2
# ANC-entry probability performance
# =============================================================================

main_doc.add_page_break()


add_table_caption(
    main_doc,
    "2",
    (
        "Held-out probability-based performance at the ANC-entry prediction "
        "horizon and incremental value of early ANC initiation."
    )
)


# -----------------------------------------------------------------------------
# Panel A: baseline and extended model performance
# -----------------------------------------------------------------------------

panel_a_rows = []


for algorithm in [
    "Logistic regression",
    "Random Forest",
    "XGBoost"
]:

    r = t2.loc[
        t2[
            "Algorithm"
        ].eq(
            algorithm
        )
    ].iloc[0]


    panel_a_rows.append({
        "Algorithm":
            algorithm,

        "Specification":
            "Baseline",

        "AP (95% CI)":
            ci_text(
                r[
                    "Baseline_Weighted_AP"
                ],
                r[
                    "Baseline_AP_CI_lower"
                ],
                r[
                    "Baseline_AP_CI_upper"
                ]
            ),

        "AUROC (95% CI)":
            ci_text(
                r[
                    "Baseline_Weighted_AUROC"
                ],
                r[
                    "Baseline_AUROC_CI_lower"
                ],
                r[
                    "Baseline_AUROC_CI_upper"
                ]
            ),

        "Brier (95% CI)":
            ci_text(
                r[
                    "Baseline_Weighted_Brier"
                ],
                r[
                    "Baseline_Brier_CI_lower"
                ],
                r[
                    "Baseline_Brier_CI_upper"
                ]
            )
    })


    panel_a_rows.append({
        "Algorithm":
            algorithm,

        "Specification":
            "+ early ANC",

        "AP (95% CI)":
            ci_text(
                r[
                    "Extended_Weighted_AP"
                ],
                r[
                    "Extended_AP_CI_lower"
                ],
                r[
                    "Extended_AP_CI_upper"
                ]
            ),

        "AUROC (95% CI)":
            ci_text(
                r[
                    "Extended_Weighted_AUROC"
                ],
                r[
                    "Extended_AUROC_CI_lower"
                ],
                r[
                    "Extended_AUROC_CI_upper"
                ]
            ),

        "Brier (95% CI)":
            ci_text(
                r[
                    "Extended_Weighted_Brier"
                ],
                r[
                    "Extended_Brier_CI_lower"
                ],
                r[
                    "Extended_Brier_CI_upper"
                ]
            )
    })


panel_a = pd.DataFrame(
    panel_a_rows
)


panel_heading = main_doc.add_paragraph()

panel_heading.paragraph_format.space_before = Pt(2)
panel_heading.paragraph_format.space_after = Pt(3)
panel_heading.paragraph_format.keep_with_next = True

run = panel_heading.add_run(
    "Panel A. Model-specific held-out performance"
)

run.bold = True
run.font.name = "Arial"
run.font.size = Pt(9)


add_dataframe_table(
    main_doc,
    panel_a,
    display_columns=list(
        panel_a.columns
    ),
    column_labels=list(
        panel_a.columns
    ),
    stub_columns=2,
    font_size=8.8
)


# -----------------------------------------------------------------------------
# Panel B: paired incremental differences
# -----------------------------------------------------------------------------

panel_b_rows = []


for _, r in t2.iterrows():

    panel_b_rows.append({
        "Algorithm":
            r[
                "Algorithm"
            ],

        "Δ AP (95% CI)":
            signed_ci_text(
                r[
                    "Delta_AP"
                ],
                r[
                    "Delta_AP_CI_lower"
                ],
                r[
                    "Delta_AP_CI_upper"
                ]
            ),

        "Δ AUROC (95% CI)":
            signed_ci_text(
                r[
                    "Delta_AUROC"
                ],
                r[
                    "Delta_AUROC_CI_lower"
                ],
                r[
                    "Delta_AUROC_CI_upper"
                ]
            ),

        "Brier improvement (95% CI)":
            signed_ci_text(
                r[
                    "Brier_improvement"
                ],
                r[
                    "Brier_improvement_CI_lower"
                ],
                r[
                    "Brier_improvement_CI_upper"
                ]
            )
    })


panel_b = pd.DataFrame(
    panel_b_rows
)


panel_heading = main_doc.add_paragraph()

panel_heading.paragraph_format.space_before = Pt(9)
panel_heading.paragraph_format.space_after = Pt(3)
panel_heading.paragraph_format.keep_with_next = True

run = panel_heading.add_run(
    "Panel B. Paired incremental value of adding early ANC"
)

run.bold = True
run.font.name = "Arial"
run.font.size = Pt(9)


add_dataframe_table(
    main_doc,
    panel_b,
    display_columns=list(
        panel_b.columns
    ),
    column_labels=list(
        panel_b.columns
    ),
    stub_columns=1,
    font_size=8.8
)


add_table_note(
    main_doc,
    (
        "All models were evaluated on the same 984 held-out ANC-entry women. "
        "The weighted ANC8 prevalence was 20.9%. Positive ΔAP and ΔAUROC favor "
        "the extended model. Positive Brier improvement indicates a lower "
        "(better) Brier score for the extended model. Confidence intervals "
        "were obtained from 2,000 paired stratified-PSU bootstrap replicates. "
        "All paired difference confidence intervals included zero."
    )
)


# =============================================================================
# MAIN TABLE 3
# Calibration and threshold-based performance
# =============================================================================

main_doc.add_page_break()


table3_display = pd.DataFrame({
    "Algorithm":
        t3[
            "Algorithm"
        ],

    "Specification":
        t3[
            "Specification"
        ].replace({
            "Extended":
                "+ early ANC"
        }),

    "CITL":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "CITL"
            ]
        ],

    "Calibration slope":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "Calibration_slope"
            ]
        ],

    "Threshold":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "Threshold"
            ]
        ],

    "Sensitivity":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "Sensitivity"
            ]
        ],

    "Specificity":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "Specificity"
            ]
        ],

    "PPV":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "PPV"
            ]
        ],

    "NPV":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "NPV"
            ]
        ],

    "F1":
        [
            fmt3(
                value
            )
            for value
            in t3[
                "F1"
            ]
        ]
})


add_table_caption(
    main_doc,
    "3",
    (
        "Calibration and threshold-based held-out performance of ANC-entry "
        "prediction models."
    )
)


add_dataframe_table(
    main_doc,
    table3_display,
    display_columns=list(
        table3_display.columns
    ),
    column_labels=list(
        table3_display.columns
    ),
    stub_columns=2,
    font_size=8.0
)


add_table_note(
    main_doc,
    (
        "CITL = calibration-in-the-large; PPV = positive predictive value; "
        "NPV = negative predictive value. Ideal CITL = 0 and ideal calibration "
        "slope = 1. Operating thresholds were selected exclusively from "
        "development out-of-fold predictions using the survey-weighted Youden "
        "index and were applied unchanged to the held-out test population. "
        "Threshold-based measures are secondary to probability-based "
        "performance."
    )
)


# =============================================================================
# SAVE MAIN DOCUMENT
# =============================================================================

main_doc.save(
    MAIN_DOCX
)


# =============================================================================
# 5. SUPPLEMENTARY DOCUMENT
# =============================================================================

supp_doc = Document()

configure_document_styles(
    supp_doc
)

setup_section(
    supp_doc.sections[0],
    landscape=True
)

add_document_title(
    supp_doc,
    "ANC8 Prediction Analysis",
    "Supplementary Tables"
)


# =============================================================================
# S1 Full-cohort development thresholds
# =============================================================================

s1_display = s1.copy()

for column in [
    "Threshold",
    "Development_sensitivity",
    "Development_specificity",
    "Development_Youden_J"
]:

    s1_display[
        column
    ] = s1_display[
        column
    ].map(
        fmt3
    )


add_table_caption(
    supp_doc,
    "S1",
    (
        "Development-derived operating thresholds for full-cohort models."
    )
)


add_dataframe_table(
    supp_doc,
    s1_display,
    display_columns=list(
        s1_display.columns
    ),
    column_labels=[
        "Model",
        "Threshold",
        "Sensitivity",
        "Specificity",
        "Youden J"
    ],
    stub_columns=1,
    font_size=8.8
)


add_table_note(
    supp_doc,
    (
        "Thresholds were selected from development out-of-fold predictions "
        "using the survey-weighted Youden index."
    )
)


# =============================================================================
# S2 ANC-entry development thresholds
# =============================================================================

supp_doc.add_page_break()

s2_display = s2.copy()

s2_display[
    "Specification"
] = s2_display[
    "Specification"
].replace({
    "Extended":
        "+ early ANC"
})


for column in [
    "Threshold",
    "Development_sensitivity",
    "Development_specificity",
    "Development_Youden_J"
]:

    s2_display[
        column
    ] = s2_display[
        column
    ].map(
        fmt3
    )


add_table_caption(
    supp_doc,
    "S2",
    (
        "Development-derived operating thresholds for ANC-entry models."
    )
)


add_dataframe_table(
    supp_doc,
    s2_display,
    display_columns=list(
        s2_display.columns
    ),
    column_labels=[
        "Algorithm",
        "Specification",
        "Threshold",
        "Sensitivity",
        "Specificity",
        "Youden J"
    ],
    stub_columns=2,
    font_size=8.5
)


add_table_note(
    supp_doc,
    (
        "Each baseline and extended model received its own threshold derived "
        "solely from development out-of-fold predictions."
    )
)


# =============================================================================
# S3 Full-cohort pairwise model comparisons
# =============================================================================

supp_doc.add_page_break()

s3_display = pd.DataFrame({
    "Comparison":
        s3[
            "Comparison"
        ],

    "Metric":
        s3[
            "Metric"
        ],

    "XGBoost advantage":
        [
            f"{value:+.3f}"
            for value
            in s3[
                "XGBoost_advantage"
            ]
        ],

    "95% CI":
        [
            f"{low:+.3f} to {high:+.3f}"
            for low, high
            in zip(
                s3[
                    "CI_low"
                ],
                s3[
                    "CI_high"
                ]
            )
        ],

    "CI excludes 0":
        s3[
            "CI_excludes_zero"
        ].map({
            True:
                "Yes",

            False:
                "No"
        })
})


add_table_caption(
    supp_doc,
    "S3",
    (
        "Paired bootstrap comparison of XGBoost with Logistic regression and "
        "Random Forest in the full-cohort prediction horizon."
    )
)


add_dataframe_table(
    supp_doc,
    s3_display,
    display_columns=list(
        s3_display.columns
    ),
    column_labels=list(
        s3_display.columns
    ),
    stub_columns=2,
    font_size=8.6
)


add_table_note(
    supp_doc,
    (
        "For AP and AUROC, positive values favor XGBoost. For Brier score, "
        "positive values indicate lower Brier score for XGBoost. Confidence "
        "intervals are empirical 95% percentile intervals from paired "
        "stratified-PSU bootstrap replicates."
    )
)


# =============================================================================
# S4 ANC-entry consolidated incremental point estimates
# =============================================================================

supp_doc.add_page_break()

s4_display = s4.copy()

for column in [
    "Delta_AP",
    "Delta_AUROC",
    "Brier_improvement",
    "CITL_improvement",
    "Slope_improvement",
    "Delta_sensitivity",
    "Delta_specificity",
    "Delta_PPV",
    "Delta_NPV",
    "Delta_F1"
]:

    s4_display[
        column
    ] = [
        f"{value:+.3f}"
        for value
        in s4_display[
            column
        ]
    ]


add_table_caption(
    supp_doc,
    "S4",
    (
        "Consolidated held-out incremental point estimates after adding early "
        "ANC initiation."
    )
)


add_dataframe_table(
    supp_doc,
    s4_display,
    display_columns=list(
        s4_display.columns
    ),
    column_labels=[
        "Algorithm",
        "Δ AP",
        "Δ AUROC",
        "Brier improvement",
        "CITL improvement",
        "Slope improvement",
        "Δ Sens.",
        "Δ Spec.",
        "Δ PPV",
        "Δ NPV",
        "Δ F1"
    ],
    stub_columns=1,
    font_size=7.6
)


add_table_note(
    supp_doc,
    (
        "Positive AP, AUROC, and Brier-improvement values favor the extended "
        "model. Positive CITL improvement indicates movement closer to 0; "
        "positive slope improvement indicates movement closer to 1. "
        "Threshold-metric changes are Extended minus Baseline."
    )
)


# =============================================================================
# S5 ANC-entry model-specific bootstrap CIs
# =============================================================================

supp_doc.add_page_break()

s5_display = pd.DataFrame({
    "Algorithm":
        s5[
            "Algorithm"
        ],

    "Specification":
        s5[
            "Specification"
        ].replace({
            "Extended":
                "+ early ANC"
        }),

    "Metric":
        s5[
            "Metric"
        ],

    "Point estimate":
        [
            fmt3(
                value
            )
            for value
            in s5[
                "Point_estimate"
            ]
        ],

    "95% CI":
        [
            f"{low:.3f}–{high:.3f}"
            for low, high
            in zip(
                s5[
                    "CI_lower"
                ],
                s5[
                    "CI_upper"
                ]
            )
        ]
})


add_table_caption(
    supp_doc,
    "S5",
    (
        "Held-out ANC-entry model performance with stratified-PSU bootstrap "
        "confidence intervals."
    )
)


add_dataframe_table(
    supp_doc,
    s5_display,
    display_columns=list(
        s5_display.columns
    ),
    column_labels=list(
        s5_display.columns
    ),
    stub_columns=3,
    font_size=8.4
)


add_table_note(
    supp_doc,
    (
        "Confidence intervals are empirical 95% percentile intervals from "
        "2,000 stratified-PSU bootstrap replicates."
    )
)


# =============================================================================
# S6 ANC-entry paired difference CIs
# =============================================================================

supp_doc.add_page_break()

s6_display = pd.DataFrame({
    "Algorithm":
        s6[
            "Algorithm"
        ],

    "Metric":
        s6[
            "Metric"
        ],

    "Point estimate":
        [
            f"{value:+.3f}"
            for value
            in s6[
                "Point_estimate"
            ]
        ],

    "95% CI":
        [
            f"{low:+.3f} to {high:+.3f}"
            for low, high
            in zip(
                s6[
                    "CI_lower"
                ],
                s6[
                    "CI_upper"
                ]
            )
        ],

    "CI excludes 0":
        s6[
            "CI_excludes_zero"
        ].map({
            True:
                "Yes",

            False:
                "No"
        })
})


add_table_caption(
    supp_doc,
    "S6",
    (
        "Paired stratified-PSU bootstrap uncertainty in the incremental value "
        "of early ANC initiation."
    )
)


add_dataframe_table(
    supp_doc,
    s6_display,
    display_columns=list(
        s6_display.columns
    ),
    column_labels=list(
        s6_display.columns
    ),
    stub_columns=2,
    font_size=8.6
)


add_table_note(
    supp_doc,
    (
        "Positive ΔAP and ΔAUROC values favor the extended model. Positive "
        "Brier improvement indicates a lower Brier score for the extended "
        "model. None of the 95% paired-difference confidence intervals "
        "excluded zero."
    )
)


# =============================================================================
# S7 Held-out design audit
# =============================================================================

supp_doc.add_page_break()

s7_display = s7.copy()

if "Survey_weight_sum" in (
    s7_display.columns
):

    s7_display[
        "Survey_weight_sum"
    ] = s7_display[
        "Survey_weight_sum"
    ].map(
        lambda x:
            f"{x:.2f}"
    )


add_table_caption(
    supp_doc,
    "S7",
    (
        "Survey-design structure of the ANC-entry held-out test population "
        "used for stratified PSU bootstrap resampling."
    )
)


add_dataframe_table(
    supp_doc,
    s7_display,
    display_columns=list(
        s7_display.columns
    ),
    column_labels=[
        "Stratum",
        "Women",
        "PSUs",
        "Survey-weight sum"
    ],
    stub_columns=1,
    font_size=8.4
)


add_table_note(
    supp_doc,
    (
        "The held-out ANC-entry population contained 984 women from 157 PSUs "
        "across 27 strata. No stratum contained a single PSU; two strata "
        "contained exactly two PSUs."
    )
)


# =============================================================================
# SAVE SUPPLEMENTARY DOCUMENT
# =============================================================================

supp_doc.save(
    SUPP_DOCX
)


# =============================================================================
# 6. FILE AUDIT
# =============================================================================

assert MAIN_DOCX.exists()
assert SUPP_DOCX.exists()

assert MAIN_DOCX.stat().st_size > 0
assert SUPP_DOCX.stat().st_size > 0


print("=" * 126)
print("PUBLICATION-QUALITY WORD TABLES GENERATED")
print("=" * 126)

print(
    "Main tables:",
    MAIN_DOCX.resolve()
)

print(
    "Supplementary tables:",
    SUPP_DOCX.resolve()
)

print(
    "\nMain document tables:",
    len(
        main_doc.tables
    )
)

print(
    "Supplementary document tables:",
    len(
        supp_doc.tables
    )
)

print(
    "\nFormatting:"
)

print(
    "  Landscape layout for wide tables: Yes"
)

print(
    "  Professional borders: Yes"
)

print(
    "  Header shading: Yes"
)

print(
    "  Cell padding / indentation: Yes"
)

print(
    "  Repeated table headers: Yes"
)

print(
    "  Table notes and abbreviation definitions: Yes"
)

print(
    "  Consistent decimal precision: Yes"
)

print(
    "  Page numbers: Yes"
)

print(
    "\nSTATUS: PUBLICATION WORD TABLES EXPORTED"
)

## 2.46 Publication-quality model-evaluation figures

Final model-evaluation figures are generated from the locked held-out
predictions and frozen bootstrap results.

Only figures that provide information complementary to the manuscript tables
are retained:

1. full-cohort held-out calibration;
2. paired ANC-entry calibration comparing baseline and extended models;
3. paired incremental predictive value of early ANC initiation with
   stratified-PSU bootstrap confidence intervals.

The figures use consistent manuscript styling and are exported in vector and
high-resolution raster formats.

Calibration curves are descriptive held-out evaluations. Smoothed calibration
curves are estimated using survey-weighted logistic regression with a restricted
cubic spline representation of predicted risk. The same fitted spline basis is
used for estimation and plotting.

No model is refitted, recalibrated, or retuned.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import statsmodels.api as sm

from patsy import dmatrix, build_design_matrices


ROOT = Path("outputs")

HANDOFF_DIR = (
    ROOT
    / "notebook3_explainability_handoff"
)

PUBLICATION_DIR = (
    ROOT
    / "publication_results"
)

FIGURE_DIR = (
    PUBLICATION_DIR
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Locked held-out predictions
full_predictions = pd.read_csv(
    HANDOFF_DIR
    / "full_cohort_locked_predictions.csv"
)

entry_predictions = pd.read_csv(
    HANDOFF_DIR
    / "anc_entry_locked_predictions.csv"
)


# Frozen paired bootstrap CIs
entry_difference_ci = pd.read_csv(
    PUBLICATION_DIR
    / "supplementary_tables"
    / "Table_S6_anc_entry_paired_difference_bootstrap_CI.csv"
)


print("Full cohort predictions:", full_predictions.shape)
print("ANC-entry predictions:", entry_predictions.shape)
print("Paired CI table:", entry_difference_ci.shape)

print("\nFull-cohort columns:")
print(full_predictions.columns.tolist())

print("\nANC-entry columns:")
print(entry_predictions.columns.tolist())

print(
    "\nSTATUS: FROZEN FIGURE INPUTS RELOADED"
)

In [ ]:
# =============================================================================
# 2.46.1 PUBLICATION FIGURE SETUP
#
# Uses ONLY frozen files reloaded after the kernel restart.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import statsmodels.api as sm

from pathlib import Path
from patsy import dmatrix, build_design_matrices


# =============================================================================
# 1. FIGURE OUTPUT DIRECTORY
# =============================================================================

FIGURE_DIR = (
    Path("outputs")
    / "publication_results"
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. PUBLICATION TYPOGRAPHY
# =============================================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# =============================================================================
# 3. MODEL STYLES
#
# Distinct colors + distinct line types support both color and grayscale use.
# =============================================================================

MODEL_STYLE = {
    "Logistic regression": {
        "color": "#0072B2",
        "linestyle": "-"
    },

    "Random Forest": {
        "color": "#D55E00",
        "linestyle": "--"
    },

    "XGBoost": {
        "color": "#009E73",
        "linestyle": "-."
    }
}


# =============================================================================
# 4. DESCRIPTIVE EMPIRICAL CALIBRATION POINTS
# =============================================================================

def weighted_calibration_groups(
    probability,
    outcome,
    weight,
    n_groups=10
):
    """
    Divide observations into quantile groups of predicted probability.

    Within each group:
      x = survey-weighted mean predicted probability
      y = survey-weighted observed ANC8 prevalence

    These points are descriptive and are not model fitting targets.
    """

    df = pd.DataFrame({
        "probability": np.asarray(
            probability,
            dtype=float
        ),
        "outcome": np.asarray(
            outcome,
            dtype=int
        ),
        "weight": np.asarray(
            weight,
            dtype=float
        )
    })


    df["risk_group"] = pd.qcut(
        df["probability"],
        q=n_groups,
        labels=False,
        duplicates="drop"
    )


    rows = []


    for group_id, group_df in df.groupby(
        "risk_group",
        observed=True
    ):

        rows.append({
            "group": int(
                group_id
            ),

            "mean_predicted": np.average(
                group_df["probability"],
                weights=group_df["weight"]
            ),

            "observed": np.average(
                group_df["outcome"],
                weights=group_df["weight"]
            )
        })


    return pd.DataFrame(
        rows
    )


# =============================================================================
# 5. WEIGHTED SPLINE CALIBRATION CURVE
#
# IMPORTANT:
# The spline basis is constructed once on the observed predicted logits.
# The exact same patsy design_info is reused for the plotting grid.
# =============================================================================

def weighted_spline_calibration_curve(
    probability,
    outcome,
    weight,
    df_spline=4
):

    probability = np.asarray(
        probability,
        dtype=float
    )

    outcome = np.asarray(
        outcome,
        dtype=int
    )

    weight = np.asarray(
        weight,
        dtype=float
    )


    probability_safe = np.clip(
        probability,
        1e-6,
        1 - 1e-6
    )


    predicted_logit = np.log(
        probability_safe
        /
        (
            1 - probability_safe
        )
    )


    # -------------------------------------------------------------------------
    # Fit spline basis ONCE.
    # -------------------------------------------------------------------------

    spline_design = dmatrix(
        (
            f"bs(x, df={df_spline}, degree=3, "
            "include_intercept=False)"
        ),
        {
            "x": predicted_logit
        },
        return_type="dataframe"
    )


    design_info = (
        spline_design.design_info
    )


    # -------------------------------------------------------------------------
    # Survey-weighted binomial GLM.
    # -------------------------------------------------------------------------

    calibration_fit = sm.GLM(
        outcome,
        spline_design,
        family=sm.families.Binomial(),
        freq_weights=weight
    ).fit()


    # -------------------------------------------------------------------------
    # Plot over central 1st–99th percentile prediction range.
    # This avoids allowing a few extreme predictions to stretch the figure.
    # -------------------------------------------------------------------------

    grid_probability = np.linspace(
        np.quantile(
            probability,
            0.01
        ),
        np.quantile(
            probability,
            0.99
        ),
        250
    )


    grid_safe = np.clip(
        grid_probability,
        1e-6,
        1 - 1e-6
    )


    grid_logit = np.log(
        grid_safe
        /
        (
            1 - grid_safe
        )
    )


    # -------------------------------------------------------------------------
    # Reuse SAME fitted spline basis.
    # -------------------------------------------------------------------------

    grid_design = build_design_matrices(
        [
            design_info
        ],
        {
            "x": grid_logit
        }
    )[0]


    smooth_observed = (
        calibration_fit.predict(
            grid_design
        )
    )


    return (
        grid_probability,
        np.asarray(
            smooth_observed
        )
    )


print(
    "STATUS: PUBLICATION FIGURE HELPERS READY"
)

In [ ]:
# =============================================================================
# FIGURE 1. FULL-COHORT HELD-OUT CALIBRATION
#
# Data source:
#   frozen full_cohort_locked_predictions.csv
#
# No model fitting or recalibration occurs.
# =============================================================================


# =============================================================================
# 1. VERIFY REQUIRED FROZEN COLUMNS
# =============================================================================

FULL_CALIBRATION_COLUMNS = {
    "Logistic regression":
        "prob_logistic",

    "Random Forest":
        "prob_random_forest",

    "XGBoost":
        "prob_xgboost"
}


required_columns = {
    "anc8plus_observed",
    "survey_weight",
    *FULL_CALIBRATION_COLUMNS.values()
}


missing_columns = (
    required_columns
    - set(
        full_predictions.columns
    )
)


assert not missing_columns, (
    "Missing required columns: "
    + str(
        missing_columns
    )
)


# =============================================================================
# 2. HELD-OUT OUTCOME AND SURVEY WEIGHTS
# =============================================================================

y_full_fig = (
    full_predictions[
        "anc8plus_observed"
    ]
    .astype(int)
    .to_numpy()
)


w_full_fig = (
    full_predictions[
        "survey_weight"
    ]
    .astype(float)
    .to_numpy()
)


# =============================================================================
# 3. COMMON AXIS LIMIT
#
# Use the maximum 99th percentile across the three models, rounded upward
# to the nearest 0.10.
# =============================================================================

probability_upper = max(
    np.quantile(
        full_predictions[
            probability_column
        ],
        0.99
    )
    for probability_column
    in FULL_CALIBRATION_COLUMNS.values()
)


axis_max = min(
    1.0,
    np.ceil(
        probability_upper
        * 10
    )
    / 10
)


# Avoid an unnecessarily narrow figure.
axis_max = max(
    axis_max,
    0.6
)


# =============================================================================
# 4. CREATE FIGURE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        6.4,
        5.2
    )
)


# Perfect calibration reference.
ax.plot(
    [
        0,
        axis_max
    ],
    [
        0,
        axis_max
    ],
    color="#444444",
    linestyle=":",
    linewidth=1.2,
    label="Perfect calibration"
)


# =============================================================================
# 5. MODEL CALIBRATION CURVES + EMPIRICAL POINTS
# =============================================================================

for model_name, probability_column in (
    FULL_CALIBRATION_COLUMNS.items()
):

    probability = (
        full_predictions[
            probability_column
        ]
        .astype(float)
        .to_numpy()
    )


    grid_probability, smooth_observed = (
        weighted_spline_calibration_curve(
            probability=probability,
            outcome=y_full_fig,
            weight=w_full_fig,
            df_spline=4
        )
    )


    empirical_points = (
        weighted_calibration_groups(
            probability=probability,
            outcome=y_full_fig,
            weight=w_full_fig,
            n_groups=10
        )
    )


    style = MODEL_STYLE[
        model_name
    ]


    # Smooth curve.
    ax.plot(
        grid_probability,
        smooth_observed,
        color=style[
            "color"
        ],
        linestyle=style[
            "linestyle"
        ],
        linewidth=2.0,
        label=model_name
    )


    # Empirical grouped calibration points.
    ax.scatter(
        empirical_points[
            "mean_predicted"
        ],
        empirical_points[
            "observed"
        ],
        color=style[
            "color"
        ],
        s=26,
        edgecolor="white",
        linewidth=0.55,
        zorder=3
    )


# =============================================================================
# 6. AXES AND LABELS
# =============================================================================

ax.set_xlim(
    0,
    axis_max
)

ax.set_ylim(
    0,
    axis_max
)


ticks = np.linspace(
    0,
    axis_max,
    6
)


ax.set_xticks(
    ticks
)

ax.set_yticks(
    ticks
)


ax.xaxis.set_major_formatter(
    mpl.ticker.PercentFormatter(
        xmax=1,
        decimals=0
    )
)

ax.yaxis.set_major_formatter(
    mpl.ticker.PercentFormatter(
        xmax=1,
        decimals=0
    )
)


ax.set_xlabel(
    "Predicted probability of ANC8+"
)

ax.set_ylabel(
    "Observed probability of ANC8+"
)


ax.grid(
    visible=True,
    linewidth=0.45,
    alpha=0.25
)


ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)


# =============================================================================
# 7. LEGEND ORDER
# =============================================================================

handles, labels = (
    ax.get_legend_handles_labels()
)


desired_order = [
    "Logistic regression",
    "Random Forest",
    "XGBoost",
    "Perfect calibration"
]


ordered_handles = [
    handles[
        labels.index(
            label
        )
    ]
    for label
    in desired_order
]


ax.legend(
    ordered_handles,
    desired_order,
    loc="upper left",
    frameon=False,
    handlelength=2.8
)


fig.tight_layout()


# =============================================================================
# 8. CRASH-SAFE EXPORT
#
# First export only:
#   PDF = vector manuscript file
#   PNG = high-quality inspection/manuscript-drafting file
#
# We will create TIFF separately after visual approval.
# =============================================================================

figure_stem = (
    FIGURE_DIR
    / "Figure_1_full_cohort_calibration"
)


fig.savefig(
    figure_stem.with_suffix(
        ".pdf"
    ),
    bbox_inches="tight"
)


fig.savefig(
    figure_stem.with_suffix(
        ".png"
    ),
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# Explicitly release memory after rendering.
plt.close(
    fig
)


print(
    "Figure 1 exported:"
)

print(
    figure_stem.with_suffix(
        ".pdf"
    ).resolve()
)

print(
    figure_stem.with_suffix(
        ".png"
    ).resolve()
)

print(
    "\nSTATUS: FIGURE 1 EXPORTED SUCCESSFULLY"
)

In [ ]:
# =============================================================================
# 2.46.2 FIGURE 2. ANC-ENTRY HELD-OUT CALIBRATION
#
# Three panels:
#   A. Logistic regression
#   B. Random Forest
#   C. XGBoost
#
# Baseline vs + early ANC
#
# Data source:
#   frozen anc_entry_locked_predictions.csv
# =============================================================================


# =============================================================================
# 1. VERIFY REQUIRED COLUMNS
# =============================================================================

ENTRY_CALIBRATION_COLUMNS = {
    "Logistic regression": {
        "Baseline":
            "prob_logistic_baseline",

        "+ early ANC":
            "prob_logistic_extended"
    },

    "Random Forest": {
        "Baseline":
            "prob_rf_baseline",

        "+ early ANC":
            "prob_rf_extended"
    },

    "XGBoost": {
        "Baseline":
            "prob_xgb_baseline",

        "+ early ANC":
            "prob_xgb_extended"
    }
}


required_entry_columns = {
    "anc8plus_observed",
    "survey_weight"
}


for algorithm_map in (
    ENTRY_CALIBRATION_COLUMNS.values()
):

    required_entry_columns.update(
        algorithm_map.values()
    )


missing_entry_columns = (
    required_entry_columns
    - set(
        entry_predictions.columns
    )
)


assert not missing_entry_columns, (
    "Missing required columns: "
    + str(
        missing_entry_columns
    )
)


# =============================================================================
# 2. OUTCOME AND WEIGHTS
# =============================================================================

y_entry_fig = (
    entry_predictions[
        "anc8plus_observed"
    ]
    .astype(int)
    .to_numpy()
)


w_entry_fig = (
    entry_predictions[
        "survey_weight"
    ]
    .astype(float)
    .to_numpy()
)


# =============================================================================
# 3. SPECIFICATION STYLES
#
# Color + line style + marker differ, so the figure remains interpretable
# in grayscale as well as color.
# =============================================================================

ENTRY_SPEC_STYLE = {
    "Baseline": {
        "color": "#6B7280",
        "linestyle": "--",
        "marker": "o"
    },

    "+ early ANC": {
        "color": "#0072B2",
        "linestyle": "-",
        "marker": "s"
    }
}


# =============================================================================
# 4. COMMON AXIS LIMIT ACROSS ALL SIX MODELS
# =============================================================================

entry_probability_columns = [
    column
    for algorithm_map
    in ENTRY_CALIBRATION_COLUMNS.values()
    for column
    in algorithm_map.values()
]


entry_upper = max(
    np.quantile(
        entry_predictions[
            column
        ],
        0.99
    )
    for column
    in entry_probability_columns
)


entry_axis_max = min(
    1.0,
    np.ceil(
        entry_upper
        * 10
    )
    / 10
)


entry_axis_max = max(
    entry_axis_max,
    0.7
)


# =============================================================================
# 5. CREATE THREE-PANEL FIGURE
# =============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(
        10.5,
        3.9
    ),
    sharex=True,
    sharey=True
)


panel_labels = [
    "A",
    "B",
    "C"
]


# =============================================================================
# 6. DRAW EACH ALGORITHM PANEL
# =============================================================================

for ax, panel_label, (
    algorithm,
    specification_map
) in zip(
    axes,
    panel_labels,
    ENTRY_CALIBRATION_COLUMNS.items()
):

    # -------------------------------------------------------------------------
    # Perfect-calibration reference
    # -------------------------------------------------------------------------

    ax.plot(
        [
            0,
            entry_axis_max
        ],
        [
            0,
            entry_axis_max
        ],
        color="#444444",
        linestyle=":",
        linewidth=1.2,
        zorder=1
    )


    # -------------------------------------------------------------------------
    # Baseline and extended specification
    # -------------------------------------------------------------------------

    for specification, probability_column in (
        specification_map.items()
    ):

        probability = (
            entry_predictions[
                probability_column
            ]
            .astype(float)
            .to_numpy()
        )


        grid_probability, smooth_observed = (
            weighted_spline_calibration_curve(
                probability=probability,
                outcome=y_entry_fig,
                weight=w_entry_fig,
                df_spline=4
            )
        )


        empirical_points = (
            weighted_calibration_groups(
                probability=probability,
                outcome=y_entry_fig,
                weight=w_entry_fig,
                n_groups=10
            )
        )


        style = (
            ENTRY_SPEC_STYLE[
                specification
            ]
        )


        # Smooth calibration curve
        ax.plot(
            grid_probability,
            smooth_observed,
            color=style[
                "color"
            ],
            linestyle=style[
                "linestyle"
            ],
            linewidth=2.0,
            label=specification
        )


        # Empirical grouped points
        ax.scatter(
            empirical_points[
                "mean_predicted"
            ],
            empirical_points[
                "observed"
            ],
            color=style[
                "color"
            ],
            marker=style[
                "marker"
            ],
            s=25,
            edgecolor="white",
            linewidth=0.5,
            zorder=3
        )


    # -------------------------------------------------------------------------
    # Axis formatting
    # -------------------------------------------------------------------------

    ax.set_xlim(
        0,
        entry_axis_max
    )

    ax.set_ylim(
        0,
        entry_axis_max
    )


    ticks = np.linspace(
        0,
        entry_axis_max,
        6
    )


    ax.set_xticks(
        ticks
    )

    ax.set_yticks(
        ticks
    )


    ax.xaxis.set_major_formatter(
        mpl.ticker.PercentFormatter(
            xmax=1,
            decimals=0
        )
    )

    ax.yaxis.set_major_formatter(
        mpl.ticker.PercentFormatter(
            xmax=1,
            decimals=0
        )
    )


    ax.grid(
        visible=True,
        linewidth=0.45,
        alpha=0.25
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


    # -------------------------------------------------------------------------
    # Panel label
    # -------------------------------------------------------------------------

    ax.text(
        -0.13,
        1.04,
        panel_label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="bottom"
    )


    # -------------------------------------------------------------------------
    # Algorithm heading
    # -------------------------------------------------------------------------

    ax.text(
        0.5,
        1.03,
        algorithm,
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold"
    )


# =============================================================================
# 7. SHARED AXIS LABELS
# =============================================================================

fig.supxlabel(
    "Predicted probability of ANC8+",
    fontsize=10,
    y=0.08
)


fig.supylabel(
    "Observed probability of ANC8+",
    fontsize=10,
    x=0.015
)


# =============================================================================
# 8. SHARED LEGEND
# =============================================================================

legend_handles = [
    mpl.lines.Line2D(
        [],
        [],
        color=ENTRY_SPEC_STYLE[
            "Baseline"
        ][
            "color"
        ],
        linestyle=ENTRY_SPEC_STYLE[
            "Baseline"
        ][
            "linestyle"
        ],
        marker=ENTRY_SPEC_STYLE[
            "Baseline"
        ][
            "marker"
        ],
        markersize=5,
        label="Baseline"
    ),

    mpl.lines.Line2D(
        [],
        [],
        color=ENTRY_SPEC_STYLE[
            "+ early ANC"
        ][
            "color"
        ],
        linestyle=ENTRY_SPEC_STYLE[
            "+ early ANC"
        ][
            "linestyle"
        ],
        marker=ENTRY_SPEC_STYLE[
            "+ early ANC"
        ][
            "marker"
        ],
        markersize=5,
        label="+ early ANC"
    ),

    mpl.lines.Line2D(
        [],
        [],
        color="#444444",
        linestyle=":",
        linewidth=1.2,
        label="Perfect calibration"
    )
]


fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(
        0.5,
        -0.01
    ),
    ncol=3,
    frameon=False,
    handlelength=2.8
)


# =============================================================================
# 9. FINAL LAYOUT
# =============================================================================

fig.tight_layout(
    rect=[
        0.025,
        0.12,
        1,
        1
    ]
)


# =============================================================================
# 10. CRASH-SAFE EXPORT
# =============================================================================

figure_stem = (
    FIGURE_DIR
    / "Figure_2_anc_entry_calibration"
)


fig.savefig(
    figure_stem.with_suffix(
        ".pdf"
    ),
    bbox_inches="tight"
)


fig.savefig(
    figure_stem.with_suffix(
        ".png"
    ),
    dpi=300,
    bbox_inches="tight"
)


plt.show()


plt.close(
    fig
)


print(
    "Figure 2 exported:"
)

print(
    figure_stem.with_suffix(
        ".pdf"
    ).resolve()
)

print(
    figure_stem.with_suffix(
        ".png"
    ).resolve()
)

print(
    "\nSTATUS: FIGURE 2 EXPORTED SUCCESSFULLY"
)

In [ ]:
# =============================================================================
# 2.46.3 FIGURE 3. INCREMENTAL PREDICTIVE VALUE OF EARLY ANC
#
# Paired held-out differences:
#   A. Average Precision
#   B. AUROC
#   C. Brier score improvement
#
# Error bars:
#   95% empirical percentile CI from 2,000 paired stratified-PSU
#   bootstrap replicates.
#
# Positive values favor the + early ANC model.
# =============================================================================


# =============================================================================
# 1. VERIFY FROZEN INPUT
# =============================================================================

required_difference_columns = {
    "Algorithm",
    "Metric",
    "Point_estimate",
    "CI_lower",
    "CI_upper",
    "CI_excludes_zero"
}


missing_difference_columns = (
    required_difference_columns
    - set(
        entry_difference_ci.columns
    )
)


assert not missing_difference_columns, (
    "Missing required columns: "
    + str(
        missing_difference_columns
    )
)


# =============================================================================
# 2. DISPLAY ORDER AND LABELS
# =============================================================================

algorithm_order = [
    "Logistic regression",
    "Random Forest",
    "XGBoost"
]


metric_configuration = {
    "Delta_AP": {
        "panel":
            "A",

        "title":
            "Average Precision",

        "xlabel":
            "Change in Average Precision"
    },

    "Delta_AUROC": {
        "panel":
            "B",

        "title":
            "AUROC",

        "xlabel":
            "Change in AUROC"
    },

    "Brier_improvement": {
        "panel":
            "C",

        "title":
            "Brier score",

        "xlabel":
            "Brier score improvement"
    }
}


# =============================================================================
# 3. CREATE FIGURE
# =============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(
        10.5,
        3.7
    ),
    sharey=True
)


# =============================================================================
# 4. DRAW EACH METRIC PANEL
# =============================================================================

for ax, (
    metric_name,
    config
) in zip(
    axes,
    metric_configuration.items()
):

    metric_df = (
        entry_difference_ci[
            entry_difference_ci[
                "Metric"
            ].eq(
                metric_name
            )
        ]
        .set_index(
            "Algorithm"
        )
        .loc[
            algorithm_order
        ]
        .reset_index()
    )


    point_estimates = (
        metric_df[
            "Point_estimate"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_lower = (
        metric_df[
            "CI_lower"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_upper = (
        metric_df[
            "CI_upper"
        ]
        .astype(float)
        .to_numpy()
    )


    y_positions = np.arange(
        len(
            algorithm_order
        )
    )


    # -------------------------------------------------------------------------
    # Reference line: no incremental predictive value
    # -------------------------------------------------------------------------

    ax.axvline(
        x=0,
        color="#444444",
        linestyle=":",
        linewidth=1.2,
        zorder=1
    )


    # -------------------------------------------------------------------------
    # Plot one model at a time so algorithm colors remain consistent with
    # Figures 1 and 2.
    # -------------------------------------------------------------------------

    for i, algorithm in enumerate(
        algorithm_order
    ):

        style = (
            MODEL_STYLE[
                algorithm
            ]
        )


        lower_error = (
            point_estimates[
                i
            ]
            -
            ci_lower[
                i
            ]
        )


        upper_error = (
            ci_upper[
                i
            ]
            -
            point_estimates[
                i
            ]
        )


        ax.errorbar(
            x=point_estimates[
                i
            ],
            y=y_positions[
                i
            ],
            xerr=np.array([
                [
                    lower_error
                ],
                [
                    upper_error
                ]
            ]),
            fmt="o",
            markersize=6.5,
            color=style[
                "color"
            ],
            ecolor=style[
                "color"
            ],
            elinewidth=1.7,
            capsize=3.5,
            capthick=1.2,
            zorder=3
        )


    # -------------------------------------------------------------------------
    # Y-axis
    # -------------------------------------------------------------------------

    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        algorithm_order
    )

    ax.invert_yaxis()


    # -------------------------------------------------------------------------
    # Panel heading
    # -------------------------------------------------------------------------

    ax.text(
        -0.13,
        1.04,
        config[
            "panel"
        ],
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="bottom"
    )


    ax.text(
        0.5,
        1.03,
        config[
            "title"
        ],
        transform=ax.transAxes,
        fontsize=9.5,
        fontweight="bold",
        ha="center",
        va="bottom"
    )


    # -------------------------------------------------------------------------
    # X-axis
    # -------------------------------------------------------------------------

    ax.set_xlabel(
        config[
            "xlabel"
        ]
    )


    ax.xaxis.set_major_formatter(
        mpl.ticker.FormatStrFormatter(
            "%+.03f"
        )
    )


    ax.grid(
        axis="x",
        linewidth=0.45,
        alpha=0.25
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# =============================================================================
# 5. REMOVE REPEATED Y LABELS FROM PANELS B AND C
# =============================================================================

for ax in axes[
    1:
]:

    ax.tick_params(
        labelleft=False
    )


# =============================================================================
# 6. ADD INTERPRETIVE DIRECTION LABEL
#
# Keep this subtle. It helps the reader understand the sign without having
# to consult the caption.
# =============================================================================

for ax in axes:

    ax.text(
        0.98,
        -0.23,
        "Positive values favor + early ANC",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=7.8,
        color="#555555"
    )


# =============================================================================
# 7. FINAL LAYOUT
# =============================================================================

fig.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1
    ]
)


# =============================================================================
# 8. CRASH-SAFE EXPORT
# =============================================================================

figure_stem = (
    FIGURE_DIR
    / "Figure_3_early_anc_incremental_value"
)


fig.savefig(
    figure_stem.with_suffix(
        ".pdf"
    ),
    bbox_inches="tight"
)


fig.savefig(
    figure_stem.with_suffix(
        ".png"
    ),
    dpi=300,
    bbox_inches="tight"
)


plt.show()


plt.close(
    fig
)


print(
    "Figure 3 exported:"
)

print(
    figure_stem.with_suffix(
        ".pdf"
    ).resolve()
)

print(
    figure_stem.with_suffix(
        ".png"
    ).resolve()
)

print(
    "\nSTATUS: FIGURE 3 EXPORTED SUCCESSFULLY"
)

In [ ]:
# =============================================================================
# 2.47 FINAL NOTEBOOK 2 CLOSURE
#
# PURPOSE
# -------
# 1. Save final figure captions.
# 2. Verify essential Notebook-3 and publication outputs exist.
# 3. Create SHA-256 integrity manifest for the final publication package.
#
# NO model fitting.
# NO prediction.
# NO bootstrap.
# NO explainability.
# =============================================================================

from pathlib import Path
import hashlib
import pandas as pd


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path("outputs")

HANDOFF_DIR = (
    ROOT
    / "notebook3_explainability_handoff"
)

PUBLICATION_DIR = (
    ROOT
    / "publication_results"
)

FIGURE_DIR = (
    PUBLICATION_DIR
    / "figures"
)

WORD_DIR = (
    PUBLICATION_DIR
    / "word_tables"
)


# =============================================================================
# 2. FINAL FIGURE CAPTIONS
# =============================================================================

figure_captions = """
Figure 1. Held-out calibration of the full-cohort ANC8 prediction models.
Curves show survey-weighted spline-smoothed observed versus predicted
probabilities for Logistic regression, Random Forest, and XGBoost. Points
represent survey-weighted empirical observed risk within groups of predicted
probability. The diagonal line indicates perfect calibration.

Figure 2. Held-out calibration at the ANC-entry prediction horizon.
Panels compare the baseline 21-predictor model with the extended model
including early ANC initiation for Logistic regression (A), Random Forest (B),
and XGBoost (C). Curves show survey-weighted spline-smoothed calibration and
points represent survey-weighted empirical observed risk within groups of
predicted probability. The diagonal line indicates perfect calibration.

Figure 3. Incremental predictive value of early ANC initiation in the held-out
ANC-entry population. Points show paired differences between extended and
baseline models for Average Precision (A), AUROC (B), and Brier score
improvement (C). Error bars show empirical 95% percentile confidence intervals
from 2,000 paired stratified-PSU bootstrap replicates. Positive values favor
the extended model. All confidence intervals included zero.
""".strip()


with open(
    FIGURE_DIR / "figure_captions.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        figure_captions
    )


# =============================================================================
# 3. ESSENTIAL OUTPUT CHECK
# =============================================================================

essential_files = [
    # Notebook 3 handoff
    HANDOFF_DIR / "full_cohort_final_models.joblib",
    HANDOFF_DIR / "anc_entry_final_models.joblib",
    HANDOFF_DIR / "full_cohort_heldout.pkl",
    HANDOFF_DIR / "anc_entry_heldout.pkl",
    HANDOFF_DIR / "full_cohort_feature_lineage.csv",
    HANDOFF_DIR / "anc_entry_feature_lineage.csv",
    HANDOFF_DIR / "SHA256_MANIFEST.csv",

    # Publication Word tables
    WORD_DIR / "ANC8_Main_Tables.docx",
    WORD_DIR / "ANC8_Supplementary_Tables.docx",

    # Publication figures
    FIGURE_DIR / "Figure_1_full_cohort_calibration.pdf",
    FIGURE_DIR / "Figure_1_full_cohort_calibration.png",

    FIGURE_DIR / "Figure_2_anc_entry_calibration.pdf",
    FIGURE_DIR / "Figure_2_anc_entry_calibration.png",

    FIGURE_DIR / "Figure_3_early_anc_incremental_value.pdf",
    FIGURE_DIR / "Figure_3_early_anc_incremental_value.png",

    FIGURE_DIR / "figure_captions.txt"
]


missing_files = [
    str(path)
    for path in essential_files
    if not path.exists()
]


assert not missing_files, (
    "Missing essential closure files:\n"
    + "\n".join(
        missing_files
    )
)


# =============================================================================
# 4. SHA-256 MANIFEST FOR PUBLICATION PACKAGE
#
# This hashes files already on disk only.
# =============================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


manifest_rows = []


for path in sorted(
    PUBLICATION_DIR.rglob("*")
):

    if (
        path.is_file()
        and path.name
        != "FINAL_PUBLICATION_MANIFEST_SHA256.csv"
    ):

        manifest_rows.append({
            "Relative_path":
                str(
                    path.relative_to(
                        PUBLICATION_DIR
                    )
                ),

            "Bytes":
                path.stat().st_size,

            "SHA256":
                sha256_file(
                    path
                )
        })


final_publication_manifest = pd.DataFrame(
    manifest_rows
)


final_publication_manifest.to_csv(
    PUBLICATION_DIR
    / "FINAL_PUBLICATION_MANIFEST_SHA256.csv",
    index=False
)


# =============================================================================
# 5. FINAL STATUS
# =============================================================================

print("=" * 126)
print("NOTEBOOK 2 FINAL CLOSURE")
print("=" * 126)

print(
    "Essential files verified:",
    len(
        essential_files
    )
)

print(
    "Publication files hashed:",
    len(
        final_publication_manifest
    )
)

print(
    "\nNotebook 3 handoff:",
    "READY"
)

print(
    "Publication source tables:",
    "READY"
)

print(
    "Word tables:",
    "READY"
)

print(
    "Figures 1-3:",
    "READY"
)

print(
    "Figure captions:",
    "READY"
)

print(
    "Final publication manifest:",
    "READY"
)

print(
    "\nSTATUS: NOTEBOOK 2 CLOSED"
)

## Notebook 2 closure

Notebook 2 is complete.

The final model-development and evaluation workflow has been frozen, including
the full-cohort and ANC-entry prediction horizons, held-out evaluation,
calibration, development-derived operating thresholds, stratified-PSU
bootstrap uncertainty, and paired model comparisons.

Publication-source tables, formatted Word tables, and final model-evaluation
figures have been exported.

A self-contained frozen handoff has also been created for Notebook 3,
containing the fitted models, governed analytic datasets, locked predictions,
predictor architecture, and transformed-feature lineage required for
explainability analyses.

**No further model tuning, model selection, performance evaluation, or
prediction regeneration will be conducted in this notebook.**

Notebook 3 will begin from the frozen handoff and will be restricted to
explainable-machine-learning analyses and interpretation.